# Notebook 10 — Secondary Machine-Learning Model Robustness Study

## Purpose

Notebook 09 completed the predefined primary forecasting experiment using:

- Ridge Regression;
- Histogram Gradient Boosting Regression;
- the controlled F0 → F4 feature-ablation hierarchy;
- 15-, 30-, and 60-minute forecast horizons;
- chronological validation and final test evaluation;
- paired daily-block bootstrap uncertainty analysis.

The primary experiment is now frozen.

This notebook begins a **secondary model-robustness study**.

The purpose is to determine whether the conclusions obtained in Notebook 09
remain similar when additional established machine-learning algorithms are
used.

---

## Central Question

The main scientific comparison remains:

$$
F3 \rightarrow F4
$$

where:

- F3 contains historical, Gregorian, solar, and continuous lunar /
  Sun–Moon astronomical information;
- F4 adds the discrete Panchang representation.

The purpose of adding new models is **not** to search until a favorable
Panchang result is obtained.

Instead, the objective is to test whether the F3 → F4 result is robust to
different modelling assumptions.

---

## Models Planned

The initial secondary machine-learning models will be evaluated in the
following order:

1. Extra Trees Regressor;
2. XGBoost Regressor;
3. LightGBM Regressor;
4. Random Forest Regressor.

CatBoost may later be considered as a sensitivity analysis, particularly
for experiments using native categorical Panchang variables.

---

## Why These Models?

### Extra Trees Regressor

Extra Trees provides a strongly randomized tree ensemble.

It is suitable for the SOLETE modelling dataset because it can learn:

- nonlinear relationships;
- interactions among lagged variables;
- interactions between meteorology and deterministic features;
- complex responses to solar geometry;
- high-dimensional F0–F4 feature sets.

It also requires no feature standardization.

---

### XGBoost

XGBoost is a widely used gradient-boosted decision-tree method.

It provides an additional strong nonlinear benchmark and can model complex
interactions among:

- historical photovoltaic output;
- irradiance;
- meteorological conditions;
- Gregorian time features;
- solar astronomy;
- lunar astronomy;
- Panchang indicators.

---

### LightGBM

LightGBM provides another efficient gradient-boosting implementation.

It is particularly appropriate for a dataset of approximately:

$$
131{,}000
$$

rows with approximately:

$$
118\text{–}244
$$

features.

---

### Random Forest

Random Forest provides a conventional bagged-tree ensemble reference.

Comparing Random Forest with Extra Trees helps distinguish the effect of
standard bootstrap-based tree ensembles from the stronger randomization used
by Extra Trees.

---

## Experimental Relationship to Notebook 09

This notebook is a **secondary experiment**.

The following primary definitions remain unchanged:

- target: `P_Solar[kW]`;
- resolution: 5 minutes;
- forecast horizons: 15, 30, and 60 minutes;
- primary horizon: 60 minutes;
- chronological train / validation / test periods;
- forecast-valid split assignment;
- F0–F4 feature definitions;
- fixed Panchang one-hot encoding;
- MAE as the primary metric;
- all-time and daylight-only evaluation;
- final test set remains chronologically later than model-development data.

The results from Notebook 09 will not be retroactively modified.

---

## Model-Robustness Principle

For each additional model, the same hierarchy will be examined:

$$
F0 \rightarrow F1 \rightarrow F2 \rightarrow F3 \rightarrow F4
$$

The central model-robustness question is:

> Does adding F4 Panchang information beyond F3 continuous astronomy
> produce a similar, different, or model-dependent forecasting effect?

Any difference between models will be reported rather than hidden.

---

## Advanced Models Deferred

Deep-learning and hybrid models will be handled separately after the
secondary tabular-machine-learning study.

These include:

- LSTM;
- GRU;
- BiLSTM;
- TCN;
- CNN-LSTM;
- physics-informed / residual hybrid models;
- VMD-based hybrid models.

They will not be mixed into the present tabular-model robustness experiment.

In [1]:
# ============================================================
# Notebook 10 — Secondary Machine-Learning Model Robustness
# Step 1: Load the frozen primary protocol and SOLETE dataset
# ============================================================

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Locate the project root
# ------------------------------------------------------------

search_locations = [
    Path.cwd(),
    *Path.cwd().parents,
]

PROJECT_ROOT = next(
    path
    for path in search_locations
    if (path / "src").is_dir()
    and (path / "solete_dataset").is_dir()
    and (path / "results").is_dir()
)

print("Project root:")
print(PROJECT_ROOT)


# ------------------------------------------------------------
# 2. Make the reusable src package importable
# ------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


# ------------------------------------------------------------
# 3. Load the frozen Notebook 09 experiment protocol
# ------------------------------------------------------------
#
# This JSON file was saved before beginning any secondary
# modelling experiment.
#
# It therefore acts as the source of truth for:
#
#   - target;
#   - resolution;
#   - forecast horizons;
#   - lag structure;
#   - split boundaries;
#   - feature counts;
#   - astronomy timing;
#   - evaluation conventions.
# ------------------------------------------------------------

PRIMARY_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "notebook09_primary_experiment"
)

PROTOCOL_PATH = (
    PRIMARY_RESULTS_DIR
    / "experiment_protocol.json"
)

assert PROTOCOL_PATH.is_file(), (
    f"Primary experiment protocol not found:\n"
    f"{PROTOCOL_PATH}"
)

with open(
    PROTOCOL_PATH,
    "r",
    encoding="utf-8",
) as file:
    primary_protocol = json.load(file)


print("\nFrozen primary protocol loaded.")

print(
    "Target:",
    primary_protocol["target"],
)

print(
    "Resolution:",
    primary_protocol["resolution_minutes"],
    "minutes",
)

print(
    "Forecast horizons:",
    primary_protocol["forecast_horizons_minutes"],
)

print(
    "Primary horizon:",
    primary_protocol["primary_horizon_minutes"],
    "minutes",
)

print(
    "Feature counts:",
    primary_protocol["feature_counts"],
)


# ------------------------------------------------------------
# 4. Verify the frozen protocol values expected by Notebook 10
# ------------------------------------------------------------

assert primary_protocol["target"] == "P_Solar[kW]"

assert primary_protocol["resolution_minutes"] == 5

assert primary_protocol[
    "forecast_horizons_minutes"
] == [
    15,
    30,
    60,
]

assert primary_protocol[
    "primary_horizon_minutes"
] == 60

assert primary_protocol[
    "feature_counts"
] == {
    "F0": 118,
    "F1": 133,
    "F2": 138,
    "F3": 147,
    "F4": 244,
}

assert primary_protocol[
    "daily_pv_lag_steps"
] == 288

assert primary_protocol[
    "solete_astronomy_offset_minutes"
] == 2

print(
    "\nFrozen protocol consistency check: PASSED"
)


# ------------------------------------------------------------
# 5. Import and locate the canonical 5-minute dataset
# ------------------------------------------------------------

from src.data_loader import load_solete


DATA_DIR = (
    PROJECT_ROOT
    / "solete_dataset"
)

five_minute_files = [
    path
    for path in DATA_DIR.rglob("*.h5")
    if "5min" in path.name.lower().replace("_", "")
]

assert len(five_minute_files) == 1, (
    "Expected exactly one canonical 5-minute SOLETE file, "
    f"found {len(five_minute_files)}."
)

DATA_PATH_5MIN = five_minute_files[0]

print("\nCanonical 5-minute dataset:")
print(
    DATA_PATH_5MIN.relative_to(
        PROJECT_ROOT
    )
)


# ------------------------------------------------------------
# 6. Load the canonical dataset
# ------------------------------------------------------------

df_5min = load_solete(
    DATA_PATH_5MIN
)


# ------------------------------------------------------------
# 7. Verify that the dataset is the same one used in Notebook 09
# ------------------------------------------------------------

assert df_5min.shape == (
    131_617,
    11,
)

assert df_5min.index.min() == pd.Timestamp(
    "2018-06-01 00:00:00",
    tz="UTC",
)

assert df_5min.index.max() == pd.Timestamp(
    "2019-09-01 00:00:00",
    tz="UTC",
)

assert df_5min.index.is_monotonic_increasing

assert not df_5min.index.has_duplicates


print("\nCanonical dataset verification: PASSED")

print(f"Shape: {df_5min.shape}")
print(f"Start: {df_5min.index.min()}")
print(f"End:   {df_5min.index.max()}")

Project root:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project

Frozen primary protocol loaded.
Target: P_Solar[kW]
Resolution: 5 minutes
Forecast horizons: [15, 30, 60]
Primary horizon: 60 minutes
Feature counts: {'F0': 118, 'F1': 133, 'F2': 138, 'F3': 147, 'F4': 244}

Frozen protocol consistency check: PASSED

Canonical 5-minute dataset:
solete_dataset/SOLETE_Pombo_5min.h5

Canonical dataset verification: PASSED
Shape: (131617, 11)
Start: 2018-06-01 00:00:00+00:00
End:   2019-09-01 00:00:00+00:00


In [2]:
# ============================================================
# Step 2: Rebuild the cleaned SOLETE feature table
# ============================================================
#
# We deliberately reconstruct the modelling inputs from the
# reusable source code rather than depending on in-memory
# objects from Notebook 09.
#
# This makes Notebook 10 independently reproducible while still
# preserving the frozen preprocessing definitions.
# ============================================================

from src.preprocessing import build_cleaned_solete_features


# ------------------------------------------------------------
# 1. Apply the reusable cleaning pipeline
# ------------------------------------------------------------

df_clean = build_cleaned_solete_features(
    df_5min
)


# ------------------------------------------------------------
# 2. Basic integrity checks
# ------------------------------------------------------------

assert df_clean.index.equals(
    df_5min.index
)

assert len(df_clean) == len(
    df_5min
)

assert df_clean.index.is_monotonic_increasing

assert not df_clean.index.has_duplicates


# ------------------------------------------------------------
# 3. Verify expected dimensions
# ------------------------------------------------------------
#
# Original SOLETE variables: 11
#
# Reusable cleaned additions: 5
#
# Expected total:
#
#     11 + 5 = 16 columns
# ------------------------------------------------------------

assert df_clean.shape == (
    131_617,
    16,
)


# ------------------------------------------------------------
# 4. Check missing values
# ------------------------------------------------------------

missing_total = int(
    df_clean.isna().sum().sum()
)

assert missing_total == 0


# ------------------------------------------------------------
# 5. Show the cleaned schema
# ------------------------------------------------------------

print(
    "Cleaned SOLETE reconstruction: PASSED"
)

print(
    "\nShape:",
    df_clean.shape,
)

print(
    "Missing values:",
    missing_total,
)

print(
    "\nColumns:"
)

for i, column in enumerate(
    df_clean.columns,
    start=1,
):
    print(
        f"{i:2d}. {column}"
    )


# ------------------------------------------------------------
# 6. Identify which columns were added by preprocessing
# ------------------------------------------------------------

cleaned_additions = [
    column
    for column in df_clean.columns
    if column not in df_5min.columns
]

print(
    "\nColumns added by reusable preprocessing:"
)

for column in cleaned_additions:
    print(
        " -",
        column,
    )

Cleaned SOLETE reconstruction: PASSED

Shape: (131617, 16)
Missing values: 0

Columns:
 1. TEMPERATURE[degC]
 2. HUMIDITY[%]
 3. WIND_SPEED[m1s]
 4. WIND_DIR[deg]
 5. GHI[kW1m2]
 6. POA Irr[kW1m2]
 7. P_Gaia[kW]
 8. P_Solar[kW]
 9. Pressure[mbar]
10. Azimuth[deg]
11. Elevation[deg]
12. Pressure_clean[mbar]
13. Humidity_clean
14. Wind_dir_clean[deg]
15. wind_direction_sin
16. wind_direction_cos

Columns added by reusable preprocessing:
 - Pressure_clean[mbar]
 - Humidity_clean
 - Wind_dir_clean[deg]
 - wind_direction_sin
 - wind_direction_cos


In [3]:
# ============================================================
# Step 3: Reconstruct the frozen F0 historical feature set
# ============================================================
#
# F0 contains only information available by forecast issue
# time t.
#
# For each of 9 observed variables we use:
#
#   t, t-5, t-10, ..., t-60 minutes
#
# This gives:
#
#   9 variables × 13 historical observations = 117 features
#
# We additionally include:
#
#   P_Solar(t-24h)
#
# corresponding to a 288-step lag at 5-minute resolution.
#
# Total F0 feature count:
#
#   117 + 1 = 118
#
# No observation later than issue time t is used.
# ============================================================


# ------------------------------------------------------------
# 1. Frozen historical variables
# ------------------------------------------------------------

F0_HISTORY_VARIABLES = [
    "P_Solar[kW]",
    "GHI[kW1m2]",
    "POA Irr[kW1m2]",
    "TEMPERATURE[degC]",
    "Humidity_clean",
    "Pressure_clean[mbar]",
    "WIND_SPEED[m1s]",
    "wind_direction_sin",
    "wind_direction_cos",
]


# ------------------------------------------------------------
# 2. Historical lag structure
# ------------------------------------------------------------
#
# At 5-minute resolution:
#
#   lag 0  = t
#   lag 1  = t - 5 min
#   ...
#   lag 12 = t - 60 min
# ------------------------------------------------------------

HISTORY_STEP_LAGS = list(
    range(13)
)

HISTORY_MINUTE_LAGS = [
    step * 5
    for step in HISTORY_STEP_LAGS
]

DAILY_PV_LAG_STEPS = 288


# ------------------------------------------------------------
# 3. Verify required source columns
# ------------------------------------------------------------

missing_source_columns = [
    column
    for column in F0_HISTORY_VARIABLES
    if column not in df_clean.columns
]

assert not missing_source_columns, (
    "Missing F0 source columns: "
    f"{missing_source_columns}"
)


# ------------------------------------------------------------
# 4. Verify the source timeline is exactly 5-minute regular
# ------------------------------------------------------------

expected_index = pd.date_range(
    start=df_clean.index.min(),
    end=df_clean.index.max(),
    freq="5min",
    tz="UTC",
)

assert df_clean.index.equals(
    expected_index
)

print(
    "Exact 5-minute timeline check: PASSED"
)


# ------------------------------------------------------------
# 5. Construct the 13-point historical window
# ------------------------------------------------------------

f0_parts = []

for variable in F0_HISTORY_VARIABLES:

    for step_lag in HISTORY_STEP_LAGS:

        minute_lag = step_lag * 5

        feature_name = (
            f"{variable}__lag_{minute_lag:03d}min"
        )

        feature = (
            df_clean[variable]
            .shift(step_lag)
            .rename(feature_name)
        )

        f0_parts.append(
            feature
        )


# ------------------------------------------------------------
# 6. Add the separate previous-day PV feature
# ------------------------------------------------------------

daily_pv_feature = (
    df_clean["P_Solar[kW]"]
    .shift(DAILY_PV_LAG_STEPS)
    .rename(
        "P_Solar[kW]__lag_1440min"
    )
)

f0_parts.append(
    daily_pv_feature
)


# ------------------------------------------------------------
# 7. Combine into the complete F0 table
# ------------------------------------------------------------

df_f0_history = pd.concat(
    f0_parts,
    axis=1,
)


# ------------------------------------------------------------
# 8. Structural verification
# ------------------------------------------------------------

assert df_f0_history.index.equals(
    df_clean.index
)

assert df_f0_history.shape == (
    131_617,
    118,
)

assert df_f0_history.columns.is_unique

assert len(
    df_f0_history.columns
) == primary_protocol[
    "feature_counts"
]["F0"]


# ------------------------------------------------------------
# 9. Determine rows where all F0 history is available
# ------------------------------------------------------------

f0_complete_mask = (
    df_f0_history
    .notna()
    .all(axis=1)
)

n_complete = int(
    f0_complete_mask.sum()
)

n_incomplete = int(
    (~f0_complete_mask).sum()
)


# The 24-hour lag is the longest required history.
assert n_incomplete == 288

assert n_complete == 131_329

assert (
    df_f0_history.index[
        f0_complete_mask
    ][0]
    == pd.Timestamp(
        "2018-06-02 00:00:00",
        tz="UTC",
    )
)


# ------------------------------------------------------------
# 10. Display verification summary
# ------------------------------------------------------------

print(
    "\nF0 historical reconstruction: PASSED"
)

print(
    "Historical variables:",
    len(F0_HISTORY_VARIABLES),
)

print(
    "Observations per variable:",
    len(HISTORY_STEP_LAGS),
)

print(
    "Recent-history features:",
    len(F0_HISTORY_VARIABLES)
    * len(HISTORY_STEP_LAGS),
)

print(
    "Previous-day PV features:",
    1,
)

print(
    "Total F0 features:",
    df_f0_history.shape[1],
)

print(
    "Rows with incomplete history:",
    n_incomplete,
)

print(
    "Rows with complete F0 history:",
    n_complete,
)

print(
    "First complete issue time:",
    df_f0_history.index[
        f0_complete_mask
    ][0],
)

print(
    "\nFirst 10 F0 feature names:"
)

for column in df_f0_history.columns[:10]:
    print(
        " -",
        column,
    )

print(
    "\nLast F0 feature:"
)

print(
    " -",
    df_f0_history.columns[-1],
)

Exact 5-minute timeline check: PASSED

F0 historical reconstruction: PASSED
Historical variables: 9
Observations per variable: 13
Recent-history features: 117
Previous-day PV features: 1
Total F0 features: 118
Rows with incomplete history: 288
Rows with complete F0 history: 131329
First complete issue time: 2018-06-02 00:00:00+00:00

First 10 F0 feature names:
 - P_Solar[kW]__lag_000min
 - P_Solar[kW]__lag_005min
 - P_Solar[kW]__lag_010min
 - P_Solar[kW]__lag_015min
 - P_Solar[kW]__lag_020min
 - P_Solar[kW]__lag_025min
 - P_Solar[kW]__lag_030min
 - P_Solar[kW]__lag_035min
 - P_Solar[kW]__lag_040min
 - P_Solar[kW]__lag_045min

Last F0 feature:
 - P_Solar[kW]__lag_1440min


In [4]:
# ============================================================
# Step 4: Reconstruct direct forecast targets
# ============================================================
#
# For each forecast horizon h, the model is issued at time t
# and predicts:
#
#     P_Solar(t + h)
#
# The feature row remains indexed by the forecast issue time t.
#
# We additionally store the corresponding forecast-valid time:
#
#     valid_time = t + h
#
# This distinction is important because the chronological
# train / validation / test split is defined using forecast-valid
# time, not issue time.
# ============================================================


# ------------------------------------------------------------
# 1. Frozen forecast horizons
# ------------------------------------------------------------

FORECAST_HORIZONS_MINUTES = (
    primary_protocol[
        "forecast_horizons_minutes"
    ]
)

TARGET_COLUMN = (
    primary_protocol["target"]
)


# ------------------------------------------------------------
# 2. Build one target table per forecast horizon
# ------------------------------------------------------------

target_tables = {}


for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    # Number of 5-minute steps between issue time and target.
    horizon_steps = (
        horizon_minutes
        // primary_protocol[
            "resolution_minutes"
        ]
    )

    assert (
        horizon_minutes
        % primary_protocol[
            "resolution_minutes"
        ]
        == 0
    )

    # --------------------------------------------------------
    # Direct target:
    #
    # At issue time t, shift(-horizon_steps) places the
    # observation from t+h onto row t.
    # --------------------------------------------------------

    target = (
        df_clean[TARGET_COLUMN]
        .shift(-horizon_steps)
        .rename("target")
    )

    # --------------------------------------------------------
    # Forecast-valid timestamp associated with each issue row.
    # --------------------------------------------------------

    valid_time = pd.Series(
        df_clean.index
        + pd.Timedelta(
            minutes=horizon_minutes
        ),
        index=df_clean.index,
        name="valid_time",
    )

    target_table = pd.concat(
        [
            target,
            valid_time,
        ],
        axis=1,
    )

    target_tables[
        horizon_minutes
    ] = target_table


# ------------------------------------------------------------
# 3. Verify target availability
# ------------------------------------------------------------

expected_available_rows = {
    15: 131_614,
    30: 131_611,
    60: 131_605,
}

expected_last_issue_times = {
    15: pd.Timestamp(
        "2019-08-31 23:45:00",
        tz="UTC",
    ),
    30: pd.Timestamp(
        "2019-08-31 23:30:00",
        tz="UTC",
    ),
    60: pd.Timestamp(
        "2019-08-31 23:00:00",
        tz="UTC",
    ),
}


for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    table = target_tables[
        horizon_minutes
    ]

    available_mask = (
        table["target"].notna()
    )

    n_available = int(
        available_mask.sum()
    )

    last_issue_time = (
        table.index[
            available_mask
        ][-1]
    )

    last_valid_time = (
        table.loc[
            last_issue_time,
            "valid_time",
        ]
    )

    assert (
        n_available
        == expected_available_rows[
            horizon_minutes
        ]
    )

    assert (
        last_issue_time
        == expected_last_issue_times[
            horizon_minutes
        ]
    )

    assert (
        last_valid_time
        == pd.Timestamp(
            "2019-09-01 00:00:00",
            tz="UTC",
        )
    )

    print(
        f"{horizon_minutes:2d}-minute horizon"
    )

    print(
        "  available targets:",
        n_available,
    )

    print(
        "  last issue time:   ",
        last_issue_time,
    )

    print(
        "  last valid time:   ",
        last_valid_time,
    )

    print()


print(
    "Direct target reconstruction: PASSED"
)

15-minute horizon
  available targets: 131614
  last issue time:    2019-08-31 23:45:00+00:00
  last valid time:    2019-09-01 00:00:00+00:00

30-minute horizon
  available targets: 131611
  last issue time:    2019-08-31 23:30:00+00:00
  last valid time:    2019-09-01 00:00:00+00:00

60-minute horizon
  available targets: 131605
  last issue time:    2019-08-31 23:00:00+00:00
  last valid time:    2019-09-01 00:00:00+00:00

Direct target reconstruction: PASSED


In [5]:
# ============================================================
# Step 5: Reconstruct the frozen modelling populations
#         and chronological train / validation / test splits
# ============================================================
#
# A row is eligible for modelling only when:
#
#   1. all 118 F0 historical features are available;
#   2. the direct future target P_Solar(t+h) is available;
#   3. its forecast-valid time lies inside the frozen
#      experiment interval:
#
#          valid_time < 2019-09-01 00:00 UTC
#
# The final timestamp in the source dataset exists so that
# forecasts valid exactly at 2019-09-01 00:00 can be formed,
# but that boundary point was excluded from the primary
# modelling population in Notebook 09.
#
# Splits are assigned using FORECAST-VALID TIME:
#
# Train:
#   [2018-06-01, 2019-04-01)
#
# Validation:
#   [2019-04-01, 2019-06-01)
#
# Test:
#   [2019-06-01, 2019-09-01)
# ============================================================


# ------------------------------------------------------------
# 1. Frozen split boundaries
# ------------------------------------------------------------

TRAIN_START = pd.Timestamp(
    "2018-06-01 00:00:00",
    tz="UTC",
)

VALIDATION_START = pd.Timestamp(
    "2019-04-01 00:00:00",
    tz="UTC",
)

TEST_START = pd.Timestamp(
    "2019-06-01 00:00:00",
    tz="UTC",
)

EXPERIMENT_END = pd.Timestamp(
    "2019-09-01 00:00:00",
    tz="UTC",
)


# ------------------------------------------------------------
# 2. Expected row counts from the frozen primary experiment
# ------------------------------------------------------------

EXPECTED_SPLIT_COUNTS = {
    15: {
        "total": 131_325,
        "train": 87_261,
        "validation": 17_568,
        "test": 26_496,
    },
    30: {
        "total": 131_322,
        "train": 87_258,
        "validation": 17_568,
        "test": 26_496,
    },
    60: {
        "total": 131_316,
        "train": 87_252,
        "validation": 17_568,
        "test": 26_496,
    },
}


# ------------------------------------------------------------
# 3. Build one canonical modelling table per horizon
# ------------------------------------------------------------

base_modelling_tables = {}


for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    target_table = target_tables[
        horizon_minutes
    ]

    # --------------------------------------------------------
    # Eligible rows:
    #
    # - complete historical F0 information;
    # - available target;
    # - forecast-valid timestamp strictly before the frozen
    #   experiment end boundary.
    # --------------------------------------------------------

    eligible_mask = (
        f0_complete_mask
        & target_table["target"].notna()
        & (
            target_table["valid_time"]
            < EXPERIMENT_END
        )
    )

    issue_index = df_clean.index[
        eligible_mask
    ]

    table = pd.DataFrame(
        index=issue_index
    )

    table.index.name = "issue_time"

    table["target"] = (
        target_table
        .loc[
            issue_index,
            "target",
        ]
        .astype(float)
    )

    table["valid_time"] = (
        target_table
        .loc[
            issue_index,
            "valid_time",
        ]
    )


    # --------------------------------------------------------
    # 4. Assign chronological split by FORECAST-VALID time
    # --------------------------------------------------------

    valid_time = table[
        "valid_time"
    ]

    train_mask = (
        (valid_time >= TRAIN_START)
        & (valid_time < VALIDATION_START)
    )

    validation_mask = (
        (valid_time >= VALIDATION_START)
        & (valid_time < TEST_START)
    )

    test_mask = (
        (valid_time >= TEST_START)
        & (valid_time < EXPERIMENT_END)
    )


    split = pd.Series(
        index=table.index,
        dtype="object",
        name="split",
    )

    split.loc[
        train_mask
    ] = "train"

    split.loc[
        validation_mask
    ] = "validation"

    split.loc[
        test_mask
    ] = "test"

    table["split"] = split


    # --------------------------------------------------------
    # 5. Integrity checks
    # --------------------------------------------------------

    assert table[
        "split"
    ].notna().all()

    assert not (
        train_mask
        & validation_mask
    ).any()

    assert not (
        train_mask
        & test_mask
    ).any()

    assert not (
        validation_mask
        & test_mask
    ).any()

    expected = EXPECTED_SPLIT_COUNTS[
        horizon_minutes
    ]

    assert len(table) == expected[
        "total"
    ]

    assert int(
        train_mask.sum()
    ) == expected["train"]

    assert int(
        validation_mask.sum()
    ) == expected[
        "validation"
    ]

    assert int(
        test_mask.sum()
    ) == expected["test"]


    # --------------------------------------------------------
    # 6. Store canonical horizon table
    # --------------------------------------------------------

    base_modelling_tables[
        horizon_minutes
    ] = table


    # --------------------------------------------------------
    # 7. Display verification
    # --------------------------------------------------------

    print(
        f"{horizon_minutes:2d}-minute horizon"
    )

    print(
        "  total rows:      ",
        len(table),
    )

    print(
        "  train rows:      ",
        int(train_mask.sum()),
    )

    print(
        "  validation rows: ",
        int(validation_mask.sum()),
    )

    print(
        "  test rows:       ",
        int(test_mask.sum()),
    )

    print(
        "  first issue time:",
        table.index[0],
    )

    print(
        "  first valid time:",
        table["valid_time"].iloc[0],
    )

    print(
        "  last valid time: ",
        table["valid_time"].iloc[-1],
    )

    print()


# ------------------------------------------------------------
# 8. Additional primary-horizon boundary verification
# ------------------------------------------------------------

table_60 = base_modelling_tables[60]

train_60 = (
    table_60[
        table_60["split"] == "train"
    ]
)

validation_60 = (
    table_60[
        table_60["split"] == "validation"
    ]
)

test_60 = (
    table_60[
        table_60["split"] == "test"
    ]
)


assert (
    train_60["valid_time"].iloc[-1]
    == pd.Timestamp(
        "2019-03-31 23:55:00",
        tz="UTC",
    )
)

assert (
    validation_60["valid_time"].iloc[0]
    == pd.Timestamp(
        "2019-04-01 00:00:00",
        tz="UTC",
    )
)

assert (
    test_60["valid_time"].iloc[0]
    == pd.Timestamp(
        "2019-06-01 00:00:00",
        tz="UTC",
    )
)


print(
    "Frozen modelling populations and split "
    "reconstruction: PASSED"
)

15-minute horizon
  total rows:       131325
  train rows:       87261
  validation rows:  17568
  test rows:        26496
  first issue time: 2018-06-02 00:00:00+00:00
  first valid time: 2018-06-02 00:15:00+00:00
  last valid time:  2019-08-31 23:55:00+00:00

30-minute horizon
  total rows:       131322
  train rows:       87258
  validation rows:  17568
  test rows:        26496
  first issue time: 2018-06-02 00:00:00+00:00
  first valid time: 2018-06-02 00:30:00+00:00
  last valid time:  2019-08-31 23:55:00+00:00

60-minute horizon
  total rows:       131316
  train rows:       87252
  validation rows:  17568
  test rows:        26496
  first issue time: 2018-06-02 00:00:00+00:00
  first valid time: 2018-06-02 01:00:00+00:00
  last valid time:  2019-08-31 23:55:00+00:00

Frozen modelling populations and split reconstruction: PASSED


In [6]:
# ============================================================
# Step 6: Inspect the reusable Gregorian feature builder
# ============================================================

import inspect

from src.preprocessing import build_gregorian_features


print(
    "Function signature:"
)

print(
    inspect.signature(
        build_gregorian_features
    )
)


print(
    "\nDocstring:"
)

print(
    inspect.getdoc(
        build_gregorian_features
    )
)

Function signature:
(timestamps: 'pd.DatetimeIndex') -> 'pd.DataFrame'

Docstring:
Build the complete Gregorian feature family used by F1.


In [7]:
# ============================================================
# Step 7: Reconstruct F1 Gregorian features
#         at forecast-valid time
# ============================================================
#
# Important:
#
# The historical F0 features belong to forecast issue time t.
#
# Gregorian features are deterministic future-known quantities,
# so for a direct forecast at horizon h they are evaluated at:
#
#     t + h
#
# i.e. at FORECAST-VALID time.
#
# This follows exactly the information scenario frozen in
# Notebook 09.
# ============================================================


# ------------------------------------------------------------
# 1. Build Gregorian valid-time features for every horizon
# ------------------------------------------------------------

gregorian_valid_features = {}


for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    table = base_modelling_tables[
        horizon_minutes
    ]

    # Convert the stored forecast-valid timestamps into the
    # DatetimeIndex expected by the reusable feature builder.
    valid_times = pd.DatetimeIndex(
        table["valid_time"]
    )

    # Build the complete 15-feature Gregorian family.
    gregorian_features = (
        build_gregorian_features(
            valid_times
        )
    )

    # --------------------------------------------------------
    # The reusable builder naturally indexes its output by the
    # timestamps supplied to it, i.e. forecast-valid time.
    #
    # For modelling, however, every feature table is organized
    # by ISSUE TIME.
    #
    # Therefore we explicitly replace the output index with
    # the corresponding issue-time index while preserving the
    # feature values computed at t+h.
    # --------------------------------------------------------

    gregorian_features.index = (
        table.index
    )

    gregorian_features.index.name = (
        "issue_time"
    )


    # --------------------------------------------------------
    # 2. Integrity checks
    # --------------------------------------------------------

    assert gregorian_features.index.equals(
        table.index
    )

    assert len(
        gregorian_features
    ) == len(table)

    assert gregorian_features.shape[1] == 15

    assert (
        gregorian_features
        .isna()
        .sum()
        .sum()
        == 0
    )

    assert (
        gregorian_features
        .columns
        .is_unique
    )


    # --------------------------------------------------------
    # 3. Verify F0 + F1 feature count
    # --------------------------------------------------------

    expected_f1_count = (
        primary_protocol[
            "feature_counts"
        ]["F1"]
    )

    assert (
        df_f0_history.shape[1]
        + gregorian_features.shape[1]
        == expected_f1_count
    )


    # --------------------------------------------------------
    # 4. Store horizon-specific valid-time features
    # --------------------------------------------------------

    gregorian_valid_features[
        horizon_minutes
    ] = gregorian_features


    # --------------------------------------------------------
    # 5. Display verification
    # --------------------------------------------------------

    print(
        f"{horizon_minutes:2d}-minute horizon"
    )

    print(
        "  rows:",
        len(gregorian_features),
    )

    print(
        "  Gregorian features:",
        gregorian_features.shape[1],
    )

    print(
        "  resulting F1 count:",
        df_f0_history.shape[1]
        + gregorian_features.shape[1],
    )

    print(
        "  missing values:",
        int(
            gregorian_features
            .isna()
            .sum()
            .sum()
        ),
    )

    print()


# ------------------------------------------------------------
# 6. Inspect the canonical Gregorian feature names
# ------------------------------------------------------------

reference_f1 = (
    gregorian_valid_features[60]
)

print(
    "F1 Gregorian feature names:"
)

for i, column in enumerate(
    reference_f1.columns,
    start=1,
):
    print(
        f"{i:2d}. {column}"
    )


# ------------------------------------------------------------
# 7. Verify a primary-horizon alignment example
# ------------------------------------------------------------

example_issue_time = (
    base_modelling_tables[60]
    .index[0]
)

example_valid_time = (
    base_modelling_tables[60]
    .loc[
        example_issue_time,
        "valid_time",
    ]
)

print(
    "\n60-minute alignment example:"
)

print(
    "  issue time:",
    example_issue_time,
)

print(
    "  valid time:",
    example_valid_time,
)

print(
    "\nF1 Gregorian reconstruction: PASSED"
)

15-minute horizon
  rows: 131325
  Gregorian features: 15
  resulting F1 count: 133
  missing values: 0

30-minute horizon
  rows: 131322
  Gregorian features: 15
  resulting F1 count: 133
  missing values: 0

60-minute horizon
  rows: 131316
  Gregorian features: 15
  resulting F1 count: 133
  missing values: 0

F1 Gregorian feature names:
 1. utc_year
 2. utc_month
 3. utc_day
 4. utc_day_of_week
 5. utc_day_of_year
 6. utc_hour
 7. utc_minute
 8. utc_minute_of_day
 9. utc_is_weekend
10. utc_time_sin
11. utc_time_cos
12. utc_weekday_sin
13. utc_weekday_cos
14. utc_year_sin
15. utc_year_cos

60-minute alignment example:
  issue time: 2018-06-02 00:00:00+00:00
  valid time: 2018-06-02 01:00:00+00:00

F1 Gregorian reconstruction: PASSED


In [8]:
# ============================================================
# Step 8: Inspect reusable F2 solar astronomy functions
# ============================================================

import inspect
import src.astronomy_features as astro


# ------------------------------------------------------------
# 1. Find reusable functions related to solar features
# ------------------------------------------------------------

solar_functions = [
    name
    for name in dir(astro)
    if (
        "solar" in name.lower()
        and callable(getattr(astro, name))
        and not name.startswith("_")
    )
]


print("Reusable solar-related functions:")

for name in solar_functions:

    function = getattr(
        astro,
        name,
    )

    try:
        signature = inspect.signature(
            function
        )
    except (TypeError, ValueError):
        signature = "<signature unavailable>"

    print(
        f"\n{name}{signature}"
    )

    docstring = inspect.getdoc(
        function
    )

    if docstring:
        print(
            "  ",
            docstring.splitlines()[0],
        )

Reusable solar-related functions:

build_solar_f2_features(timestamps: 'pd.DatetimeIndex') -> 'pd.DataFrame'
   Build the five deterministic solar-geometry features used by F2.

solar_position_spa(timestamps: 'pd.DatetimeIndex') -> 'pd.DataFrame'
   Compute solar position at the validated SOLETE reference times.


In [9]:
# ============================================================
# Step 9: Reconstruct F2 solar-astronomy features
#         at forecast-valid time
# ============================================================
#
# F2 adds five deterministic solar-geometry features.
#
# These features are known in advance and are therefore
# evaluated at forecast-valid time:
#
#     t + h
#
# The reusable builder internally applies the validated SOLETE
# astronomical reference convention:
#
#     SOLETE timestamp + 2 minutes
#
# We compute the solar features ONCE for the complete canonical
# 5-minute timeline and then select the required valid-time rows
# for each forecast horizon.
# ============================================================


# ------------------------------------------------------------
# 1. Compute F2 solar features once for the full SOLETE timeline
# ------------------------------------------------------------

df_solar_f2_full = (
    astro.build_solar_f2_features(
        df_clean.index
    )
)


# ------------------------------------------------------------
# 2. Verify the full deterministic solar table
# ------------------------------------------------------------

assert df_solar_f2_full.index.equals(
    df_clean.index
)

assert df_solar_f2_full.shape == (
    131_617,
    5,
)

assert (
    df_solar_f2_full
    .isna()
    .sum()
    .sum()
    == 0
)

assert df_solar_f2_full.columns.is_unique


print(
    "Full F2 solar table: PASSED"
)

print(
    "Shape:",
    df_solar_f2_full.shape,
)

print(
    "Missing values:",
    int(
        df_solar_f2_full
        .isna()
        .sum()
        .sum()
    ),
)


# ------------------------------------------------------------
# 3. Show the canonical F2 feature names
# ------------------------------------------------------------

print(
    "\nF2 solar feature names:"
)

for i, column in enumerate(
    df_solar_f2_full.columns,
    start=1,
):
    print(
        f"{i:2d}. {column}"
    )


# ------------------------------------------------------------
# 4. Select solar features at forecast-valid time for each
#    forecast horizon
# ------------------------------------------------------------

solar_valid_features = {}


for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    table = base_modelling_tables[
        horizon_minutes
    ]

    valid_times = pd.DatetimeIndex(
        table["valid_time"]
    )


    # --------------------------------------------------------
    # Select rows using forecast-valid timestamps.
    #
    # The values therefore correspond to solar geometry at
    # t+h, while the final modelling table remains indexed by
    # issue time t.
    # --------------------------------------------------------

    solar_features = (
        df_solar_f2_full
        .loc[
            valid_times
        ]
        .copy()
    )

    solar_features.index = (
        table.index
    )

    solar_features.index.name = (
        "issue_time"
    )


    # --------------------------------------------------------
    # 5. Integrity checks
    # --------------------------------------------------------

    assert solar_features.index.equals(
        table.index
    )

    assert solar_features.shape == (
        len(table),
        5,
    )

    assert (
        solar_features
        .isna()
        .sum()
        .sum()
        == 0
    )


    # F1 = 133 features
    # F2 adds 5
    # therefore F2 = 138.
    assert (
        df_f0_history.shape[1]
        + gregorian_valid_features[
            horizon_minutes
        ].shape[1]
        + solar_features.shape[1]
        == primary_protocol[
            "feature_counts"
        ]["F2"]
    )


    solar_valid_features[
        horizon_minutes
    ] = solar_features


    print(
        f"\n{horizon_minutes:2d}-minute horizon"
    )

    print(
        "  rows:",
        len(solar_features),
    )

    print(
        "  solar features:",
        solar_features.shape[1],
    )

    print(
        "  resulting F2 count:",
        (
            df_f0_history.shape[1]
            + gregorian_valid_features[
                horizon_minutes
            ].shape[1]
            + solar_features.shape[1]
        ),
    )

    print(
        "  missing values:",
        int(
            solar_features
            .isna()
            .sum()
            .sum()
        ),
    )


# ------------------------------------------------------------
# 6. Verify one 60-minute alignment example
# ------------------------------------------------------------

example_issue_time = (
    base_modelling_tables[60]
    .index[0]
)

example_valid_time = (
    base_modelling_tables[60]
    .loc[
        example_issue_time,
        "valid_time",
    ]
)

example_astronomy_time = (
    example_valid_time
    + pd.Timedelta(
        minutes=2
    )
)


print(
    "\n60-minute F2 alignment example:"
)

print(
    "  issue time:       ",
    example_issue_time,
)

print(
    "  forecast-valid:   ",
    example_valid_time,
)

print(
    "  astronomy reference:",
    example_astronomy_time,
)


print(
    "\nSolar features for this forecast:"
)

print(
    solar_valid_features[60]
    .loc[
        example_issue_time
    ]
)


print(
    "\nF2 solar reconstruction: PASSED"
)

Full F2 solar table: PASSED
Shape: (131617, 5)
Missing values: 0

F2 solar feature names:
 1. solar_elevation_deg
 2. solar_cos_zenith
 3. solar_azimuth_sin
 4. solar_azimuth_cos
 5. solar_daylight

15-minute horizon
  rows: 131325
  solar features: 5
  resulting F2 count: 138
  missing values: 0

30-minute horizon
  rows: 131322
  solar features: 5
  resulting F2 count: 138
  missing values: 0

60-minute horizon
  rows: 131316
  solar features: 5
  resulting F2 count: 138
  missing values: 0

60-minute F2 alignment example:
  issue time:        2018-06-02 00:00:00+00:00
  forecast-valid:    2018-06-02 01:00:00+00:00
  astronomy reference: 2018-06-02 01:02:00+00:00

Solar features for this forecast:
solar_elevation_deg   -8.571247
solar_cos_zenith      -0.149039
solar_azimuth_sin      0.441480
solar_azimuth_cos      0.897271
solar_daylight         0.000000
Name: 2018-06-02 00:00:00+00:00, dtype: float64

F2 solar reconstruction: PASSED


In [10]:
# ============================================================
# Step 10: Inspect reusable F3 lunar / Sun-Moon astronomy API
# ============================================================
#
# F3 adds continuous astronomical quantities beyond the
# Gregorian and solar feature families.
#
# We inspect the reusable functions first so that Notebook 10
# calls exactly the implementation already validated in
# Notebooks 07–09.
# ============================================================

import inspect
import src.astronomy_features as astro


# ------------------------------------------------------------
# 1. Find public functions related to lunar / Moon features
# ------------------------------------------------------------

lunar_functions = [
    name
    for name in dir(astro)
    if (
        (
            "lunar" in name.lower()
            or "moon" in name.lower()
            or "ephemeris" in name.lower()
        )
        and callable(getattr(astro, name))
        and not name.startswith("_")
    )
]


print(
    "Reusable lunar / Moon / ephemeris-related functions:"
)


for name in lunar_functions:

    function = getattr(
        astro,
        name,
    )

    try:
        signature = inspect.signature(
            function
        )
    except (TypeError, ValueError):
        signature = "<signature unavailable>"

    print(
        f"\n{name}{signature}"
    )

    docstring = inspect.getdoc(
        function
    )

    if docstring:
        print(
            "  ",
            docstring.splitlines()[0],
        )

Reusable lunar / Moon / ephemeris-related functions:

build_lunar_f3_features(timestamps: 'pd.DatetimeIndex', ephemeris) -> 'pd.DataFrame'
   Build the nine continuous lunar/Sun-Moon features used by F3.

load_jpl_ephemeris(ephemeris_path: 'str | Path')
   Load a JPL BSP ephemeris file with Skyfield.

moon_ecliptic_latitude_and_distance(ephemeris, skyfield_times) -> 'tuple[np.ndarray, np.ndarray]'
   Return apparent Moon ecliptic latitude (deg) and Earth-Moon distance (km).

moon_illumination_fraction(ephemeris, skyfield_times) -> 'np.ndarray'
   Return the apparent illuminated fraction of the Moon in [0, 1].

moon_sun_phase_angle_deg(sun_longitude_deg: 'float | np.ndarray | pd.Series', moon_longitude_deg: 'float | np.ndarray | pd.Series') -> 'float | np.ndarray | pd.Series'
   Return Moon-Sun ecliptic longitude separation in [0, 360).


In [11]:
# ============================================================
# Step 11: Reconstruct F3 continuous lunar / Sun-Moon features
#          at forecast-valid time
# ============================================================
#
# F3 adds nine continuous astronomical controls beyond F2.
#
# These quantities are deterministic and known in advance, so
# for each forecast issued at time t they are evaluated at:
#
#     forecast-valid time = t + h
#
# The reusable implementation internally applies the validated
# SOLETE astronomical reference convention:
#
#     SOLETE timestamp + 2 minutes
#
# As with F2, the expensive ephemeris calculations are performed
# ONCE for the complete 5-minute timeline and then selected for
# each forecast horizon.
# ============================================================


# ------------------------------------------------------------
# 1. Locate and verify the local JPL DE440s ephemeris
# ------------------------------------------------------------

EPHEMERIS_PATH = (
    PROJECT_ROOT
    / "ephemeris"
    / "de440s.bsp"
)

assert EPHEMERIS_PATH.is_file(), (
    "JPL ephemeris file not found:\n"
    f"{EPHEMERIS_PATH}"
)

print(
    "JPL ephemeris:"
)

print(
    EPHEMERIS_PATH.relative_to(
        PROJECT_ROOT
    )
)


# ------------------------------------------------------------
# 2. Load the ephemeris once
# ------------------------------------------------------------

jpl_ephemeris = (
    astro.load_jpl_ephemeris(
        EPHEMERIS_PATH
    )
)

print(
    "\nJPL DE440s ephemeris loaded."
)


# ------------------------------------------------------------
# 3. Compute the complete F3 lunar / Sun-Moon feature table
# ------------------------------------------------------------
#
# This may take a little longer than the Gregorian or solar
# feature calculations because Skyfield evaluates the JPL
# ephemeris across all 131,617 timestamps.
# ------------------------------------------------------------

df_lunar_f3_full = (
    astro.build_lunar_f3_features(
        df_clean.index,
        jpl_ephemeris,
    )
)


# ------------------------------------------------------------
# 4. Verify the complete F3 table
# ------------------------------------------------------------

assert df_lunar_f3_full.index.equals(
    df_clean.index
)

assert df_lunar_f3_full.shape == (
    131_617,
    9,
)

assert (
    df_lunar_f3_full
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    df_lunar_f3_full
    .columns
    .is_unique
)


print(
    "\nFull F3 lunar / Sun-Moon table: PASSED"
)

print(
    "Shape:",
    df_lunar_f3_full.shape,
)

print(
    "Missing values:",
    int(
        df_lunar_f3_full
        .isna()
        .sum()
        .sum()
    ),
)


# ------------------------------------------------------------
# 5. Show the canonical nine F3 feature names
# ------------------------------------------------------------

print(
    "\nF3 lunar / Sun-Moon feature names:"
)

for i, column in enumerate(
    df_lunar_f3_full.columns,
    start=1,
):
    print(
        f"{i:2d}. {column}"
    )


# ------------------------------------------------------------
# 6. Select F3 features at forecast-valid time for each horizon
# ------------------------------------------------------------

lunar_valid_features = {}


for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    table = base_modelling_tables[
        horizon_minutes
    ]

    valid_times = pd.DatetimeIndex(
        table["valid_time"]
    )


    # Select astronomy values at forecast-valid time t+h.
    lunar_features = (
        df_lunar_f3_full
        .loc[
            valid_times
        ]
        .copy()
    )

    # Re-index by forecast issue time t so that the table aligns
    # directly with F0, F1, F2 and the target.
    lunar_features.index = (
        table.index
    )

    lunar_features.index.name = (
        "issue_time"
    )


    # --------------------------------------------------------
    # 7. Integrity checks
    # --------------------------------------------------------

    assert lunar_features.index.equals(
        table.index
    )

    assert lunar_features.shape == (
        len(table),
        9,
    )

    assert (
        lunar_features
        .isna()
        .sum()
        .sum()
        == 0
    )


    # F2 = 138 features
    # F3 adds 9 continuous astronomy features
    # therefore:
    #
    #     F3 = 147
    # --------------------------------------------------------

    resulting_f3_count = (
        df_f0_history.shape[1]
        + gregorian_valid_features[
            horizon_minutes
        ].shape[1]
        + solar_valid_features[
            horizon_minutes
        ].shape[1]
        + lunar_features.shape[1]
    )

    assert (
        resulting_f3_count
        == primary_protocol[
            "feature_counts"
        ]["F3"]
    )


    lunar_valid_features[
        horizon_minutes
    ] = lunar_features


    print(
        f"\n{horizon_minutes:2d}-minute horizon"
    )

    print(
        "  rows:",
        len(lunar_features),
    )

    print(
        "  lunar / Sun-Moon features:",
        lunar_features.shape[1],
    )

    print(
        "  resulting F3 count:",
        resulting_f3_count,
    )

    print(
        "  missing values:",
        int(
            lunar_features
            .isna()
            .sum()
            .sum()
        ),
    )


# ------------------------------------------------------------
# 8. Verify one 60-minute timing example
# ------------------------------------------------------------

example_issue_time = (
    base_modelling_tables[60]
    .index[0]
)

example_valid_time = (
    base_modelling_tables[60]
    .loc[
        example_issue_time,
        "valid_time",
    ]
)

example_astronomy_time = (
    example_valid_time
    + pd.Timedelta(
        minutes=2
    )
)


print(
    "\n60-minute F3 alignment example:"
)

print(
    "  issue time:          ",
    example_issue_time,
)

print(
    "  forecast-valid time: ",
    example_valid_time,
)

print(
    "  astronomy reference: ",
    example_astronomy_time,
)


print(
    "\nF3 features for this forecast:"
)

print(
    lunar_valid_features[60]
    .loc[
        example_issue_time
    ]
)


print(
    "\nF3 lunar / Sun-Moon reconstruction: PASSED"
)

JPL ephemeris:
ephemeris/de440s.bsp

JPL DE440s ephemeris loaded.

Full F3 lunar / Sun-Moon table: PASSED
Shape: (131617, 9)
Missing values: 0

F3 lunar / Sun-Moon feature names:
 1. moon_phase_sin
 2. moon_phase_cos
 3. moon_longitude_sin
 4. moon_longitude_cos
 5. sun_ecliptic_longitude_sin
 6. sun_ecliptic_longitude_cos
 7. moon_ecliptic_latitude_deg
 8. moon_illumination_fraction
 9. earth_moon_distance_km

15-minute horizon
  rows: 131325
  lunar / Sun-Moon features: 9
  resulting F3 count: 147
  missing values: 0

30-minute horizon
  rows: 131322
  lunar / Sun-Moon features: 9
  resulting F3 count: 147
  missing values: 0

60-minute horizon
  rows: 131316
  lunar / Sun-Moon features: 9
  resulting F3 count: 147
  missing values: 0

60-minute F3 alignment example:
  issue time:           2018-06-02 00:00:00+00:00
  forecast-valid time:  2018-06-02 01:00:00+00:00
  astronomy reference:  2018-06-02 01:02:00+00:00

F3 features for this forecast:
moon_phase_sin                    -0.6

In [12]:
# ============================================================
# Step 12: Inspect reusable F4 Panchang API and load the
#          frozen Notebook 09 categorical vocabulary
# ============================================================
#
# F4 adds five raw Panchang categories:
#
#   - Tithi
#   - Paksha
#   - Karana
#   - Nakshatra
#   - Yoga
#
# These are later encoded using the exact fixed vocabulary
# frozen in Notebook 09.
#
# We first inspect the reusable API and load that vocabulary
# rather than reconstructing category order from the data.
# ============================================================

import inspect
import src.vedic_calendar as vedic


# ------------------------------------------------------------
# 1. Inspect public Panchang-related reusable functions
# ------------------------------------------------------------

panchang_functions = [
    name
    for name in dir(vedic)
    if (
        (
            "panchang" in name.lower()
            or "ayanamsha" in name.lower()
            or "sidereal" in name.lower()
        )
        and callable(getattr(vedic, name))
        and not name.startswith("_")
    )
]


print(
    "Reusable Panchang / sidereal functions:"
)

for name in panchang_functions:

    function = getattr(
        vedic,
        name,
    )

    try:
        signature = inspect.signature(
            function
        )
    except (TypeError, ValueError):
        signature = "<signature unavailable>"

    print(
        f"\n{name}{signature}"
    )

    docstring = inspect.getdoc(
        function
    )

    if docstring:
        print(
            "  ",
            docstring.splitlines()[0],
        )


# ------------------------------------------------------------
# 2. Load the frozen Panchang vocabulary from Notebook 09
# ------------------------------------------------------------

PANCHANG_VOCABULARY_PATH = (
    PRIMARY_RESULTS_DIR
    / "panchang_vocabulary.json"
)

assert PANCHANG_VOCABULARY_PATH.is_file(), (
    "Frozen Panchang vocabulary not found:\n"
    f"{PANCHANG_VOCABULARY_PATH}"
)

with open(
    PANCHANG_VOCABULARY_PATH,
    "r",
    encoding="utf-8",
) as file:
    panchang_vocabulary = json.load(file)


print(
    "\nFrozen Panchang vocabulary loaded."
)

print(
    "\nVocabulary families:"
)

for family, categories in panchang_vocabulary.items():

    print(
        f"{family:12s}: "
        f"{len(categories):2d} categories"
    )


# ------------------------------------------------------------
# 3. Verify total one-hot width
# ------------------------------------------------------------

total_panchang_indicators = sum(
    len(categories)
    for categories
    in panchang_vocabulary.values()
)

print(
    "\nTotal Panchang one-hot indicators:",
    total_panchang_indicators,
)

assert total_panchang_indicators == 97

assert (
    primary_protocol["feature_counts"]["F3"]
    + total_panchang_indicators
    == primary_protocol["feature_counts"]["F4"]
)


print(
    "\nFrozen F4 vocabulary verification: PASSED"
)

Reusable Panchang / sidereal functions:

build_panchang_f4_raw(sun_tropical_longitude_deg: 'pd.Series', moon_tropical_longitude_deg: 'pd.Series', ayanamsha_deg: 'pd.Series') -> 'pd.DataFrame'
   Build the canonical raw F4 Panchang feature table.

lahiri_ayanamsha_extended_deg(julian_day_ut: 'float') -> 'tuple[float, int]'
   Compute the canonical extended Lahiri ayanamsha for one UT Julian day.

lahiri_ayanamsha_for_utc_times(timestamps: 'pd.DatetimeIndex') -> 'np.ndarray'
   Compute canonical extended Lahiri ayanamsha for UTC timestamps.

nakshatra_number_from_sidereal_moon(moon_sidereal_longitude_deg: 'float | np.ndarray | pd.Series') -> 'int | np.ndarray | pd.Series'
   Convert Lahiri sidereal Moon longitude into Nakshatra number.

nakshatra_progress_from_sidereal_moon(moon_sidereal_longitude_deg: 'float | np.ndarray | pd.Series') -> 'float | np.ndarray | pd.Series'
   Return fractional progress through the current Nakshatra.

sidereal_longitude_from_tropical(tropical_longitude_deg:

In [13]:
# ============================================================
# Step 13: Inspect reusable tropical Sun/Moon longitude API
# ============================================================
#
# Raw Panchang construction requires:
#
#   - tropical Sun ecliptic longitude;
#   - tropical Moon ecliptic longitude;
#   - canonical Lahiri ayanamsha.
#
# We inspect the astronomy module first so that we call the
# exact reusable implementation already validated previously.
# ============================================================

import inspect
import src.astronomy_features as astro


longitude_functions = [
    name
    for name in dir(astro)
    if (
        "longitude" in name.lower()
        and callable(getattr(astro, name))
        and not name.startswith("_")
    )
]


print(
    "Reusable longitude-related astronomy functions:"
)


for name in longitude_functions:

    function = getattr(
        astro,
        name,
    )

    try:
        signature = inspect.signature(
            function
        )
    except (TypeError, ValueError):
        signature = "<signature unavailable>"

    print(
        f"\n{name}{signature}"
    )

    docstring = inspect.getdoc(
        function
    )

    if docstring:
        print(
            "  ",
            docstring.splitlines()[0],
        )

Reusable longitude-related astronomy functions:

apparent_tropical_ecliptic_longitudes(ephemeris, skyfield_times) -> 'tuple[np.ndarray, np.ndarray]'
   Return apparent geocentric tropical ecliptic longitudes of Sun and Moon.

longitude_cyclic_features(longitude_deg: 'float | np.ndarray | pd.Series') -> 'tuple[float | np.ndarray | pd.Series, float | np.ndarray | pd.Series]'
   Return sine and cosine encoding of an angular longitude.

tropical_longitudes_for_solete(timestamps: 'pd.DatetimeIndex', ephemeris) -> 'pd.DataFrame'
   Return apparent tropical Sun/Moon longitudes at SOLETE reference times.


In [14]:
# ============================================================
# Step 14: Reconstruct the astronomical inputs required by F4
# ============================================================
#
# The raw Panchang categories require:
#
#   1. apparent tropical Sun longitude;
#   2. apparent tropical Moon longitude;
#   3. canonical extended Lahiri ayanamsha.
#
# Sun/Moon tropical longitudes are evaluated using the reusable
# SOLETE astronomy convention:
#
#     timestamp + 2 minutes
#
# The Lahiri ayanamsha must be evaluated at those SAME
# astronomical reference timestamps.
# ============================================================


# ------------------------------------------------------------
# 1. Compute apparent tropical Sun/Moon longitudes
# ------------------------------------------------------------

df_tropical_longitudes_full = (
    astro.tropical_longitudes_for_solete(
        df_clean.index,
        jpl_ephemeris,
    )
)


# ------------------------------------------------------------
# 2. Verify the longitude table
# ------------------------------------------------------------

assert df_tropical_longitudes_full.index.equals(
    df_clean.index
)

assert df_tropical_longitudes_full.shape == (
    131_617,
    2,
)

assert (
    df_tropical_longitudes_full
    .isna()
    .sum()
    .sum()
    == 0
)


print(
    "Full tropical-longitude table: PASSED"
)

print(
    "Shape:",
    df_tropical_longitudes_full.shape,
)

print(
    "Missing values:",
    int(
        df_tropical_longitudes_full
        .isna()
        .sum()
        .sum()
    ),
)


print(
    "\nTropical-longitude columns:"
)

for column in df_tropical_longitudes_full.columns:
    print(
        " -",
        column,
    )


# ------------------------------------------------------------
# 3. Construct the exact SOLETE astronomical reference times
# ------------------------------------------------------------

astronomy_reference_times = (
    astro.solete_astronomy_reference_times(
        df_clean.index
    )
)


assert len(
    astronomy_reference_times
) == len(df_clean)

assert (
    astronomy_reference_times[0]
    == df_clean.index[0]
    + pd.Timedelta(minutes=2)
)


print(
    "\nFirst SOLETE timestamp:       ",
    df_clean.index[0],
)

print(
    "First astronomy reference:   ",
    astronomy_reference_times[0],
)


# ------------------------------------------------------------
# 4. Compute canonical extended Lahiri ayanamsha
# ------------------------------------------------------------
#
# Important:
#
# We pass the +2-minute astronomy reference timestamps here.
# This keeps the ayanamsha temporally aligned with the tropical
# Sun and Moon longitudes.
#
# This operation uses the reusable canonical extended-Lahiri
# convention established in Notebook 08.
# ------------------------------------------------------------

ayanamsha_values = (
    vedic.lahiri_ayanamsha_for_utc_times(
        astronomy_reference_times
    )
)


# Convert to a Series indexed by the canonical SOLETE timestamp.
#
# Values correspond to timestamp + 2 minutes, but keeping the
# SOLETE timestamp as the table index makes later alignment
# straightforward.
ayanamsha_full = pd.Series(
    ayanamsha_values,
    index=df_clean.index,
    name="lahiri_ayanamsha_deg",
)


# ------------------------------------------------------------
# 5. Verify the ayanamsha timeline
# ------------------------------------------------------------

assert ayanamsha_full.index.equals(
    df_clean.index
)

assert len(
    ayanamsha_full
) == 131_617

assert not ayanamsha_full.isna().any()

assert np.isfinite(
    ayanamsha_full.to_numpy()
).all()


print(
    "\nCanonical Lahiri ayanamsha: PASSED"
)

print(
    "First value:",
    ayanamsha_full.iloc[0],
)

print(
    "Minimum:",
    ayanamsha_full.min(),
)

print(
    "Maximum:",
    ayanamsha_full.max(),
)


# ------------------------------------------------------------
# 6. Strong consistency checks against Notebook 08
# ------------------------------------------------------------

assert np.isclose(
    ayanamsha_full.iloc[0],
    24.110343268621794,
    atol=1e-10,
)

assert np.isclose(
    ayanamsha_full.max(),
    24.12737225615406,
    atol=1e-10,
)


print(
    "\nF4 astronomical-input reconstruction: PASSED"
)

Full tropical-longitude table: PASSED
Shape: (131617, 2)
Missing values: 0

Tropical-longitude columns:
 - sun_tropical_longitude_deg
 - moon_tropical_longitude_deg

First SOLETE timestamp:        2018-06-01 00:00:00+00:00
First astronomy reference:    2018-06-01 00:02:00+00:00

Canonical Lahiri ayanamsha: PASSED
First value: 24.110343268621794
Minimum: 24.110343268621794
Maximum: 24.12737225615406

F4 astronomical-input reconstruction: PASSED


In [15]:
# ============================================================
# Step 15: Reconstruct the canonical raw F4 Panchang features
# ============================================================
#
# The five raw Panchang categories are:
#
#   1. Tithi
#   2. Paksha
#   3. Karana
#   4. Nakshatra
#   5. Yoga
#
# These are derived from:
#
#   - apparent tropical Sun longitude;
#   - apparent tropical Moon longitude;
#   - canonical extended Lahiri ayanamsha.
#
# All three astronomical inputs correspond to the validated
# SOLETE reference time:
#
#     timestamp + 2 minutes
#
# We first construct the raw categorical table for the entire
# canonical timeline. Encoding into the frozen 97-column F4
# representation will be done in the next step.
# ============================================================


# ------------------------------------------------------------
# 1. Extract the verified tropical longitude series
# ------------------------------------------------------------

sun_tropical_full = (
    df_tropical_longitudes_full[
        "sun_tropical_longitude_deg"
    ]
)

moon_tropical_full = (
    df_tropical_longitudes_full[
        "moon_tropical_longitude_deg"
    ]
)


# ------------------------------------------------------------
# 2. Build the complete raw Panchang table
# ------------------------------------------------------------

df_panchang_f4_raw_full = (
    vedic.build_panchang_f4_raw(
        sun_tropical_full,
        moon_tropical_full,
        ayanamsha_full,
    )
)


# ------------------------------------------------------------
# 3. Verify structure
# ------------------------------------------------------------

EXPECTED_PANCHANG_COLUMNS = [
    "tithi",
    "paksha",
    "karana",
    "nakshatra",
    "yoga",
]


assert df_panchang_f4_raw_full.index.equals(
    df_clean.index
)

assert df_panchang_f4_raw_full.shape == (
    131_617,
    5,
)

assert list(
    df_panchang_f4_raw_full.columns
) == EXPECTED_PANCHANG_COLUMNS

assert (
    df_panchang_f4_raw_full
    .isna()
    .sum()
    .sum()
    == 0
)


print(
    "Full raw F4 Panchang table: PASSED"
)

print(
    "Shape:",
    df_panchang_f4_raw_full.shape,
)

print(
    "Missing values:",
    int(
        df_panchang_f4_raw_full
        .isna()
        .sum()
        .sum()
    ),
)


# ------------------------------------------------------------
# 4. Verify the canonical first reference row
# ------------------------------------------------------------
#
# This is a strong regression check against the canonical
# Notebook 08 calculation at:
#
# SOLETE timestamp:
#     2018-06-01 00:00 UTC
#
# astronomy reference:
#     2018-06-01 00:02 UTC
# ------------------------------------------------------------

first_timestamp = (
    df_panchang_f4_raw_full.index[0]
)

first_panchang = (
    df_panchang_f4_raw_full
    .iloc[0]
)


assert int(
    first_panchang["tithi"]
) == 18

assert (
    first_panchang["paksha"]
    == "Krishna"
)

assert (
    first_panchang["karana"]
    == "Vanija"
)

assert (
    first_panchang["nakshatra"]
    == "Mula"
)

assert (
    first_panchang["yoga"]
    == "Shubha"
)


print(
    "\nCanonical first Panchang row:"
)

print(
    "SOLETE timestamp:",
    first_timestamp,
)

print(
    "Astronomy reference:",
    first_timestamp
    + pd.Timedelta(minutes=2),
)

print(
    first_panchang
)


# ------------------------------------------------------------
# 5. Verify that every observed category belongs to the frozen
#    Notebook 09 vocabulary
# ------------------------------------------------------------

for family in EXPECTED_PANCHANG_COLUMNS:

    observed_categories = set(
        df_panchang_f4_raw_full[
            family
        ].tolist()
    )

    frozen_categories = set(
        panchang_vocabulary[
            family
        ]
    )

    unexpected_categories = (
        observed_categories
        - frozen_categories
    )

    assert not unexpected_categories, (
        f"Unexpected {family} categories: "
        f"{unexpected_categories}"
    )

    print(
        f"{family:12s}: "
        f"{len(observed_categories):2d} "
        f"observed categories"
    )


# ------------------------------------------------------------
# 6. Select raw Panchang features at forecast-valid time
#    for each horizon
# ------------------------------------------------------------

panchang_raw_valid_features = {}


for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    table = base_modelling_tables[
        horizon_minutes
    ]

    valid_times = pd.DatetimeIndex(
        table["valid_time"]
    )

    panchang_features = (
        df_panchang_f4_raw_full
        .loc[
            valid_times
        ]
        .copy()
    )

    # Values correspond to t+h, but modelling rows are indexed
    # by forecast issue time t.
    panchang_features.index = (
        table.index
    )

    panchang_features.index.name = (
        "issue_time"
    )


    assert panchang_features.index.equals(
        table.index
    )

    assert panchang_features.shape == (
        len(table),
        5,
    )

    assert (
        panchang_features
        .isna()
        .sum()
        .sum()
        == 0
    )


    panchang_raw_valid_features[
        horizon_minutes
    ] = panchang_features


    print(
        f"\n{horizon_minutes:2d}-minute horizon"
    )

    print(
        "  rows:",
        len(panchang_features),
    )

    print(
        "  raw Panchang features:",
        panchang_features.shape[1],
    )


# ------------------------------------------------------------
# 7. Inspect the first 60-minute valid-time Panchang row
# ------------------------------------------------------------

example_issue_time = (
    base_modelling_tables[60]
    .index[0]
)

example_valid_time = (
    base_modelling_tables[60]
    .loc[
        example_issue_time,
        "valid_time",
    ]
)


print(
    "\nFirst 60-minute forecast:"
)

print(
    "  issue time:",
    example_issue_time,
)

print(
    "  valid time:",
    example_valid_time,
)

print(
    "\nPanchang at forecast-valid time:"
)

print(
    panchang_raw_valid_features[60]
    .loc[
        example_issue_time
    ]
)


print(
    "\nRaw F4 Panchang reconstruction: PASSED"
)

Full raw F4 Panchang table: PASSED
Shape: (131617, 5)
Missing values: 0

Canonical first Panchang row:
SOLETE timestamp: 2018-06-01 00:00:00+00:00
Astronomy reference: 2018-06-01 00:02:00+00:00
tithi             18
paksha       Krishna
karana        Vanija
nakshatra       Mula
yoga          Shubha
Name: 2018-06-01 00:00:00+00:00, dtype: object
tithi       : 30 observed categories
paksha      :  2 observed categories
karana      : 11 observed categories
nakshatra   : 27 observed categories
yoga        : 27 observed categories

15-minute horizon
  rows: 131325
  raw Panchang features: 5

30-minute horizon
  rows: 131322
  raw Panchang features: 5

60-minute horizon
  rows: 131316
  raw Panchang features: 5

First 60-minute forecast:
  issue time: 2018-06-02 00:00:00+00:00
  valid time: 2018-06-02 01:00:00+00:00

Panchang at forecast-valid time:
tithi                   19
paksha             Krishna
karana                Bava
nakshatra    Purva Ashadha
yoga                Shukla
Name: 2018

In [16]:
# ============================================================
# Step 16: Encode raw Panchang categories into the frozen
#          97-column F4 one-hot representation
# ============================================================
#
# Important design choice:
#
# We DO NOT discover categories independently from each
# train/validation/test subset or forecast horizon.
#
# Instead, the exact categorical vocabulary frozen in
# Notebook 09 is used for every row and every horizon.
#
# This guarantees:
#
#   Tithi       = 30 indicators
#   Paksha      =  2 indicators
#   Karana      = 11 indicators
#   Nakshatra   = 27 indicators
#   Yoga        = 27 indicators
#                  --
#                  97 total
#
# Since each Panchang family contributes exactly one category,
# every row must contain exactly five active indicators.
# ============================================================


# ------------------------------------------------------------
# 1. Canonical Panchang family order
# ------------------------------------------------------------

PANCHANG_FAMILIES = [
    "tithi",
    "paksha",
    "karana",
    "nakshatra",
    "yoga",
]


# ------------------------------------------------------------
# 2. Helper for frozen-vocabulary one-hot encoding
# ------------------------------------------------------------

def encode_frozen_panchang(
    raw_panchang: pd.DataFrame,
    vocabulary: dict,
) -> pd.DataFrame:
    """
    Encode the five raw Panchang categorical variables using
    the exact category vocabulary frozen in Notebook 09.

    The output always contains the same 97 columns in the same
    deterministic family/category order.
    """

    encoded_parts = []

    for family in PANCHANG_FAMILIES:

        categories = vocabulary[
            family
        ]

        # Explicitly assign the frozen category set.
        categorical = pd.Series(
            pd.Categorical(
                raw_panchang[family],
                categories=categories,
                ordered=False,
            ),
            index=raw_panchang.index,
            name=family,
        )

        # If an input value were absent from the frozen
        # vocabulary, pd.Categorical would convert it to NaN.
        assert not categorical.isna().any(), (
            f"Unrecognized category found in {family}"
        )

        family_dummies = pd.get_dummies(
            categorical,
            prefix=family,
            prefix_sep="__",
            dtype=np.uint8,
        )

        # Force deterministic column order from the frozen
        # vocabulary rather than relying on pandas ordering.
        expected_columns = [
            f"{family}__{category}"
            for category in categories
        ]

        family_dummies = (
            family_dummies
            .reindex(
                columns=expected_columns,
                fill_value=0,
            )
        )

        encoded_parts.append(
            family_dummies
        )

    encoded = pd.concat(
        encoded_parts,
        axis=1,
    )

    return encoded


# ------------------------------------------------------------
# 3. Encode F4 separately for each forecast horizon
# ------------------------------------------------------------

panchang_encoded_valid_features = {}


for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    raw_features = (
        panchang_raw_valid_features[
            horizon_minutes
        ]
    )

    encoded_features = (
        encode_frozen_panchang(
            raw_features,
            panchang_vocabulary,
        )
    )


    # --------------------------------------------------------
    # 4. Structural checks
    # --------------------------------------------------------

    assert encoded_features.index.equals(
        raw_features.index
    )

    assert encoded_features.shape == (
        len(raw_features),
        97,
    )

    assert encoded_features.columns.is_unique

    assert (
        encoded_features
        .isna()
        .sum()
        .sum()
        == 0
    )


    # --------------------------------------------------------
    # 5. Every row must activate exactly one category from
    #    each of the five Panchang families.
    # --------------------------------------------------------

    active_indicator_count = (
        encoded_features.sum(axis=1)
    )

    assert (
        active_indicator_count == 5
    ).all()


    # --------------------------------------------------------
    # 6. Verify final F4 width
    # --------------------------------------------------------

    resulting_f4_count = (
        primary_protocol[
            "feature_counts"
        ]["F3"]
        + encoded_features.shape[1]
    )

    assert (
        resulting_f4_count
        == primary_protocol[
            "feature_counts"
        ]["F4"]
    )


    panchang_encoded_valid_features[
        horizon_minutes
    ] = encoded_features


    print(
        f"{horizon_minutes:2d}-minute horizon"
    )

    print(
        "  rows:",
        len(encoded_features),
    )

    print(
        "  Panchang indicators:",
        encoded_features.shape[1],
    )

    print(
        "  active indicators per row:",
        int(
            active_indicator_count.iloc[0]
        ),
    )

    print(
        "  resulting F4 count:",
        resulting_f4_count,
    )

    print()


# ------------------------------------------------------------
# 7. Verify family widths explicitly
# ------------------------------------------------------------

reference_encoded = (
    panchang_encoded_valid_features[60]
)

print(
    "Frozen family widths:"
)

for family in PANCHANG_FAMILIES:

    family_columns = [
        column
        for column in reference_encoded.columns
        if column.startswith(
            f"{family}__"
        )
    ]

    print(
        f"  {family:12s}: "
        f"{len(family_columns):2d}"
    )


# ------------------------------------------------------------
# 8. Inspect active indicators for the first 60-minute row
# ------------------------------------------------------------

example_issue_time = (
    reference_encoded.index[0]
)

example_row = (
    reference_encoded
    .loc[
        example_issue_time
    ]
)

active_columns = (
    example_row[
        example_row == 1
    ]
    .index
    .tolist()
)


print(
    "\nFirst 60-minute forecast active F4 indicators:"
)

for column in active_columns:
    print(
        " -",
        column,
    )


assert len(active_columns) == 5


print(
    "\nFrozen 97-column F4 encoding: PASSED"
)

15-minute horizon
  rows: 131325
  Panchang indicators: 97
  active indicators per row: 5
  resulting F4 count: 244

30-minute horizon
  rows: 131322
  Panchang indicators: 97
  active indicators per row: 5
  resulting F4 count: 244

60-minute horizon
  rows: 131316
  Panchang indicators: 97
  active indicators per row: 5
  resulting F4 count: 244

Frozen family widths:
  tithi       : 30
  paksha      :  2
  karana      : 11
  nakshatra   : 27
  yoga        : 27

First 60-minute forecast active F4 indicators:
 - tithi__19
 - paksha__Krishna
 - karana__Bava
 - nakshatra__Purva Ashadha
 - yoga__Shukla

Frozen 97-column F4 encoding: PASSED


In [17]:
# ============================================================
# Step 17: Assemble the canonical nested F0-F4 model matrices
# ============================================================
#
# For every forecast horizon we now combine:
#
#   F0 = historical observations
#
#   F1 = F0
#        + Gregorian valid-time features
#
#   F2 = F1
#        + solar astronomy
#
#   F3 = F2
#        + continuous lunar / Sun-Moon astronomy
#
#   F4 = F3
#        + frozen 97-column Panchang one-hot representation
#
# All matrices are indexed by FORECAST ISSUE TIME.
#
# The target and chronological split metadata are taken from
# base_modelling_tables.
# ============================================================


# ------------------------------------------------------------
# 1. Expected feature counts
# ------------------------------------------------------------

EXPECTED_FEATURE_COUNTS = {
    "F0": 118,
    "F1": 133,
    "F2": 138,
    "F3": 147,
    "F4": 244,
}


# ------------------------------------------------------------
# 2. Build nested feature matrices for every horizon
# ------------------------------------------------------------

feature_matrices = {}


for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    table = base_modelling_tables[
        horizon_minutes
    ]

    issue_index = table.index


    # --------------------------------------------------------
    # F0
    # --------------------------------------------------------

    X_f0 = (
        df_f0_history
        .loc[
            issue_index
        ]
        .copy()
    )


    # --------------------------------------------------------
    # F1 = F0 + Gregorian
    # --------------------------------------------------------

    X_f1 = pd.concat(
        [
            X_f0,
            gregorian_valid_features[
                horizon_minutes
            ],
        ],
        axis=1,
    )


    # --------------------------------------------------------
    # F2 = F1 + solar astronomy
    # --------------------------------------------------------

    X_f2 = pd.concat(
        [
            X_f1,
            solar_valid_features[
                horizon_minutes
            ],
        ],
        axis=1,
    )


    # --------------------------------------------------------
    # F3 = F2 + lunar / Sun-Moon astronomy
    # --------------------------------------------------------

    X_f3 = pd.concat(
        [
            X_f2,
            lunar_valid_features[
                horizon_minutes
            ],
        ],
        axis=1,
    )


    # --------------------------------------------------------
    # F4 = F3 + Panchang indicators
    # --------------------------------------------------------

    X_f4 = pd.concat(
        [
            X_f3,
            panchang_encoded_valid_features[
                horizon_minutes
            ],
        ],
        axis=1,
    )


    horizon_matrices = {
        "F0": X_f0,
        "F1": X_f1,
        "F2": X_f2,
        "F3": X_f3,
        "F4": X_f4,
    }


    # --------------------------------------------------------
    # 3. Verify dimensions, alignment, missing values,
    #    uniqueness and numeric compatibility
    # --------------------------------------------------------

    for family_name, X in horizon_matrices.items():

        assert X.index.equals(
            issue_index
        )

        assert X.shape == (
            len(table),
            EXPECTED_FEATURE_COUNTS[
                family_name
            ],
        )

        assert (
            X.shape[1]
            == primary_protocol[
                "feature_counts"
            ][family_name]
        )

        assert X.columns.is_unique

        assert (
            X.isna()
            .sum()
            .sum()
            == 0
        )

        # All final model inputs must be numeric.
        non_numeric_columns = (
            X.select_dtypes(
                exclude=np.number
            )
            .columns
            .tolist()
        )

        assert not non_numeric_columns, (
            f"{family_name} contains non-numeric "
            f"columns: {non_numeric_columns}"
        )


    # --------------------------------------------------------
    # 4. Verify strict nesting
    # --------------------------------------------------------
    #
    # Each previous feature family must appear as the exact
    # leading block of the next one.
    # --------------------------------------------------------

    assert list(
        X_f1.columns[
            :X_f0.shape[1]
        ]
    ) == list(
        X_f0.columns
    )

    assert list(
        X_f2.columns[
            :X_f1.shape[1]
        ]
    ) == list(
        X_f1.columns
    )

    assert list(
        X_f3.columns[
            :X_f2.shape[1]
        ]
    ) == list(
        X_f2.columns
    )

    assert list(
        X_f4.columns[
            :X_f3.shape[1]
        ]
    ) == list(
        X_f3.columns
    )


    feature_matrices[
        horizon_minutes
    ] = horizon_matrices


    # --------------------------------------------------------
    # 5. Display horizon summary
    # --------------------------------------------------------

    print(
        f"{horizon_minutes:2d}-minute horizon"
    )

    for family_name in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        X = horizon_matrices[
            family_name
        ]

        print(
            f"  {family_name}: "
            f"{X.shape[0]:6d} rows × "
            f"{X.shape[1]:3d} features"
        )

    print()


# ------------------------------------------------------------
# 6. Build target and split containers
# ------------------------------------------------------------

targets = {}
split_labels = {}


for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    table = base_modelling_tables[
        horizon_minutes
    ]

    targets[
        horizon_minutes
    ] = (
        table["target"]
        .copy()
    )

    split_labels[
        horizon_minutes
    ] = (
        table["split"]
        .copy()
    )


# ------------------------------------------------------------
# 7. Verify final alignment between X, y and split metadata
# ------------------------------------------------------------

for horizon_minutes in FORECAST_HORIZONS_MINUTES:

    reference_index = (
        targets[
            horizon_minutes
        ].index
    )

    assert split_labels[
        horizon_minutes
    ].index.equals(
        reference_index
    )

    for family_name in EXPECTED_FEATURE_COUNTS:

        assert feature_matrices[
            horizon_minutes
        ][family_name].index.equals(
            reference_index
        )


print(
    "Canonical F0-F4 modelling matrices: PASSED"
)

15-minute horizon
  F0: 131325 rows × 118 features
  F1: 131325 rows × 133 features
  F2: 131325 rows × 138 features
  F3: 131325 rows × 147 features
  F4: 131325 rows × 244 features

30-minute horizon
  F0: 131322 rows × 118 features
  F1: 131322 rows × 133 features
  F2: 131322 rows × 138 features
  F3: 131322 rows × 147 features
  F4: 131322 rows × 244 features

60-minute horizon
  F0: 131316 rows × 118 features
  F1: 131316 rows × 133 features
  F2: 131316 rows × 138 features
  F3: 131316 rows × 147 features
  F4: 131316 rows × 244 features

Canonical F0-F4 modelling matrices: PASSED


## Extra Trees Regressor — Frozen Secondary-Model Protocol

### Purpose

Extra Trees is the first additional machine-learning model evaluated in this
secondary robustness study.

It provides a useful contrast with the Histogram Gradient Boosting Regressor
used in the primary experiment because Extra Trees is a randomized ensemble
of independently constructed decision trees rather than a sequential
boosting method.

The objective is not merely to obtain a lower forecasting error.

The principal scientific question remains whether the effect of adding the
Panchang representation:

$$
F3 \rightarrow F4
$$

is consistent under a substantially different nonlinear learning algorithm.

---

### Data and Information Protocol

The definitions established in Notebook 09 remain unchanged:

- resolution: 5 minutes;
- target: `P_Solar[kW]`;
- horizons: 15, 30, and 60 minutes;
- primary horizon: 60 minutes;
- F0–F4 feature definitions are unchanged;
- train / validation / test assignment is based on forecast-valid time;
- future observed meteorological or irradiance measurements are not used;
- deterministic Gregorian, solar, lunar, and Panchang quantities are evaluated
  at forecast-valid time;
- the test period is not used for Extra Trees hyperparameter selection.

Extra Trees does not require feature standardization because its decision
rules are invariant to monotonic rescaling of individual numeric predictors.

---

### Hyperparameter Search

A deliberately compact grid is frozen before examining validation results.

The following parameters will be evaluated:

| Hyperparameter | Values |
|---|---|
| `n_estimators` | `300` |
| `max_features` | `0.5`, `1.0` |
| `min_samples_leaf` | `1`, `3` |
| `max_depth` | `None`, `30` |
| `criterion` | `"squared_error"` |
| `bootstrap` | `False` |
| `random_state` | `42` |
| `n_jobs` | `-1` |

This produces:

$$
1 \times 2 \times 2 \times 2 = 8
$$

candidate configurations.

The number of trees is fixed at 300 rather than treated as a major tuning
dimension. The objective is to obtain a sufficiently stable ensemble while
keeping the computational burden reasonable.

---

### Hyperparameter Selection Criterion

For each feature family, the preferred Extra Trees configuration will be the
configuration with the lowest **validation MAE**.

RMSE will also be recorded, but it will not determine hyperparameter
selection.

No test-set metric will be used during selection.

---

### Evaluation Sequence

Extra Trees will be evaluated in stages.

First, a sanity experiment will be performed on:

$$
60\text{-minute horizon} + F0
$$

This verifies runtime, memory usage, prediction behaviour, and the frozen
search implementation before performing the complete ablation.

After that, the same frozen grid will be applied to:

$$
F0,\;F1,\;F2,\;F3,\;F4
$$

for the 60-minute horizon.

The 15- and 30-minute horizons will then be evaluated using the same frozen
procedure.

Only after validation-based model selection has been completed will the
selected configurations be refitted on:

$$
\text{train} + \text{validation}
$$

and evaluated on the secondary-model test period.

---

### Interpretation Rule

Particular attention will again be given to:

$$
\Delta_{\mathrm{F3\rightarrow F4}}
=
\frac{\mathrm{MAE}_{F3}-\mathrm{MAE}_{F4}}
     {\mathrm{MAE}_{F3}}
\times100
$$

where a positive value indicates lower MAE after adding F4.

However, a positive point estimate alone will not be interpreted as evidence
of a robust Panchang advantage.

The result will later be considered together with:

- performance across forecast horizons;
- consistency across model families;
- daylight-only performance;
- and paired uncertainty analysis where appropriate.

The frozen Extra Trees grid will not be expanded merely because a preferred
configuration lies on a grid boundary.

In [18]:
# ============================================================
# Step 18: Extra Trees sanity experiment
#          60-minute horizon + F0 only
# ============================================================
#
# Purpose:
#
#   - verify that Extra Trees runs correctly on our modelling
#     matrices;
#   - measure runtime and memory practicality;
#   - evaluate the 8 frozen hyperparameter configurations;
#   - select the configuration with the lowest validation MAE.
#
# IMPORTANT:
#
# The TEST period is NOT used anywhere in this cell.
# ============================================================


import gc
import time
from itertools import product

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)


# ------------------------------------------------------------
# 1. Select the frozen sanity-experiment dataset
# ------------------------------------------------------------

SANITY_HORIZON = 60
SANITY_FEATURE_FAMILY = "F0"

X_60_f0 = (
    feature_matrices[
        SANITY_HORIZON
    ][
        SANITY_FEATURE_FAMILY
    ]
)

y_60 = (
    targets[
        SANITY_HORIZON
    ]
)

splits_60 = (
    split_labels[
        SANITY_HORIZON
    ]
)


# ------------------------------------------------------------
# 2. Construct train and validation masks
# ------------------------------------------------------------

train_mask_60 = (
    splits_60 == "train"
)

validation_mask_60 = (
    splits_60 == "validation"
)


assert int(
    train_mask_60.sum()
) == 87_252

assert int(
    validation_mask_60.sum()
) == 17_568


# ------------------------------------------------------------
# 3. Extract train / validation data
# ------------------------------------------------------------
#
# Extra Trees internally works efficiently with float32 data.
# Explicit conversion reduces memory usage during repeated tree
# fitting without changing the underlying feature definitions.
# ------------------------------------------------------------

X_train = (
    X_60_f0
    .loc[train_mask_60]
    .to_numpy(
        dtype=np.float32
    )
)

X_validation = (
    X_60_f0
    .loc[validation_mask_60]
    .to_numpy(
        dtype=np.float32
    )
)

y_train = (
    y_60
    .loc[train_mask_60]
    .to_numpy(
        dtype=np.float32
    )
)

y_validation = (
    y_60
    .loc[validation_mask_60]
    .to_numpy(
        dtype=np.float32
    )
)


# ------------------------------------------------------------
# 4. Final leakage / shape checks
# ------------------------------------------------------------

assert X_train.shape == (
    87_252,
    118,
)

assert X_validation.shape == (
    17_568,
    118,
)

assert y_train.shape == (
    87_252,
)

assert y_validation.shape == (
    17_568,
)

assert np.isfinite(
    X_train
).all()

assert np.isfinite(
    X_validation
).all()

assert np.isfinite(
    y_train
).all()

assert np.isfinite(
    y_validation
).all()


print(
    "Extra Trees sanity dataset:"
)

print(
    "  Horizon:",
    SANITY_HORIZON,
    "minutes",
)

print(
    "  Feature family:",
    SANITY_FEATURE_FAMILY,
)

print(
    "  Train:",
    X_train.shape,
)

print(
    "  Validation:",
    X_validation.shape,
)


# ------------------------------------------------------------
# 5. Frozen Extra Trees hyperparameter grid
# ------------------------------------------------------------

EXTRA_TREES_GRID = {
    "n_estimators": [
        300,
    ],
    "max_features": [
        0.5,
        1.0,
    ],
    "min_samples_leaf": [
        1,
        3,
    ],
    "max_depth": [
        None,
        30,
    ],
}


extra_trees_configs = list(
    product(
        EXTRA_TREES_GRID[
            "n_estimators"
        ],
        EXTRA_TREES_GRID[
            "max_features"
        ],
        EXTRA_TREES_GRID[
            "min_samples_leaf"
        ],
        EXTRA_TREES_GRID[
            "max_depth"
        ],
    )
)


assert len(
    extra_trees_configs
) == 8


print(
    "\nFrozen configurations:",
    len(extra_trees_configs),
)


# ------------------------------------------------------------
# 6. Fit and evaluate every frozen configuration
# ------------------------------------------------------------

extra_trees_sanity_results = []

sanity_start_time = (
    time.perf_counter()
)


for config_number, (
    n_estimators,
    max_features,
    min_samples_leaf,
    max_depth,
) in enumerate(
    extra_trees_configs,
    start=1,
):

    print(
        f"\nConfiguration "
        f"{config_number}/"
        f"{len(extra_trees_configs)}"
    )

    print(
        "  n_estimators      =",
        n_estimators,
    )

    print(
        "  max_features      =",
        max_features,
    )

    print(
        "  min_samples_leaf  =",
        min_samples_leaf,
    )

    print(
        "  max_depth         =",
        max_depth,
    )


    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        min_samples_leaf=min_samples_leaf,
        max_depth=max_depth,
        criterion="squared_error",
        bootstrap=False,
        random_state=42,
        n_jobs=-1,
    )


    fit_start_time = (
        time.perf_counter()
    )


    model.fit(
        X_train,
        y_train,
    )


    predictions = model.predict(
        X_validation
    )


    elapsed_seconds = (
        time.perf_counter()
        - fit_start_time
    )


    validation_mae = (
        mean_absolute_error(
            y_validation,
            predictions,
        )
    )

    validation_rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            predictions,
        )
    )


    extra_trees_sanity_results.append(
        {
            "horizon_minutes":
                SANITY_HORIZON,

            "feature_family":
                SANITY_FEATURE_FAMILY,

            "n_estimators":
                n_estimators,

            "max_features":
                max_features,

            "min_samples_leaf":
                min_samples_leaf,

            "max_depth":
                max_depth,

            "validation_mae":
                validation_mae,

            "validation_rmse":
                validation_rmse,

            "fit_predict_seconds":
                elapsed_seconds,
        }
    )


    print(
        f"  validation MAE  = "
        f"{validation_mae:.6f}"
    )

    print(
        f"  validation RMSE = "
        f"{validation_rmse:.6f}"
    )

    print(
        f"  runtime         = "
        f"{elapsed_seconds:.1f} s"
    )


    # We do not retain all eight fitted ensembles in memory.
    del model
    del predictions

    gc.collect()


# ------------------------------------------------------------
# 7. Convert results to a table
# ------------------------------------------------------------

extra_trees_sanity_df = (
    pd.DataFrame(
        extra_trees_sanity_results
    )
    .sort_values(
        [
            "validation_mae",
            "validation_rmse",
        ],
        ascending=True,
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 8. Select the best configuration by frozen criterion:
#
#           minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f0_60 = (
    extra_trees_sanity_df
    .iloc[0]
)


total_sanity_seconds = (
    time.perf_counter()
    - sanity_start_time
)


# ------------------------------------------------------------
# 9. Display complete ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 60-minute F0 validation results"
)

print(
    "============================================\n"
)


display(
    extra_trees_sanity_df
)


print(
    "\nBest frozen-grid configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f0_60[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f0_60[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f0_60[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    best_extra_trees_f0_60[
        "max_depth"
    ],
)

print(
    "  validation MAE:",
    best_extra_trees_f0_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f0_60[
        "validation_rmse"
    ],
)


print(
    "\nTotal sanity-grid runtime:"
)

print(
    f"  {total_sanity_seconds:.1f} seconds"
)

print(
    f"  {total_sanity_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 60-minute F0 sanity experiment: PASSED"
)

Extra Trees sanity dataset:
  Horizon: 60 minutes
  Feature family: F0
  Train: (87252, 118)
  Validation: (17568, 118)

Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.471633
  validation RMSE = 0.813662
  runtime         = 93.3 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.472207
  validation RMSE = 0.814869
  runtime         = 84.7 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.453436
  validation RMSE = 0.800208
  runtime         = 74.0 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.450864
  validation RMSE = 0.798697
  runtime         = 66.2 s

Configuration 5/8
  n_estimat

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,60,F0,300,1.0,3,NaN,0.446048,0.797113,139.315789
1,60,F0,300,1.0,3,30.0,0.447581,0.798680,128.899573
2,60,F0,300,0.5,3,30.0,0.450864,0.798697,66.221819
3,60,F0,300,0.5,3,NaN,0.453436,0.800208,74.031893
4,60,F0,300,1.0,1,30.0,0.461394,0.808376,166.318110
5,60,F0,300,1.0,1,NaN,0.463547,0.810945,175.536023
6,60,F0,300,0.5,1,NaN,0.471633,0.813662,93.313620
7,60,F0,300,0.5,1,30.0,0.472207,0.814869,84.669087



Best frozen-grid configuration:
  n_estimators: 300
  max_features: 1.0
  min_samples_leaf: 3
  max_depth: nan
  validation MAE: 0.44604823894105927
  validation RMSE: 0.7971126857437592

Total sanity-grid runtime:
  928.8 seconds
  15.48 minutes

Extra Trees 60-minute F0 sanity experiment: PASSED


In [19]:
# ============================================================
# Step 19: Extra Trees validation
#          60-minute horizon + F1
# ============================================================
#
# F1 = F0 historical information
#      + 15 Gregorian forecast-valid-time features
#
# The exact same 8-configuration Extra Trees grid frozen
# before the F0 sanity experiment is used here.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Select the 60-minute F1 dataset
# ------------------------------------------------------------

HORIZON = 60
FEATURE_FAMILY = "F1"

X_60_f1 = (
    feature_matrices[
        HORIZON
    ][
        FEATURE_FAMILY
    ]
)

y_60 = targets[
    HORIZON
]

splits_60 = split_labels[
    HORIZON
]


train_mask_60 = (
    splits_60 == "train"
)

validation_mask_60 = (
    splits_60 == "validation"
)


# ------------------------------------------------------------
# 2. Convert train / validation matrices to float32
# ------------------------------------------------------------

X_train_f1 = (
    X_60_f1
    .loc[train_mask_60]
    .to_numpy(
        dtype=np.float32
    )
)

X_validation_f1 = (
    X_60_f1
    .loc[validation_mask_60]
    .to_numpy(
        dtype=np.float32
    )
)

y_train_f1 = (
    y_60
    .loc[train_mask_60]
    .to_numpy(
        dtype=np.float32
    )
)

y_validation_f1 = (
    y_60
    .loc[validation_mask_60]
    .to_numpy(
        dtype=np.float32
    )
)


assert X_train_f1.shape == (
    87_252,
    133,
)

assert X_validation_f1.shape == (
    17_568,
    133,
)


print(
    "Extra Trees F1 dataset:"
)

print(
    "  Horizon:",
    HORIZON,
    "minutes",
)

print(
    "  Feature family:",
    FEATURE_FAMILY,
)

print(
    "  Train:",
    X_train_f1.shape,
)

print(
    "  Validation:",
    X_validation_f1.shape,
)

print(
    "  Frozen configurations:",
    len(extra_trees_configs),
)


# ------------------------------------------------------------
# 3. Evaluate the frozen 8-configuration grid
# ------------------------------------------------------------

extra_trees_f1_results = []

f1_start_time = (
    time.perf_counter()
)


for config_number, (
    n_estimators,
    max_features,
    min_samples_leaf,
    max_depth,
) in enumerate(
    extra_trees_configs,
    start=1,
):

    print(
        f"\nConfiguration "
        f"{config_number}/"
        f"{len(extra_trees_configs)}"
    )

    print(
        "  n_estimators      =",
        n_estimators,
    )

    print(
        "  max_features      =",
        max_features,
    )

    print(
        "  min_samples_leaf  =",
        min_samples_leaf,
    )

    print(
        "  max_depth         =",
        max_depth,
    )


    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        min_samples_leaf=min_samples_leaf,
        max_depth=max_depth,
        criterion="squared_error",
        bootstrap=False,
        random_state=42,
        n_jobs=-1,
    )


    fit_start_time = (
        time.perf_counter()
    )


    model.fit(
        X_train_f1,
        y_train_f1,
    )


    predictions = model.predict(
        X_validation_f1
    )


    elapsed_seconds = (
        time.perf_counter()
        - fit_start_time
    )


    validation_mae = (
        mean_absolute_error(
            y_validation_f1,
            predictions,
        )
    )

    validation_rmse = np.sqrt(
        mean_squared_error(
            y_validation_f1,
            predictions,
        )
    )


    extra_trees_f1_results.append(
        {
            "horizon_minutes":
                HORIZON,

            "feature_family":
                FEATURE_FAMILY,

            "n_estimators":
                n_estimators,

            "max_features":
                max_features,

            "min_samples_leaf":
                min_samples_leaf,

            "max_depth":
                max_depth,

            "validation_mae":
                validation_mae,

            "validation_rmse":
                validation_rmse,

            "fit_predict_seconds":
                elapsed_seconds,
        }
    )


    print(
        f"  validation MAE  = "
        f"{validation_mae:.6f}"
    )

    print(
        f"  validation RMSE = "
        f"{validation_rmse:.6f}"
    )

    print(
        f"  runtime         = "
        f"{elapsed_seconds:.1f} s"
    )


    del model
    del predictions

    gc.collect()


# ------------------------------------------------------------
# 4. Rank configurations using the frozen selection criterion
# ------------------------------------------------------------

extra_trees_f1_df = (
    pd.DataFrame(
        extra_trees_f1_results
    )
    .sort_values(
        [
            "validation_mae",
            "validation_rmse",
        ]
    )
    .reset_index(
        drop=True
    )
)


best_extra_trees_f1_60 = (
    extra_trees_f1_df.iloc[0]
)


f1_total_seconds = (
    time.perf_counter()
    - f1_start_time
)


# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 60-minute F1 validation results"
)

print(
    "============================================\n"
)


display(
    extra_trees_f1_df
)


print(
    "\nSelected F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f1_60[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f1_60[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f1_60[
            "min_samples_leaf"
        ]
    ),
)

selected_depth = (
    None
    if pd.isna(
        best_extra_trees_f1_60[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f1_60[
            "max_depth"
        ]
    )
)

print(
    "  max_depth:",
    selected_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f1_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f1_60[
        "validation_rmse"
    ],
)


print(
    "\nTotal F1 grid runtime:"
)

print(
    f"  {f1_total_seconds:.1f} seconds"
)

print(
    f"  {f1_total_seconds / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 6. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_mae = float(
    best_extra_trees_f0_60[
        "validation_mae"
    ]
)

f1_mae = float(
    best_extra_trees_f1_60[
        "validation_mae"
    ]
)

f0_to_f1_relative_improvement = (
    (f0_mae - f1_mae)
    / f0_mae
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_mae:.6f}"
)

print(
    f"  F1 MAE: {f1_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_relative_improvement:+.4f}%"
)


print(
    "\nExtra Trees 60-minute F1 experiment: PASSED"
)

Extra Trees F1 dataset:
  Horizon: 60 minutes
  Feature family: F1
  Train: (87252, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.377859
  validation RMSE = 0.721643
  runtime         = 97.0 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.379191
  validation RMSE = 0.723689
  runtime         = 92.6 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.369309
  validation RMSE = 0.719392
  runtime         = 62.2 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.368185
  validation RMSE = 0.717665
  runtime         = 66.9 s

Configuration 5/8
  n_estimators

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,60,F1,300,0.5,3,30.0,0.368185,0.717665,66.921235
1,60,F1,300,1.0,3,30.0,0.369161,0.723057,117.423776
2,60,F1,300,0.5,3,NaN,0.369309,0.719392,62.166271
3,60,F1,300,1.0,3,NaN,0.369703,0.724000,120.915601
4,60,F1,300,0.5,1,NaN,0.377859,0.721643,97.004885
5,60,F1,300,1.0,1,30.0,0.378879,0.726319,152.253151
6,60,F1,300,0.5,1,30.0,0.379191,0.723689,92.595978
7,60,F1,300,1.0,1,NaN,0.379976,0.724730,168.119215



Selected F1 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.36818544974657363
  validation RMSE: 0.7176654463682638

Total F1 grid runtime:
  877.9 seconds
  14.63 minutes

F0 -> F1 validation comparison:
  F0 MAE: 0.446048
  F1 MAE: 0.368185
  Relative MAE improvement: +17.4561%

Extra Trees 60-minute F1 experiment: PASSED


In [20]:
# ============================================================
# Step 20: Reusable Extra Trees frozen-grid validation helper
#          + 60-minute F2 experiment
# ============================================================
#
# The F0 and F1 experiments established that the fitting and
# evaluation procedure works correctly.
#
# From this point onward we encapsulate exactly the same
# procedure in a helper function to avoid duplicated code.
#
# The frozen hyperparameter grid is NOT changed.
# The TEST period is NOT accessed.
# ============================================================


def run_extra_trees_validation_grid(
    horizon_minutes: int,
    feature_family: str,
) -> tuple[pd.DataFrame, float]:
    """
    Fit all frozen Extra Trees configurations on the training
    period and evaluate them on the fixed chronological
    validation period.

    Selection is based only on validation MAE.

    Returns
    -------
    results_df
        Configurations sorted by validation MAE and then RMSE.

    total_seconds
        Total elapsed time for the complete frozen grid.
    """

    # --------------------------------------------------------
    # 1. Retrieve the requested modelling matrix
    # --------------------------------------------------------

    X = feature_matrices[
        horizon_minutes
    ][
        feature_family
    ]

    y = targets[
        horizon_minutes
    ]

    split = split_labels[
        horizon_minutes
    ]


    # --------------------------------------------------------
    # 2. Chronological train / validation masks
    # --------------------------------------------------------

    train_mask = (
        split == "train"
    )

    validation_mask = (
        split == "validation"
    )


    # --------------------------------------------------------
    # 3. Convert to compact numeric arrays
    # --------------------------------------------------------

    X_train_local = (
        X.loc[train_mask]
        .to_numpy(
            dtype=np.float32
        )
    )

    X_validation_local = (
        X.loc[validation_mask]
        .to_numpy(
            dtype=np.float32
        )
    )

    y_train_local = (
        y.loc[train_mask]
        .to_numpy(
            dtype=np.float32
        )
    )

    y_validation_local = (
        y.loc[validation_mask]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # 4. Integrity checks
    # --------------------------------------------------------

    expected_feature_count = (
        primary_protocol[
            "feature_counts"
        ][feature_family]
    )

    assert (
        X_train_local.shape[1]
        == expected_feature_count
    )

    assert (
        X_validation_local.shape[1]
        == expected_feature_count
    )

    assert np.isfinite(
        X_train_local
    ).all()

    assert np.isfinite(
        X_validation_local
    ).all()

    assert np.isfinite(
        y_train_local
    ).all()

    assert np.isfinite(
        y_validation_local
    ).all()


    print(
        "Extra Trees validation dataset:"
    )

    print(
        "  Horizon:",
        horizon_minutes,
        "minutes",
    )

    print(
        "  Feature family:",
        feature_family,
    )

    print(
        "  Train:",
        X_train_local.shape,
    )

    print(
        "  Validation:",
        X_validation_local.shape,
    )

    print(
        "  Frozen configurations:",
        len(extra_trees_configs),
    )


    # --------------------------------------------------------
    # 5. Run the complete frozen grid
    # --------------------------------------------------------

    results = []

    grid_start_time = (
        time.perf_counter()
    )


    for config_number, (
        n_estimators,
        max_features,
        min_samples_leaf,
        max_depth,
    ) in enumerate(
        extra_trees_configs,
        start=1,
    ):

        print(
            f"\nConfiguration "
            f"{config_number}/"
            f"{len(extra_trees_configs)}"
        )

        print(
            "  n_estimators      =",
            n_estimators,
        )

        print(
            "  max_features      =",
            max_features,
        )

        print(
            "  min_samples_leaf  =",
            min_samples_leaf,
        )

        print(
            "  max_depth         =",
            max_depth,
        )


        model = ExtraTreesRegressor(
            n_estimators=n_estimators,
            max_features=max_features,
            min_samples_leaf=min_samples_leaf,
            max_depth=max_depth,
            criterion="squared_error",
            bootstrap=False,
            random_state=42,
            n_jobs=-1,
        )


        fit_start_time = (
            time.perf_counter()
        )


        model.fit(
            X_train_local,
            y_train_local,
        )


        predictions = (
            model.predict(
                X_validation_local
            )
        )


        elapsed_seconds = (
            time.perf_counter()
            - fit_start_time
        )


        validation_mae = (
            mean_absolute_error(
                y_validation_local,
                predictions,
            )
        )

        validation_rmse = np.sqrt(
            mean_squared_error(
                y_validation_local,
                predictions,
            )
        )


        results.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_estimators":
                    n_estimators,

                "max_features":
                    max_features,

                "min_samples_leaf":
                    min_samples_leaf,

                "max_depth":
                    max_depth,

                "validation_mae":
                    validation_mae,

                "validation_rmse":
                    validation_rmse,

                "fit_predict_seconds":
                    elapsed_seconds,
            }
        )


        print(
            f"  validation MAE  = "
            f"{validation_mae:.6f}"
        )

        print(
            f"  validation RMSE = "
            f"{validation_rmse:.6f}"
        )

        print(
            f"  runtime         = "
            f"{elapsed_seconds:.1f} s"
        )


        del model
        del predictions

        gc.collect()


    # --------------------------------------------------------
    # 6. Rank using the frozen selection criterion
    # --------------------------------------------------------

    results_df = (
        pd.DataFrame(
            results
        )
        .sort_values(
            [
                "validation_mae",
                "validation_rmse",
            ],
            ascending=True,
        )
        .reset_index(
            drop=True
        )
    )


    total_seconds = (
        time.perf_counter()
        - grid_start_time
    )


    return (
        results_df,
        total_seconds,
    )


# ============================================================
# 7. Run the helper for:
#
#       60-minute horizon + F2
#
# F2 = F1 + five deterministic solar-geometry features
# ============================================================

extra_trees_f2_df, f2_total_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=60,
        feature_family="F2",
    )
)


best_extra_trees_f2_60 = (
    extra_trees_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 8. Display ordered F2 results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 60-minute F2 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_f2_df
)


# ------------------------------------------------------------
# 9. Decode selected configuration cleanly
# ------------------------------------------------------------

selected_f2_depth = (
    None
    if pd.isna(
        best_extra_trees_f2_60[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f2_60[
            "max_depth"
        ]
    )
)


print(
    "\nSelected F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f2_60[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f2_60[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f2_60[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f2_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f2_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f2_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 10. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_mae = float(
    best_extra_trees_f1_60[
        "validation_mae"
    ]
)

f2_mae = float(
    best_extra_trees_f2_60[
        "validation_mae"
    ]
)

f1_to_f2_relative_improvement = (
    (f1_mae - f2_mae)
    / f1_mae
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_mae:.6f}"
)

print(
    f"  F2 MAE: {f2_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_relative_improvement:+.4f}%"
)


print(
    "\nTotal F2 grid runtime:"
)

print(
    f"  {f2_total_seconds:.1f} seconds"
)

print(
    f"  {f2_total_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 60-minute F2 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 60 minutes
  Feature family: F2
  Train: (87252, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.380697
  validation RMSE = 0.724916
  runtime         = 91.0 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.377222
  validation RMSE = 0.721272
  runtime         = 85.9 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.368060
  validation RMSE = 0.717971
  runtime         = 71.5 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.367791
  validation RMSE = 0.718632
  runtime         = 78.8 s

Configuration 5/8
  n_es

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,60,F2,300,1.0,3,NaN,0.367777,0.721399,141.477078
1,60,F2,300,0.5,3,30.0,0.367791,0.718632,78.833025
2,60,F2,300,0.5,3,NaN,0.368060,0.717971,71.498840
3,60,F2,300,1.0,3,30.0,0.368310,0.722283,137.058627
4,60,F2,300,1.0,1,30.0,0.376057,0.723498,194.998753
5,60,F2,300,0.5,1,30.0,0.377222,0.721272,85.867239
6,60,F2,300,1.0,1,NaN,0.377644,0.726368,206.247507
7,60,F2,300,0.5,1,NaN,0.380697,0.724916,90.960393



Selected F2 configuration:
  n_estimators: 300
  max_features: 1.0
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.3677767365738547
  validation RMSE: 0.7213993233217392

F1 -> F2 validation comparison:
  F1 MAE: 0.368185
  F2 MAE: 0.367777
  Relative MAE improvement: +0.1110%

Total F2 grid runtime:
  1007.6 seconds
  16.79 minutes

Extra Trees 60-minute F2 experiment: PASSED


In [21]:
# ============================================================
# Step 21: Extra Trees validation
#          60-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + nine continuous lunar / Sun-Moon astronomical features
#
# The same frozen 8-configuration Extra Trees grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_f3_df, f3_total_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=60,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select the configuration with minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f3_60 = (
    extra_trees_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 60-minute F3 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_f3_df
)


# ------------------------------------------------------------
# 4. Decode the selected max_depth value
# ------------------------------------------------------------

selected_f3_depth = (
    None
    if pd.isna(
        best_extra_trees_f3_60[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f3_60[
            "max_depth"
        ]
    )
)


print(
    "\nSelected F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f3_60[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f3_60[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f3_60[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f3_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f3_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f3_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_mae = float(
    best_extra_trees_f2_60[
        "validation_mae"
    ]
)

f3_mae = float(
    best_extra_trees_f3_60[
        "validation_mae"
    ]
)

f2_to_f3_relative_improvement = (
    (f2_mae - f3_mae)
    / f2_mae
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_mae:.6f}"
)

print(
    f"  F3 MAE: {f3_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_relative_improvement:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal F3 grid runtime:"
)

print(
    f"  {f3_total_seconds:.1f} seconds"
)

print(
    f"  {f3_total_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 60-minute F3 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 60 minutes
  Feature family: F3
  Train: (87252, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.381306
  validation RMSE = 0.723964
  runtime         = 101.4 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.382169
  validation RMSE = 0.722411
  runtime         = 100.7 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.370669
  validation RMSE = 0.719561
  runtime         = 76.2 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.369305
  validation RMSE = 0.716883
  runtime         = 73.4 s

Configuration 5/8
  n_

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,60,F3,300,0.5,3,30.0,0.369305,0.716883,73.410743
1,60,F3,300,1.0,3,NaN,0.370442,0.720098,146.337160
2,60,F3,300,1.0,3,30.0,0.370491,0.721726,145.774653
3,60,F3,300,0.5,3,NaN,0.370669,0.719561,76.169404
4,60,F3,300,1.0,1,30.0,0.379824,0.724081,186.963434
5,60,F3,300,0.5,1,NaN,0.381306,0.723964,101.390946
6,60,F3,300,1.0,1,NaN,0.381386,0.724766,177.859166
7,60,F3,300,0.5,1,30.0,0.382169,0.722411,100.713152



Selected F3 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.36930474760996146
  validation RMSE: 0.7168828682363658

F2 -> F3 validation comparison:
  F2 MAE: 0.367777
  F3 MAE: 0.369305
  Relative MAE improvement: -0.4155%

Total F3 grid runtime:
  1009.2 seconds
  16.82 minutes

Extra Trees 60-minute F3 experiment: PASSED


In [23]:
# ============================================================
# Step 22: Extra Trees validation
#          60-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + 97 frozen Panchang one-hot indicators
#
# This is the central Extra Trees comparison:
#
#          F3  ->  F4
#
# The same frozen 8-configuration Extra Trees grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_f4_df, f4_total_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=60,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f4_60 = (
    extra_trees_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 60-minute F4 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_f4_df
)


# ------------------------------------------------------------
# 4. Decode selected max_depth cleanly
# ------------------------------------------------------------

selected_f4_depth = (
    None
    if pd.isna(
        best_extra_trees_f4_60[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f4_60[
            "max_depth"
        ]
    )
)


print(
    "\nSelected F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f4_60[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f4_60[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f4_60[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f4_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f4_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f4_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Central F3 -> F4 comparison
# ------------------------------------------------------------

f3_mae = float(
    best_extra_trees_f3_60[
        "validation_mae"
    ]
)

f4_mae = float(
    best_extra_trees_f4_60[
        "validation_mae"
    ]
)

f3_to_f4_absolute_change = (
    f3_mae - f4_mae
)

f3_to_f4_relative_improvement = (
    f3_to_f4_absolute_change
    / f3_mae
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_mae:.6f}"
)

print(
    f"  F4 MAE: {f4_mae:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_absolute_change:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_relative_improvement:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal F4 grid runtime:"
)

print(
    f"  {f4_total_seconds:.1f} seconds"
)

print(
    f"  {f4_total_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 60-minute F4 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 60 minutes
  Feature family: F4
  Train: (87252, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.387418
  validation RMSE = 0.733711
  runtime         = 105.6 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.384982
  validation RMSE = 0.732285
  runtime         = 68.9 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.375706
  validation RMSE = 0.727527
  runtime         = 59.2 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.374657
  validation RMSE = 0.726774
  runtime         = 58.4 s

Configuration 5/8
  n_e

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,60,F4,300,0.5,3,30.0,0.374657,0.726774,58.427562
1,60,F4,300,1.0,3,30.0,0.375594,0.731343,162.441002
2,60,F4,300,0.5,3,NaN,0.375706,0.727527,59.208173
3,60,F4,300,1.0,3,NaN,0.376026,0.732620,169.764419
4,60,F4,300,0.5,1,30.0,0.384982,0.732285,68.943189
5,60,F4,300,1.0,1,30.0,0.387154,0.738273,189.971047
6,60,F4,300,0.5,1,NaN,0.387418,0.733711,105.615374
7,60,F4,300,1.0,1,NaN,0.388219,0.739576,193.580414



Selected F4 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.3746573999457742
  validation RMSE: 0.7267741762580785

F3 -> F4 validation comparison:
  F3 MAE: 0.369305
  F4 MAE: 0.374657
  Absolute MAE reduction: -0.005353
  Relative MAE improvement: -1.4494%

Total F4 grid runtime:
  1008.4 seconds
  16.81 minutes

Extra Trees 60-minute F4 experiment: PASSED


In [24]:
# ============================================================
# Step 23: Summarize the complete 60-minute Extra Trees
#          F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect the selected configuration for each feature family
# ------------------------------------------------------------

best_rows_60 = {
    "F0": best_extra_trees_f0_60,
    "F1": best_extra_trees_f1_60,
    "F2": best_extra_trees_f2_60,
    "F3": best_extra_trees_f3_60,
    "F4": best_extra_trees_f4_60,
}


summary_rows = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_60[
        feature_family
    ]

    max_depth = (
        None
        if pd.isna(
            row["max_depth"]
        )
        else int(
            row["max_depth"]
        )
    )

    summary_rows.append(
        {
            "horizon_minutes": 60,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][feature_family],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "max_features":
                float(
                    row[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    row[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                max_depth,

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


extra_trees_60_summary = (
    pd.DataFrame(
        summary_rows
    )
)


# ------------------------------------------------------------
# 2. Add incremental MAE change
# ------------------------------------------------------------
#
# Positive percentage:
#     new feature family reduced MAE.
#
# Negative percentage:
#     new feature family increased MAE.
# ------------------------------------------------------------

extra_trees_60_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(extra_trees_60_summary),
):

    previous_mae = (
        extra_trees_60_summary
        .loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = (
        extra_trees_60_summary
        .loc[
            i,
            "validation_mae",
        ]
    )

    extra_trees_60_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (previous_mae - current_mae)
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Verify the central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae = (
    extra_trees_60_summary
    .loc[
        extra_trees_60_summary[
            "feature_family"
        ] == "F3",
        "validation_mae",
    ]
    .iloc[0]
)

f4_summary_mae = (
    extra_trees_60_summary
    .loc[
        extra_trees_60_summary[
            "feature_family"
        ] == "F4",
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_60 = (
    (f3_summary_mae - f4_summary_mae)
    / f3_summary_mae
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_60,
    -1.4494,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display the complete 60-minute summary
# ------------------------------------------------------------

print(
    "Extra Trees — 60-minute validation summary\n"
)

display(
    extra_trees_60_summary
)


# ------------------------------------------------------------
# 5. Key comparisons
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)

for i in range(
    1,
    len(extra_trees_60_summary),
):

    previous_family = (
        extra_trees_60_summary
        .loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        extra_trees_60_summary
        .loc[
            i,
            "feature_family",
        ]
    )

    change = (
        extra_trees_60_summary
        .loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )

    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_60 = (
    extra_trees_60_summary
    .sort_values(
        "validation_mae"
    )
    .iloc[0]
)


print(
    "\nLowest 60-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_60[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_60[
        "validation_mae"
    ],
)


print(
    "\nCentral Extra Trees comparison:"
)

print(
    f"  F3 MAE: {f3_summary_mae:.6f}"
)

print(
    f"  F4 MAE: {f4_summary_mae:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_60:+.4f}%"
)


print(
    "\nExtra Trees 60-minute summary: PASSED"
)

Extra Trees — 60-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,60,F0,118,300,1.0,3,NaN,0.446048,0.797113,NaN
1,60,F1,133,300,0.5,3,30.0,0.368185,0.717665,17.456136
2,60,F2,138,300,1.0,3,NaN,0.367777,0.721399,0.111007
3,60,F3,147,300,0.5,3,30.0,0.369305,0.716883,-0.415472
4,60,F4,244,300,0.5,3,30.0,0.374657,0.726774,-1.449386


Incremental validation MAE changes:
  F0 -> F1: +17.4561%
  F1 -> F2: +0.1110%
  F2 -> F3: -0.4155%
  F3 -> F4: -1.4494%

Lowest 60-minute validation MAE:
  Feature family: F2
  MAE: 0.3677767365738547

Central Extra Trees comparison:
  F3 MAE: 0.369305
  F4 MAE: 0.374657
  F3 -> F4 change: -1.4494%

Extra Trees 60-minute summary: PASSED


In [25]:
# ============================================================
# Step 24: Extra Trees validation
#          30-minute horizon + F0
# ============================================================
#
# We now repeat the frozen Extra Trees protocol at the
# 30-minute forecast horizon.
#
# F0 contains only historical information available at
# forecast issue time.
#
# The exact same 8-configuration grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_30_f0_df, extra_trees_30_f0_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=30,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f0_30 = (
    extra_trees_30_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 30-minute F0 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_30_f0_df
)


# ------------------------------------------------------------
# 4. Decode max_depth
# ------------------------------------------------------------

selected_f0_30_depth = (
    None
    if pd.isna(
        best_extra_trees_f0_30[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f0_30[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 30-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f0_30[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f0_30[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f0_30[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f0_30_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f0_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f0_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F0 grid runtime:"
)

print(
    f"  {extra_trees_30_f0_seconds:.1f} seconds"
)

print(
    f"  {extra_trees_30_f0_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 30-minute F0 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 30 minutes
  Feature family: F0
  Train: (87258, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.355075
  validation RMSE = 0.683853
  runtime         = 42.8 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.356221
  validation RMSE = 0.684052
  runtime         = 38.8 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.341728
  validation RMSE = 0.673713
  runtime         = 31.9 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.342635
  validation RMSE = 0.674013
  runtime         = 30.4 s

Configuration 5/8
  n_es

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,30,F0,300,1.0,3,NaN,0.338981,0.673867,74.123886
1,30,F0,300,1.0,3,30.0,0.339559,0.674028,69.770112
2,30,F0,300,0.5,3,NaN,0.341728,0.673713,31.893029
3,30,F0,300,0.5,3,30.0,0.342635,0.674013,30.357197
4,30,F0,300,1.0,1,NaN,0.352466,0.682446,85.764290
5,30,F0,300,1.0,1,30.0,0.353335,0.683511,87.242034
6,30,F0,300,0.5,1,NaN,0.355075,0.683853,42.810170
7,30,F0,300,0.5,1,30.0,0.356221,0.684052,38.835007



Selected 30-minute F0 configuration:
  n_estimators: 300
  max_features: 1.0
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.3389812167088785
  validation RMSE: 0.6738670263982838

Total 30-minute F0 grid runtime:
  461.2 seconds
  7.69 minutes

Extra Trees 30-minute F0 experiment: PASSED


In [26]:
# ============================================================
# Step 25: Extra Trees validation
#          30-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + 15 Gregorian forecast-valid-time features
#
# The same frozen 8-configuration Extra Trees grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_30_f1_df, extra_trees_30_f1_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=30,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f1_30 = (
    extra_trees_30_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 30-minute F1 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_30_f1_df
)


# ------------------------------------------------------------
# 4. Decode selected max_depth
# ------------------------------------------------------------

selected_f1_30_depth = (
    None
    if pd.isna(
        best_extra_trees_f1_30[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f1_30[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 30-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f1_30[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f1_30[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f1_30[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f1_30_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f1_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f1_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_30_mae = float(
    best_extra_trees_f0_30[
        "validation_mae"
    ]
)

f1_30_mae = float(
    best_extra_trees_f1_30[
        "validation_mae"
    ]
)

f0_to_f1_30_pct = (
    (f0_30_mae - f1_30_mae)
    / f0_30_mae
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_30_mae:.6f}"
)

print(
    f"  F1 MAE: {f1_30_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_30_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F1 grid runtime:"
)

print(
    f"  {extra_trees_30_f1_seconds:.1f} seconds"
)

print(
    f"  {extra_trees_30_f1_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 30-minute F1 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 30 minutes
  Feature family: F1
  Train: (87258, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.321927
  validation RMSE = 0.645695
  runtime         = 48.2 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.321433
  validation RMSE = 0.644023
  runtime         = 42.5 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.307264
  validation RMSE = 0.638857
  runtime         = 34.3 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.308131
  validation RMSE = 0.639053
  runtime         = 35.8 s

Configuration 5/8
  n_es

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,30,F1,300,0.5,3,NaN,0.307264,0.638857,34.324569
1,30,F1,300,0.5,3,30.0,0.308131,0.639053,35.762253
2,30,F1,300,1.0,3,30.0,0.308857,0.640917,86.770496
3,30,F1,300,1.0,3,NaN,0.310194,0.641811,86.311399
4,30,F1,300,1.0,1,30.0,0.320343,0.646284,115.959242
5,30,F1,300,1.0,1,NaN,0.320401,0.646018,129.834977
6,30,F1,300,0.5,1,30.0,0.321433,0.644023,42.482798
7,30,F1,300,0.5,1,NaN,0.321927,0.645695,48.156739



Selected 30-minute F1 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.3072636810232968
  validation RMSE: 0.6388574102812851

F0 -> F1 validation comparison:
  F0 MAE: 0.338981
  F1 MAE: 0.307264
  Relative MAE improvement: +9.3567%

Total 30-minute F1 grid runtime:
  580.0 seconds
  9.67 minutes

Extra Trees 30-minute F1 experiment: PASSED


In [27]:
# ============================================================
# Step 26: Extra Trees validation
#          30-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + five deterministic solar-geometry features
#
# The same frozen 8-configuration Extra Trees grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_30_f2_df, extra_trees_30_f2_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=30,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f2_30 = (
    extra_trees_30_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 30-minute F2 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_30_f2_df
)


# ------------------------------------------------------------
# 4. Decode selected max_depth
# ------------------------------------------------------------

selected_f2_30_depth = (
    None
    if pd.isna(
        best_extra_trees_f2_30[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f2_30[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 30-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f2_30[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f2_30[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f2_30[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f2_30_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f2_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f2_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_30_mae = float(
    best_extra_trees_f1_30[
        "validation_mae"
    ]
)

f2_30_mae = float(
    best_extra_trees_f2_30[
        "validation_mae"
    ]
)

f1_to_f2_30_pct = (
    (f1_30_mae - f2_30_mae)
    / f1_30_mae
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_30_mae:.6f}"
)

print(
    f"  F2 MAE: {f2_30_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_30_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F2 grid runtime:"
)

print(
    f"  {extra_trees_30_f2_seconds:.1f} seconds"
)

print(
    f"  {extra_trees_30_f2_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 30-minute F2 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 30 minutes
  Feature family: F2
  Train: (87258, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.320102
  validation RMSE = 0.642728
  runtime         = 44.3 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.318738
  validation RMSE = 0.643667
  runtime         = 43.7 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.307399
  validation RMSE = 0.638451
  runtime         = 34.8 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.307064
  validation RMSE = 0.637557
  runtime         = 36.0 s

Configuration 5/8
  n_es

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,30,F2,300,1.0,3,NaN,0.306829,0.638223,89.005705
1,30,F2,300,0.5,3,30.0,0.307064,0.637557,36.043459
2,30,F2,300,1.0,3,30.0,0.307360,0.639819,86.273007
3,30,F2,300,0.5,3,NaN,0.307399,0.638451,34.791026
4,30,F2,300,1.0,1,30.0,0.317836,0.643464,115.857498
5,30,F2,300,0.5,1,30.0,0.318738,0.643667,43.744507
6,30,F2,300,1.0,1,NaN,0.319934,0.643928,117.797140
7,30,F2,300,0.5,1,NaN,0.320102,0.642728,44.259117



Selected 30-minute F2 configuration:
  n_estimators: 300
  max_features: 1.0
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.3068289613739121
  validation RMSE: 0.6382226027215651

F1 -> F2 validation comparison:
  F1 MAE: 0.307264
  F2 MAE: 0.306829
  Relative MAE improvement: +0.1415%

Total 30-minute F2 grid runtime:
  568.2 seconds
  9.47 minutes

Extra Trees 30-minute F2 experiment: PASSED


In [28]:
# ============================================================
# Step 27: Extra Trees validation
#          30-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + nine continuous lunar / Sun-Moon astronomical features
#
# The same frozen 8-configuration Extra Trees grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_30_f3_df, extra_trees_30_f3_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=30,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f3_30 = (
    extra_trees_30_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 30-minute F3 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_30_f3_df
)


# ------------------------------------------------------------
# 4. Decode selected max_depth
# ------------------------------------------------------------

selected_f3_30_depth = (
    None
    if pd.isna(
        best_extra_trees_f3_30[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f3_30[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 30-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f3_30[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f3_30[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f3_30[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f3_30_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f3_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f3_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_30_mae = float(
    best_extra_trees_f2_30[
        "validation_mae"
    ]
)

f3_30_mae = float(
    best_extra_trees_f3_30[
        "validation_mae"
    ]
)

f2_to_f3_30_pct = (
    (f2_30_mae - f3_30_mae)
    / f2_30_mae
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_30_mae:.6f}"
)

print(
    f"  F3 MAE: {f3_30_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_30_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F3 grid runtime:"
)

print(
    f"  {extra_trees_30_f3_seconds:.1f} seconds"
)

print(
    f"  {extra_trees_30_f3_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 30-minute F3 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 30 minutes
  Feature family: F3
  Train: (87258, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.320484
  validation RMSE = 0.644464
  runtime         = 48.1 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.323051
  validation RMSE = 0.645077
  runtime         = 47.7 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.308672
  validation RMSE = 0.637670
  runtime         = 37.5 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.308265
  validation RMSE = 0.637826
  runtime         = 41.0 s

Configuration 5/8
  n_es

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,30,F3,300,0.5,3,30.0,0.308265,0.637826,40.976627
1,30,F3,300,0.5,3,NaN,0.308672,0.637670,37.537736
2,30,F3,300,1.0,3,30.0,0.309466,0.639784,98.280255
3,30,F3,300,1.0,3,NaN,0.310524,0.639502,97.302167
4,30,F3,300,0.5,1,NaN,0.320484,0.644464,48.077915
5,30,F3,300,1.0,1,NaN,0.320922,0.644644,129.961868
6,30,F3,300,0.5,1,30.0,0.323051,0.645077,47.684593
7,30,F3,300,1.0,1,30.0,0.323153,0.645751,128.632482



Selected 30-minute F3 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.3082650984323502
  validation RMSE: 0.6378258928981099

F2 -> F3 validation comparison:
  F2 MAE: 0.306829
  F3 MAE: 0.308265
  Relative MAE improvement: -0.4681%

Total 30-minute F3 grid runtime:
  628.9 seconds
  10.48 minutes

Extra Trees 30-minute F3 experiment: PASSED


In [29]:
# ============================================================
# Step 28: Extra Trees validation
#          30-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + 97 frozen Panchang one-hot indicators
#
# This gives the central 30-minute comparison:
#
#          F3 -> F4
#
# The same frozen 8-configuration Extra Trees grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_30_f4_df, extra_trees_30_f4_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=30,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f4_30 = (
    extra_trees_30_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 30-minute F4 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_30_f4_df
)


# ------------------------------------------------------------
# 4. Decode selected max_depth
# ------------------------------------------------------------

selected_f4_30_depth = (
    None
    if pd.isna(
        best_extra_trees_f4_30[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f4_30[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 30-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f4_30[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f4_30[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f4_30[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f4_30_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f4_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f4_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Central F3 -> F4 comparison
# ------------------------------------------------------------

f3_30_mae = float(
    best_extra_trees_f3_30[
        "validation_mae"
    ]
)

f4_30_mae = float(
    best_extra_trees_f4_30[
        "validation_mae"
    ]
)

f3_to_f4_30_absolute_change = (
    f3_30_mae - f4_30_mae
)

f3_to_f4_30_pct = (
    f3_to_f4_30_absolute_change
    / f3_30_mae
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_30_mae:.6f}"
)

print(
    f"  F4 MAE: {f4_30_mae:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_30_absolute_change:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_30_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F4 grid runtime:"
)

print(
    f"  {extra_trees_30_f4_seconds:.1f} seconds"
)

print(
    f"  {extra_trees_30_f4_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 30-minute F4 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 30 minutes
  Feature family: F4
  Train: (87258, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.322574
  validation RMSE = 0.650357
  runtime         = 76.6 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.321705
  validation RMSE = 0.650208
  runtime         = 77.5 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.308679
  validation RMSE = 0.641719
  runtime         = 104.5 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.308147
  validation RMSE = 0.641338
  runtime         = 83.2 s

Configuration 5/8
  n_e

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,30,F4,300,0.5,3,30.0,0.308147,0.641338,83.201704
1,30,F4,300,0.5,3,NaN,0.308679,0.641719,104.493933
2,30,F4,300,1.0,3,NaN,0.309871,0.644834,162.430606
3,30,F4,300,1.0,3,30.0,0.310718,0.646228,162.669195
4,30,F4,300,0.5,1,30.0,0.321705,0.650208,77.450865
5,30,F4,300,0.5,1,NaN,0.322574,0.650357,76.619147
6,30,F4,300,1.0,1,30.0,0.323651,0.653828,188.134422
7,30,F4,300,1.0,1,NaN,0.324643,0.654712,204.734995



Selected 30-minute F4 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.30814711159335934
  validation RMSE: 0.6413380754755147

F3 -> F4 validation comparison:
  F3 MAE: 0.308265
  F4 MAE: 0.308147
  Absolute MAE reduction: +0.000118
  Relative MAE improvement: +0.0383%

Total 30-minute F4 grid runtime:
  1060.2 seconds
  17.67 minutes

Extra Trees 30-minute F4 experiment: PASSED


In [30]:
# ============================================================
# Step 29: Summarize the complete 30-minute Extra Trees
#          F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect the selected configuration for each feature family
# ------------------------------------------------------------

best_rows_30 = {
    "F0": best_extra_trees_f0_30,
    "F1": best_extra_trees_f1_30,
    "F2": best_extra_trees_f2_30,
    "F3": best_extra_trees_f3_30,
    "F4": best_extra_trees_f4_30,
}


summary_rows_30 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_30[
        feature_family
    ]

    max_depth = (
        None
        if pd.isna(
            row["max_depth"]
        )
        else int(
            row["max_depth"]
        )
    )

    summary_rows_30.append(
        {
            "horizon_minutes": 30,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][feature_family],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "max_features":
                float(
                    row[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    row[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                max_depth,

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


extra_trees_30_summary = (
    pd.DataFrame(
        summary_rows_30
    )
)


# ------------------------------------------------------------
# 2. Add incremental MAE change
# ------------------------------------------------------------
#
# Positive:
#     new feature family reduced MAE.
#
# Negative:
#     new feature family increased MAE.
# ------------------------------------------------------------

extra_trees_30_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(extra_trees_30_summary),
):

    previous_mae = (
        extra_trees_30_summary
        .loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = (
        extra_trees_30_summary
        .loc[
            i,
            "validation_mae",
        ]
    )

    extra_trees_30_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (previous_mae - current_mae)
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Verify central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_30 = (
    extra_trees_30_summary
    .loc[
        extra_trees_30_summary[
            "feature_family"
        ] == "F3",
        "validation_mae",
    ]
    .iloc[0]
)

f4_summary_mae_30 = (
    extra_trees_30_summary
    .loc[
        extra_trees_30_summary[
            "feature_family"
        ] == "F4",
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_30 = (
    (
        f3_summary_mae_30
        - f4_summary_mae_30
    )
    / f3_summary_mae_30
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_30,
    0.0383,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete 30-minute summary
# ------------------------------------------------------------

print(
    "Extra Trees — 30-minute validation summary\n"
)

display(
    extra_trees_30_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)

for i in range(
    1,
    len(extra_trees_30_summary),
):

    previous_family = (
        extra_trees_30_summary
        .loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        extra_trees_30_summary
        .loc[
            i,
            "feature_family",
        ]
    )

    change = (
        extra_trees_30_summary
        .loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )

    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_30 = (
    extra_trees_30_summary
    .sort_values(
        "validation_mae"
    )
    .iloc[0]
)


print(
    "\nLowest 30-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_30[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_30[
        "validation_mae"
    ],
)


print(
    "\nCentral Extra Trees comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_30:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_30:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_30:+.4f}%"
)


print(
    "\nExtra Trees 30-minute summary: PASSED"
)

Extra Trees — 30-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,30,F0,118,300,1.0,3,NaN,0.338981,0.673867,NaN
1,30,F1,133,300,0.5,3,NaN,0.307264,0.638857,9.356724
2,30,F2,138,300,1.0,3,NaN,0.306829,0.638223,0.141481
3,30,F3,147,300,0.5,3,30.0,0.308265,0.637826,-0.468058
4,30,F4,244,300,0.5,3,30.0,0.308147,0.641338,0.038274


Incremental validation MAE changes:
  F0 -> F1: +9.3567%
  F1 -> F2: +0.1415%
  F2 -> F3: -0.4681%
  F3 -> F4: +0.0383%

Lowest 30-minute validation MAE:
  Feature family: F2
  MAE: 0.3068289613739121

Central Extra Trees comparison:
  F3 MAE: 0.308265
  F4 MAE: 0.308147
  F3 -> F4 change: +0.0383%

Extra Trees 30-minute summary: PASSED


In [31]:
# ============================================================
# Step 30: Extra Trees validation
#          15-minute horizon + F0
# ============================================================
#
# We now repeat the frozen Extra Trees protocol at the
# 15-minute forecast horizon.
#
# F0 contains only historical information available at
# forecast issue time.
#
# The exact same 8-configuration grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_15_f0_df, extra_trees_15_f0_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=15,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f0_15 = (
    extra_trees_15_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 15-minute F0 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_15_f0_df
)


# ------------------------------------------------------------
# 4. Decode selected max_depth
# ------------------------------------------------------------

selected_f0_15_depth = (
    None
    if pd.isna(
        best_extra_trees_f0_15[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f0_15[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 15-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f0_15[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f0_15[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f0_15[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f0_15_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f0_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f0_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F0 grid runtime:"
)

print(
    f"  {extra_trees_15_f0_seconds:.1f} seconds"
)

print(
    f"  {extra_trees_15_f0_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 15-minute F0 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 15 minutes
  Feature family: F0
  Train: (87261, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.292519
  validation RMSE = 0.607578
  runtime         = 96.0 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.291426
  validation RMSE = 0.607346
  runtime         = 87.0 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.275840
  validation RMSE = 0.595753
  runtime         = 70.0 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.276605
  validation RMSE = 0.596118
  runtime         = 68.3 s

Configuration 5/8
  n_es

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,15,F0,300,0.5,3,NaN,0.275840,0.595753,70.016076
1,15,F0,300,1.0,3,NaN,0.276242,0.596769,123.239142
2,15,F0,300,1.0,3,30.0,0.276423,0.598033,119.106706
3,15,F0,300,0.5,3,30.0,0.276605,0.596118,68.312705
4,15,F0,300,0.5,1,30.0,0.291426,0.607346,87.048116
5,15,F0,300,0.5,1,NaN,0.292519,0.607578,96.049440
6,15,F0,300,1.0,1,NaN,0.293038,0.608925,173.364578
7,15,F0,300,1.0,1,30.0,0.293175,0.609779,175.475955



Selected 15-minute F0 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.2758396361321829
  validation RMSE: 0.5957525658762308

Total 15-minute F0 grid runtime:
  913.3 seconds
  15.22 minutes

Extra Trees 15-minute F0 experiment: PASSED


In [32]:
# ============================================================
# Step 31: Extra Trees validation
#          15-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + 15 Gregorian forecast-valid-time features
#
# The same frozen 8-configuration Extra Trees grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_15_f1_df, extra_trees_15_f1_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=15,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f1_15 = (
    extra_trees_15_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 15-minute F1 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_15_f1_df
)


# ------------------------------------------------------------
# 4. Decode selected max_depth
# ------------------------------------------------------------

selected_f1_15_depth = (
    None
    if pd.isna(
        best_extra_trees_f1_15[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f1_15[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 15-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f1_15[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f1_15[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f1_15[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f1_15_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f1_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f1_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_15_mae = float(
    best_extra_trees_f0_15[
        "validation_mae"
    ]
)

f1_15_mae = float(
    best_extra_trees_f1_15[
        "validation_mae"
    ]
)

f0_to_f1_15_pct = (
    (f0_15_mae - f1_15_mae)
    / f0_15_mae
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_15_mae:.6f}"
)

print(
    f"  F1 MAE: {f1_15_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_15_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F1 grid runtime:"
)

print(
    f"  {extra_trees_15_f1_seconds:.1f} seconds"
)

print(
    f"  {extra_trees_15_f1_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 15-minute F1 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 15 minutes
  Feature family: F1
  Train: (87261, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.269116
  validation RMSE = 0.580188
  runtime         = 105.4 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.269458
  validation RMSE = 0.581325
  runtime         = 95.0 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.256789
  validation RMSE = 0.578273
  runtime         = 87.6 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.257255
  validation RMSE = 0.577001
  runtime         = 78.8 s

Configuration 5/8
  n_e

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,15,F1,300,0.5,3,NaN,0.256789,0.578273,87.641193
1,15,F1,300,1.0,3,NaN,0.257038,0.578301,185.868323
2,15,F1,300,0.5,3,30.0,0.257255,0.577001,78.774241
3,15,F1,300,1.0,3,30.0,0.257659,0.579069,157.485722
4,15,F1,300,1.0,1,NaN,0.267161,0.581549,202.499110
5,15,F1,300,1.0,1,30.0,0.269054,0.580974,200.533798
6,15,F1,300,0.5,1,NaN,0.269116,0.580188,105.382587
7,15,F1,300,0.5,1,30.0,0.269458,0.581325,94.967606



Selected 15-minute F1 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.25678865625382724
  validation RMSE: 0.578272596285303

F0 -> F1 validation comparison:
  F0 MAE: 0.275840
  F1 MAE: 0.256789
  Relative MAE improvement: +6.9065%

Total 15-minute F1 grid runtime:
  1113.8 seconds
  18.56 minutes

Extra Trees 15-minute F1 experiment: PASSED


In [33]:
# ============================================================
# Step 32: Extra Trees validation
#          15-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + five deterministic solar-geometry features
#
# The same frozen 8-configuration Extra Trees grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_15_f2_df, extra_trees_15_f2_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=15,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f2_15 = (
    extra_trees_15_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 15-minute F2 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_15_f2_df
)


# ------------------------------------------------------------
# 4. Decode selected max_depth
# ------------------------------------------------------------

selected_f2_15_depth = (
    None
    if pd.isna(
        best_extra_trees_f2_15[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f2_15[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 15-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f2_15[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f2_15[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f2_15[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f2_15_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f2_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f2_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_15_mae = float(
    best_extra_trees_f1_15[
        "validation_mae"
    ]
)

f2_15_mae = float(
    best_extra_trees_f2_15[
        "validation_mae"
    ]
)

f1_to_f2_15_pct = (
    (f1_15_mae - f2_15_mae)
    / f1_15_mae
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_15_mae:.6f}"
)

print(
    f"  F2 MAE: {f2_15_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_15_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F2 grid runtime:"
)

print(
    f"  {extra_trees_15_f2_seconds:.1f} seconds"
)

print(
    f"  {extra_trees_15_f2_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 15-minute F2 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 15 minutes
  Feature family: F2
  Train: (87261, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.270688
  validation RMSE = 0.580440
  runtime         = 99.8 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.267414
  validation RMSE = 0.579280
  runtime         = 97.7 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.257373
  validation RMSE = 0.576795
  runtime         = 78.1 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.256894
  validation RMSE = 0.576788
  runtime         = 77.5 s

Configuration 5/8
  n_es

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,15,F2,300,1.0,3,30.0,0.255868,0.577324,164.610543
1,15,F2,300,1.0,3,NaN,0.256564,0.577558,151.333322
2,15,F2,300,0.5,3,30.0,0.256894,0.576788,77.504283
3,15,F2,300,0.5,3,NaN,0.257373,0.576795,78.080363
4,15,F2,300,1.0,1,30.0,0.266734,0.579521,193.918250
5,15,F2,300,0.5,1,30.0,0.267414,0.579280,97.664238
6,15,F2,300,1.0,1,NaN,0.268300,0.581230,197.327024
7,15,F2,300,0.5,1,NaN,0.270688,0.580440,99.823047



Selected 15-minute F2 configuration:
  n_estimators: 300
  max_features: 1.0
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.25586848424514574
  validation RMSE: 0.577323736493238

F1 -> F2 validation comparison:
  F1 MAE: 0.256789
  F2 MAE: 0.255868
  Relative MAE improvement: +0.3583%

Total 15-minute F2 grid runtime:
  1060.9 seconds
  17.68 minutes

Extra Trees 15-minute F2 experiment: PASSED


In [34]:
# ============================================================
# Step 33: Extra Trees validation
#          15-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + nine continuous lunar / Sun-Moon astronomical features
#
# The same frozen 8-configuration Extra Trees grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_15_f3_df, extra_trees_15_f3_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=15,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f3_15 = (
    extra_trees_15_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 15-minute F3 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_15_f3_df
)


# ------------------------------------------------------------
# 4. Decode selected max_depth
# ------------------------------------------------------------

selected_f3_15_depth = (
    None
    if pd.isna(
        best_extra_trees_f3_15[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f3_15[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 15-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f3_15[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f3_15[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f3_15[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f3_15_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f3_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f3_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_15_mae = float(
    best_extra_trees_f2_15[
        "validation_mae"
    ]
)

f3_15_mae = float(
    best_extra_trees_f3_15[
        "validation_mae"
    ]
)

f2_to_f3_15_pct = (
    (f2_15_mae - f3_15_mae)
    / f2_15_mae
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_15_mae:.6f}"
)

print(
    f"  F3 MAE: {f3_15_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_15_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F3 grid runtime:"
)

print(
    f"  {extra_trees_15_f3_seconds:.1f} seconds"
)

print(
    f"  {extra_trees_15_f3_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 15-minute F3 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 15 minutes
  Feature family: F3
  Train: (87261, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.269162
  validation RMSE = 0.580915
  runtime         = 117.8 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.269633
  validation RMSE = 0.580919
  runtime         = 129.9 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.257788
  validation RMSE = 0.576744
  runtime         = 94.2 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.257498
  validation RMSE = 0.576430
  runtime         = 106.2 s

Configuration 5/8
  n

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,15,F3,300,1.0,3,30.0,0.255376,0.576080,169.277009
1,15,F3,300,0.5,3,30.0,0.257498,0.576430,106.166726
2,15,F3,300,1.0,3,NaN,0.257532,0.576508,189.554979
3,15,F3,300,0.5,3,NaN,0.257788,0.576744,94.199006
4,15,F3,300,1.0,1,NaN,0.268054,0.579443,243.710313
5,15,F3,300,1.0,1,30.0,0.268228,0.580571,230.737947
6,15,F3,300,0.5,1,NaN,0.269162,0.580915,117.818200
7,15,F3,300,0.5,1,30.0,0.269633,0.580919,129.865881



Selected 15-minute F3 configuration:
  n_estimators: 300
  max_features: 1.0
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.2553764141350846
  validation RMSE: 0.5760796037373559

F2 -> F3 validation comparison:
  F2 MAE: 0.255868
  F3 MAE: 0.255376
  Relative MAE improvement: +0.1923%

Total 15-minute F3 grid runtime:
  1282.4 seconds
  21.37 minutes

Extra Trees 15-minute F3 experiment: PASSED


In [35]:
# ============================================================
# Step 34: Extra Trees validation
#          15-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + 97 frozen Panchang one-hot indicators
#
# This gives the central 15-minute comparison:
#
#          F3 -> F4
#
# The same frozen 8-configuration Extra Trees grid is used.
#
# The TEST period remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run the frozen validation grid
# ------------------------------------------------------------

extra_trees_15_f4_df, extra_trees_15_f4_seconds = (
    run_extra_trees_validation_grid(
        horizon_minutes=15,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select configuration using minimum validation MAE
# ------------------------------------------------------------

best_extra_trees_f4_15 = (
    extra_trees_15_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Extra Trees — 15-minute F4 validation results"
)

print(
    "============================================\n"
)

display(
    extra_trees_15_f4_df
)


# ------------------------------------------------------------
# 4. Decode selected max_depth
# ------------------------------------------------------------

selected_f4_15_depth = (
    None
    if pd.isna(
        best_extra_trees_f4_15[
            "max_depth"
        ]
    )
    else int(
        best_extra_trees_f4_15[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 15-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_extra_trees_f4_15[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    best_extra_trees_f4_15[
        "max_features"
    ],
)

print(
    "  min_samples_leaf:",
    int(
        best_extra_trees_f4_15[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_f4_15_depth,
)

print(
    "  validation MAE:",
    best_extra_trees_f4_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_extra_trees_f4_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Central F3 -> F4 comparison
# ------------------------------------------------------------

f3_15_mae = float(
    best_extra_trees_f3_15[
        "validation_mae"
    ]
)

f4_15_mae = float(
    best_extra_trees_f4_15[
        "validation_mae"
    ]
)

f3_to_f4_15_absolute_change = (
    f3_15_mae - f4_15_mae
)

f3_to_f4_15_pct = (
    f3_to_f4_15_absolute_change
    / f3_15_mae
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_15_mae:.6f}"
)

print(
    f"  F4 MAE: {f4_15_mae:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_15_absolute_change:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_15_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F4 grid runtime:"
)

print(
    f"  {extra_trees_15_f4_seconds:.1f} seconds"
)

print(
    f"  {extra_trees_15_f4_seconds / 60:.2f} minutes"
)


print(
    "\nExtra Trees 15-minute F4 experiment: PASSED"
)

Extra Trees validation dataset:
  Horizon: 15 minutes
  Feature family: F4
  Train: (87261, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.273867
  validation RMSE = 0.585697
  runtime         = 173.7 s

Configuration 2/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.274182
  validation RMSE = 0.585769
  runtime         = 156.5 s

Configuration 3/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.258943
  validation RMSE = 0.579089
  runtime         = 138.9 s

Configuration 4/8
  n_estimators      = 300
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.259226
  validation RMSE = 0.578726
  runtime         = 136.4 s

Configuration 5/8
  

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,fit_predict_seconds
0,15,F4,300,1.0,3,30.0,0.257614,0.579830,284.478893
1,15,F4,300,1.0,3,NaN,0.258235,0.579655,322.435601
2,15,F4,300,0.5,3,NaN,0.258943,0.579089,138.903041
3,15,F4,300,0.5,3,30.0,0.259226,0.578726,136.424100
4,15,F4,300,1.0,1,NaN,0.273497,0.588270,338.120096
5,15,F4,300,1.0,1,30.0,0.273662,0.588114,307.068441
6,15,F4,300,0.5,1,NaN,0.273867,0.585697,173.731933
7,15,F4,300,0.5,1,30.0,0.274182,0.585769,156.469865



Selected 15-minute F4 configuration:
  n_estimators: 300
  max_features: 1.0
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.25761417405226716
  validation RMSE: 0.5798300584217423

F3 -> F4 validation comparison:
  F3 MAE: 0.255376
  F4 MAE: 0.257614
  Absolute MAE reduction: -0.002238
  Relative MAE improvement: -0.8763%

Total 15-minute F4 grid runtime:
  1858.3 seconds
  30.97 minutes

Extra Trees 15-minute F4 experiment: PASSED


In [36]:
# ============================================================
# Step 35: Summarize the complete 15-minute Extra Trees
#          F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect selected configurations
# ------------------------------------------------------------

best_rows_15 = {
    "F0": best_extra_trees_f0_15,
    "F1": best_extra_trees_f1_15,
    "F2": best_extra_trees_f2_15,
    "F3": best_extra_trees_f3_15,
    "F4": best_extra_trees_f4_15,
}


summary_rows_15 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_15[
        feature_family
    ]

    max_depth = (
        None
        if pd.isna(
            row["max_depth"]
        )
        else int(
            row["max_depth"]
        )
    )

    summary_rows_15.append(
        {
            "horizon_minutes": 15,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][feature_family],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "max_features":
                float(
                    row[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    row[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                max_depth,

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


extra_trees_15_summary = (
    pd.DataFrame(
        summary_rows_15
    )
)


# ------------------------------------------------------------
# 2. Add incremental MAE improvement
# ------------------------------------------------------------
#
# Positive:
#     new feature family reduced MAE.
#
# Negative:
#     new feature family increased MAE.
# ------------------------------------------------------------

extra_trees_15_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(extra_trees_15_summary),
):

    previous_mae = (
        extra_trees_15_summary
        .loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = (
        extra_trees_15_summary
        .loc[
            i,
            "validation_mae",
        ]
    )

    extra_trees_15_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (previous_mae - current_mae)
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Verify central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_15 = (
    extra_trees_15_summary
    .loc[
        extra_trees_15_summary[
            "feature_family"
        ] == "F3",
        "validation_mae",
    ]
    .iloc[0]
)

f4_summary_mae_15 = (
    extra_trees_15_summary
    .loc[
        extra_trees_15_summary[
            "feature_family"
        ] == "F4",
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_15 = (
    (
        f3_summary_mae_15
        - f4_summary_mae_15
    )
    / f3_summary_mae_15
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_15,
    -0.8763,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete summary
# ------------------------------------------------------------

print(
    "Extra Trees — 15-minute validation summary\n"
)

display(
    extra_trees_15_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)

for i in range(
    1,
    len(extra_trees_15_summary),
):

    previous_family = (
        extra_trees_15_summary
        .loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        extra_trees_15_summary
        .loc[
            i,
            "feature_family",
        ]
    )

    change = (
        extra_trees_15_summary
        .loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )

    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_15 = (
    extra_trees_15_summary
    .sort_values(
        "validation_mae"
    )
    .iloc[0]
)


print(
    "\nLowest 15-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_15[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_15[
        "validation_mae"
    ],
)


print(
    "\nCentral Extra Trees comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_15:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_15:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_15:+.4f}%"
)


print(
    "\nExtra Trees 15-minute summary: PASSED"
)

Extra Trees — 15-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,15,F0,118,300,0.5,3,NaN,0.275840,0.595753,NaN
1,15,F1,133,300,0.5,3,NaN,0.256789,0.578273,6.906542
2,15,F2,138,300,1.0,3,30.0,0.255868,0.577324,0.358338
3,15,F3,147,300,1.0,3,30.0,0.255376,0.576080,0.192314
4,15,F4,244,300,1.0,3,30.0,0.257614,0.579830,-0.876259


Incremental validation MAE changes:
  F0 -> F1: +6.9065%
  F1 -> F2: +0.3583%
  F2 -> F3: +0.1923%
  F3 -> F4: -0.8763%

Lowest 15-minute validation MAE:
  Feature family: F3
  MAE: 0.2553764141350846

Central Extra Trees comparison:
  F3 MAE: 0.255376
  F4 MAE: 0.257614
  F3 -> F4 change: -0.8763%

Extra Trees 15-minute summary: PASSED


In [37]:
# ============================================================
# Step 36: Combine the complete Extra Trees validation study
#          and freeze all selected configurations
# ============================================================
#
# At this point:
#
#   - all 15 validation experiments are complete;
#   - no Extra Trees test-set result has been examined;
#   - hyperparameter selection was based exclusively on
#     chronological validation MAE.
#
# We now freeze the selected configuration for every:
#
#     horizon × feature family
#
# combination before any train+validation refitting or
# test-period evaluation.
# ============================================================


# ------------------------------------------------------------
# 1. Combine the three horizon summaries
# ------------------------------------------------------------

extra_trees_validation_summary = (
    pd.concat(
        [
            extra_trees_15_summary,
            extra_trees_30_summary,
            extra_trees_60_summary,
        ],
        axis=0,
        ignore_index=True,
    )
)


# Preserve the natural F0 -> F4 family order.
feature_family_order = {
    "F0": 0,
    "F1": 1,
    "F2": 2,
    "F3": 3,
    "F4": 4,
}


extra_trees_validation_summary[
    "_family_order"
] = (
    extra_trees_validation_summary[
        "feature_family"
    ]
    .map(
        feature_family_order
    )
)


extra_trees_validation_summary = (
    extra_trees_validation_summary
    .sort_values(
        [
            "horizon_minutes",
            "_family_order",
        ]
    )
    .drop(
        columns="_family_order"
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        extra_trees_validation_summary
    )
    == 15
)


print(
    "Combined Extra Trees validation summary:\n"
)

display(
    extra_trees_validation_summary
)


# ------------------------------------------------------------
# 2. Build explicit F3 -> F4 validation comparison by horizon
# ------------------------------------------------------------

f3_f4_validation_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_summary = (
        extra_trees_validation_summary[
            extra_trees_validation_summary[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
    )

    f3_mae = float(
        horizon_summary.loc[
            horizon_summary[
                "feature_family"
            ]
            == "F3",
            "validation_mae",
        ]
        .iloc[0]
    )

    f4_mae = float(
        horizon_summary.loc[
            horizon_summary[
                "feature_family"
            ]
            == "F4",
            "validation_mae",
        ]
        .iloc[0]
    )

    absolute_reduction = (
        f3_mae - f4_mae
    )

    relative_improvement_pct = (
        absolute_reduction
        / f3_mae
        * 100.0
    )

    f3_f4_validation_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "f3_validation_mae":
                f3_mae,

            "f4_validation_mae":
                f4_mae,

            "absolute_mae_reduction":
                absolute_reduction,

            "relative_mae_improvement_pct":
                relative_improvement_pct,
        }
    )


extra_trees_f3_f4_validation = (
    pd.DataFrame(
        f3_f4_validation_rows
    )
)


print(
    "\nExtra Trees F3 -> F4 validation comparison:\n"
)

display(
    extra_trees_f3_f4_validation
)


# ------------------------------------------------------------
# 3. Verify the known F3 -> F4 validation effects
# ------------------------------------------------------------

expected_f3_f4_effects = {
    15: -0.876259,
    30: +0.038274,
    60: -1.449386,
}


for horizon_minutes, expected_value in (
    expected_f3_f4_effects.items()
):

    observed_value = float(
        extra_trees_f3_f4_validation.loc[
            extra_trees_f3_f4_validation[
                "horizon_minutes"
            ]
            == horizon_minutes,
            "relative_mae_improvement_pct",
        ]
        .iloc[0]
    )

    assert np.isclose(
        observed_value,
        expected_value,
        atol=1e-5,
    )


# ------------------------------------------------------------
# 4. Gather the actual selected validation rows
# ------------------------------------------------------------

best_rows_by_horizon = {
    15: best_rows_15,
    30: best_rows_30,
    60: best_rows_60,
}


# ------------------------------------------------------------
# 5. Freeze the selected Extra Trees configuration registry
# ------------------------------------------------------------
#
# Important:
#
# max_depth is explicitly converted back to either:
#
#     None
#
# or:
#
#     30
#
# rather than retaining pandas NaN.
# ------------------------------------------------------------

extra_trees_selected_configs = {}


for horizon_minutes in [
    15,
    30,
    60,
]:

    extra_trees_selected_configs[
        horizon_minutes
    ] = {}

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        selected_row = (
            best_rows_by_horizon[
                horizon_minutes
            ][
                feature_family
            ]
        )

        selected_max_depth = (
            None
            if pd.isna(
                selected_row[
                    "max_depth"
                ]
            )
            else int(
                selected_row[
                    "max_depth"
                ]
            )
        )

        extra_trees_selected_configs[
            horizon_minutes
        ][
            feature_family
        ] = {
            "n_estimators":
                int(
                    selected_row[
                        "n_estimators"
                    ]
                ),

            "max_features":
                float(
                    selected_row[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    selected_row[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                selected_max_depth,

            "criterion":
                "squared_error",

            "bootstrap":
                False,

            "random_state":
                42,

            "n_jobs":
                -1,
        }


# ------------------------------------------------------------
# 6. Create a human-readable frozen-selection table
# ------------------------------------------------------------

frozen_selection_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        config = (
            extra_trees_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )

        validation_row = (
            extra_trees_validation_summary[
                (
                    extra_trees_validation_summary[
                        "horizon_minutes"
                    ]
                    == horizon_minutes
                )
                &
                (
                    extra_trees_validation_summary[
                        "feature_family"
                    ]
                    == feature_family
                )
            ]
            .iloc[0]
        )

        frozen_selection_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_features":
                    primary_protocol[
                        "feature_counts"
                    ][feature_family],

                "n_estimators":
                    config[
                        "n_estimators"
                    ],

                "max_features":
                    config[
                        "max_features"
                    ],

                "min_samples_leaf":
                    config[
                        "min_samples_leaf"
                    ],

                # Use a readable string here only for display.
                # The actual registry above retains Python None.
                "max_depth":
                    (
                        "None"
                        if config[
                            "max_depth"
                        ] is None
                        else str(
                            config[
                                "max_depth"
                            ]
                        )
                    ),

                "validation_mae":
                    float(
                        validation_row[
                            "validation_mae"
                        ]
                    ),

                "validation_rmse":
                    float(
                        validation_row[
                            "validation_rmse"
                        ]
                    ),
            }
        )


extra_trees_frozen_selection = (
    pd.DataFrame(
        frozen_selection_rows
    )
)


# ------------------------------------------------------------
# 7. Strong registry checks
# ------------------------------------------------------------

assert (
    extra_trees_frozen_selection.shape[0]
    == 15
)

assert set(
    extra_trees_frozen_selection[
        "horizon_minutes"
    ]
) == {
    15,
    30,
    60,
}

assert set(
    extra_trees_frozen_selection[
        "feature_family"
    ]
) == {
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
}


for horizon_minutes in [
    15,
    30,
    60,
]:

    assert (
        len(
            extra_trees_selected_configs[
                horizon_minutes
            ]
        )
        == 5
    )


print(
    "\nFrozen Extra Trees configuration registry:\n"
)

display(
    extra_trees_frozen_selection
)


# ------------------------------------------------------------
# 8. Identify the lowest validation MAE within each horizon
# ------------------------------------------------------------

print(
    "\nLowest validation MAE by horizon:"
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    best_row = (
        extra_trees_validation_summary[
            extra_trees_validation_summary[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
        .sort_values(
            "validation_mae"
        )
        .iloc[0]
    )

    print(
        f"  {horizon_minutes:2d} min: "
        f"{best_row['feature_family']} "
        f"(MAE = "
        f"{best_row['validation_mae']:.6f})"
    )


# ------------------------------------------------------------
# 9. Final pre-test checkpoint
# ------------------------------------------------------------

print(
    "\nF3 -> F4 validation effects:"
)

for _, row in (
    extra_trees_f3_f4_validation
    .iterrows()
):

    print(
        f"  {int(row['horizon_minutes']):2d} min: "
        f"{row['relative_mae_improvement_pct']:+.4f}%"
    )


print(
    "\nAll 15 Extra Trees configurations are now frozen."
)

print(
    "No Extra Trees test-set metric has been used for selection."
)

print(
    "\nExtra Trees validation-selection checkpoint: PASSED"
)

Combined Extra Trees validation summary:



,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,15,F0,118,300,0.5,3,NaN,0.275840,0.595753,NaN
1,15,F1,133,300,0.5,3,NaN,0.256789,0.578273,6.906542
2,15,F2,138,300,1.0,3,30.0,0.255868,0.577324,0.358338
3,15,F3,147,300,1.0,3,30.0,0.255376,0.576080,0.192314
4,15,F4,244,300,1.0,3,30.0,0.257614,0.579830,-0.876259
5,30,F0,118,300,1.0,3,NaN,0.338981,0.673867,NaN
6,30,F1,133,300,0.5,3,NaN,0.307264,0.638857,9.356724
7,30,F2,138,300,1.0,3,NaN,0.306829,0.638223,0.141481
8,30,F3,147,300,0.5,3,30.0,0.308265,0.637826,-0.468058
9,30,F4,244,300,0.5,3,30.0,0.308147,0.641338,0.038274



Extra Trees F3 -> F4 validation comparison:



,horizon_minutes,f3_validation_mae,f4_validation_mae,absolute_mae_reduction,relative_mae_improvement_pct
0,15,0.255376,0.257614,-0.002238,-0.876259
1,30,0.308265,0.308147,0.000118,0.038274
2,60,0.369305,0.374657,-0.005353,-1.449386



Frozen Extra Trees configuration registry:



,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse
0,15,F0,118,300,0.5,3,None,0.275840,0.595753
1,15,F1,133,300,0.5,3,None,0.256789,0.578273
2,15,F2,138,300,1.0,3,30,0.255868,0.577324
3,15,F3,147,300,1.0,3,30,0.255376,0.576080
4,15,F4,244,300,1.0,3,30,0.257614,0.579830
5,30,F0,118,300,1.0,3,None,0.338981,0.673867
6,30,F1,133,300,0.5,3,None,0.307264,0.638857
7,30,F2,138,300,1.0,3,None,0.306829,0.638223
8,30,F3,147,300,0.5,3,30,0.308265,0.637826
9,30,F4,244,300,0.5,3,30,0.308147,0.641338



Lowest validation MAE by horizon:
  15 min: F3 (MAE = 0.255376)
  30 min: F2 (MAE = 0.306829)
  60 min: F2 (MAE = 0.367777)

F3 -> F4 validation effects:
  15 min: -0.8763%
  30 min: +0.0383%
  60 min: -1.4494%

All 15 Extra Trees configurations are now frozen.
No Extra Trees test-set metric has been used for selection.

Extra Trees validation-selection checkpoint: PASSED


In [38]:
# ============================================================
# Step 37: Final Extra Trees test evaluation
# ============================================================
#
# IMPORTANT:
#
# All 15 Extra Trees configurations were frozen using
# validation MAE before entering this cell.
#
# For each:
#
#     horizon × feature family
#
# we now:
#
#   1. refit the selected model on TRAIN + VALIDATION;
#   2. evaluate exactly once on TEST;
#   3. record all-time metrics;
#   4. record daylight-only metrics;
#   5. retain timestamp-aligned predictions for later
#      paired F3-vs-F4 uncertainty analysis.
#
# No hyperparameters will be changed based on these test
# results.
# ============================================================


from sklearn.metrics import r2_score


# ------------------------------------------------------------
# 1. Fixed nominal PV capacity for normalized metrics
# ------------------------------------------------------------

PV_NOMINAL_KW = 10.0


# ------------------------------------------------------------
# 2. Containers for final results and predictions
# ------------------------------------------------------------

extra_trees_test_results = []

extra_trees_test_predictions = {}


# ------------------------------------------------------------
# 3. Start complete final-evaluation timer
# ------------------------------------------------------------

final_test_start_time = (
    time.perf_counter()
)


# ------------------------------------------------------------
# 4. Evaluate every frozen horizon × feature-family model
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    y = targets[
        horizon_minutes
    ]

    split = split_labels[
        horizon_minutes
    ]


    # --------------------------------------------------------
    # Final model-development population:
    #
    #     train + validation
    #
    # Test remains chronologically later.
    # --------------------------------------------------------

    train_validation_mask = (
        split.isin(
            [
                "train",
                "validation",
            ]
        )
    )

    test_mask = (
        split == "test"
    )


    assert int(
        test_mask.sum()
    ) == 26_496


    # --------------------------------------------------------
    # Daylight definition
    #
    # This is the same deterministic solar-elevation criterion
    # used in Notebook 09:
    #
    #     solar elevation > 0 degrees
    #
    # evaluated at forecast-valid time.
    # --------------------------------------------------------

    daylight_test_mask = (
        solar_valid_features[
            horizon_minutes
        ][
            "solar_elevation_deg"
        ]
        > 0.0
    ) & test_mask


    n_test = int(
        test_mask.sum()
    )

    n_test_daylight = int(
        daylight_test_mask.sum()
    )


    print(
        "\n"
        "============================================"
    )

    print(
        f"{horizon_minutes}-minute horizon"
    )

    print(
        "============================================"
    )

    print(
        "Test rows:",
        n_test,
    )

    print(
        "Daylight test rows:",
        n_test_daylight,
    )


    # --------------------------------------------------------
    # Evaluate F0 -> F4
    # --------------------------------------------------------

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        print(
            f"\n{horizon_minutes} min — "
            f"{feature_family}"
        )


        # ----------------------------------------------------
        # Retrieve frozen configuration
        # ----------------------------------------------------

        config = (
            extra_trees_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )


        # ----------------------------------------------------
        # Retrieve feature matrix
        # ----------------------------------------------------

        X = (
            feature_matrices[
                horizon_minutes
            ][
                feature_family
            ]
        )


        # ----------------------------------------------------
        # Build final development and test arrays
        # ----------------------------------------------------

        X_train_validation = (
            X.loc[
                train_validation_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        X_test = (
            X.loc[
                test_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        y_train_validation = (
            y.loc[
                train_validation_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        y_test = (
            y.loc[
                test_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        # ----------------------------------------------------
        # Integrity checks
        # ----------------------------------------------------

        assert (
            X_train_validation.shape[1]
            == primary_protocol[
                "feature_counts"
            ][
                feature_family
            ]
        )

        assert (
            X_test.shape
            == (
                26_496,
                primary_protocol[
                    "feature_counts"
                ][
                    feature_family
                ],
            )
        )

        assert np.isfinite(
            X_train_validation
        ).all()

        assert np.isfinite(
            X_test
        ).all()

        assert np.isfinite(
            y_train_validation
        ).all()

        assert np.isfinite(
            y_test
        ).all()


        # ----------------------------------------------------
        # Fit frozen Extra Trees configuration on
        # train + validation
        # ----------------------------------------------------

        model = ExtraTreesRegressor(
            **config
        )


        fit_start_time = (
            time.perf_counter()
        )


        model.fit(
            X_train_validation,
            y_train_validation,
        )


        test_predictions = (
            model.predict(
                X_test
            )
        )


        fit_predict_seconds = (
            time.perf_counter()
            - fit_start_time
        )


        # ----------------------------------------------------
        # All-time test metrics
        # ----------------------------------------------------

        test_mae = (
            mean_absolute_error(
                y_test,
                test_predictions,
            )
        )

        test_rmse = np.sqrt(
            mean_squared_error(
                y_test,
                test_predictions,
            )
        )

        test_nmae = (
            test_mae
            / PV_NOMINAL_KW
        )

        test_nrmse = (
            test_rmse
            / PV_NOMINAL_KW
        )

        test_r2 = (
            r2_score(
                y_test,
                test_predictions,
            )
        )


        # ----------------------------------------------------
        # Daylight-only test metrics
        # ----------------------------------------------------
        #
        # Convert the daylight Series onto the exact test-row
        # ordering used by y_test / test_predictions.
        # ----------------------------------------------------

        test_issue_index = (
            y.index[
                test_mask
            ]
        )

        daylight_on_test = (
            daylight_test_mask
            .loc[
                test_issue_index
            ]
            .to_numpy(
                dtype=bool
            )
        )


        assert int(
            daylight_on_test.sum()
        ) == n_test_daylight


        y_test_daylight = (
            y_test[
                daylight_on_test
            ]
        )

        predictions_daylight = (
            test_predictions[
                daylight_on_test
            ]
        )


        daylight_mae = (
            mean_absolute_error(
                y_test_daylight,
                predictions_daylight,
            )
        )

        daylight_rmse = np.sqrt(
            mean_squared_error(
                y_test_daylight,
                predictions_daylight,
            )
        )

        daylight_nmae = (
            daylight_mae
            / PV_NOMINAL_KW
        )

        daylight_nrmse = (
            daylight_rmse
            / PV_NOMINAL_KW
        )

        daylight_r2 = (
            r2_score(
                y_test_daylight,
                predictions_daylight,
            )
        )


        # ----------------------------------------------------
        # Record final metrics
        # ----------------------------------------------------

        extra_trees_test_results.append(
            {
                "model":
                    "ExtraTreesRegressor",

                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_features":
                    X.shape[1],

                "n_estimators":
                    config[
                        "n_estimators"
                    ],

                "max_features":
                    config[
                        "max_features"
                    ],

                "min_samples_leaf":
                    config[
                        "min_samples_leaf"
                    ],

                "max_depth":
                    config[
                        "max_depth"
                    ],

                "test_rows":
                    n_test,

                "test_mae":
                    test_mae,

                "test_rmse":
                    test_rmse,

                "test_nmae":
                    test_nmae,

                "test_nrmse":
                    test_nrmse,

                "test_r2":
                    test_r2,

                "daylight_test_rows":
                    n_test_daylight,

                "daylight_mae":
                    daylight_mae,

                "daylight_rmse":
                    daylight_rmse,

                "daylight_nmae":
                    daylight_nmae,

                "daylight_nrmse":
                    daylight_nrmse,

                "daylight_r2":
                    daylight_r2,

                "fit_predict_seconds":
                    fit_predict_seconds,
            }
        )


        # ----------------------------------------------------
        # Retain timestamp-aligned predictions
        #
        # These will later support:
        #
        #   - F3 vs F4 paired error comparison;
        #   - daily-block bootstrap;
        #   - plotting;
        #   - saving reproducible secondary-model artifacts.
        # ----------------------------------------------------

        prediction_table = pd.DataFrame(
            {
                "valid_time":
                    base_modelling_tables[
                        horizon_minutes
                    ]
                    .loc[
                        test_issue_index,
                        "valid_time",
                    ],

                "y_true":
                    y_test,

                "y_pred":
                    test_predictions,

                "absolute_error":
                    np.abs(
                        y_test
                        - test_predictions
                    ),

                "daylight":
                    daylight_on_test,
            },
            index=test_issue_index,
        )

        prediction_table.index.name = (
            "issue_time"
        )


        extra_trees_test_predictions[
            (
                horizon_minutes,
                feature_family,
            )
        ] = prediction_table


        # ----------------------------------------------------
        # Display compact result immediately
        # ----------------------------------------------------

        print(
            f"  all-time MAE:     "
            f"{test_mae:.6f}"
        )

        print(
            f"  all-time RMSE:    "
            f"{test_rmse:.6f}"
        )

        print(
            f"  daylight MAE:     "
            f"{daylight_mae:.6f}"
        )

        print(
            f"  daylight RMSE:    "
            f"{daylight_rmse:.6f}"
        )

        print(
            f"  runtime:          "
            f"{fit_predict_seconds:.1f} s"
        )


        # Do not retain fitted ensembles unnecessarily.
        del model
        del X_train_validation
        del X_test
        del y_train_validation
        del y_test
        del test_predictions

        gc.collect()


# ------------------------------------------------------------
# 5. Construct final test-results table
# ------------------------------------------------------------

extra_trees_final_test_results = (
    pd.DataFrame(
        extra_trees_test_results
    )
)


assert (
    len(
        extra_trees_final_test_results
    )
    == 15
)


# ------------------------------------------------------------
# 6. Preserve natural horizon / family ordering
# ------------------------------------------------------------

extra_trees_final_test_results[
    "_family_order"
] = (
    extra_trees_final_test_results[
        "feature_family"
    ]
    .map(
        feature_family_order
    )
)


extra_trees_final_test_results = (
    extra_trees_final_test_results
    .sort_values(
        [
            "horizon_minutes",
            "_family_order",
        ]
    )
    .drop(
        columns="_family_order"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 7. Final integrity checks
# ------------------------------------------------------------

assert len(
    extra_trees_test_predictions
) == 15

assert (
    extra_trees_final_test_results[
        "test_rows"
    ]
    == 26_496
).all()

assert (
    extra_trees_final_test_results[
        "test_mae"
    ]
    .notna()
    .all()
)

assert (
    extra_trees_final_test_results[
        "daylight_mae"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 8. Display final Extra Trees test results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "FINAL EXTRA TREES TEST RESULTS"
)

print(
    "============================================\n"
)


display(
    extra_trees_final_test_results
)


# ------------------------------------------------------------
# 9. Explicit F3 -> F4 test comparison
# ------------------------------------------------------------

extra_trees_f3_f4_test_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_results = (
        extra_trees_final_test_results[
            extra_trees_final_test_results[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
    )


    f3_row = (
        horizon_results[
            horizon_results[
                "feature_family"
            ]
            == "F3"
        ]
        .iloc[0]
    )

    f4_row = (
        horizon_results[
            horizon_results[
                "feature_family"
            ]
            == "F4"
        ]
        .iloc[0]
    )


    all_time_relative_pct = (
        (
            f3_row["test_mae"]
            - f4_row["test_mae"]
        )
        / f3_row["test_mae"]
        * 100.0
    )

    daylight_relative_pct = (
        (
            f3_row["daylight_mae"]
            - f4_row["daylight_mae"]
        )
        / f3_row["daylight_mae"]
        * 100.0
    )


    extra_trees_f3_f4_test_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "f3_test_mae":
                f3_row[
                    "test_mae"
                ],

            "f4_test_mae":
                f4_row[
                    "test_mae"
                ],

            "f3_to_f4_test_improvement_pct":
                all_time_relative_pct,

            "f3_daylight_mae":
                f3_row[
                    "daylight_mae"
                ],

            "f4_daylight_mae":
                f4_row[
                    "daylight_mae"
                ],

            "f3_to_f4_daylight_improvement_pct":
                daylight_relative_pct,
        }
    )


extra_trees_f3_f4_test_summary = (
    pd.DataFrame(
        extra_trees_f3_f4_test_rows
    )
)


print(
    "\nExtra Trees F3 -> F4 FINAL test comparison:\n"
)

display(
    extra_trees_f3_f4_test_summary
)


# ------------------------------------------------------------
# 10. Runtime
# ------------------------------------------------------------

final_test_total_seconds = (
    time.perf_counter()
    - final_test_start_time
)


print(
    "\nTotal final Extra Trees evaluation runtime:"
)

print(
    f"  {final_test_total_seconds:.1f} seconds"
)

print(
    f"  {final_test_total_seconds / 60:.2f} minutes"
)


print(
    "\nFinal Extra Trees test evaluation: PASSED"
)


15-minute horizon
Test rows: 26496
Daylight test rows: 17851

15 min — F0
  all-time MAE:     0.268254
  all-time RMSE:    0.577067
  daylight MAE:     0.397619
  daylight RMSE:    0.703043
  runtime:          96.6 s

15 min — F1
  all-time MAE:     0.255136
  all-time RMSE:    0.560227
  daylight MAE:     0.378315
  daylight RMSE:    0.682530
  runtime:          107.2 s

15 min — F2
  all-time MAE:     0.254105
  all-time RMSE:    0.559004
  daylight MAE:     0.376882
  daylight RMSE:    0.681041
  runtime:          195.2 s

15 min — F3
  all-time MAE:     0.254847
  all-time RMSE:    0.558407
  daylight MAE:     0.377985
  daylight RMSE:    0.680313
  runtime:          226.2 s

15 min — F4
  all-time MAE:     0.255909
  all-time RMSE:    0.560455
  daylight MAE:     0.379563
  daylight RMSE:    0.682808
  runtime:          315.5 s

30-minute horizon
Test rows: 26496
Daylight test rows: 17851

30 min — F0
  all-time MAE:     0.323862
  all-time RMSE:    0.651994
  daylight MAE:     0

,model,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,test_rows,test_mae,...,test_nmae,test_nrmse,test_r2,daylight_test_rows,daylight_mae,daylight_rmse,daylight_nmae,daylight_nrmse,daylight_r2,fit_predict_seconds
0,ExtraTreesRegressor,15,F0,118,300,0.5,3,NaN,26496,0.268254,...,0.026825,0.057707,0.877781,17851,0.397619,0.703043,0.039762,0.070304,0.834404,96.604616
1,ExtraTreesRegressor,15,F1,133,300,0.5,3,NaN,26496,0.255136,...,0.025514,0.056023,0.884810,17851,0.378315,0.682530,0.037832,0.068253,0.843926,107.200217
2,ExtraTreesRegressor,15,F2,138,300,1.0,3,30.0,26496,0.254105,...,0.025411,0.055900,0.885312,17851,0.376882,0.681041,0.037688,0.068104,0.844606,195.154242
3,ExtraTreesRegressor,15,F3,147,300,1.0,3,30.0,26496,0.254847,...,0.025485,0.055841,0.885557,17851,0.377985,0.680313,0.037798,0.068031,0.844938,226.183244
4,ExtraTreesRegressor,15,F4,244,300,1.0,3,30.0,26496,0.255909,...,0.025591,0.056045,0.884716,17851,0.379563,0.682808,0.037956,0.068281,0.843799,315.501039
5,ExtraTreesRegressor,30,F0,118,300,1.0,3,NaN,26496,0.323862,...,0.032386,0.065199,0.843983,17851,0.479164,0.794308,0.047916,0.079431,0.788620,116.696344
6,ExtraTreesRegressor,30,F1,133,300,0.5,3,NaN,26496,0.295281,...,0.029528,0.061997,0.858934,17851,0.437514,0.755307,0.043751,0.075531,0.808868,43.966827
7,ExtraTreesRegressor,30,F2,138,300,1.0,3,NaN,26496,0.296099,...,0.029610,0.062156,0.858210,17851,0.439076,0.757247,0.043908,0.075725,0.807885,87.047692
8,ExtraTreesRegressor,30,F3,147,300,0.5,3,30.0,26496,0.295468,...,0.029547,0.061985,0.858988,17851,0.438135,0.755165,0.043813,0.075516,0.808940,52.020153
9,ExtraTreesRegressor,30,F4,244,300,0.5,3,30.0,26496,0.296174,...,0.029617,0.062241,0.857819,17851,0.439190,0.758289,0.043919,0.075829,0.807356,91.685932



Extra Trees F3 -> F4 FINAL test comparison:



,horizon_minutes,f3_test_mae,f4_test_mae,f3_to_f4_test_improvement_pct,f3_daylight_mae,f4_daylight_mae,f3_to_f4_daylight_improvement_pct
0,15,0.254847,0.255909,-0.416511,0.377985,0.379563,-0.417672
1,30,0.295468,0.296174,-0.238948,0.438135,0.439190,-0.240892
2,60,0.338997,0.338905,0.026957,0.502679,0.502551,0.025549



Total final Extra Trees evaluation runtime:
  1725.5 seconds
  28.76 minutes

Final Extra Trees test evaluation: PASSED


In [39]:
# ============================================================
# Step 38: Recover the frozen paired-bootstrap specification
#          from the primary experiment protocol
# ============================================================
#
# Before applying uncertainty analysis to Extra Trees, we
# inspect the saved Notebook 09 protocol so that the secondary
# model uses the SAME:
#
#   - number of bootstrap repetitions;
#   - random seed;
#   - block definition;
#   - confidence interval;
#   - paired-error convention.
#
# We do not choose new bootstrap settings after seeing the
# Extra Trees test results.
# ============================================================


def show_matching_protocol_entries(
    obj,
    keywords,
    path="primary_protocol",
):
    """
    Recursively display protocol entries whose key/path contains
    one of the requested keywords.
    """

    if isinstance(obj, dict):

        for key, value in obj.items():

            current_path = (
                f"{path}.{key}"
            )

            key_lower = (
                str(key).lower()
            )

            path_lower = (
                current_path.lower()
            )

            if any(
                keyword in key_lower
                or keyword in path_lower
                for keyword in keywords
            ):
                print(
                    f"{current_path}:"
                )

                print(
                    value
                )

                print()

            if isinstance(
                value,
                (dict, list),
            ):
                show_matching_protocol_entries(
                    value,
                    keywords,
                    current_path,
                )

    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            if isinstance(
                value,
                (dict, list),
            ):
                show_matching_protocol_entries(
                    value,
                    keywords,
                    f"{path}[{i}]",
                )


show_matching_protocol_entries(
    primary_protocol,
    keywords=[
        "bootstrap",
        "seed",
        "confidence",
        "block",
        "replicate",
    ],
)

primary_protocol.bootstrap:
{'comparison': 'F3_minus_F4_MAE', 'block_unit': 'forecast-valid UTC calendar day', 'daily_blocks': 92, 'replicates': 10000, 'random_seed': 42, 'confidence_interval': '95% percentile', 'model_retraining': False}

primary_protocol.bootstrap.comparison:
F3_minus_F4_MAE

primary_protocol.bootstrap.block_unit:
forecast-valid UTC calendar day

primary_protocol.bootstrap.daily_blocks:
92

primary_protocol.bootstrap.replicates:
10000

primary_protocol.bootstrap.random_seed:
42

primary_protocol.bootstrap.confidence_interval:
95% percentile

primary_protocol.bootstrap.model_retraining:
False



In [40]:
# ============================================================
# Step 39: Recover the exact paired daily-block bootstrap
#          implementation used in Notebook 09
# ============================================================
#
# The protocol JSON tells us the frozen statistical settings,
# but we also want to reproduce the exact computational
# implementation used in the primary experiment.
#
# This prevents subtle procedural drift in:
#
#   - block construction;
#   - daylight filtering;
#   - relative-improvement calculation;
#   - random-number handling;
#   - percentile confidence intervals.
# ============================================================

import json


NOTEBOOK09_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "09_modelling_dataset_and_protocol.ipynb"
)

assert NOTEBOOK09_PATH.is_file()


with open(
    NOTEBOOK09_PATH,
    "r",
    encoding="utf-8",
) as file:
    notebook09_json = json.load(file)


matching_cells = []


for cell_number, cell in enumerate(
    notebook09_json["cells"],
    start=1,
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            [],
        )
    )

    source_lower = source.lower()

    if (
        "bootstrap" in source_lower
        and (
            "f3" in source_lower
            or "block" in source_lower
            or "absolute_error" in source_lower
        )
    ):

        matching_cells.append(
            (
                cell_number,
                source,
            )
        )


print(
    "Matching Notebook 09 bootstrap code cells:",
    len(matching_cells),
)


for cell_number, source in matching_cells:

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"NOTEBOOK 09 CELL {cell_number}"
    )

    print(
        "=" * 80
    )

    print(
        source
    )

Matching Notebook 09 bootstrap code cells: 2

NOTEBOOK 09 CELL 40
# ============================================================
# Step 25: Paired daily-block bootstrap for F3 vs F4
# ============================================================

# ------------------------------------------------------------
# 1. Frozen bootstrap settings
# ------------------------------------------------------------

N_BOOTSTRAP = 10_000
BOOTSTRAP_RANDOM_SEED = 42

BOOTSTRAP_MODELS = [
    "Ridge",
    "HGBR",
]

BOOTSTRAP_HORIZONS = [
    15,
    30,
    60,
]

BOOTSTRAP_SCOPES = [
    "all",
    "daylight",
]


# One reproducible random-number generator for the complete
# bootstrap analysis.
rng = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)


# ------------------------------------------------------------
# 2. Storage for bootstrap summaries
# ------------------------------------------------------------

bootstrap_summary_rows = []

# Keep the complete bootstrap distributions as well, in case
#

In [ ]:
# ============================================================
# Step 40: Paired daily-block bootstrap for Extra Trees
#          F3 vs F4
# ============================================================
#
# This reproduces the bootstrap procedure used in Notebook 09.
#
# Frozen specification:
#
#   comparison:
#       |error F3| - |error F4|
#
#   positive delta:
#       F4 has lower MAE than F3
#
#   resampling block:
#       forecast-valid UTC calendar day
#
#   test days:
#       92
#
#   bootstrap replicates:
#       10,000
#
#   random seed:
#       42
#
#   interval:
#       95% percentile
#
#   model retraining:
#       False
#
# Both:
#
#   - all test observations
#   - daylight-only observations
#
# are evaluated.
# ============================================================


# ------------------------------------------------------------
# 1. Recover the frozen settings directly from the primary
#    experiment protocol
# ------------------------------------------------------------

N_BOOTSTRAP = int(
    primary_protocol[
        "bootstrap"
    ][
        "replicates"
    ]
)

BOOTSTRAP_RANDOM_SEED = int(
    primary_protocol[
        "bootstrap"
    ][
        "random_seed"
    ]
)

EXPECTED_DAILY_BLOCKS = int(
    primary_protocol[
        "bootstrap"
    ][
        "daily_blocks"
    ]
)


assert N_BOOTSTRAP == 10_000
assert BOOTSTRAP_RANDOM_SEED == 42
assert EXPECTED_DAILY_BLOCKS == 92


BOOTSTRAP_HORIZONS = [
    15,
    30,
    60,
]

BOOTSTRAP_SCOPES = [
    "all",
    "daylight",
]


# One reproducible RNG for the complete Extra Trees
# bootstrap analysis, matching the Notebook 09 procedure.
rng_extra_trees = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)


# ------------------------------------------------------------
# 2. Storage
# ------------------------------------------------------------

extra_trees_bootstrap_summary_rows = []

extra_trees_bootstrap_distributions = {}


# ------------------------------------------------------------
# 3. Run one paired bootstrap for every:
#
#       horizon × scope
#
# Total:
#
#       3 × 2 = 6 analyses
# ------------------------------------------------------------

for horizon_minutes in BOOTSTRAP_HORIZONS:

    # --------------------------------------------------------
    # Retrieve the frozen final F3 and F4 prediction tables
    # --------------------------------------------------------

    df_f3 = (
        extra_trees_test_predictions[
            (
                horizon_minutes,
                "F3",
            )
        ]
        .copy()
    )

    df_f4 = (
        extra_trees_test_predictions[
            (
                horizon_minutes,
                "F4",
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # 4. Verify perfect paired alignment
    # --------------------------------------------------------

    assert df_f3.index.equals(
        df_f4.index
    )

    assert len(df_f3) == 26_496

    assert len(df_f4) == 26_496

    assert df_f3[
        "valid_time"
    ].equals(
        df_f4[
            "valid_time"
        ]
    )

    assert np.allclose(
        df_f3[
            "y_true"
        ].to_numpy(),
        df_f4[
            "y_true"
        ].to_numpy(),
        atol=0.0,
        rtol=0.0,
    )

    assert df_f3[
        "daylight"
    ].equals(
        df_f4[
            "daylight"
        ]
    )


    # --------------------------------------------------------
    # 5. Construct one paired prediction table
    # --------------------------------------------------------

    df_predictions = pd.DataFrame(
        {
            "valid_time":
                df_f3[
                    "valid_time"
                ],

            "y_true":
                df_f3[
                    "y_true"
                ],

            "F3_prediction":
                df_f3[
                    "y_pred"
                ],

            "F4_prediction":
                df_f4[
                    "y_pred"
                ],

            "daylight":
                df_f3[
                    "daylight"
                ],
        },
        index=df_f3.index,
    )


    # --------------------------------------------------------
    # 6. Verify fixed final test interval
    # --------------------------------------------------------

    assert (
        df_predictions[
            "valid_time"
        ].min()
        == pd.Timestamp(
            "2019-06-01 00:00:00",
            tz="UTC",
        )
    )

    assert (
        df_predictions[
            "valid_time"
        ].max()
        == pd.Timestamp(
            "2019-08-31 23:55:00",
            tz="UTC",
        )
    )


    # --------------------------------------------------------
    # 7. All-time and daylight scopes
    # --------------------------------------------------------

    for scope_name in BOOTSTRAP_SCOPES:

        if scope_name == "all":

            df_scope = (
                df_predictions
                .copy()
            )

        elif scope_name == "daylight":

            df_scope = (
                df_predictions.loc[
                    df_predictions[
                        "daylight"
                    ]
                ]
                .copy()
            )

        else:

            raise ValueError(
                f"Unexpected scope: {scope_name}"
            )


        # ----------------------------------------------------
        # 8. Paired absolute errors
        #
        # Positive difference means F4 is better:
        #
        #     |e_F3| - |e_F4| > 0
        # ----------------------------------------------------

        df_scope[
            "abs_error_F3"
        ] = np.abs(
            df_scope[
                "y_true"
            ]
            - df_scope[
                "F3_prediction"
            ]
        )

        df_scope[
            "abs_error_F4"
        ] = np.abs(
            df_scope[
                "y_true"
            ]
            - df_scope[
                "F4_prediction"
            ]
        )

        df_scope[
            "paired_error_difference"
        ] = (
            df_scope[
                "abs_error_F3"
            ]
            - df_scope[
                "abs_error_F4"
            ]
        )


        # ----------------------------------------------------
        # 9. Forecast-valid UTC calendar-day blocks
        # ----------------------------------------------------

        df_scope[
            "valid_date"
        ] = (
            df_scope[
                "valid_time"
            ]
            .dt.floor("D")
        )


        # ----------------------------------------------------
        # 10. Collapse each daily block into sufficient
        #     statistics exactly as in Notebook 09
        # ----------------------------------------------------

        daily_blocks = (
            df_scope
            .groupby(
                "valid_date",
                sort=True,
            )
            .agg(
                n_rows=(
                    "y_true",
                    "size",
                ),

                abs_error_F3_sum=(
                    "abs_error_F3",
                    "sum",
                ),

                abs_error_F4_sum=(
                    "abs_error_F4",
                    "sum",
                ),
            )
        )


        n_days = len(
            daily_blocks
        )


        assert n_days == EXPECTED_DAILY_BLOCKS, (
            f"Extra Trees / "
            f"{horizon_minutes} min / "
            f"{scope_name}: "
            f"expected {EXPECTED_DAILY_BLOCKS} "
            f"daily blocks, found {n_days}."
        )


        # ----------------------------------------------------
        # 11. Observed test-set point estimates
        # ----------------------------------------------------

        observed_f3_mae = (
            df_scope[
                "abs_error_F3"
            ]
            .mean()
        )

        observed_f4_mae = (
            df_scope[
                "abs_error_F4"
            ]
            .mean()
        )

        observed_delta_mae = (
            observed_f3_mae
            - observed_f4_mae
        )

        observed_relative_change = (
            observed_delta_mae
            / observed_f3_mae
            * 100.0
        )


        # ----------------------------------------------------
        # 12. Verify against the final Extra Trees result table
        # ----------------------------------------------------

        expected_row = (
            extra_trees_f3_f4_test_summary.loc[
                extra_trees_f3_f4_test_summary[
                    "horizon_minutes"
                ].eq(
                    horizon_minutes
                )
            ]
            .iloc[0]
        )


        if scope_name == "all":

            expected_f3_mae = (
                expected_row[
                    "f3_test_mae"
                ]
            )

            expected_f4_mae = (
                expected_row[
                    "f4_test_mae"
                ]
            )

            expected_relative_change = (
                expected_row[
                    "f3_to_f4_test_improvement_pct"
                ]
            )

        else:

            expected_f3_mae = (
                expected_row[
                    "f3_daylight_mae"
                ]
            )

            expected_f4_mae = (
                expected_row[
                    "f4_daylight_mae"
                ]
            )

            expected_relative_change = (
                expected_row[
                    "f3_to_f4_daylight_improvement_pct"
                ]
            )


        assert np.isclose(
            observed_f3_mae,
            expected_f3_mae,
            atol=1e-12,
            rtol=0.0,
        )

        assert np.isclose(
            observed_f4_mae,
            expected_f4_mae,
            atol=1e-12,
            rtol=0.0,
        )

        assert np.isclose(
            observed_relative_change,
            expected_relative_change,
            atol=1e-10,
            rtol=0.0,
        )


        # ----------------------------------------------------
        # 13. Daily sufficient-statistic arrays
        # ----------------------------------------------------

        daily_counts = (
            daily_blocks[
                "n_rows"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        daily_f3_error_sums = (
            daily_blocks[
                "abs_error_F3_sum"
            ]
            .to_numpy(
                dtype=float
            )
        )

        daily_f4_error_sums = (
            daily_blocks[
                "abs_error_F4_sum"
            ]
            .to_numpy(
                dtype=float
            )
        )


        # ----------------------------------------------------
        # 14. Draw 92 whole daily blocks with replacement
        #     for every bootstrap replicate
        # ----------------------------------------------------

        sampled_day_indices = (
            rng_extra_trees.integers(
                low=0,
                high=n_days,
                size=(
                    N_BOOTSTRAP,
                    n_days,
                ),
            )
        )


        # ----------------------------------------------------
        # 15. Reconstruct pooled MAE for every replicate
        # ----------------------------------------------------

        bootstrap_row_counts = (
            daily_counts[
                sampled_day_indices
            ]
            .sum(
                axis=1
            )
        )

        bootstrap_f3_error_sums = (
            daily_f3_error_sums[
                sampled_day_indices
            ]
            .sum(
                axis=1
            )
        )

        bootstrap_f4_error_sums = (
            daily_f4_error_sums[
                sampled_day_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f3_mae = (
            bootstrap_f3_error_sums
            / bootstrap_row_counts
        )

        bootstrap_f4_mae = (
            bootstrap_f4_error_sums
            / bootstrap_row_counts
        )


        # ----------------------------------------------------
        # 16. Paired F3 -> F4 difference
        # ----------------------------------------------------

        bootstrap_delta_mae = (
            bootstrap_f3_mae
            - bootstrap_f4_mae
        )

        bootstrap_relative_change = (
            bootstrap_delta_mae
            / bootstrap_f3_mae
            * 100.0
        )


        # ----------------------------------------------------
        # 17. Frozen 95% percentile intervals
        # ----------------------------------------------------

        (
            delta_ci_lower,
            delta_ci_upper,
        ) = np.percentile(
            bootstrap_delta_mae,
            [
                2.5,
                97.5,
            ],
        )


        (
            relative_ci_lower,
            relative_ci_upper,
        ) = np.percentile(
            bootstrap_relative_change,
            [
                2.5,
                97.5,
            ],
        )


        # ----------------------------------------------------
        # 18. Store complete bootstrap distribution
        # ----------------------------------------------------

        distribution_key = (
            horizon_minutes,
            scope_name,
        )


        extra_trees_bootstrap_distributions[
            distribution_key
        ] = pd.DataFrame(
            {
                "delta_MAE_kW":
                    bootstrap_delta_mae,

                "relative_MAE_change_percent":
                    bootstrap_relative_change,
            }
        )


        # ----------------------------------------------------
        # 19. Store compact summary
        # ----------------------------------------------------

        extra_trees_bootstrap_summary_rows.append(
            {
                "model":
                    "ExtraTreesRegressor",

                "horizon_minutes":
                    horizon_minutes,

                "scope":
                    scope_name,

                "n_test_rows":
                    len(df_scope),

                "n_daily_blocks":
                    n_days,

                "F3_MAE_kW":
                    observed_f3_mae,

                "F4_MAE_kW":
                    observed_f4_mae,

                "delta_MAE_kW":
                    observed_delta_mae,

                "delta_MAE_CI95_lower_kW":
                    delta_ci_lower,

                "delta_MAE_CI95_upper_kW":
                    delta_ci_upper,

                "relative_MAE_change_percent":
                    observed_relative_change,

                "relative_CI95_lower_percent":
                    relative_ci_lower,

                "relative_CI95_upper_percent":
                    relative_ci_upper,
            }
        )


# ------------------------------------------------------------
# 20. Assemble final Extra Trees bootstrap summary
# ------------------------------------------------------------

extra_trees_bootstrap_summary = (
    pd.DataFrame(
        extra_trees_bootstrap_summary_rows
    )
)


# ------------------------------------------------------------
# 21. Structural checks
# ------------------------------------------------------------

assert len(
    extra_trees_bootstrap_summary
) == 6

assert len(
    extra_trees_bootstrap_distributions
) == 6

assert (
    extra_trees_bootstrap_summary[
        "n_daily_blocks"
    ]
    .eq(
        92
    )
    .all()
)

assert np.isfinite(
    extra_trees_bootstrap_summary[
        [
            "F3_MAE_kW",
            "F4_MAE_kW",
            "delta_MAE_kW",
            "delta_MAE_CI95_lower_kW",
            "delta_MAE_CI95_upper_kW",
            "relative_MAE_change_percent",
            "relative_CI95_lower_percent",
            "relative_CI95_upper_percent",
        ]
    ]
    .to_numpy()
).all()


# ------------------------------------------------------------
# 22. Sort for interpretation
# ------------------------------------------------------------

extra_trees_bootstrap_summary = (
    extra_trees_bootstrap_summary
    .sort_values(
        [
            "scope",
            "horizon_minutes",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Extra Trees paired daily-block bootstrap: PASSED"
)


# ------------------------------------------------------------
# 23. Display final uncertainty table
# ------------------------------------------------------------

print(
    "\nExtra Trees F3 -> F4 paired daily-block "
    "bootstrap results:"
)


display(
    extra_trees_bootstrap_summary[
        [
            "model",
            "horizon_minutes",
            "scope",
            "n_test_rows",
            "n_daily_blocks",
            "F3_MAE_kW",
            "F4_MAE_kW",
            "delta_MAE_kW",
            "delta_MAE_CI95_lower_kW",
            "delta_MAE_CI95_upper_kW",
            "relative_MAE_change_percent",
            "relative_CI95_lower_percent",
            "relative_CI95_upper_percent",
        ]
    ]
)


print(
    "\nInterpretation convention:"
)

print(
    "  positive delta / relative change = F4 better"
)

print(
    "  negative delta / relative change = F4 worse"
)

print(
    "  CI containing zero = no clear paired "
    "difference at the 95% percentile level"
)

Extra Trees paired daily-block bootstrap: PASSED

Extra Trees F3 -> F4 paired daily-block bootstrap results:


,model,horizon_minutes,scope,n_test_rows,n_daily_blocks,F3_MAE_kW,F4_MAE_kW,delta_MAE_kW,delta_MAE_CI95_lower_kW,delta_MAE_CI95_upper_kW,relative_MAE_change_percent,relative_CI95_lower_percent,relative_CI95_upper_percent
0,ExtraTreesRegressor,15,all,26496,92,0.254847,0.255909,-0.001061,-0.002545,0.000313,-0.416511,-1.011273,0.123243
1,ExtraTreesRegressor,30,all,26496,92,0.295468,0.296174,-0.000706,-0.002647,0.001278,-0.238948,-0.905414,0.436091
2,ExtraTreesRegressor,60,all,26496,92,0.338997,0.338905,0.000091,-0.002892,0.003114,0.026957,-0.841275,0.953053
3,ExtraTreesRegressor,15,daylight,17851,92,0.377985,0.379563,-0.001579,-0.003827,0.000520,-0.417672,-1.022274,0.136736
4,ExtraTreesRegressor,30,daylight,17851,92,0.438135,0.439190,-0.001055,-0.003932,0.001903,-0.240892,-0.905567,0.444613
5,ExtraTreesRegressor,60,daylight,17851,92,0.502679,0.502551,0.000128,-0.004294,0.004941,0.025549,-0.839309,1.010082



Interpretation convention:
  positive delta / relative change = F4 better
  negative delta / relative change = F4 worse
  CI containing zero = no clear paired difference at the 95% percentile level


## Extra Trees — Validation, Test, and Uncertainty Results

The Extra Trees robustness experiment has now been completed for all three
forecast horizons and all five feature families.

The model-selection procedure used the same frozen eight-configuration
hyperparameter grid for every experiment. Hyperparameters were selected using
chronological validation MAE only, after which the selected models were
refitted on the combined training and validation periods and evaluated on the
chronologically later test period.

---

### Validation Behaviour

The selected validation results showed that the largest improvement generally
occurred when Gregorian time information was added:

$$
F0 \rightarrow F1
$$

The relative validation MAE improvements were approximately:

- 15 min: +6.91%
- 30 min: +9.36%
- 60 min: +17.46%

The addition of deterministic solar astronomy:

$$
F1 \rightarrow F2
$$

produced much smaller changes:

- 15 min: +0.36%
- 30 min: +0.14%
- 60 min: +0.11%

The continuous lunar / Sun–Moon astronomical family:

$$
F2 \rightarrow F3
$$

was also small and model/horizon dependent:

- 15 min: +0.19%
- 30 min: −0.47%
- 60 min: −0.42%

The central Panchang comparison:

$$
F3 \rightarrow F4
$$

gave validation changes of:

- 15 min: −0.88%
- 30 min: +0.04%
- 60 min: −1.45%

Thus, the validation results did not show a consistent benefit from adding
the discrete Panchang representation.

---

### Final Test Results

After all 15 Extra Trees configurations had been frozen, each selected model
was refitted using:

$$
\text{training} + \text{validation}
$$

and evaluated once on the final chronological test period.

The lowest Extra Trees all-time test MAE within each horizon was:

| Horizon | Feature family | Test MAE (kW) |
|---|---|---:|
| 15 min | F2 | 0.254105 |
| 30 min | F1 | 0.295281 |
| 60 min | F2 | 0.337808 |

The test-set F3 → F4 comparisons were:

| Horizon | F3 MAE | F4 MAE | Relative F3 → F4 change |
|---|---:|---:|---:|
| 15 min | 0.254847 | 0.255909 | −0.4165% |
| 30 min | 0.295468 | 0.296174 | −0.2389% |
| 60 min | 0.338997 | 0.338905 | +0.0270% |

Positive values indicate lower MAE for F4.

The corresponding daylight-only changes were:

- 15 min: −0.4177%
- 30 min: −0.2409%
- 60 min: +0.0255%

The all-time and daylight-only results therefore show essentially the same
pattern.

---

### Paired Daily-Block Bootstrap

To quantify uncertainty in the F3 → F4 comparison, the exact paired
daily-block bootstrap procedure used in Notebook 09 was reused.

The procedure used:

- 92 forecast-valid UTC calendar-day blocks;
- 10,000 bootstrap replicates;
- random seed 42;
- paired absolute-error differences;
- pooled MAE after resampling complete daily blocks;
- 95% percentile confidence intervals;
- no model retraining within bootstrap replicates.

Positive paired differences indicate lower error for F4.

For every horizon and for both evaluation scopes, the 95% confidence interval
for the F3 → F4 MAE difference contained zero.

All-time relative-change intervals were approximately:

- 15 min: −0.42%, CI [−1.01%, +0.12%]
- 30 min: −0.24%, CI [−0.91%, +0.44%]
- 60 min: +0.03%, CI [−0.84%, +0.95%]

Daylight-only intervals also contained zero at every horizon.

---

### Interpretation

Extra Trees provides no evidence of a robust forecasting advantage from
adding the discrete Panchang representation beyond the continuous astronomical
controls already contained in F3.

The point estimates themselves are small and change direction across forecast
horizons. More importantly, every paired daily-block bootstrap interval spans
zero.

Therefore, under the Extra Trees model and the present forecasting protocol,
the appropriate conclusion is:

> The F4 Panchang representation does not demonstrate a statistically clear
> improvement over F3 continuous astronomical features.

This result should not be interpreted as evidence that Panchang features can
never be useful. It applies specifically to the present SOLETE dataset,
feature representation, forecast horizons, chronological split, and Extra
Trees modelling procedure.

The result will subsequently be compared with additional nonlinear models to
determine whether the F3 → F4 conclusion remains stable across modelling
families.

In [42]:
# ============================================================
# Step 41: Persist the completed Extra Trees experiment
# ============================================================
#
# Save:
#
#   1. selected validation results;
#   2. frozen hyperparameter registry;
#   3. final test metrics;
#   4. F3 -> F4 test comparison;
#   5. bootstrap summary;
#   6. complete bootstrap distributions;
#   7. timestamp-level test predictions;
#   8. machine-readable Extra Trees protocol;
#   9. result-file manifest.
#
# These files make the Extra Trees experiment independently
# recoverable before we proceed to XGBoost.
# ============================================================

import json


# ------------------------------------------------------------
# 1. Create Notebook 10 results directory
# ------------------------------------------------------------

SECONDARY_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "notebook10_secondary_ml_models"
)

SECONDARY_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Secondary-model results directory:")
print(SECONDARY_RESULTS_DIR)


# ------------------------------------------------------------
# 2. Save compact Extra Trees result tables
# ------------------------------------------------------------

extra_trees_tables_to_save = {
    "extra_trees_validation_summary.csv":
        extra_trees_validation_summary,

    "extra_trees_frozen_selection.csv":
        extra_trees_frozen_selection,

    "extra_trees_final_test_results.csv":
        extra_trees_final_test_results,

    "extra_trees_f3_f4_test_summary.csv":
        extra_trees_f3_f4_test_summary,

    "extra_trees_bootstrap_summary.csv":
        extra_trees_bootstrap_summary,
}


for filename, dataframe in (
    extra_trees_tables_to_save.items()
):

    output_path = (
        SECONDARY_RESULTS_DIR
        / filename
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )


# ------------------------------------------------------------
# 3. Save complete bootstrap distributions
# ------------------------------------------------------------
#
# There are:
#
#   3 horizons
#   × 2 scopes
#   × 10,000 bootstrap replicates
#
# = 60,000 rows.
# ------------------------------------------------------------

bootstrap_distribution_tables = []


for (
    horizon_minutes,
    scope_name,
), df_distribution in (
    extra_trees_bootstrap_distributions.items()
):

    df_temp = (
        df_distribution.copy()
    )

    df_temp.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            1,
            len(df_temp) + 1,
        ),
    )

    df_temp.insert(
        0,
        "scope",
        scope_name,
    )

    df_temp.insert(
        0,
        "horizon_minutes",
        horizon_minutes,
    )

    df_temp.insert(
        0,
        "model",
        "ExtraTreesRegressor",
    )

    bootstrap_distribution_tables.append(
        df_temp
    )


extra_trees_bootstrap_distributions_combined = (
    pd.concat(
        bootstrap_distribution_tables,
        ignore_index=True,
    )
)


assert len(
    extra_trees_bootstrap_distributions_combined
) == (
    3
    * 2
    * N_BOOTSTRAP
)


extra_trees_bootstrap_distributions_combined.to_csv(
    SECONDARY_RESULTS_DIR
    / (
        "extra_trees_"
        "bootstrap_distributions.csv.gz"
    ),
    index=False,
    compression="gzip",
)


# ------------------------------------------------------------
# 4. Save timestamp-level final test predictions
# ------------------------------------------------------------
#
# One file per horizon.
#
# Each file contains:
#
#   issue_time
#   valid_time
#   daylight
#   observed P_Solar
#   Extra Trees F0-F4 predictions
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    prediction_tables = {
        feature_family:
            extra_trees_test_predictions[
                (
                    horizon_minutes,
                    feature_family,
                )
            ]
        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ]
    }


    # Use F0 as the canonical alignment table.
    reference_table = (
        prediction_tables["F0"]
    )


    combined_predictions = pd.DataFrame(
        {
            "valid_time":
                reference_table[
                    "valid_time"
                ],

            "daylight":
                reference_table[
                    "daylight"
                ],

            "target_P_Solar_kW":
                reference_table[
                    "y_true"
                ],
        },
        index=reference_table.index,
    )


    # Verify and append each model prediction.
    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        current_table = (
            prediction_tables[
                feature_family
            ]
        )

        assert current_table.index.equals(
            reference_table.index
        )

        assert current_table[
            "valid_time"
        ].equals(
            reference_table[
                "valid_time"
            ]
        )

        assert np.allclose(
            current_table[
                "y_true"
            ].to_numpy(),
            reference_table[
                "y_true"
            ].to_numpy(),
            atol=0.0,
            rtol=0.0,
        )

        combined_predictions[
            f"ExtraTrees_{feature_family}"
        ] = (
            current_table[
                "y_pred"
            ]
        )


    combined_predictions.index.name = (
        "issue_time"
    )


    assert len(
        combined_predictions
    ) == 26_496


    combined_predictions.to_csv(
        SECONDARY_RESULTS_DIR
        / (
            "extra_trees_test_predictions_"
            f"{horizon_minutes}min.csv"
        ),
        index=True,
    )


# ------------------------------------------------------------
# 5. Save frozen selected configurations
# ------------------------------------------------------------

with open(
    SECONDARY_RESULTS_DIR
    / "extra_trees_selected_configs.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        extra_trees_selected_configs,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 6. Save Extra Trees experiment protocol
# ------------------------------------------------------------

extra_trees_protocol = {
    "notebook":
        "10_secondary_ml_models.ipynb",

    "model":
        "ExtraTreesRegressor",

    "target":
        "P_Solar[kW]",

    "resolution_minutes":
        5,

    "forecast_horizons_minutes":
        [
            15,
            30,
            60,
        ],

    "feature_counts":
        {
            family:
                int(
                    primary_protocol[
                        "feature_counts"
                    ][family]
                )
            for family in [
                "F0",
                "F1",
                "F2",
                "F3",
                "F4",
            ]
        },

    "validation_selection_metric":
        "MAE_kW",

    "hyperparameter_grid":
        {
            "n_estimators":
                [
                    300,
                ],

            "max_features":
                [
                    0.5,
                    1.0,
                ],

            "min_samples_leaf":
                [
                    1,
                    3,
                ],

            "max_depth":
                [
                    None,
                    30,
                ],

            "criterion":
                "squared_error",

            "bootstrap":
                False,

            "random_state":
                42,

            "n_jobs":
                -1,
        },

    "number_of_grid_configurations":
        8,

    "test_refit":
        "train_plus_validation",

    "test_evaluation":
        "single chronological held-out test evaluation",

    "test_rows_per_horizon":
        26_496,

    "daylight_definition":
        primary_protocol[
            "daylight_definition"
        ],

    "bootstrap":
        {
            "comparison":
                primary_protocol[
                    "bootstrap"
                ][
                    "comparison"
                ],

            "block_unit":
                primary_protocol[
                    "bootstrap"
                ][
                    "block_unit"
                ],

            "daily_blocks":
                EXPECTED_DAILY_BLOCKS,

            "replicates":
                N_BOOTSTRAP,

            "random_seed":
                BOOTSTRAP_RANDOM_SEED,

            "confidence_interval":
                primary_protocol[
                    "bootstrap"
                ][
                    "confidence_interval"
                ],

            "model_retraining":
                False,
        },
}


with open(
    SECONDARY_RESULTS_DIR
    / "extra_trees_protocol.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        extra_trees_protocol,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 7. Build a manifest of Extra Trees artifacts
# ------------------------------------------------------------

extra_trees_required_files = [
    "extra_trees_validation_summary.csv",
    "extra_trees_frozen_selection.csv",
    "extra_trees_final_test_results.csv",
    "extra_trees_f3_f4_test_summary.csv",
    "extra_trees_bootstrap_summary.csv",
    "extra_trees_bootstrap_distributions.csv.gz",
    "extra_trees_test_predictions_15min.csv",
    "extra_trees_test_predictions_30min.csv",
    "extra_trees_test_predictions_60min.csv",
    "extra_trees_selected_configs.json",
    "extra_trees_protocol.json",
]


manifest_rows = []


for filename in extra_trees_required_files:

    path = (
        SECONDARY_RESULTS_DIR
        / filename
    )

    assert path.is_file(), (
        f"Missing Extra Trees artifact: {filename}"
    )

    manifest_rows.append(
        {
            "filename":
                filename,

            "size_bytes":
                path.stat().st_size,

            "size_kb":
                path.stat().st_size
                / 1024.0,
        }
    )


extra_trees_results_manifest = (
    pd.DataFrame(
        manifest_rows
    )
)


extra_trees_results_manifest.to_csv(
    SECONDARY_RESULTS_DIR
    / "extra_trees_results_manifest.csv",
    index=False,
)


# ------------------------------------------------------------
# 8. Final persistence audit
# ------------------------------------------------------------

assert (
    SECONDARY_RESULTS_DIR
    / "extra_trees_results_manifest.csv"
).is_file()


print(
    "\nExtra Trees persistence check: PASSED"
)

print(
    "\nSaved Extra Trees artifacts:"
)

display(
    extra_trees_results_manifest
)

print(
    "\nTotal saved before manifest:",
    len(extra_trees_required_files),
)

print(
    "Manifest:",
    "extra_trees_results_manifest.csv",
)

Secondary-model results directory:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook10_secondary_ml_models

Extra Trees persistence check: PASSED

Saved Extra Trees artifacts:


,filename,size_bytes,size_kb
0,extra_trees_validation_summary.csv,1317,1.286133
1,extra_trees_frozen_selection.csv,1060,1.035156
2,extra_trees_final_test_results.csv,4275,4.174805
3,extra_trees_f3_f4_test_summary.csv,499,0.487305
4,extra_trees_bootstrap_summary.csv,1438,1.404297
5,extra_trees_bootstrap_distributions.csv.gz,1354797,1323.043945
6,extra_trees_test_predictions_15min.csv,4353061,4251.036133
7,extra_trees_test_predictions_30min.csv,4357807,4255.670898
8,extra_trees_test_predictions_60min.csv,4365499,4263.182617
9,extra_trees_selected_configs.json,3538,3.455078



Total saved before manifest: 11
Manifest: extra_trees_results_manifest.csv


## XGBoost — Frozen Validation Protocol

The next secondary-model robustness experiment uses **XGBoost
`XGBRegressor`**.

Installed environment:

- XGBoost version: 3.4.1
- execution: CPU
- tree construction method: histogram (`hist`)
- random seed: 42

The purpose of this experiment is not to perform an unrestricted
hyperparameter search. Instead, a compact hyperparameter grid is frozen
**before examining any XGBoost validation or test result**.

This preserves the same experimental discipline used for the primary
Ridge/HGBR experiment and the completed Extra Trees experiment.

---

### Data Protocol

The forecasting protocol remains unchanged.

For every horizon:

- 15 minutes
- 30 minutes
- 60 minutes

the same chronological populations and feature matrices constructed earlier
in this notebook are reused.

Feature families remain strictly nested:

$$
F0 \subset F1 \subset F2 \subset F3 \subset F4
$$

with:

| Family | Features |
|---|---:|
| F0 | 118 |
| F1 | 133 |
| F2 | 138 |
| F3 | 147 |
| F4 | 244 |

Hyperparameter selection uses **training → validation** only.

The final chronological test period remains untouched until all XGBoost
configurations have been selected and frozen.

---

### Frozen XGBoost Grid

Three capacity / learning-rate parameters are varied:

#### Number of boosting trees

$$
n_{\text{estimators}} \in \{300,\ 600\}
$$

#### Learning rate

$$
\eta \in \{0.03,\ 0.05\}
$$

#### Maximum tree depth

$$
d_{\max} \in \{4,\ 6\}
$$

The Cartesian product therefore contains:

$$
2 \times 2 \times 2 = 8
$$

frozen configurations.

---

### Fixed XGBoost Parameters

The following parameters are fixed across every experiment:

- `objective = "reg:squarederror"`
- `tree_method = "hist"`
- `subsample = 0.8`
- `colsample_bytree = 0.8`
- `min_child_weight = 1.0`
- `reg_alpha = 0.0`
- `reg_lambda = 1.0`
- `gamma = 0.0`
- `random_state = 42`
- `n_jobs = -1`
- `verbosity = 0`

No early stopping is used.

This is deliberate: every candidate has a fully specified boosting budget,
and chronological validation MAE is used only to compare the eight frozen
configurations.

---

### Model Selection Rule

For every:

$$
\text{horizon} \times \text{feature family}
$$

combination, the selected XGBoost configuration is the one with the lowest
**validation MAE**.

Validation RMSE is recorded as a secondary metric.

If two configurations have effectively identical MAE, no additional
hyperparameter search will be introduced after observing the result.

The test set will not influence:

- grid construction;
- hyperparameter selection;
- feature-family selection;
- stopping decisions.

Only after all:

$$
3 \text{ horizons}
\times
5 \text{ feature families}
=
15
$$

validation selections have been frozen will XGBoost be refitted on
training + validation and evaluated on the final test period.

---

### Central Scientific Comparison

As in the primary experiment and Extra Trees analysis, the main ablation is:

$$
F3 \rightarrow F4
$$

where:

- **F3** contains historical, Gregorian, solar, and continuous Sun–Moon
  astronomical information;
- **F4** additionally contains the fixed-vocabulary one-hot Panchang
  representation.

Therefore, the XGBoost experiment asks whether the discrete Panchang
representation contributes predictive information beyond the continuous
astronomical controls already present in F3.

The same paired daily-block bootstrap procedure used previously will be
applied only after final XGBoost test predictions have been generated.

In [43]:
# ============================================================
# Step 42: Import XGBoost and freeze the validation grid
# ============================================================
#
# IMPORTANT:
#
# This cell defines the complete XGBoost search space BEFORE
# any XGBoost validation result is observed.
#
# Grid:
#
#   n_estimators  ∈ {300, 600}
#   learning_rate ∈ {0.03, 0.05}
#   max_depth     ∈ {4, 6}
#
# Total:
#
#   2 × 2 × 2 = 8 configurations
#
# All other parameters remain fixed.
# ============================================================


from itertools import product

import xgboost as xgb
from xgboost import XGBRegressor


# ------------------------------------------------------------
# 1. Verify installed XGBoost version
# ------------------------------------------------------------

print(
    "XGBoost version:",
    xgb.__version__,
)


assert xgb.__version__ == "3.4.1"


# ------------------------------------------------------------
# 2. Frozen hyperparameter values
# ------------------------------------------------------------

XGB_N_ESTIMATORS = [
    300,
    600,
]

XGB_LEARNING_RATES = [
    0.03,
    0.05,
]

XGB_MAX_DEPTHS = [
    4,
    6,
]


# ------------------------------------------------------------
# 3. Parameters fixed for every XGBoost experiment
# ------------------------------------------------------------

XGB_FIXED_PARAMS = {
    "objective":
        "reg:squarederror",

    "tree_method":
        "hist",

    "subsample":
        0.8,

    "colsample_bytree":
        0.8,

    "min_child_weight":
        1.0,

    "reg_alpha":
        0.0,

    "reg_lambda":
        1.0,

    "gamma":
        0.0,

    "random_state":
        42,

    "n_jobs":
        -1,

    "verbosity":
        0,
}


# ------------------------------------------------------------
# 4. Construct the complete frozen Cartesian grid
# ------------------------------------------------------------

xgboost_configs = []


for (
    n_estimators,
    learning_rate,
    max_depth,
) in product(
    XGB_N_ESTIMATORS,
    XGB_LEARNING_RATES,
    XGB_MAX_DEPTHS,
):

    config = {
        "n_estimators":
            int(n_estimators),

        "learning_rate":
            float(learning_rate),

        "max_depth":
            int(max_depth),

        **XGB_FIXED_PARAMS,
    }

    xgboost_configs.append(
        config
    )


# ------------------------------------------------------------
# 5. Structural checks
# ------------------------------------------------------------

assert len(
    xgboost_configs
) == 8


assert {
    config["n_estimators"]
    for config in xgboost_configs
} == {
    300,
    600,
}


assert {
    config["learning_rate"]
    for config in xgboost_configs
} == {
    0.03,
    0.05,
}


assert {
    config["max_depth"]
    for config in xgboost_configs
} == {
    4,
    6,
}


# ------------------------------------------------------------
# 6. Display the exact frozen grid
# ------------------------------------------------------------

xgboost_grid_table = pd.DataFrame(
    [
        {
            "configuration":
                i,

            "n_estimators":
                config[
                    "n_estimators"
                ],

            "learning_rate":
                config[
                    "learning_rate"
                ],

            "max_depth":
                config[
                    "max_depth"
                ],

            "subsample":
                config[
                    "subsample"
                ],

            "colsample_bytree":
                config[
                    "colsample_bytree"
                ],

            "min_child_weight":
                config[
                    "min_child_weight"
                ],

            "reg_alpha":
                config[
                    "reg_alpha"
                ],

            "reg_lambda":
                config[
                    "reg_lambda"
                ],

            "gamma":
                config[
                    "gamma"
                ],

            "tree_method":
                config[
                    "tree_method"
                ],
        }

        for i, config in enumerate(
            xgboost_configs,
            start=1,
        )
    ]
)


print(
    "\nFrozen XGBoost validation grid:\n"
)

display(
    xgboost_grid_table
)


print(
    "Number of frozen configurations:",
    len(xgboost_configs),
)

print(
    "\nXGBoost validation grid: FROZEN"
)

print(
    "No XGBoost validation or test result "
    "has been examined."
)

XGBoost version: 3.4.1

Frozen XGBoost validation grid:



,configuration,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,tree_method
0,1,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,hist
1,2,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,hist
2,3,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,hist
3,4,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,hist
4,5,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,hist
5,6,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,hist
6,7,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,hist
7,8,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,hist


Number of frozen configurations: 8

XGBoost validation grid: FROZEN
No XGBoost validation or test result has been examined.


In [44]:
# ============================================================
# Step 43: Reusable XGBoost validation helper
#          + first 60-minute F0 sanity experiment
# ============================================================
#
# This helper:
#
#   1. uses TRAIN only for fitting;
#   2. uses VALIDATION only for model selection;
#   3. never accesses TEST;
#   4. evaluates the frozen 8-configuration XGBoost grid;
#   5. selects the configuration with minimum validation MAE;
#   6. records validation RMSE as a secondary metric.
#
# We first apply it only to:
#
#       horizon = 60 minutes
#       family  = F0
#
# before running the remaining XGBoost experiments.
# ============================================================


# ------------------------------------------------------------
# 1. Define reusable validation-grid helper
# ------------------------------------------------------------

def run_xgboost_validation_grid(
    horizon_minutes: int,
    feature_family: str,
) -> tuple[pd.DataFrame, float]:

    """
    Run the frozen XGBoost validation grid for one
    horizon × feature-family combination.

    Model selection is based exclusively on chronological
    validation MAE.

    The final TEST partition is not accessed.
    """

    # --------------------------------------------------------
    # Retrieve modelling objects
    # --------------------------------------------------------

    X = (
        feature_matrices[
            horizon_minutes
        ][
            feature_family
        ]
    )

    y = targets[
        horizon_minutes
    ]

    split = split_labels[
        horizon_minutes
    ]


    # --------------------------------------------------------
    # Chronological train / validation masks
    # --------------------------------------------------------

    train_mask = (
        split == "train"
    )

    validation_mask = (
        split == "validation"
    )


    # --------------------------------------------------------
    # Convert to float32 NumPy arrays
    #
    # This reduces memory use and is appropriate for
    # XGBoost's histogram tree implementation.
    # --------------------------------------------------------

    X_train = (
        X.loc[
            train_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    X_validation = (
        X.loc[
            validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    y_train = (
        y.loc[
            train_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    y_validation = (
        y.loc[
            validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    expected_feature_count = (
        primary_protocol[
            "feature_counts"
        ][
            feature_family
        ]
    )


    assert (
        X_train.shape[1]
        == expected_feature_count
    )

    assert (
        X_validation.shape[1]
        == expected_feature_count
    )

    assert np.isfinite(
        X_train
    ).all()

    assert np.isfinite(
        X_validation
    ).all()

    assert np.isfinite(
        y_train
    ).all()

    assert np.isfinite(
        y_validation
    ).all()


    print(
        "\nXGBoost validation dataset:"
    )

    print(
        "  Horizon:",
        horizon_minutes,
        "minutes",
    )

    print(
        "  Feature family:",
        feature_family,
    )

    print(
        "  Train:",
        X_train.shape,
    )

    print(
        "  Validation:",
        X_validation.shape,
    )

    print(
        "  Frozen configurations:",
        len(
            xgboost_configs
        ),
    )


    # --------------------------------------------------------
    # Run frozen grid
    # --------------------------------------------------------

    result_rows = []

    grid_start_time = (
        time.perf_counter()
    )


    for config_number, config in enumerate(
        xgboost_configs,
        start=1,
    ):

        print(
            "\n"
            f"Configuration "
            f"{config_number}/"
            f"{len(xgboost_configs)}"
        )

        print(
            "  n_estimators     =",
            config[
                "n_estimators"
            ],
        )

        print(
            "  learning_rate    =",
            config[
                "learning_rate"
            ],
        )

        print(
            "  max_depth        =",
            config[
                "max_depth"
            ],
        )


        # ----------------------------------------------------
        # Build model from the frozen configuration
        # ----------------------------------------------------

        model = XGBRegressor(
            **config
        )


        fit_start_time = (
            time.perf_counter()
        )


        # ----------------------------------------------------
        # Fit using TRAIN only
        # ----------------------------------------------------

        model.fit(
            X_train,
            y_train,
        )


        # ----------------------------------------------------
        # Predict chronological VALIDATION only
        # ----------------------------------------------------

        validation_predictions = (
            model.predict(
                X_validation
            )
        )


        fit_predict_seconds = (
            time.perf_counter()
            - fit_start_time
        )


        # ----------------------------------------------------
        # Validation metrics
        # ----------------------------------------------------

        validation_mae = (
            mean_absolute_error(
                y_validation,
                validation_predictions,
            )
        )

        validation_rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                validation_predictions,
            )
        )


        print(
            f"  validation MAE  = "
            f"{validation_mae:.6f}"
        )

        print(
            f"  validation RMSE = "
            f"{validation_rmse:.6f}"
        )

        print(
            f"  runtime         = "
            f"{fit_predict_seconds:.1f} s"
        )


        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        result_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_estimators":
                    config[
                        "n_estimators"
                    ],

                "learning_rate":
                    config[
                        "learning_rate"
                    ],

                "max_depth":
                    config[
                        "max_depth"
                    ],

                "subsample":
                    config[
                        "subsample"
                    ],

                "colsample_bytree":
                    config[
                        "colsample_bytree"
                    ],

                "min_child_weight":
                    config[
                        "min_child_weight"
                    ],

                "reg_alpha":
                    config[
                        "reg_alpha"
                    ],

                "reg_lambda":
                    config[
                        "reg_lambda"
                    ],

                "gamma":
                    config[
                        "gamma"
                    ],

                "validation_mae":
                    validation_mae,

                "validation_rmse":
                    validation_rmse,

                "fit_predict_seconds":
                    fit_predict_seconds,
            }
        )


        # Avoid retaining fitted boosters unnecessarily.
        del model
        del validation_predictions

        gc.collect()


    # --------------------------------------------------------
    # Total grid runtime
    # --------------------------------------------------------

    total_grid_seconds = (
        time.perf_counter()
        - grid_start_time
    )


    # --------------------------------------------------------
    # Sort:
    #
    # primary criterion   = validation MAE
    # secondary criterion = validation RMSE
    # --------------------------------------------------------

    results_df = (
        pd.DataFrame(
            result_rows
        )
        .sort_values(
            [
                "validation_mae",
                "validation_rmse",
            ],
            ascending=True,
        )
        .reset_index(
            drop=True
        )
    )


    return (
        results_df,
        total_grid_seconds,
    )


# ============================================================
# 2. First sanity run:
#
#       60-minute horizon
#       F0
# ============================================================

xgboost_60_f0_df, xgboost_60_f0_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=60,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 3. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f0_60 = (
    xgboost_60_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 4. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 60-minute F0 validation results"
)

print(
    "============================================\n"
)


display(
    xgboost_60_f0_df
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f0_60[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f0_60[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f0_60[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f0_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f0_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F0 grid runtime:"
)

print(
    f"  {xgboost_60_f0_seconds:.1f} seconds"
)

print(
    f"  {xgboost_60_f0_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 60-minute F0 sanity experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 60 minutes
  Feature family: F0
  Train: (87252, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.458549
  validation RMSE = 0.824310
  runtime         = 2.0 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.452187
  validation RMSE = 0.817598
  runtime         = 2.8 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.464895
  validation RMSE = 0.833309
  runtime         = 1.7 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.469174
  validation RMSE = 0.844916
  runtime         = 2.7 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.469694
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,60,F0,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.452187,0.817598,2.788803
1,60,F0,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.458549,0.824310,1.961503
2,60,F0,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.464895,0.833309,1.739914
3,60,F0,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.464979,0.835265,5.246592
4,60,F0,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.469174,0.844916,2.744837
5,60,F0,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.469694,0.840357,3.191565
6,60,F0,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.481195,0.851030,3.222117
7,60,F0,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.485540,0.866589,5.060827



Selected 60-minute F0 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 6
  validation MAE: 0.4521867334842682
  validation RMSE: 0.8175975376235775

Total 60-minute F0 grid runtime:
  26.3 seconds
  0.44 minutes

XGBoost 60-minute F0 sanity experiment: PASSED


In [45]:
# ============================================================
# Step 44: XGBoost validation
#          60-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + Gregorian / conventional calendar-time features
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_60_f1_df, xgboost_60_f1_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=60,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f1_60 = (
    xgboost_60_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 60-minute F1 validation results"
)

print(
    "============================================\n"
)


display(
    xgboost_60_f1_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f1_60[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f1_60[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f1_60[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f1_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f1_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_60_mae = float(
    best_xgboost_f0_60[
        "validation_mae"
    ]
)

f1_60_mae = float(
    best_xgboost_f1_60[
        "validation_mae"
    ]
)

f0_to_f1_60_pct = (
    (f0_60_mae - f1_60_mae)
    / f0_60_mae
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_60_mae:.6f}"
)

print(
    f"  F1 MAE: {f1_60_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_60_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F1 grid runtime:"
)

print(
    f"  {xgboost_60_f1_seconds:.1f} seconds"
)

print(
    f"  {xgboost_60_f1_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 60-minute F1 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 60 minutes
  Feature family: F1
  Train: (87252, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.371761
  validation RMSE = 0.719995
  runtime         = 2.1 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.374633
  validation RMSE = 0.725187
  runtime         = 3.0 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.390407
  validation RMSE = 0.732701
  runtime         = 1.8 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.396308
  validation RMSE = 0.747396
  runtime         = 2.8 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.381652
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,60,F1,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.371761,0.719995,2.057756
1,60,F1,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.374633,0.725187,3.034599
2,60,F1,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.381652,0.725524,3.349488
3,60,F1,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.390407,0.732701,1.770964
4,60,F1,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.391542,0.740972,5.415486
5,60,F1,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.396308,0.747396,2.837019
6,60,F1,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.407818,0.753440,3.303161
7,60,F1,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.413314,0.770588,5.372706



Selected 60-minute F1 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.3717612326145172
  validation RMSE: 0.7199948766976441

F0 -> F1 validation comparison:
  F0 MAE: 0.452187
  F1 MAE: 0.371761
  Relative MAE improvement: +17.7859%

Total 60-minute F1 grid runtime:
  27.4 seconds
  0.46 minutes

XGBoost 60-minute F1 experiment: PASSED


In [46]:
# ============================================================
# Step 45: XGBoost validation
#          60-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + deterministic solar-geometry features
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_60_f2_df, xgboost_60_f2_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=60,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f2_60 = (
    xgboost_60_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 60-minute F2 validation results"
)

print(
    "============================================\n"
)


display(
    xgboost_60_f2_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f2_60[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f2_60[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f2_60[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f2_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f2_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_60_mae = float(
    best_xgboost_f1_60[
        "validation_mae"
    ]
)

f2_60_mae = float(
    best_xgboost_f2_60[
        "validation_mae"
    ]
)

f1_to_f2_60_pct = (
    (f1_60_mae - f2_60_mae)
    / f1_60_mae
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_60_mae:.6f}"
)

print(
    f"  F2 MAE: {f2_60_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_60_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F2 grid runtime:"
)

print(
    f"  {xgboost_60_f2_seconds:.1f} seconds"
)

print(
    f"  {xgboost_60_f2_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 60-minute F2 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 60 minutes
  Feature family: F2
  Train: (87252, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.380626
  validation RMSE = 0.729340
  runtime         = 2.0 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.382156
  validation RMSE = 0.729708
  runtime         = 3.0 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.395681
  validation RMSE = 0.735643
  runtime         = 1.9 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.397278
  validation RMSE = 0.757322
  runtime         = 2.8 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.391364
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,60,F2,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.380626,0.729340,2.021925
1,60,F2,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.382156,0.729708,3.000073
2,60,F2,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.391364,0.738609,3.577009
3,60,F2,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.395681,0.735643,1.850762
4,60,F2,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.397278,0.757322,2.844433
5,60,F2,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.402332,0.752894,5.747005
6,60,F2,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.413513,0.758022,3.501033
7,60,F2,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.419233,0.782290,5.491444



Selected 60-minute F2 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.3806261420249939
  validation RMSE: 0.7293401965783668

F1 -> F2 validation comparison:
  F1 MAE: 0.371761
  F2 MAE: 0.380626
  Relative MAE improvement: -2.3846%

Total 60-minute F2 grid runtime:
  28.3 seconds
  0.47 minutes

XGBoost 60-minute F2 experiment: PASSED


In [47]:
# ============================================================
# Step 46: XGBoost validation
#          60-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + continuous lunar / Sun-Moon astronomical features
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_60_f3_df, xgboost_60_f3_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=60,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f3_60 = (
    xgboost_60_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 60-minute F3 validation results"
)

print(
    "============================================\n"
)


display(
    xgboost_60_f3_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f3_60[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f3_60[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f3_60[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f3_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f3_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_60_mae = float(
    best_xgboost_f2_60[
        "validation_mae"
    ]
)

f3_60_mae = float(
    best_xgboost_f3_60[
        "validation_mae"
    ]
)

f2_to_f3_60_pct = (
    (f2_60_mae - f3_60_mae)
    / f2_60_mae
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_60_mae:.6f}"
)

print(
    f"  F3 MAE: {f3_60_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_60_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F3 grid runtime:"
)

print(
    f"  {xgboost_60_f3_seconds:.1f} seconds"
)

print(
    f"  {xgboost_60_f3_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 60-minute F3 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 60 minutes
  Feature family: F3
  Train: (87252, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.387382
  validation RMSE = 0.732092
  runtime         = 2.2 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.382230
  validation RMSE = 0.729520
  runtime         = 3.2 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.393981
  validation RMSE = 0.734107
  runtime         = 2.0 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.403948
  validation RMSE = 0.750463
  runtime         = 3.3 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.396153
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,60,F3,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.382230,0.729520,3.249732
1,60,F3,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.387382,0.732092,2.176120
2,60,F3,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.393981,0.734107,2.009027
3,60,F3,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.396153,0.737865,3.745008
4,60,F3,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.403948,0.750463,3.328943
5,60,F3,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.405557,0.750781,5.841696
6,60,F3,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.411896,0.750813,3.547037
7,60,F3,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.428223,0.772844,5.796764



Selected 60-minute F3 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 6
  validation MAE: 0.3822299540042877
  validation RMSE: 0.7295199265114954

F2 -> F3 validation comparison:
  F2 MAE: 0.380626
  F3 MAE: 0.382230
  Relative MAE improvement: -0.4214%

Total 60-minute F3 grid runtime:
  30.0 seconds
  0.50 minutes

XGBoost 60-minute F3 experiment: PASSED


In [48]:
# ============================================================
# Step 47: XGBoost validation
#          60-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + 97 frozen Panchang one-hot indicators
#
# This gives the central comparison:
#
#       F3 -> F4
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_60_f4_df, xgboost_60_f4_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=60,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f4_60 = (
    xgboost_60_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 60-minute F4 validation results"
)

print(
    "============================================\n"
)


display(
    xgboost_60_f4_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f4_60[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f4_60[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f4_60[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f4_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f4_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Central F3 -> F4 validation comparison
# ------------------------------------------------------------

f3_60_mae = float(
    best_xgboost_f3_60[
        "validation_mae"
    ]
)

f4_60_mae = float(
    best_xgboost_f4_60[
        "validation_mae"
    ]
)

f3_to_f4_60_absolute_change = (
    f3_60_mae
    - f4_60_mae
)

f3_to_f4_60_pct = (
    f3_to_f4_60_absolute_change
    / f3_60_mae
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_60_mae:.6f}"
)

print(
    f"  F4 MAE: {f4_60_mae:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_60_absolute_change:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_60_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F4 grid runtime:"
)

print(
    f"  {xgboost_60_f4_seconds:.1f} seconds"
)

print(
    f"  {xgboost_60_f4_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 60-minute F4 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 60 minutes
  Feature family: F4
  Train: (87252, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.388357
  validation RMSE = 0.735461
  runtime         = 2.7 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.388624
  validation RMSE = 0.738455
  runtime         = 3.7 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.401544
  validation RMSE = 0.743535
  runtime         = 2.5 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.401894
  validation RMSE = 0.751160
  runtime         = 3.8 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.397709
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,60,F4,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.388357,0.735461,2.658668
1,60,F4,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.388624,0.738455,3.706028
2,60,F4,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.397709,0.739852,4.443942
3,60,F4,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.401544,0.743535,2.490568
4,60,F4,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.401894,0.751160,3.806655
5,60,F4,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.411155,0.760502,6.678624
6,60,F4,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.420919,0.759808,4.451507
7,60,F4,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.421790,0.770410,6.830372



Selected 60-minute F4 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.388357013463974
  validation RMSE: 0.7354611015043687

F3 -> F4 validation comparison:
  F3 MAE: 0.382230
  F4 MAE: 0.388357
  Absolute MAE reduction: -0.006127
  Relative MAE improvement: -1.6030%

Total 60-minute F4 grid runtime:
  35.4 seconds
  0.59 minutes

XGBoost 60-minute F4 experiment: PASSED


In [49]:
# ============================================================
# Step 48: Summarize the complete 60-minute XGBoost
#          F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect selected configurations
# ------------------------------------------------------------

best_rows_xgb_60 = {
    "F0": best_xgboost_f0_60,
    "F1": best_xgboost_f1_60,
    "F2": best_xgboost_f2_60,
    "F3": best_xgboost_f3_60,
    "F4": best_xgboost_f4_60,
}


summary_rows_xgb_60 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_xgb_60[
        feature_family
    ]

    summary_rows_xgb_60.append(
        {
            "horizon_minutes":
                60,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][feature_family],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    row[
                        "learning_rate"
                    ]
                ),

            "max_depth":
                int(
                    row[
                        "max_depth"
                    ]
                ),

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


xgboost_60_summary = pd.DataFrame(
    summary_rows_xgb_60
)


# ------------------------------------------------------------
# 2. Add incremental MAE improvement
#
# Positive:
#     new feature family reduced MAE.
#
# Negative:
#     new feature family increased MAE.
# ------------------------------------------------------------

xgboost_60_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(xgboost_60_summary),
):

    previous_mae = (
        xgboost_60_summary.loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = (
        xgboost_60_summary.loc[
            i,
            "validation_mae",
        ]
    )

    xgboost_60_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (previous_mae - current_mae)
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Verify central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_xgb_60 = (
    xgboost_60_summary.loc[
        xgboost_60_summary[
            "feature_family"
        ] == "F3",
        "validation_mae",
    ]
    .iloc[0]
)

f4_summary_mae_xgb_60 = (
    xgboost_60_summary.loc[
        xgboost_60_summary[
            "feature_family"
        ] == "F4",
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_xgb_60 = (
    (
        f3_summary_mae_xgb_60
        - f4_summary_mae_xgb_60
    )
    / f3_summary_mae_xgb_60
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_xgb_60,
    -1.6030,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete summary
# ------------------------------------------------------------

print(
    "XGBoost — 60-minute validation summary\n"
)

display(
    xgboost_60_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)

for i in range(
    1,
    len(xgboost_60_summary),
):

    previous_family = (
        xgboost_60_summary.loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        xgboost_60_summary.loc[
            i,
            "feature_family",
        ]
    )

    change = (
        xgboost_60_summary.loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )

    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_xgb_60 = (
    xgboost_60_summary
    .sort_values(
        "validation_mae"
    )
    .iloc[0]
)


print(
    "\nLowest 60-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_xgb_60[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_xgb_60[
        "validation_mae"
    ],
)


print(
    "\nCentral XGBoost comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_xgb_60:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_xgb_60:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_xgb_60:+.4f}%"
)


print(
    "\nXGBoost 60-minute summary: PASSED"
)

XGBoost — 60-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,60,F0,118,300,0.03,6,0.452187,0.817598,NaN
1,60,F1,133,300,0.03,4,0.371761,0.719995,17.785905
2,60,F2,138,300,0.03,4,0.380626,0.729340,-2.384571
3,60,F3,147,300,0.03,6,0.382230,0.729520,-0.421361
4,60,F4,244,300,0.03,4,0.388357,0.735461,-1.602977


Incremental validation MAE changes:
  F0 -> F1: +17.7859%
  F1 -> F2: -2.3846%
  F2 -> F3: -0.4214%
  F3 -> F4: -1.6030%

Lowest 60-minute validation MAE:
  Feature family: F1
  MAE: 0.3717612326145172

Central XGBoost comparison:
  F3 MAE: 0.382230
  F4 MAE: 0.388357
  F3 -> F4 change: -1.6030%

XGBoost 60-minute summary: PASSED


In [50]:
# ============================================================
# Step 49: XGBoost validation
#          30-minute horizon + F0
# ============================================================
#
# F0 contains:
#
#   - historical observed measurements through issue time t;
#   - recent lags;
#   - 24-hour PV lag.
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_30_f0_df, xgboost_30_f0_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=30,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f0_30 = (
    xgboost_30_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 30-minute F0 validation results"
)

print(
    "============================================\n"
)


display(
    xgboost_30_f0_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f0_30[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f0_30[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f0_30[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f0_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f0_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F0 grid runtime:"
)

print(
    f"  {xgboost_30_f0_seconds:.1f} seconds"
)

print(
    f"  {xgboost_30_f0_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 30-minute F0 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 30 minutes
  Feature family: F0
  Train: (87258, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.358788
  validation RMSE = 0.703574
  runtime         = 1.7 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.361243
  validation RMSE = 0.708163
  runtime         = 2.8 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.360436
  validation RMSE = 0.708050
  runtime         = 1.8 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.374200
  validation RMSE = 0.725173
  runtime         = 2.5 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.366175
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,30,F0,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.358788,0.703574,1.720278
1,30,F0,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.360436,0.708050,1.761859
2,30,F0,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.361243,0.708163,2.776345
3,30,F0,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.366175,0.714601,3.094338
4,30,F0,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.371370,0.724734,3.098400
5,30,F0,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.374200,0.725173,2.544182
6,30,F0,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.376291,0.730496,5.249542
7,30,F0,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.390691,0.750733,5.174755



Selected 30-minute F0 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.358788400888443
  validation RMSE: 0.7035743760862567

Total 30-minute F0 grid runtime:
  25.7 seconds
  0.43 minutes

XGBoost 30-minute F0 experiment: PASSED


In [51]:
# ============================================================
# Step 50: XGBoost validation
#          30-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + Gregorian / conventional calendar-time features
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_30_f1_df, xgboost_30_f1_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=30,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f1_30 = (
    xgboost_30_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 30-minute F1 validation results"
)

print(
    "============================================\n"
)


display(
    xgboost_30_f1_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f1_30[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f1_30[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f1_30[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f1_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f1_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_30_mae = float(
    best_xgboost_f0_30[
        "validation_mae"
    ]
)

f1_30_mae = float(
    best_xgboost_f1_30[
        "validation_mae"
    ]
)

f0_to_f1_30_pct = (
    (f0_30_mae - f1_30_mae)
    / f0_30_mae
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_30_mae:.6f}"
)

print(
    f"  F1 MAE: {f1_30_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_30_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F1 grid runtime:"
)

print(
    f"  {xgboost_30_f1_seconds:.1f} seconds"
)

print(
    f"  {xgboost_30_f1_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 30-minute F1 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 30 minutes
  Feature family: F1
  Train: (87258, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.318399
  validation RMSE = 0.647551
  runtime         = 2.1 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.321266
  validation RMSE = 0.655865
  runtime         = 3.0 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.322161
  validation RMSE = 0.653669
  runtime         = 1.7 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.328157
  validation RMSE = 0.666074
  runtime         = 2.8 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.331060
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,30,F1,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.318399,0.647551,2.052266
1,30,F1,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.321266,0.655865,2.974232
2,30,F1,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.322161,0.653669,1.749517
3,30,F1,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.328157,0.666074,2.753257
4,30,F1,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.331060,0.659441,3.366943
5,30,F1,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.341201,0.676877,3.252610
6,30,F1,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.345204,0.691593,5.388726
7,30,F1,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.354261,0.699483,5.184131



Selected 30-minute F1 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.3183993697166443
  validation RMSE: 0.6475514022204607

F0 -> F1 validation comparison:
  F0 MAE: 0.358788
  F1 MAE: 0.318399
  Relative MAE improvement: +11.2571%

Total 30-minute F1 grid runtime:
  27.1 seconds
  0.45 minutes

XGBoost 30-minute F1 experiment: PASSED


In [52]:
# ============================================================
# Step 51: XGBoost validation
#          30-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + deterministic solar-geometry features
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_30_f2_df, xgboost_30_f2_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=30,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f2_30 = (
    xgboost_30_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 30-minute F2 validation results"
)

print(
    "============================================\n"
)


display(
    xgboost_30_f2_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f2_30[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f2_30[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f2_30[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f2_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f2_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_30_mae = float(
    best_xgboost_f1_30[
        "validation_mae"
    ]
)

f2_30_mae = float(
    best_xgboost_f2_30[
        "validation_mae"
    ]
)

f1_to_f2_30_pct = (
    (f1_30_mae - f2_30_mae)
    / f1_30_mae
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_30_mae:.6f}"
)

print(
    f"  F2 MAE: {f2_30_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_30_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F2 grid runtime:"
)

print(
    f"  {xgboost_30_f2_seconds:.1f} seconds"
)

print(
    f"  {xgboost_30_f2_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 30-minute F2 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 30 minutes
  Feature family: F2
  Train: (87258, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.318581
  validation RMSE = 0.646225
  runtime         = 2.0 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.316794
  validation RMSE = 0.650841
  runtime         = 3.0 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.316852
  validation RMSE = 0.647069
  runtime         = 2.0 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.330549
  validation RMSE = 0.666067
  runtime         = 2.9 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.330618
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,30,F2,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.316794,0.650841,2.984202
1,30,F2,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.316852,0.647069,1.977276
2,30,F2,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.318581,0.646225,1.998988
3,30,F2,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.330549,0.666067,2.925255
4,30,F2,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.330618,0.656745,3.516206
5,30,F2,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.336232,0.670356,3.300969
6,30,F2,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.340612,0.679142,5.383360
7,30,F2,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.352641,0.695805,5.287278



Selected 30-minute F2 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 6
  validation MAE: 0.3167944848537445
  validation RMSE: 0.6508409241499455

F1 -> F2 validation comparison:
  F1 MAE: 0.318399
  F2 MAE: 0.316794
  Relative MAE improvement: +0.5040%

Total 30-minute F2 grid runtime:
  27.7 seconds
  0.46 minutes

XGBoost 30-minute F2 experiment: PASSED


In [53]:
# ============================================================
# Step 52: XGBoost validation
#          30-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + continuous lunar / Sun-Moon astronomical features
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_30_f3_df, xgboost_30_f3_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=30,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f3_30 = (
    xgboost_30_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 30-minute F3 validation results"
)

print(
    "============================================\n"
)


display(
    xgboost_30_f3_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f3_30[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f3_30[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f3_30[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f3_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f3_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_30_mae = float(
    best_xgboost_f2_30[
        "validation_mae"
    ]
)

f3_30_mae = float(
    best_xgboost_f3_30[
        "validation_mae"
    ]
)

f2_to_f3_30_pct = (
    (f2_30_mae - f3_30_mae)
    / f2_30_mae
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_30_mae:.6f}"
)

print(
    f"  F3 MAE: {f3_30_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_30_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F3 grid runtime:"
)

print(
    f"  {xgboost_30_f3_seconds:.1f} seconds"
)

print(
    f"  {xgboost_30_f3_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 30-minute F3 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 30 minutes
  Feature family: F3
  Train: (87258, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.326521
  validation RMSE = 0.652593
  runtime         = 2.1 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.322430
  validation RMSE = 0.651366
  runtime         = 3.1 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.329115
  validation RMSE = 0.656970
  runtime         = 1.9 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.329998
  validation RMSE = 0.659585
  runtime         = 3.1 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.338427
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,30,F3,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.322430,0.651366,3.058343
1,30,F3,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.326521,0.652593,2.075693
2,30,F3,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.329115,0.656970,1.893766
3,30,F3,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.329998,0.659585,3.100118
4,30,F3,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.338427,0.665045,3.603980
5,30,F3,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.340630,0.674141,5.924240
6,30,F3,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.342791,0.673537,3.468488
7,30,F3,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.357045,0.692794,5.565435



Selected 30-minute F3 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 6
  validation MAE: 0.3224296569824219
  validation RMSE: 0.6513663181831015

F2 -> F3 validation comparison:
  F2 MAE: 0.316794
  F3 MAE: 0.322430
  Relative MAE improvement: -1.7788%

Total 30-minute F3 grid runtime:
  29.0 seconds
  0.48 minutes

XGBoost 30-minute F3 experiment: PASSED


In [54]:
# ============================================================
# Step 53: XGBoost validation
#          30-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + 97 frozen Panchang one-hot indicators
#
# This gives the central:
#
#       F3 -> F4
#
# comparison for the 30-minute horizon.
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_30_f4_df, xgboost_30_f4_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=30,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f4_30 = (
    xgboost_30_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 30-minute F4 validation results"
)

print(
    "============================================\n"
)

display(
    xgboost_30_f4_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f4_30[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f4_30[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f4_30[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f4_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f4_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Central F3 -> F4 validation comparison
# ------------------------------------------------------------

f3_30_mae = float(
    best_xgboost_f3_30[
        "validation_mae"
    ]
)

f4_30_mae = float(
    best_xgboost_f4_30[
        "validation_mae"
    ]
)

f3_to_f4_30_absolute_change = (
    f3_30_mae
    - f4_30_mae
)

f3_to_f4_30_pct = (
    f3_to_f4_30_absolute_change
    / f3_30_mae
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_30_mae:.6f}"
)

print(
    f"  F4 MAE: {f4_30_mae:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_30_absolute_change:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_30_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F4 grid runtime:"
)

print(
    f"  {xgboost_30_f4_seconds:.1f} seconds"
)

print(
    f"  {xgboost_30_f4_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 30-minute F4 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 30 minutes
  Feature family: F4
  Train: (87258, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.324539
  validation RMSE = 0.653677
  runtime         = 2.7 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.326037
  validation RMSE = 0.655345
  runtime         = 3.6 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.333076
  validation RMSE = 0.662391
  runtime         = 2.3 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.349092
  validation RMSE = 0.690034
  runtime         = 3.4 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.335333
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,30,F4,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.324539,0.653677,2.729540
1,30,F4,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.326037,0.655345,3.593804
2,30,F4,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.333076,0.662391,2.322903
3,30,F4,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.335333,0.664773,4.520145
4,30,F4,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.346683,0.682321,6.790695
5,30,F4,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.349092,0.690034,3.419184
6,30,F4,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.355574,0.691106,4.719151
7,30,F4,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.364987,0.709536,6.613162



Selected 30-minute F4 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.3245388865470886
  validation RMSE: 0.6536768694395374

F3 -> F4 validation comparison:
  F3 MAE: 0.322430
  F4 MAE: 0.324539
  Absolute MAE reduction: -0.002109
  Relative MAE improvement: -0.6542%

Total 30-minute F4 grid runtime:
  35.0 seconds
  0.58 minutes

XGBoost 30-minute F4 experiment: PASSED


In [55]:
# ============================================================
# Step 54: Summarize the complete 30-minute XGBoost
#          F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect selected configurations
# ------------------------------------------------------------

best_rows_xgb_30 = {
    "F0": best_xgboost_f0_30,
    "F1": best_xgboost_f1_30,
    "F2": best_xgboost_f2_30,
    "F3": best_xgboost_f3_30,
    "F4": best_xgboost_f4_30,
}


summary_rows_xgb_30 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_xgb_30[
        feature_family
    ]

    summary_rows_xgb_30.append(
        {
            "horizon_minutes":
                30,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][feature_family],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    row[
                        "learning_rate"
                    ]
                ),

            "max_depth":
                int(
                    row[
                        "max_depth"
                    ]
                ),

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


xgboost_30_summary = pd.DataFrame(
    summary_rows_xgb_30
)


# ------------------------------------------------------------
# 2. Add incremental MAE improvement
# ------------------------------------------------------------

xgboost_30_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(xgboost_30_summary),
):

    previous_mae = (
        xgboost_30_summary.loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = (
        xgboost_30_summary.loc[
            i,
            "validation_mae",
        ]
    )

    xgboost_30_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (previous_mae - current_mae)
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Verify central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_xgb_30 = (
    xgboost_30_summary.loc[
        xgboost_30_summary[
            "feature_family"
        ] == "F3",
        "validation_mae",
    ]
    .iloc[0]
)

f4_summary_mae_xgb_30 = (
    xgboost_30_summary.loc[
        xgboost_30_summary[
            "feature_family"
        ] == "F4",
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_xgb_30 = (
    (
        f3_summary_mae_xgb_30
        - f4_summary_mae_xgb_30
    )
    / f3_summary_mae_xgb_30
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_xgb_30,
    -0.6542,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete summary
# ------------------------------------------------------------

print(
    "XGBoost — 30-minute validation summary\n"
)

display(
    xgboost_30_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)

for i in range(
    1,
    len(xgboost_30_summary),
):

    previous_family = (
        xgboost_30_summary.loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        xgboost_30_summary.loc[
            i,
            "feature_family",
        ]
    )

    change = (
        xgboost_30_summary.loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )

    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_xgb_30 = (
    xgboost_30_summary
    .sort_values(
        "validation_mae"
    )
    .iloc[0]
)


print(
    "\nLowest 30-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_xgb_30[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_xgb_30[
        "validation_mae"
    ],
)


print(
    "\nCentral XGBoost comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_xgb_30:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_xgb_30:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_xgb_30:+.4f}%"
)


print(
    "\nXGBoost 30-minute summary: PASSED"
)

XGBoost — 30-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,30,F0,118,300,0.03,4,0.358788,0.703574,NaN
1,30,F1,133,300,0.03,4,0.318399,0.647551,11.257062
2,30,F2,138,300,0.03,6,0.316794,0.650841,0.504048
3,30,F3,147,300,0.03,6,0.322430,0.651366,-1.778810
4,30,F4,244,300,0.03,4,0.324539,0.653677,-0.654167


Incremental validation MAE changes:
  F0 -> F1: +11.2571%
  F1 -> F2: +0.5040%
  F2 -> F3: -1.7788%
  F3 -> F4: -0.6542%

Lowest 30-minute validation MAE:
  Feature family: F2
  MAE: 0.3167944848537445

Central XGBoost comparison:
  F3 MAE: 0.322430
  F4 MAE: 0.324539
  F3 -> F4 change: -0.6542%

XGBoost 30-minute summary: PASSED


In [56]:
# ============================================================
# Step 55: XGBoost validation
#          15-minute horizon + F0
# ============================================================
#
# F0 contains:
#
#   - historical observed measurements through issue time t;
#   - recent lags;
#   - 24-hour PV lag.
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_15_f0_df, xgboost_15_f0_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=15,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f0_15 = (
    xgboost_15_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 15-minute F0 validation results"
)

print(
    "============================================\n"
)


display(
    xgboost_15_f0_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f0_15[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f0_15[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f0_15[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f0_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f0_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F0 grid runtime:"
)

print(
    f"  {xgboost_15_f0_seconds:.1f} seconds"
)

print(
    f"  {xgboost_15_f0_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 15-minute F0 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 15 minutes
  Feature family: F0
  Train: (87261, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.287141
  validation RMSE = 0.609990
  runtime         = 1.8 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.292117
  validation RMSE = 0.615019
  runtime         = 2.7 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.293979
  validation RMSE = 0.618041
  runtime         = 1.7 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.311148
  validation RMSE = 0.644166
  runtime         = 2.6 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.291541
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,15,F0,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.287141,0.609990,1.825917
1,15,F0,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.291541,0.617935,3.223539
2,15,F0,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.292117,0.615019,2.746225
3,15,F0,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.293979,0.618041,1.722291
4,15,F0,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.304622,0.632620,4.855243
5,15,F0,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.305314,0.637868,2.910694
6,15,F0,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.311148,0.644166,2.628681
7,15,F0,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.322505,0.660011,4.826359



Selected 15-minute F0 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.287141352891922
  validation RMSE: 0.6099895166449943

Total 15-minute F0 grid runtime:
  25.0 seconds
  0.42 minutes

XGBoost 15-minute F0 experiment: PASSED


In [57]:
# ============================================================
# Step 56: XGBoost validation
#          15-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + Gregorian / conventional calendar-time features
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_15_f1_df, xgboost_15_f1_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=15,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f1_15 = (
    xgboost_15_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 15-minute F1 validation results"
)

print(
    "============================================\n"
)

display(
    xgboost_15_f1_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f1_15[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f1_15[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f1_15[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f1_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f1_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_15_mae = float(
    best_xgboost_f0_15[
        "validation_mae"
    ]
)

f1_15_mae = float(
    best_xgboost_f1_15[
        "validation_mae"
    ]
)

f0_to_f1_15_pct = (
    (f0_15_mae - f1_15_mae)
    / f0_15_mae
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_15_mae:.6f}"
)

print(
    f"  F1 MAE: {f1_15_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_15_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F1 grid runtime:"
)

print(
    f"  {xgboost_15_f1_seconds:.1f} seconds"
)

print(
    f"  {xgboost_15_f1_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 15-minute F1 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 15 minutes
  Feature family: F1
  Train: (87261, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.261262
  validation RMSE = 0.580116
  runtime         = 2.0 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.269690
  validation RMSE = 0.587073
  runtime         = 3.0 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.274923
  validation RMSE = 0.592778
  runtime         = 1.7 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.293922
  validation RMSE = 0.616318
  runtime         = 2.8 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.273983
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,15,F1,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.261262,0.580116,1.955524
1,15,F1,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.269690,0.587073,2.985541
2,15,F1,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.273983,0.589863,3.340663
3,15,F1,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.274923,0.592778,1.668275
4,15,F1,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.283311,0.601760,5.492594
5,15,F1,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.288405,0.610259,3.173857
6,15,F1,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.293922,0.616318,2.778803
7,15,F1,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.307565,0.637724,5.182486



Selected 15-minute F1 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.26126161217689514
  validation RMSE: 0.5801162707184322

F0 -> F1 validation comparison:
  F0 MAE: 0.287141
  F1 MAE: 0.261262
  Relative MAE improvement: +9.0129%

Total 15-minute F1 grid runtime:
  26.9 seconds
  0.45 minutes

XGBoost 15-minute F1 experiment: PASSED


In [58]:
# ============================================================
# Step 57: XGBoost validation
#          15-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + deterministic solar-geometry features
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_15_f2_df, xgboost_15_f2_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=15,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f2_15 = (
    xgboost_15_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 15-minute F2 validation results"
)

print(
    "============================================\n"
)

display(
    xgboost_15_f2_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f2_15[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f2_15[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f2_15[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f2_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f2_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_15_mae = float(
    best_xgboost_f1_15[
        "validation_mae"
    ]
)

f2_15_mae = float(
    best_xgboost_f2_15[
        "validation_mae"
    ]
)

f1_to_f2_15_pct = (
    (f1_15_mae - f2_15_mae)
    / f1_15_mae
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_15_mae:.6f}"
)

print(
    f"  F2 MAE: {f2_15_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_15_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F2 grid runtime:"
)

print(
    f"  {xgboost_15_f2_seconds:.1f} seconds"
)

print(
    f"  {xgboost_15_f2_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 15-minute F2 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 15 minutes
  Feature family: F2
  Train: (87261, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.266062
  validation RMSE = 0.583223
  runtime         = 2.1 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.268050
  validation RMSE = 0.586098
  runtime         = 2.9 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.274880
  validation RMSE = 0.590247
  runtime         = 1.8 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.288134
  validation RMSE = 0.609005
  runtime         = 2.8 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.275085
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,15,F2,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.266062,0.583223,2.125635
1,15,F2,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.268050,0.586098,2.934135
2,15,F2,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.274880,0.590247,1.758491
3,15,F2,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.275085,0.591092,3.388929
4,15,F2,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.280224,0.599255,5.316651
5,15,F2,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.288134,0.609005,2.796434
6,15,F2,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.292875,0.614117,3.210920
7,15,F2,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.301787,0.628628,5.247365



Selected 15-minute F2 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.266061931848526
  validation RMSE: 0.5832227761087185

F1 -> F2 validation comparison:
  F1 MAE: 0.261262
  F2 MAE: 0.266062
  Relative MAE improvement: -1.8374%

Total 15-minute F2 grid runtime:
  27.1 seconds
  0.45 minutes

XGBoost 15-minute F2 experiment: PASSED


In [59]:
# ============================================================
# Step 58: XGBoost validation
#          15-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + continuous lunar / Sun-Moon astronomical features
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_15_f3_df, xgboost_15_f3_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=15,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f3_15 = (
    xgboost_15_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 15-minute F3 validation results"
)

print(
    "============================================\n"
)

display(
    xgboost_15_f3_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f3_15[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f3_15[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f3_15[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f3_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f3_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_15_mae = float(
    best_xgboost_f2_15[
        "validation_mae"
    ]
)

f3_15_mae = float(
    best_xgboost_f3_15[
        "validation_mae"
    ]
)

f2_to_f3_15_pct = (
    (f2_15_mae - f3_15_mae)
    / f2_15_mae
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_15_mae:.6f}"
)

print(
    f"  F3 MAE: {f3_15_mae:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_15_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F3 grid runtime:"
)

print(
    f"  {xgboost_15_f3_seconds:.1f} seconds"
)

print(
    f"  {xgboost_15_f3_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 15-minute F3 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 15 minutes
  Feature family: F3
  Train: (87261, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.266845
  validation RMSE = 0.582792
  runtime         = 2.1 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.274470
  validation RMSE = 0.588397
  runtime         = 3.1 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.276234
  validation RMSE = 0.591230
  runtime         = 1.9 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.288570
  validation RMSE = 0.602220
  runtime         = 3.1 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.282169
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,15,F3,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.266845,0.582792,2.063620
1,15,F3,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.274470,0.588397,3.112991
2,15,F3,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.276234,0.591230,1.881152
3,15,F3,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.282169,0.594764,3.695577
4,15,F3,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.288570,0.602220,3.119667
5,15,F3,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.291210,0.606536,5.594239
6,15,F3,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.299660,0.612567,3.465741
7,15,F3,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.304321,0.622508,5.429176



Selected 15-minute F3 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.26684466004371643
  validation RMSE: 0.5827916445848592

F2 -> F3 validation comparison:
  F2 MAE: 0.266062
  F3 MAE: 0.266845
  Relative MAE improvement: -0.2942%

Total 15-minute F3 grid runtime:
  28.7 seconds
  0.48 minutes

XGBoost 15-minute F3 experiment: PASSED


In [60]:
# ============================================================
# Step 59: XGBoost validation
#          15-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + 97 frozen Panchang one-hot indicators
#
# This gives the central:
#
#       F3 -> F4
#
# comparison for the 15-minute horizon.
#
# The same frozen 8-configuration XGBoost grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen validation grid
# ------------------------------------------------------------

xgboost_15_f4_df, xgboost_15_f4_seconds = (
    run_xgboost_validation_grid(
        horizon_minutes=15,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_xgboost_f4_15 = (
    xgboost_15_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBoost — 15-minute F4 validation results"
)

print(
    "============================================\n"
)

display(
    xgboost_15_f4_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_xgboost_f4_15[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_xgboost_f4_15[
            "learning_rate"
        ]
    ),
)

print(
    "  max_depth:",
    int(
        best_xgboost_f4_15[
            "max_depth"
        ]
    ),
)

print(
    "  validation MAE:",
    best_xgboost_f4_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_xgboost_f4_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Central F3 -> F4 validation comparison
# ------------------------------------------------------------

f3_15_mae = float(
    best_xgboost_f3_15[
        "validation_mae"
    ]
)

f4_15_mae = float(
    best_xgboost_f4_15[
        "validation_mae"
    ]
)

f3_to_f4_15_absolute_change = (
    f3_15_mae
    - f4_15_mae
)

f3_to_f4_15_pct = (
    f3_to_f4_15_absolute_change
    / f3_15_mae
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_15_mae:.6f}"
)

print(
    f"  F4 MAE: {f4_15_mae:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_15_absolute_change:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_15_pct:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F4 grid runtime:"
)

print(
    f"  {xgboost_15_f4_seconds:.1f} seconds"
)

print(
    f"  {xgboost_15_f4_seconds / 60:.2f} minutes"
)


print(
    "\nXGBoost 15-minute F4 experiment: PASSED"
)


XGBoost validation dataset:
  Horizon: 15 minutes
  Feature family: F4
  Train: (87261, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.267433
  validation RMSE = 0.582476
  runtime         = 2.7 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  max_depth        = 6
  validation MAE  = 0.277304
  validation RMSE = 0.592083
  runtime         = 3.5 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 4
  validation MAE  = 0.282191
  validation RMSE = 0.597844
  runtime         = 2.4 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  max_depth        = 6
  validation MAE  = 0.292809
  validation RMSE = 0.605952
  runtime         = 3.5 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  max_depth        = 4
  validation MAE  = 0.284169
  validation RMSE = 0.

,horizon_minutes,feature_family,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,gamma,validation_mae,validation_rmse,fit_predict_seconds
0,15,F4,300,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.267433,0.582476,2.660062
1,15,F4,300,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.277304,0.592083,3.517651
2,15,F4,300,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.282191,0.597844,2.356299
3,15,F4,600,0.03,4,0.8,0.8,1.0,0.0,1.0,0.0,0.284169,0.597580,4.329732
4,15,F4,300,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.292809,0.605952,3.515162
5,15,F4,600,0.03,6,0.8,0.8,1.0,0.0,1.0,0.0,0.294065,0.610052,6.434093
6,15,F4,600,0.05,4,0.8,0.8,1.0,0.0,1.0,0.0,0.303393,0.626486,4.342409
7,15,F4,600,0.05,6,0.8,0.8,1.0,0.0,1.0,0.0,0.311306,0.628963,6.430427



Selected 15-minute F4 configuration:
  n_estimators: 300
  learning_rate: 0.03
  max_depth: 4
  validation MAE: 0.26743337512016296
  validation RMSE: 0.5824757357279289

F3 -> F4 validation comparison:
  F3 MAE: 0.266845
  F4 MAE: 0.267433
  Absolute MAE reduction: -0.000589
  Relative MAE improvement: -0.2206%

Total 15-minute F4 grid runtime:
  33.9 seconds
  0.56 minutes

XGBoost 15-minute F4 experiment: PASSED


In [61]:
# ============================================================
# Step 60: Summarize the complete 15-minute XGBoost
#          F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect selected configurations
# ------------------------------------------------------------

best_rows_xgb_15 = {
    "F0": best_xgboost_f0_15,
    "F1": best_xgboost_f1_15,
    "F2": best_xgboost_f2_15,
    "F3": best_xgboost_f3_15,
    "F4": best_xgboost_f4_15,
}


summary_rows_xgb_15 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_xgb_15[
        feature_family
    ]

    summary_rows_xgb_15.append(
        {
            "horizon_minutes":
                15,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][feature_family],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    row[
                        "learning_rate"
                    ]
                ),

            "max_depth":
                int(
                    row[
                        "max_depth"
                    ]
                ),

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


xgboost_15_summary = pd.DataFrame(
    summary_rows_xgb_15
)


# ------------------------------------------------------------
# 2. Add incremental MAE improvement
#
# Positive:
#     new feature family reduced MAE.
#
# Negative:
#     new feature family increased MAE.
# ------------------------------------------------------------

xgboost_15_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(xgboost_15_summary),
):

    previous_mae = (
        xgboost_15_summary.loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = (
        xgboost_15_summary.loc[
            i,
            "validation_mae",
        ]
    )

    xgboost_15_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (previous_mae - current_mae)
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Verify central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_xgb_15 = (
    xgboost_15_summary.loc[
        xgboost_15_summary[
            "feature_family"
        ] == "F3",
        "validation_mae",
    ]
    .iloc[0]
)

f4_summary_mae_xgb_15 = (
    xgboost_15_summary.loc[
        xgboost_15_summary[
            "feature_family"
        ] == "F4",
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_xgb_15 = (
    (
        f3_summary_mae_xgb_15
        - f4_summary_mae_xgb_15
    )
    / f3_summary_mae_xgb_15
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_xgb_15,
    -0.2206,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete summary
# ------------------------------------------------------------

print(
    "XGBoost — 15-minute validation summary\n"
)

display(
    xgboost_15_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)

for i in range(
    1,
    len(xgboost_15_summary),
):

    previous_family = (
        xgboost_15_summary.loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        xgboost_15_summary.loc[
            i,
            "feature_family",
        ]
    )

    change = (
        xgboost_15_summary.loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )

    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_xgb_15 = (
    xgboost_15_summary
    .sort_values(
        "validation_mae"
    )
    .iloc[0]
)


print(
    "\nLowest 15-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_xgb_15[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_xgb_15[
        "validation_mae"
    ],
)


print(
    "\nCentral XGBoost comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_xgb_15:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_xgb_15:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_xgb_15:+.4f}%"
)


print(
    "\nXGBoost 15-minute summary: PASSED"
)

XGBoost — 15-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,15,F0,118,300,0.03,4,0.287141,0.609990,NaN
1,15,F1,133,300,0.03,4,0.261262,0.580116,9.012892
2,15,F2,138,300,0.03,4,0.266062,0.583223,-1.837361
3,15,F3,147,300,0.03,4,0.266845,0.582792,-0.294190
4,15,F4,244,300,0.03,4,0.267433,0.582476,-0.220621


Incremental validation MAE changes:
  F0 -> F1: +9.0129%
  F1 -> F2: -1.8374%
  F2 -> F3: -0.2942%
  F3 -> F4: -0.2206%

Lowest 15-minute validation MAE:
  Feature family: F1
  MAE: 0.26126161217689514

Central XGBoost comparison:
  F3 MAE: 0.266845
  F4 MAE: 0.267433
  F3 -> F4 change: -0.2206%

XGBoost 15-minute summary: PASSED


In [62]:
# ============================================================
# Step 61: Combine the complete XGBoost validation study
#          and freeze all selected configurations
# ============================================================
#
# At this checkpoint:
#
#   - all 15 XGBoost validation experiments are complete;
#   - model selection used chronological validation MAE only;
#   - no XGBoost test metric has been examined.
#
# We now freeze one selected configuration for every:
#
#       horizon × feature family
#
# combination.
#
# Only after this registry is frozen may we refit on
# TRAIN + VALIDATION and evaluate the TEST period.
# ============================================================


# ------------------------------------------------------------
# 1. Combine 15-, 30-, and 60-minute summaries
# ------------------------------------------------------------

xgboost_validation_summary = (
    pd.concat(
        [
            xgboost_15_summary,
            xgboost_30_summary,
            xgboost_60_summary,
        ],
        axis=0,
        ignore_index=True,
    )
)


# Natural F0 -> F4 ordering
feature_family_order = {
    "F0": 0,
    "F1": 1,
    "F2": 2,
    "F3": 3,
    "F4": 4,
}


xgboost_validation_summary[
    "_family_order"
] = (
    xgboost_validation_summary[
        "feature_family"
    ]
    .map(
        feature_family_order
    )
)


xgboost_validation_summary = (
    xgboost_validation_summary
    .sort_values(
        [
            "horizon_minutes",
            "_family_order",
        ]
    )
    .drop(
        columns="_family_order"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    xgboost_validation_summary
) == 15


print(
    "Combined XGBoost validation summary:\n"
)

display(
    xgboost_validation_summary
)


# ------------------------------------------------------------
# 2. Explicit F3 -> F4 validation comparison
# ------------------------------------------------------------

xgboost_f3_f4_validation_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_summary = (
        xgboost_validation_summary[
            xgboost_validation_summary[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
    )


    f3_mae = float(
        horizon_summary.loc[
            horizon_summary[
                "feature_family"
            ]
            == "F3",
            "validation_mae",
        ]
        .iloc[0]
    )


    f4_mae = float(
        horizon_summary.loc[
            horizon_summary[
                "feature_family"
            ]
            == "F4",
            "validation_mae",
        ]
        .iloc[0]
    )


    absolute_reduction = (
        f3_mae
        - f4_mae
    )


    relative_improvement_pct = (
        absolute_reduction
        / f3_mae
        * 100.0
    )


    xgboost_f3_f4_validation_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "f3_validation_mae":
                f3_mae,

            "f4_validation_mae":
                f4_mae,

            "absolute_mae_reduction":
                absolute_reduction,

            "relative_mae_improvement_pct":
                relative_improvement_pct,
        }
    )


xgboost_f3_f4_validation = (
    pd.DataFrame(
        xgboost_f3_f4_validation_rows
    )
)


print(
    "\nXGBoost F3 -> F4 validation comparison:\n"
)

display(
    xgboost_f3_f4_validation
)


# ------------------------------------------------------------
# 3. Verify already-observed F3 -> F4 effects
# ------------------------------------------------------------

expected_xgb_f3_f4_effects = {
    15: -0.220621,
    30: -0.654167,
    60: -1.602977,
}


for (
    horizon_minutes,
    expected_value,
) in expected_xgb_f3_f4_effects.items():

    observed_value = float(
        xgboost_f3_f4_validation.loc[
            xgboost_f3_f4_validation[
                "horizon_minutes"
            ]
            == horizon_minutes,
            "relative_mae_improvement_pct",
        ]
        .iloc[0]
    )

    assert np.isclose(
        observed_value,
        expected_value,
        atol=1e-5,
    )


# ------------------------------------------------------------
# 4. Gather selected rows from each horizon
# ------------------------------------------------------------

best_rows_xgb_by_horizon = {
    15: best_rows_xgb_15,
    30: best_rows_xgb_30,
    60: best_rows_xgb_60,
}


# ------------------------------------------------------------
# 5. Freeze selected executable configurations
# ------------------------------------------------------------

xgboost_selected_configs = {}


for horizon_minutes in [
    15,
    30,
    60,
]:

    xgboost_selected_configs[
        horizon_minutes
    ] = {}


    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        selected_row = (
            best_rows_xgb_by_horizon[
                horizon_minutes
            ][
                feature_family
            ]
        )


        xgboost_selected_configs[
            horizon_minutes
        ][
            feature_family
        ] = {
            "n_estimators":
                int(
                    selected_row[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    selected_row[
                        "learning_rate"
                    ]
                ),

            "max_depth":
                int(
                    selected_row[
                        "max_depth"
                    ]
                ),

            # All remaining parameters retain the values
            # frozen before any XGBoost result was observed.
            **XGB_FIXED_PARAMS,
        }


# ------------------------------------------------------------
# 6. Create human-readable frozen-selection table
# ------------------------------------------------------------

xgboost_frozen_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        config = (
            xgboost_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )


        validation_row = (
            xgboost_validation_summary[
                (
                    xgboost_validation_summary[
                        "horizon_minutes"
                    ]
                    == horizon_minutes
                )
                &
                (
                    xgboost_validation_summary[
                        "feature_family"
                    ]
                    == feature_family
                )
            ]
            .iloc[0]
        )


        xgboost_frozen_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_features":
                    primary_protocol[
                        "feature_counts"
                    ][
                        feature_family
                    ],

                "n_estimators":
                    config[
                        "n_estimators"
                    ],

                "learning_rate":
                    config[
                        "learning_rate"
                    ],

                "max_depth":
                    config[
                        "max_depth"
                    ],

                "subsample":
                    config[
                        "subsample"
                    ],

                "colsample_bytree":
                    config[
                        "colsample_bytree"
                    ],

                "validation_mae":
                    float(
                        validation_row[
                            "validation_mae"
                        ]
                    ),

                "validation_rmse":
                    float(
                        validation_row[
                            "validation_rmse"
                        ]
                    ),
            }
        )


xgboost_frozen_selection = (
    pd.DataFrame(
        xgboost_frozen_rows
    )
)


# ------------------------------------------------------------
# 7. Strong integrity checks
# ------------------------------------------------------------

assert (
    xgboost_frozen_selection.shape[0]
    == 15
)

assert set(
    xgboost_frozen_selection[
        "horizon_minutes"
    ]
) == {
    15,
    30,
    60,
}

assert set(
    xgboost_frozen_selection[
        "feature_family"
    ]
) == {
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
}


for horizon_minutes in [
    15,
    30,
    60,
]:

    assert len(
        xgboost_selected_configs[
            horizon_minutes
        ]
    ) == 5


# Every selected configuration must still belong to the
# originally frozen 8-configuration grid.
for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        selected_config = (
            xgboost_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )

        assert any(
            (
                selected_config[
                    "n_estimators"
                ]
                == grid_config[
                    "n_estimators"
                ]
            )
            and
            (
                selected_config[
                    "learning_rate"
                ]
                == grid_config[
                    "learning_rate"
                ]
            )
            and
            (
                selected_config[
                    "max_depth"
                ]
                == grid_config[
                    "max_depth"
                ]
            )
            for grid_config in xgboost_configs
        )


print(
    "\nFrozen XGBoost configuration registry:\n"
)

display(
    xgboost_frozen_selection
)


# ------------------------------------------------------------
# 8. Lowest validation MAE within each horizon
# ------------------------------------------------------------

print(
    "\nLowest validation MAE by horizon:"
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    best_row = (
        xgboost_validation_summary[
            xgboost_validation_summary[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
        .sort_values(
            "validation_mae"
        )
        .iloc[0]
    )


    print(
        f"  {horizon_minutes:2d} min: "
        f"{best_row['feature_family']} "
        f"(MAE = "
        f"{best_row['validation_mae']:.6f})"
    )


# ------------------------------------------------------------
# 9. Final F3 -> F4 pre-test checkpoint
# ------------------------------------------------------------

print(
    "\nF3 -> F4 validation effects:"
)


for _, row in (
    xgboost_f3_f4_validation
    .iterrows()
):

    print(
        f"  "
        f"{int(row['horizon_minutes']):2d} min: "
        f"{row['relative_mae_improvement_pct']:+.4f}%"
    )


print(
    "\nAll 15 XGBoost configurations are now frozen."
)

print(
    "No XGBoost test-set metric has been used for selection."
)

print(
    "\nXGBoost validation-selection checkpoint: PASSED"
)

Combined XGBoost validation summary:



,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,15,F0,118,300,0.03,4,0.287141,0.609990,NaN
1,15,F1,133,300,0.03,4,0.261262,0.580116,9.012892
2,15,F2,138,300,0.03,4,0.266062,0.583223,-1.837361
3,15,F3,147,300,0.03,4,0.266845,0.582792,-0.294190
4,15,F4,244,300,0.03,4,0.267433,0.582476,-0.220621
5,30,F0,118,300,0.03,4,0.358788,0.703574,NaN
6,30,F1,133,300,0.03,4,0.318399,0.647551,11.257062
7,30,F2,138,300,0.03,6,0.316794,0.650841,0.504048
8,30,F3,147,300,0.03,6,0.322430,0.651366,-1.778810
9,30,F4,244,300,0.03,4,0.324539,0.653677,-0.654167



XGBoost F3 -> F4 validation comparison:



,horizon_minutes,f3_validation_mae,f4_validation_mae,absolute_mae_reduction,relative_mae_improvement_pct
0,15,0.266845,0.267433,-0.000589,-0.220621
1,30,0.322430,0.324539,-0.002109,-0.654167
2,60,0.382230,0.388357,-0.006127,-1.602977



Frozen XGBoost configuration registry:



,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,validation_mae,validation_rmse
0,15,F0,118,300,0.03,4,0.8,0.8,0.287141,0.609990
1,15,F1,133,300,0.03,4,0.8,0.8,0.261262,0.580116
2,15,F2,138,300,0.03,4,0.8,0.8,0.266062,0.583223
3,15,F3,147,300,0.03,4,0.8,0.8,0.266845,0.582792
4,15,F4,244,300,0.03,4,0.8,0.8,0.267433,0.582476
5,30,F0,118,300,0.03,4,0.8,0.8,0.358788,0.703574
6,30,F1,133,300,0.03,4,0.8,0.8,0.318399,0.647551
7,30,F2,138,300,0.03,6,0.8,0.8,0.316794,0.650841
8,30,F3,147,300,0.03,6,0.8,0.8,0.322430,0.651366
9,30,F4,244,300,0.03,4,0.8,0.8,0.324539,0.653677



Lowest validation MAE by horizon:
  15 min: F1 (MAE = 0.261262)
  30 min: F2 (MAE = 0.316794)
  60 min: F1 (MAE = 0.371761)

F3 -> F4 validation effects:
  15 min: -0.2206%
  30 min: -0.6542%
  60 min: -1.6030%

All 15 XGBoost configurations are now frozen.
No XGBoost test-set metric has been used for selection.

XGBoost validation-selection checkpoint: PASSED


In [63]:
# ============================================================
# Step 62: Final XGBoost test evaluation
# ============================================================
#
# IMPORTANT:
#
# All 15 XGBoost configurations were frozen using
# chronological validation MAE before this cell.
#
# For every:
#
#       horizon × feature family
#
# we now:
#
#   1. refit the frozen XGBoost model on TRAIN + VALIDATION;
#   2. predict the held-out chronological TEST period once;
#   3. calculate all-time test metrics;
#   4. calculate daylight-only test metrics;
#   5. retain timestamp-level predictions for the later
#      paired F3-vs-F4 daily-block bootstrap.
#
# No hyperparameter will be modified using test performance.
# ============================================================


from sklearn.metrics import r2_score


# ------------------------------------------------------------
# 1. Rated PV capacity for normalized metrics
# ------------------------------------------------------------

XGB_PV_RATED_KW = float(
    primary_protocol[
        "pv_rated_kw"
    ]
)

assert XGB_PV_RATED_KW == 10.0


# ------------------------------------------------------------
# 2. Result and prediction containers
# ------------------------------------------------------------

xgboost_test_result_rows = []

xgboost_test_predictions = {}


# ------------------------------------------------------------
# 3. Complete final-evaluation timer
# ------------------------------------------------------------

xgboost_final_test_start_time = (
    time.perf_counter()
)


# ------------------------------------------------------------
# 4. Evaluate all frozen horizon × family configurations
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    y = targets[
        horizon_minutes
    ]

    split = split_labels[
        horizon_minutes
    ]


    # --------------------------------------------------------
    # Final development population:
    #
    #     TRAIN + VALIDATION
    #
    # Test remains chronologically later.
    # --------------------------------------------------------

    train_validation_mask = (
        split.isin(
            [
                "train",
                "validation",
            ]
        )
    )

    test_mask = (
        split == "test"
    )


    assert int(
        test_mask.sum()
    ) == 26_496


    # --------------------------------------------------------
    # Daylight definition
    #
    # Same frozen definition used in Notebook 09 and the
    # Extra Trees analysis:
    #
    #     forecast-valid solar elevation > 0 degrees
    # --------------------------------------------------------

    daylight_test_mask = (
        solar_valid_features[
            horizon_minutes
        ][
            "solar_elevation_deg"
        ]
        > 0.0
    ) & test_mask


    n_test = int(
        test_mask.sum()
    )

    n_test_daylight = int(
        daylight_test_mask.sum()
    )


    print(
        "\n"
        "============================================"
    )

    print(
        f"{horizon_minutes}-minute horizon"
    )

    print(
        "============================================"
    )

    print(
        "Test rows:",
        n_test,
    )

    print(
        "Daylight test rows:",
        n_test_daylight,
    )


    # --------------------------------------------------------
    # F0 -> F4
    # --------------------------------------------------------

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        print(
            f"\n{horizon_minutes} min — "
            f"{feature_family}"
        )


        # ----------------------------------------------------
        # Frozen configuration
        # ----------------------------------------------------

        config = (
            xgboost_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )


        # ----------------------------------------------------
        # Feature matrix
        # ----------------------------------------------------

        X = (
            feature_matrices[
                horizon_minutes
            ][
                feature_family
            ]
        )


        # ----------------------------------------------------
        # Final train+validation arrays
        # ----------------------------------------------------

        X_train_validation = (
            X.loc[
                train_validation_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        X_test = (
            X.loc[
                test_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        y_train_validation = (
            y.loc[
                train_validation_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        y_test = (
            y.loc[
                test_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        # ----------------------------------------------------
        # Integrity checks
        # ----------------------------------------------------

        expected_feature_count = (
            primary_protocol[
                "feature_counts"
            ][
                feature_family
            ]
        )


        assert (
            X_train_validation.shape[1]
            == expected_feature_count
        )

        assert (
            X_test.shape
            == (
                26_496,
                expected_feature_count,
            )
        )

        assert np.isfinite(
            X_train_validation
        ).all()

        assert np.isfinite(
            X_test
        ).all()

        assert np.isfinite(
            y_train_validation
        ).all()

        assert np.isfinite(
            y_test
        ).all()


        # ----------------------------------------------------
        # Refit frozen XGBoost model
        # ----------------------------------------------------

        model = XGBRegressor(
            **config
        )


        fit_start_time = (
            time.perf_counter()
        )


        model.fit(
            X_train_validation,
            y_train_validation,
        )


        test_predictions = (
            model.predict(
                X_test
            )
        )


        fit_predict_seconds = (
            time.perf_counter()
            - fit_start_time
        )


        # ----------------------------------------------------
        # All-time metrics
        # ----------------------------------------------------

        test_mae = (
            mean_absolute_error(
                y_test,
                test_predictions,
            )
        )

        test_rmse = np.sqrt(
            mean_squared_error(
                y_test,
                test_predictions,
            )
        )

        test_nmae = (
            test_mae
            / XGB_PV_RATED_KW
        )

        test_nrmse = (
            test_rmse
            / XGB_PV_RATED_KW
        )

        test_r2 = (
            r2_score(
                y_test,
                test_predictions,
            )
        )


        # ----------------------------------------------------
        # Daylight-only metrics
        # ----------------------------------------------------

        test_issue_index = (
            y.index[
                test_mask
            ]
        )


        daylight_on_test = (
            daylight_test_mask
            .loc[
                test_issue_index
            ]
            .to_numpy(
                dtype=bool
            )
        )


        assert int(
            daylight_on_test.sum()
        ) == n_test_daylight


        y_test_daylight = (
            y_test[
                daylight_on_test
            ]
        )

        predictions_daylight = (
            test_predictions[
                daylight_on_test
            ]
        )


        daylight_mae = (
            mean_absolute_error(
                y_test_daylight,
                predictions_daylight,
            )
        )

        daylight_rmse = np.sqrt(
            mean_squared_error(
                y_test_daylight,
                predictions_daylight,
            )
        )

        daylight_nmae = (
            daylight_mae
            / XGB_PV_RATED_KW
        )

        daylight_nrmse = (
            daylight_rmse
            / XGB_PV_RATED_KW
        )

        daylight_r2 = (
            r2_score(
                y_test_daylight,
                predictions_daylight,
            )
        )


        # ----------------------------------------------------
        # Store final metrics
        # ----------------------------------------------------

        xgboost_test_result_rows.append(
            {
                "model":
                    "XGBRegressor",

                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_features":
                    X.shape[1],

                "n_estimators":
                    config[
                        "n_estimators"
                    ],

                "learning_rate":
                    config[
                        "learning_rate"
                    ],

                "max_depth":
                    config[
                        "max_depth"
                    ],

                "subsample":
                    config[
                        "subsample"
                    ],

                "colsample_bytree":
                    config[
                        "colsample_bytree"
                    ],

                "test_rows":
                    n_test,

                "test_mae":
                    test_mae,

                "test_rmse":
                    test_rmse,

                "test_nmae":
                    test_nmae,

                "test_nrmse":
                    test_nrmse,

                "test_r2":
                    test_r2,

                "daylight_test_rows":
                    n_test_daylight,

                "daylight_mae":
                    daylight_mae,

                "daylight_rmse":
                    daylight_rmse,

                "daylight_nmae":
                    daylight_nmae,

                "daylight_nrmse":
                    daylight_nrmse,

                "daylight_r2":
                    daylight_r2,

                "fit_predict_seconds":
                    fit_predict_seconds,
            }
        )


        # ----------------------------------------------------
        # Retain timestamp-level predictions
        # ----------------------------------------------------

        prediction_table = pd.DataFrame(
            {
                "valid_time":
                    base_modelling_tables[
                        horizon_minutes
                    ]
                    .loc[
                        test_issue_index,
                        "valid_time",
                    ],

                "y_true":
                    y_test,

                "y_pred":
                    test_predictions,

                "absolute_error":
                    np.abs(
                        y_test
                        - test_predictions
                    ),

                "daylight":
                    daylight_on_test,
            },
            index=test_issue_index,
        )


        prediction_table.index.name = (
            "issue_time"
        )


        xgboost_test_predictions[
            (
                horizon_minutes,
                feature_family,
            )
        ] = prediction_table


        # ----------------------------------------------------
        # Compact live output
        # ----------------------------------------------------

        print(
            f"  all-time MAE:     "
            f"{test_mae:.6f}"
        )

        print(
            f"  all-time RMSE:    "
            f"{test_rmse:.6f}"
        )

        print(
            f"  daylight MAE:     "
            f"{daylight_mae:.6f}"
        )

        print(
            f"  daylight RMSE:    "
            f"{daylight_rmse:.6f}"
        )

        print(
            f"  runtime:          "
            f"{fit_predict_seconds:.1f} s"
        )


        # ----------------------------------------------------
        # Release fitted model memory
        # ----------------------------------------------------

        del model
        del X_train_validation
        del X_test
        del y_train_validation
        del y_test
        del test_predictions

        gc.collect()


# ------------------------------------------------------------
# 5. Assemble final results table
# ------------------------------------------------------------

xgboost_final_test_results = (
    pd.DataFrame(
        xgboost_test_result_rows
    )
)


assert len(
    xgboost_final_test_results
) == 15


# ------------------------------------------------------------
# 6. Natural horizon / family ordering
# ------------------------------------------------------------

xgboost_final_test_results[
    "_family_order"
] = (
    xgboost_final_test_results[
        "feature_family"
    ]
    .map(
        feature_family_order
    )
)


xgboost_final_test_results = (
    xgboost_final_test_results
    .sort_values(
        [
            "horizon_minutes",
            "_family_order",
        ]
    )
    .drop(
        columns="_family_order"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 7. Final structural checks
# ------------------------------------------------------------

assert len(
    xgboost_test_predictions
) == 15

assert (
    xgboost_final_test_results[
        "test_rows"
    ]
    .eq(
        26_496
    )
    .all()
)

assert (
    xgboost_final_test_results[
        "test_mae"
    ]
    .notna()
    .all()
)

assert (
    xgboost_final_test_results[
        "daylight_mae"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 8. Display complete final test table
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "FINAL XGBOOST TEST RESULTS"
)

print(
    "============================================\n"
)


display(
    xgboost_final_test_results
)


# ------------------------------------------------------------
# 9. Explicit F3 -> F4 test comparison
# ------------------------------------------------------------

xgboost_f3_f4_test_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_results = (
        xgboost_final_test_results[
            xgboost_final_test_results[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
    )


    f3_row = (
        horizon_results[
            horizon_results[
                "feature_family"
            ]
            == "F3"
        ]
        .iloc[0]
    )


    f4_row = (
        horizon_results[
            horizon_results[
                "feature_family"
            ]
            == "F4"
        ]
        .iloc[0]
    )


    all_time_relative_pct = (
        (
            f3_row[
                "test_mae"
            ]
            - f4_row[
                "test_mae"
            ]
        )
        / f3_row[
            "test_mae"
        ]
        * 100.0
    )


    daylight_relative_pct = (
        (
            f3_row[
                "daylight_mae"
            ]
            - f4_row[
                "daylight_mae"
            ]
        )
        / f3_row[
            "daylight_mae"
        ]
        * 100.0
    )


    xgboost_f3_f4_test_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "f3_test_mae":
                f3_row[
                    "test_mae"
                ],

            "f4_test_mae":
                f4_row[
                    "test_mae"
                ],

            "f3_to_f4_test_improvement_pct":
                all_time_relative_pct,

            "f3_daylight_mae":
                f3_row[
                    "daylight_mae"
                ],

            "f4_daylight_mae":
                f4_row[
                    "daylight_mae"
                ],

            "f3_to_f4_daylight_improvement_pct":
                daylight_relative_pct,
        }
    )


xgboost_f3_f4_test_summary = (
    pd.DataFrame(
        xgboost_f3_f4_test_rows
    )
)


print(
    "\nXGBoost F3 -> F4 FINAL test comparison:\n"
)


display(
    xgboost_f3_f4_test_summary
)


# ------------------------------------------------------------
# 10. Lowest final test MAE within each horizon
# ------------------------------------------------------------

print(
    "\nLowest XGBoost test MAE by horizon:"
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    best_test_row = (
        xgboost_final_test_results[
            xgboost_final_test_results[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
        .sort_values(
            "test_mae"
        )
        .iloc[0]
    )


    print(
        f"  {horizon_minutes:2d} min: "
        f"{best_test_row['feature_family']} "
        f"(MAE = "
        f"{best_test_row['test_mae']:.6f})"
    )


# ------------------------------------------------------------
# 11. Runtime
# ------------------------------------------------------------

xgboost_final_test_total_seconds = (
    time.perf_counter()
    - xgboost_final_test_start_time
)


print(
    "\nTotal final XGBoost evaluation runtime:"
)

print(
    f"  "
    f"{xgboost_final_test_total_seconds:.1f} "
    f"seconds"
)

print(
    f"  "
    f"{xgboost_final_test_total_seconds / 60:.2f} "
    f"minutes"
)


print(
    "\nFinal XGBoost test evaluation: PASSED"
)


15-minute horizon
Test rows: 26496
Daylight test rows: 17851

15 min — F0
  all-time MAE:     0.277332
  all-time RMSE:    0.580335
  daylight MAE:     0.408569
  daylight RMSE:    0.707005
  runtime:          2.1 s

15 min — F1
  all-time MAE:     0.259730
  all-time RMSE:    0.561436
  daylight MAE:     0.382363
  daylight RMSE:    0.683980
  runtime:          2.1 s

15 min — F2
  all-time MAE:     0.259387
  all-time RMSE:    0.560853
  daylight MAE:     0.382865
  daylight RMSE:    0.683281
  runtime:          2.1 s

15 min — F3
  all-time MAE:     0.259552
  all-time RMSE:    0.560503
  daylight MAE:     0.382892
  daylight RMSE:    0.682852
  runtime:          2.2 s

15 min — F4
  all-time MAE:     0.258857
  all-time RMSE:    0.560523
  daylight MAE:     0.382024
  daylight RMSE:    0.682878
  runtime:          2.7 s

30-minute horizon
Test rows: 26496
Daylight test rows: 17851

30 min — F0
  all-time MAE:     0.338429
  all-time RMSE:    0.661344
  daylight MAE:     0.496626
 

,model,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,test_rows,...,test_nmae,test_nrmse,test_r2,daylight_test_rows,daylight_mae,daylight_rmse,daylight_nmae,daylight_nrmse,daylight_r2,fit_predict_seconds
0,XGBRegressor,15,F0,118,300,0.03,4,0.8,0.8,26496,...,0.027733,0.058033,0.876393,17851,0.408569,0.707005,0.040857,0.070701,0.832532,2.068769
1,XGBRegressor,15,F1,133,300,0.03,4,0.8,0.8,26496,...,0.025973,0.056144,0.884312,17851,0.382363,0.683980,0.038236,0.068398,0.843262,2.073538
2,XGBRegressor,15,F2,138,300,0.03,4,0.8,0.8,26496,...,0.025939,0.056085,0.884552,17851,0.382865,0.683281,0.038286,0.068328,0.843583,2.052299
3,XGBRegressor,15,F3,147,300,0.03,4,0.8,0.8,26496,...,0.025955,0.056050,0.884697,17851,0.382892,0.682852,0.038289,0.068285,0.843779,2.163336
4,XGBRegressor,15,F4,244,300,0.03,4,0.8,0.8,26496,...,0.025886,0.056052,0.884688,17851,0.382024,0.682878,0.038202,0.068288,0.843767,2.710526
5,XGBRegressor,30,F0,118,300,0.03,4,0.8,0.8,26496,...,0.033843,0.066134,0.839475,17851,0.496626,0.805626,0.049663,0.080563,0.782553,1.828430
6,XGBRegressor,30,F1,133,300,0.03,4,0.8,0.8,26496,...,0.030042,0.062309,0.857510,17851,0.441186,0.759044,0.044119,0.075904,0.806972,1.908004
7,XGBRegressor,30,F2,138,300,0.03,6,0.8,0.8,26496,...,0.029682,0.061895,0.859398,17851,0.439158,0.754062,0.043916,0.075406,0.809497,3.036020
8,XGBRegressor,30,F3,147,300,0.03,6,0.8,0.8,26496,...,0.030026,0.062090,0.858508,17851,0.444263,0.756443,0.044426,0.075644,0.808293,3.639553
9,XGBRegressor,30,F4,244,300,0.03,4,0.8,0.8,26496,...,0.029827,0.062149,0.858238,17851,0.439193,0.757139,0.043919,0.075714,0.807940,2.897991



XGBoost F3 -> F4 FINAL test comparison:



,horizon_minutes,f3_test_mae,f4_test_mae,f3_to_f4_test_improvement_pct,f3_daylight_mae,f4_daylight_mae,f3_to_f4_daylight_improvement_pct
0,15,0.259552,0.258857,0.267857,0.382892,0.382024,0.226624
1,30,0.300259,0.298266,0.663741,0.444263,0.439193,1.141297
2,60,0.342523,0.342375,0.043052,0.506239,0.501526,0.931018



Lowest XGBoost test MAE by horizon:
  15 min: F4 (MAE = 0.258857)
  30 min: F2 (MAE = 0.296823)
  60 min: F2 (MAE = 0.340921)

Total final XGBoost evaluation runtime:
  39.7 seconds
  0.66 minutes

Final XGBoost test evaluation: PASSED


In [72]:
# ============================================================
# Step 63: Paired daily-block bootstrap for XGBoost
#          F3 vs F4
# ============================================================
#
# Purpose
# -------
# Quantify uncertainty in the final XGBoost F3 -> F4 test
# comparison while respecting temporal dependence.
#
# We reproduce the same frozen uncertainty procedure used for:
#
#   - Notebook 09 primary models;
#   - Extra Trees in Notebook 10.
#
#
# Central comparison
# ------------------
#
#       |error_F3| - |error_F4|
#
# Therefore:
#
#   positive delta:
#       F4 has LOWER MAE than F3
#
#   negative delta:
#       F4 has HIGHER MAE than F3
#
#
# Frozen bootstrap specification
# ------------------------------
#
#   block unit:
#       forecast-valid UTC calendar day
#
#   number of daily blocks:
#       92
#
#   bootstrap replicates:
#       10,000
#
#   random seed:
#       42
#
#   confidence interval:
#       95% percentile interval
#
#   model retraining:
#       NO
#
#   scopes:
#       - all test observations
#       - daylight-only test observations
#
#
# Important numerical detail
# --------------------------
#
# XGBoost predictions are float32.
#
# The observed MAEs produced from those predictions can
# therefore be NumPy float32 scalars. We explicitly cast the
# observed F3 and F4 MAEs to Python float before calculating
# the relative percentage.
#
# This makes the percentage calculation double precision and
# avoids meaningless ~1e-8 percentage-point differences when
# cross-checking against the stored final-test summary.
#
# This numerical cast DOES NOT change:
#
#   - predictions;
#   - test errors;
#   - bootstrap blocks;
#   - bootstrap samples;
#   - seed;
#   - confidence interval;
#   - statistical procedure.
# ============================================================


# ------------------------------------------------------------
# 1. Recover frozen bootstrap settings
# ------------------------------------------------------------

N_BOOTSTRAP = int(
    primary_protocol[
        "bootstrap"
    ][
        "replicates"
    ]
)

BOOTSTRAP_RANDOM_SEED = int(
    primary_protocol[
        "bootstrap"
    ][
        "random_seed"
    ]
)

EXPECTED_DAILY_BLOCKS = int(
    primary_protocol[
        "bootstrap"
    ][
        "daily_blocks"
    ]
)


assert N_BOOTSTRAP == 10_000

assert BOOTSTRAP_RANDOM_SEED == 42

assert EXPECTED_DAILY_BLOCKS == 92


# ------------------------------------------------------------
# Numerical tolerance used ONLY for cross-checking the stored
# relative percentage.
#
# The underlying F3 and F4 MAEs are checked separately using
# the stricter atol = 1e-12.
# ------------------------------------------------------------

XGB_RELATIVE_CHECK_ATOL = 1e-8


BOOTSTRAP_HORIZONS = [
    15,
    30,
    60,
]

BOOTSTRAP_SCOPES = [
    "all",
    "daylight",
]


# ------------------------------------------------------------
# One reproducible RNG for the complete XGBoost bootstrap.
#
# Re-running this entire cell resets the RNG to seed 42.
# ------------------------------------------------------------

rng_xgboost = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)


# ------------------------------------------------------------
# 2. Storage
# ------------------------------------------------------------

xgboost_bootstrap_summary_rows = []

xgboost_bootstrap_distributions = {}


# ------------------------------------------------------------
# 3. Run one paired bootstrap for every:
#
#       horizon × scope
#
# Total analyses:
#
#       3 horizons × 2 scopes = 6
# ------------------------------------------------------------

for horizon_minutes in BOOTSTRAP_HORIZONS:

    # --------------------------------------------------------
    # Retrieve final frozen F3 and F4 TEST predictions
    # --------------------------------------------------------

    df_f3 = (
        xgboost_test_predictions[
            (
                horizon_minutes,
                "F3",
            )
        ]
        .copy()
    )

    df_f4 = (
        xgboost_test_predictions[
            (
                horizon_minutes,
                "F4",
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # 4. Verify exact paired alignment
    # --------------------------------------------------------

    assert df_f3.index.equals(
        df_f4.index
    )

    assert len(df_f3) == 26_496

    assert len(df_f4) == 26_496


    assert df_f3[
        "valid_time"
    ].equals(
        df_f4[
            "valid_time"
        ]
    )


    assert np.allclose(
        df_f3[
            "y_true"
        ].to_numpy(),
        df_f4[
            "y_true"
        ].to_numpy(),
        atol=0.0,
        rtol=0.0,
    )


    assert df_f3[
        "daylight"
    ].equals(
        df_f4[
            "daylight"
        ]
    )


    # --------------------------------------------------------
    # 5. Construct paired prediction table
    # --------------------------------------------------------

    df_predictions = pd.DataFrame(
        {
            "valid_time":
                df_f3[
                    "valid_time"
                ],

            "y_true":
                df_f3[
                    "y_true"
                ],

            "F3_prediction":
                df_f3[
                    "y_pred"
                ],

            "F4_prediction":
                df_f4[
                    "y_pred"
                ],

            "daylight":
                df_f3[
                    "daylight"
                ],
        },
        index=df_f3.index,
    )


    # --------------------------------------------------------
    # 6. Verify fixed chronological test interval
    # --------------------------------------------------------

    assert (
        df_predictions[
            "valid_time"
        ].min()
        == pd.Timestamp(
            "2019-06-01 00:00:00",
            tz="UTC",
        )
    )


    assert (
        df_predictions[
            "valid_time"
        ].max()
        == pd.Timestamp(
            "2019-08-31 23:55:00",
            tz="UTC",
        )
    )


    # --------------------------------------------------------
    # 7. Evaluate both scopes
    # --------------------------------------------------------

    for scope_name in BOOTSTRAP_SCOPES:

        # ----------------------------------------------------
        # All-time scope
        # ----------------------------------------------------

        if scope_name == "all":

            df_scope = (
                df_predictions
                .copy()
            )


        # ----------------------------------------------------
        # Daylight-only scope
        #
        # Filtering occurs BEFORE daily-block construction.
        # ----------------------------------------------------

        elif scope_name == "daylight":

            df_scope = (
                df_predictions.loc[
                    df_predictions[
                        "daylight"
                    ]
                ]
                .copy()
            )


        else:

            raise ValueError(
                f"Unexpected scope: {scope_name}"
            )


        # ----------------------------------------------------
        # 8. Calculate paired absolute errors
        #
        # Positive:
        #
        #       |e_F3| - |e_F4| > 0
        #
        # means F4 is better.
        # ----------------------------------------------------

        df_scope[
            "abs_error_F3"
        ] = np.abs(
            df_scope[
                "y_true"
            ]
            - df_scope[
                "F3_prediction"
            ]
        )


        df_scope[
            "abs_error_F4"
        ] = np.abs(
            df_scope[
                "y_true"
            ]
            - df_scope[
                "F4_prediction"
            ]
        )


        df_scope[
            "paired_error_difference"
        ] = (
            df_scope[
                "abs_error_F3"
            ]
            - df_scope[
                "abs_error_F4"
            ]
        )


        # ----------------------------------------------------
        # 9. Assign forecast-valid UTC calendar-day blocks
        # ----------------------------------------------------

        df_scope[
            "valid_date"
        ] = (
            df_scope[
                "valid_time"
            ]
            .dt.floor("D")
        )


        # ----------------------------------------------------
        # 10. Collapse each day into sufficient statistics
        #
        # We retain:
        #
        #   - number of rows in the day;
        #   - total F3 absolute error;
        #   - total F4 absolute error.
        #
        # This allows each bootstrap replicate to reconstruct
        # the POOLED MAE across sampled days.
        #
        # We do NOT average daily MAEs equally.
        # ----------------------------------------------------

        daily_blocks = (
            df_scope
            .groupby(
                "valid_date",
                sort=True,
            )
            .agg(
                n_rows=(
                    "y_true",
                    "size",
                ),

                abs_error_F3_sum=(
                    "abs_error_F3",
                    "sum",
                ),

                abs_error_F4_sum=(
                    "abs_error_F4",
                    "sum",
                ),
            )
        )


        n_days = len(
            daily_blocks
        )


        assert n_days == EXPECTED_DAILY_BLOCKS, (
            f"XGBoost / "
            f"{horizon_minutes} min / "
            f"{scope_name}: "
            f"expected {EXPECTED_DAILY_BLOCKS} "
            f"daily blocks, found {n_days}."
        )


        # ----------------------------------------------------
        # 11. Observed test-set point estimates
        #
        # IMPORTANT NUMERICAL FIX:
        #
        # XGBoost prediction/error arrays are float32.
        #
        # Convert the mean MAEs explicitly to Python float
        # before calculating delta and relative percentage.
        #
        # This produces a double-precision percentage and
        # avoids tiny float32-vs-float64 discrepancies.
        # ----------------------------------------------------

        observed_f3_mae = float(
            df_scope[
                "abs_error_F3"
            ]
            .mean()
        )


        observed_f4_mae = float(
            df_scope[
                "abs_error_F4"
            ]
            .mean()
        )


        observed_delta_mae = (
            observed_f3_mae
            - observed_f4_mae
        )


        observed_relative_change = (
            observed_delta_mae
            / observed_f3_mae
            * 100.0
        )


        # ----------------------------------------------------
        # 12. Verify point estimates against the previously
        #     frozen final XGBoost test summary
        # ----------------------------------------------------

        expected_row = (
            xgboost_f3_f4_test_summary.loc[
                xgboost_f3_f4_test_summary[
                    "horizon_minutes"
                ].eq(
                    horizon_minutes
                )
            ]
            .iloc[0]
        )


        if scope_name == "all":

            expected_f3_mae = float(
                expected_row[
                    "f3_test_mae"
                ]
            )

            expected_f4_mae = float(
                expected_row[
                    "f4_test_mae"
                ]
            )

            expected_relative_change = float(
                expected_row[
                    "f3_to_f4_test_improvement_pct"
                ]
            )


        else:

            expected_f3_mae = float(
                expected_row[
                    "f3_daylight_mae"
                ]
            )

            expected_f4_mae = float(
                expected_row[
                    "f4_daylight_mae"
                ]
            )

            expected_relative_change = float(
                expected_row[
                    "f3_to_f4_daylight_improvement_pct"
                ]
            )


        # ----------------------------------------------------
        # Underlying MAEs should agree extremely closely.
        # ----------------------------------------------------

        assert np.isclose(
            observed_f3_mae,
            expected_f3_mae,
            atol=1e-12,
            rtol=0.0,
        )


        assert np.isclose(
            observed_f4_mae,
            expected_f4_mae,
            atol=1e-12,
            rtol=0.0,
        )


        # ----------------------------------------------------
        # Relative percentage cross-check.
        #
        # 1e-8 is used only because the stored value originated
        # from float32 XGBoost prediction arithmetic.
        # ----------------------------------------------------

        assert np.isclose(
            observed_relative_change,
            expected_relative_change,
            atol=XGB_RELATIVE_CHECK_ATOL,
            rtol=0.0,
        )


        # ----------------------------------------------------
        # 13. Convert daily sufficient statistics to arrays
        #
        # Explicit float conversion gives double-precision
        # bootstrap arithmetic.
        # ----------------------------------------------------

        daily_counts = (
            daily_blocks[
                "n_rows"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )


        daily_f3_error_sums = (
            daily_blocks[
                "abs_error_F3_sum"
            ]
            .to_numpy(
                dtype=float
            )
        )


        daily_f4_error_sums = (
            daily_blocks[
                "abs_error_F4_sum"
            ]
            .to_numpy(
                dtype=float
            )
        )


        # ----------------------------------------------------
        # 14. Sample 92 complete daily blocks with replacement
        #
        # Shape:
        #
        #       10,000 × 92
        #
        # Each row is one bootstrap replicate.
        # ----------------------------------------------------

        sampled_day_indices = (
            rng_xgboost.integers(
                low=0,
                high=n_days,
                size=(
                    N_BOOTSTRAP,
                    n_days,
                ),
            )
        )


        # ----------------------------------------------------
        # 15. Reconstruct pooled MAE for every replicate
        #
        # Because sampled days can have different row counts,
        # especially in daylight-only analysis, the correct
        # pooled MAE is:
        #
        #       total sampled absolute error
        #       ----------------------------
        #       total sampled observations
        #
        # rather than an unweighted average of daily MAEs.
        # ----------------------------------------------------

        bootstrap_row_counts = (
            daily_counts[
                sampled_day_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f3_error_sums = (
            daily_f3_error_sums[
                sampled_day_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f4_error_sums = (
            daily_f4_error_sums[
                sampled_day_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f3_mae = (
            bootstrap_f3_error_sums
            / bootstrap_row_counts
        )


        bootstrap_f4_mae = (
            bootstrap_f4_error_sums
            / bootstrap_row_counts
        )


        # ----------------------------------------------------
        # 16. Paired F3 -> F4 difference
        #
        # Positive delta:
        #       F4 lower MAE / better
        #
        # Negative delta:
        #       F4 higher MAE / worse
        # ----------------------------------------------------

        bootstrap_delta_mae = (
            bootstrap_f3_mae
            - bootstrap_f4_mae
        )


        bootstrap_relative_change = (
            bootstrap_delta_mae
            / bootstrap_f3_mae
            * 100.0
        )


        # ----------------------------------------------------
        # 17. Frozen 95% percentile confidence intervals
        # ----------------------------------------------------

        (
            delta_ci_lower,
            delta_ci_upper,
        ) = np.percentile(
            bootstrap_delta_mae,
            [
                2.5,
                97.5,
            ],
        )


        (
            relative_ci_lower,
            relative_ci_upper,
        ) = np.percentile(
            bootstrap_relative_change,
            [
                2.5,
                97.5,
            ],
        )


        # ----------------------------------------------------
        # 18. Store complete bootstrap distribution
        # ----------------------------------------------------

        distribution_key = (
            horizon_minutes,
            scope_name,
        )


        xgboost_bootstrap_distributions[
            distribution_key
        ] = pd.DataFrame(
            {
                "delta_MAE_kW":
                    bootstrap_delta_mae,

                "relative_MAE_change_percent":
                    bootstrap_relative_change,
            }
        )


        # ----------------------------------------------------
        # 19. Store summary row
        # ----------------------------------------------------

        xgboost_bootstrap_summary_rows.append(
            {
                "model":
                    "XGBRegressor",

                "horizon_minutes":
                    horizon_minutes,

                "scope":
                    scope_name,

                "n_test_rows":
                    len(
                        df_scope
                    ),

                "n_daily_blocks":
                    n_days,

                "F3_MAE_kW":
                    observed_f3_mae,

                "F4_MAE_kW":
                    observed_f4_mae,

                "delta_MAE_kW":
                    observed_delta_mae,

                "delta_MAE_CI95_lower_kW":
                    delta_ci_lower,

                "delta_MAE_CI95_upper_kW":
                    delta_ci_upper,

                "relative_MAE_change_percent":
                    observed_relative_change,

                "relative_CI95_lower_percent":
                    relative_ci_lower,

                "relative_CI95_upper_percent":
                    relative_ci_upper,
            }
        )


# ------------------------------------------------------------
# 20. Assemble final XGBoost bootstrap summary
# ------------------------------------------------------------

xgboost_bootstrap_summary = (
    pd.DataFrame(
        xgboost_bootstrap_summary_rows
    )
)


# ------------------------------------------------------------
# 21. Structural and numerical checks
# ------------------------------------------------------------

assert len(
    xgboost_bootstrap_summary
) == 6


assert len(
    xgboost_bootstrap_distributions
) == 6


assert (
    xgboost_bootstrap_summary[
        "n_daily_blocks"
    ]
    .eq(
        92
    )
    .all()
)


bootstrap_numeric_columns = [
    "F3_MAE_kW",
    "F4_MAE_kW",
    "delta_MAE_kW",
    "delta_MAE_CI95_lower_kW",
    "delta_MAE_CI95_upper_kW",
    "relative_MAE_change_percent",
    "relative_CI95_lower_percent",
    "relative_CI95_upper_percent",
]


assert np.isfinite(
    xgboost_bootstrap_summary[
        bootstrap_numeric_columns
    ]
    .to_numpy()
).all()


# ------------------------------------------------------------
# Every stored distribution must contain exactly 10,000
# bootstrap replicates.
# ------------------------------------------------------------

for distribution_key, distribution_df in (
    xgboost_bootstrap_distributions.items()
):

    assert (
        len(
            distribution_df
        )
        == N_BOOTSTRAP
    ), (
        f"Unexpected bootstrap length for "
        f"{distribution_key}"
    )


# ------------------------------------------------------------
# 22. Sort for interpretation
#
# Output order:
#
#   all:
#       15, 30, 60
#
#   daylight:
#       15, 30, 60
# ------------------------------------------------------------

scope_order = {
    "all": 0,
    "daylight": 1,
}


xgboost_bootstrap_summary[
    "_scope_order"
] = (
    xgboost_bootstrap_summary[
        "scope"
    ]
    .map(
        scope_order
    )
)


xgboost_bootstrap_summary = (
    xgboost_bootstrap_summary
    .sort_values(
        [
            "_scope_order",
            "horizon_minutes",
        ]
    )
    .drop(
        columns="_scope_order"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 23. Confirm whether each CI contains zero
# ------------------------------------------------------------

xgboost_bootstrap_summary[
    "delta_CI_contains_zero"
] = (
    (
        xgboost_bootstrap_summary[
            "delta_MAE_CI95_lower_kW"
        ]
        <= 0.0
    )
    &
    (
        xgboost_bootstrap_summary[
            "delta_MAE_CI95_upper_kW"
        ]
        >= 0.0
    )
)


xgboost_bootstrap_summary[
    "relative_CI_contains_zero"
] = (
    (
        xgboost_bootstrap_summary[
            "relative_CI95_lower_percent"
        ]
        <= 0.0
    )
    &
    (
        xgboost_bootstrap_summary[
            "relative_CI95_upper_percent"
        ]
        >= 0.0
    )
)


# ------------------------------------------------------------
# 24. Final validation
# ------------------------------------------------------------

assert (
    xgboost_bootstrap_summary[
        "delta_CI_contains_zero"
    ]
    .equals(
        xgboost_bootstrap_summary[
            "relative_CI_contains_zero"
        ]
    )
)


print(
    "XGBoost paired daily-block bootstrap: PASSED"
)


# ------------------------------------------------------------
# 25. Display final uncertainty table
# ------------------------------------------------------------

print(
    "\n"
    "XGBoost F3 -> F4 paired daily-block "
    "bootstrap results:\n"
)


display(
    xgboost_bootstrap_summary[
        [
            "model",
            "horizon_minutes",
            "scope",
            "n_test_rows",
            "n_daily_blocks",
            "F3_MAE_kW",
            "F4_MAE_kW",
            "delta_MAE_kW",
            "delta_MAE_CI95_lower_kW",
            "delta_MAE_CI95_upper_kW",
            "relative_MAE_change_percent",
            "relative_CI95_lower_percent",
            "relative_CI95_upper_percent",
            "delta_CI_contains_zero",
        ]
    ]
)


# ------------------------------------------------------------
# 26. Compact interpretation output
# ------------------------------------------------------------

print(
    "\nInterpretation convention:"
)

print(
    "  positive delta / relative change = F4 better"
)

print(
    "  negative delta / relative change = F4 worse"
)

print(
    "  CI containing zero = no clear paired difference "
    "at the 95% percentile level"
)


print(
    "\nF3 -> F4 bootstrap summary:"
)


for _, row in (
    xgboost_bootstrap_summary
    .iterrows()
):

    ci_status = (
        "contains zero"
        if row[
            "delta_CI_contains_zero"
        ]
        else
        "excludes zero"
    )

    print(
        f"  "
        f"{int(row['horizon_minutes']):2d} min | "
        f"{row['scope']:8s} | "
        f"point = "
        f"{row['relative_MAE_change_percent']:+.4f}% | "
        f"95% CI = ["
        f"{row['relative_CI95_lower_percent']:+.4f}%, "
        f"{row['relative_CI95_upper_percent']:+.4f}%] | "
        f"{ci_status}"
    )


print(
    "\nXGBoost bootstrap analysis: PASSED"
)

XGBoost paired daily-block bootstrap: PASSED

XGBoost F3 -> F4 paired daily-block bootstrap results:



,model,horizon_minutes,scope,n_test_rows,n_daily_blocks,F3_MAE_kW,F4_MAE_kW,delta_MAE_kW,delta_MAE_CI95_lower_kW,delta_MAE_CI95_upper_kW,relative_MAE_change_percent,relative_CI95_lower_percent,relative_CI95_upper_percent,delta_CI_contains_zero
0,XGBRegressor,15,all,26496,92,0.259552,0.258857,0.000695,-0.000109,0.001494,0.267857,-0.042346,0.584600,True
1,XGBRegressor,30,all,26496,92,0.300259,0.298266,0.001993,-0.001644,0.006058,0.663741,-0.536353,2.066308,True
2,XGBRegressor,60,all,26496,92,0.342523,0.342375,0.000147,-0.004330,0.004988,0.043052,-1.260916,1.456227,True
3,XGBRegressor,15,daylight,17851,92,0.382892,0.382024,0.000868,-0.000337,0.002075,0.226624,-0.089156,0.550161,True
4,XGBRegressor,30,daylight,17851,92,0.444263,0.439193,0.005070,-0.000343,0.011149,1.141297,-0.076064,2.551971,True
5,XGBRegressor,60,daylight,17851,92,0.506239,0.501526,0.004713,-0.002143,0.011634,0.931018,-0.428051,2.337723,True



Interpretation convention:
  positive delta / relative change = F4 better
  negative delta / relative change = F4 worse
  CI containing zero = no clear paired difference at the 95% percentile level

F3 -> F4 bootstrap summary:
  15 min | all      | point = +0.2679% | 95% CI = [-0.0423%, +0.5846%] | contains zero
  30 min | all      | point = +0.6637% | 95% CI = [-0.5364%, +2.0663%] | contains zero
  60 min | all      | point = +0.0431% | 95% CI = [-1.2609%, +1.4562%] | contains zero
  15 min | daylight | point = +0.2266% | 95% CI = [-0.0892%, +0.5502%] | contains zero
  30 min | daylight | point = +1.1413% | 95% CI = [-0.0761%, +2.5520%] | contains zero
  60 min | daylight | point = +0.9310% | 95% CI = [-0.4281%, +2.3377%] | contains zero

XGBoost bootstrap analysis: PASSED


In [69]:
# ============================================================
# XGBoost bootstrap numerical-tolerance correction
# ============================================================
#
# Diagnostic result:
#
# observed relative change:
#     0.26785680651664734
#
# stored final-test relative change:
#     0.2678568045422672
#
# absolute difference:
#     1.974380159364131e-09 percentage points
#
# The underlying F3 and F4 MAEs match exactly. Therefore this
# is only floating-point propagation from the float32 XGBoost
# predictions, not a disagreement in the test results.
#
# For the relative-percentage cross-check only, use:
#
#     atol = 1e-8
#
# The MAE checks remain at their stricter atol = 1e-12.
# ============================================================

XGB_RELATIVE_CHECK_ATOL = 1e-8

print(
    "XGBoost relative-change assertion tolerance:",
    XGB_RELATIVE_CHECK_ATOL,
)

print(
    "Bootstrap settings and statistical procedure unchanged."
)

XGBoost relative-change assertion tolerance: 1e-08
Bootstrap settings and statistical procedure unchanged.


## XGBoost: interpretation of the F3 → F4 experiment

The XGBoost experiment used the same controlled feature-family ablation as the
primary experiment and Extra Trees:

- **F0:** historical/autoregressive and meteorological predictors;
- **F1:** F0 + conventional Gregorian calendar features;
- **F2:** F1 + independently computed solar-geometry features;
- **F3:** F2 + continuous lunar and Sun–Moon astronomical features;
- **F4:** F3 + one-hot encoded Vedic/Panchang categories.

Hyperparameters were selected using the chronological validation period only.
All 15 horizon × feature-family configurations were frozen before any XGBoost
test metric was inspected. Each selected model was then refitted once on the
combined training + validation population and evaluated on the held-out
chronological test period.

### Validation behaviour

The incremental **F3 → F4** validation MAE changes were:

| Horizon | F3 → F4 validation MAE change |
|---:|---:|
| 15 min | -0.2206% |
| 30 min | -0.6542% |
| 60 min | -1.6030% |

Thus, adding the Panchang representation slightly worsened validation MAE at
all three horizons.

The lowest validation MAE occurred for:

- **15 min:** F1;
- **30 min:** F2;
- **60 min:** F1.

### Final held-out test behaviour

On the final test period, the direction of the F3 → F4 point estimate reversed:

| Horizon | All-time F3 → F4 | Daylight F3 → F4 |
|---:|---:|---:|
| 15 min | +0.2679% | +0.2266% |
| 30 min | +0.6637% | +1.1413% |
| 60 min | +0.0431% | +0.9310% |

Positive values indicate lower MAE for F4.

However, these improvements were small and their direction was not consistent
with the validation-period results. In addition, F4 was not the lowest-MAE
feature family at the 30- or 60-minute horizons:

- **15 min:** F4 had the lowest test MAE, 0.258857 kW;
- **30 min:** F2 had the lowest test MAE, 0.296823 kW;
- **60 min:** F2 had the lowest test MAE, 0.340921 kW.

### Paired daily-block bootstrap

To account for temporal dependence, F3 and F4 were compared using the frozen
paired daily-block bootstrap procedure:

- block unit: forecast-valid UTC calendar day;
- number of test-day blocks: 92;
- bootstrap replicates: 10,000;
- random seed: 42;
- interval: 95% percentile interval;
- no model retraining during bootstrap.

The resulting relative MAE changes were:

| Horizon | Scope | Point estimate | 95% bootstrap CI |
|---:|:---|---:|---:|
| 15 min | All | +0.2679% | [-0.0423%, +0.5846%] |
| 30 min | All | +0.6637% | [-0.5364%, +2.0663%] |
| 60 min | All | +0.0431% | [-1.2609%, +1.4562%] |
| 15 min | Daylight | +0.2266% | [-0.0892%, +0.5502%] |
| 30 min | Daylight | +1.1413% | [-0.0761%, +2.5520%] |
| 60 min | Daylight | +0.9310% | [-0.4281%, +2.3377%] |

Every 95% bootstrap interval contains zero.

Therefore, although the final XGBoost test point estimates are positive for
F3 → F4, the paired day-level uncertainty analysis does **not provide clear
evidence of a robust Panchang improvement beyond the continuous astronomical
F3 controls**.

This result is consistent with the broader interpretation of the experiment:
small isolated F4 improvements should not be treated as evidence for an
independent Panchang forecasting contribution unless they are stable across
validation/test periods, forecasting horizons, model classes, and uncertainty
analysis.

In [73]:
# ============================================================
# Step 64: Persist complete XGBoost experiment artifacts
# ============================================================
#
# Save the complete frozen XGBoost experiment so that:
#
#   - validation selections are preserved;
#   - final held-out test results are preserved;
#   - F3 -> F4 comparisons are preserved;
#   - bootstrap uncertainty results are preserved;
#   - timestamp-level test predictions are preserved;
#   - the exact selected model configurations are preserved;
#   - the experiment protocol is documented;
#   - every artifact is recorded in a manifest.
#
# This is performed BEFORE beginning LightGBM.
# ============================================================


from pathlib import Path
import json
import hashlib
import xgboost as xgb


# ------------------------------------------------------------
# 1. Locate project root robustly
# ------------------------------------------------------------
#
# Depending on how VS Code / Jupyter was started, the current
# working directory may be either:
#
#   Project/
#
# or:
#
#   Project/notebooks/
#
# Detect the project root rather than assuming one.
# ------------------------------------------------------------

cwd = Path.cwd().resolve()


if (
    (cwd / "src").exists()
    and
    (cwd / "notebooks").exists()
):

    project_root = cwd

elif (
    (cwd.parent / "src").exists()
    and
    (cwd.parent / "notebooks").exists()
):

    project_root = cwd.parent

else:

    raise RuntimeError(
        "Could not identify project root from the current "
        "working directory."
    )


results_dir = (
    project_root
    / "results"
    / "notebook10_secondary_ml_models"
)

results_dir.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Project root:",
    project_root,
)

print(
    "Results directory:",
    results_dir,
)


# ------------------------------------------------------------
# 2. Define XGBoost artifact paths
# ------------------------------------------------------------

artifact_paths = {
    "validation_summary":
        results_dir
        / "xgboost_validation_summary.csv",

    "frozen_selection":
        results_dir
        / "xgboost_frozen_selection.csv",

    "final_test_results":
        results_dir
        / "xgboost_final_test_results.csv",

    "f3_f4_test_summary":
        results_dir
        / "xgboost_f3_f4_test_summary.csv",

    "bootstrap_summary":
        results_dir
        / "xgboost_bootstrap_summary.csv",

    "bootstrap_distributions":
        results_dir
        / "xgboost_bootstrap_distributions.csv.gz",

    "selected_configs":
        results_dir
        / "xgboost_selected_configs.json",

    "protocol":
        results_dir
        / "xgboost_protocol.json",

    "manifest":
        results_dir
        / "xgboost_results_manifest.csv",
}


for horizon_minutes in [
    15,
    30,
    60,
]:

    artifact_paths[
        f"test_predictions_{horizon_minutes}"
    ] = (
        results_dir
        / (
            "xgboost_test_predictions_"
            f"{horizon_minutes}min.csv"
        )
    )


# ------------------------------------------------------------
# 3. Save validation summary
# ------------------------------------------------------------

xgboost_validation_summary.to_csv(
    artifact_paths[
        "validation_summary"
    ],
    index=False,
)


# ------------------------------------------------------------
# 4. Save frozen configuration-selection table
# ------------------------------------------------------------

xgboost_frozen_selection.to_csv(
    artifact_paths[
        "frozen_selection"
    ],
    index=False,
)


# ------------------------------------------------------------
# 5. Save final held-out test results
# ------------------------------------------------------------

xgboost_final_test_results.to_csv(
    artifact_paths[
        "final_test_results"
    ],
    index=False,
)


# ------------------------------------------------------------
# 6. Save central F3 -> F4 final-test summary
# ------------------------------------------------------------

xgboost_f3_f4_test_summary.to_csv(
    artifact_paths[
        "f3_f4_test_summary"
    ],
    index=False,
)


# ------------------------------------------------------------
# 7. Save paired bootstrap summary
# ------------------------------------------------------------

xgboost_bootstrap_summary.to_csv(
    artifact_paths[
        "bootstrap_summary"
    ],
    index=False,
)


# ------------------------------------------------------------
# 8. Combine and save all six bootstrap distributions
#
# Each row retains:
#
#   - horizon;
#   - scope;
#   - replicate number;
#   - absolute paired MAE difference;
#   - relative paired MAE difference.
# ------------------------------------------------------------

bootstrap_distribution_frames = []


for (
    horizon_minutes,
    scope_name,
), distribution_df in (
    xgboost_bootstrap_distributions.items()
):

    df_distribution = (
        distribution_df
        .copy()
    )

    df_distribution.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            1,
            len(df_distribution) + 1,
            dtype=np.int64,
        ),
    )

    df_distribution.insert(
        0,
        "scope",
        scope_name,
    )

    df_distribution.insert(
        0,
        "horizon_minutes",
        horizon_minutes,
    )


    bootstrap_distribution_frames.append(
        df_distribution
    )


xgboost_bootstrap_distributions_table = (
    pd.concat(
        bootstrap_distribution_frames,
        axis=0,
        ignore_index=True,
    )
)


assert (
    len(
        xgboost_bootstrap_distributions_table
    )
    == 6 * 10_000
)


xgboost_bootstrap_distributions_table.to_csv(
    artifact_paths[
        "bootstrap_distributions"
    ],
    index=False,
    compression="gzip",
)


# ------------------------------------------------------------
# 9. Save timestamp-level TEST predictions
#
# Store all five feature-family predictions side by side for
# each horizon.
#
# This avoids writing five separate files per horizon while
# still retaining every final XGBoost test prediction.
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    prediction_tables = []


    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        df_pred = (
            xgboost_test_predictions[
                (
                    horizon_minutes,
                    feature_family,
                )
            ]
            .copy()
        )


        # Shared columns are taken from the first family.
        if feature_family == "F0":

            prediction_output = pd.DataFrame(
                {
                    "issue_time":
                        df_pred.index,

                    "valid_time":
                        df_pred[
                            "valid_time"
                        ].to_numpy(),

                    "y_true":
                        df_pred[
                            "y_true"
                        ].to_numpy(),

                    "daylight":
                        df_pred[
                            "daylight"
                        ].to_numpy(),
                }
            )


        prediction_output[
            f"{feature_family}_prediction"
        ] = (
            df_pred[
                "y_pred"
            ]
            .to_numpy()
        )


        prediction_output[
            f"{feature_family}_absolute_error"
        ] = (
            df_pred[
                "absolute_error"
            ]
            .to_numpy()
        )


    # --------------------------------------------------------
    # Prediction-table integrity
    # --------------------------------------------------------

    assert len(
        prediction_output
    ) == 26_496

    assert prediction_output[
        "issue_time"
    ].is_unique

    assert prediction_output[
        "valid_time"
    ].is_unique

    assert prediction_output[
        "y_true"
    ].notna().all()


    prediction_output.to_csv(
        artifact_paths[
            f"test_predictions_{horizon_minutes}"
        ],
        index=False,
    )


# ------------------------------------------------------------
# 10. Save exact selected configurations as JSON
# ------------------------------------------------------------
#
# Convert integer horizon keys to strings explicitly so the
# JSON representation is unambiguous.
# ------------------------------------------------------------

selected_configs_json = {}


for horizon_minutes in [
    15,
    30,
    60,
]:

    selected_configs_json[
        str(
            horizon_minutes
        )
    ] = {}


    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        config = (
            xgboost_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )


        selected_configs_json[
            str(
                horizon_minutes
            )
        ][
            feature_family
        ] = {
            key:
                (
                    value.item()
                    if isinstance(
                        value,
                        np.generic,
                    )
                    else value
                )
            for key, value
            in config.items()
        }


with open(
    artifact_paths[
        "selected_configs"
    ],
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        selected_configs_json,
        f,
        indent=2,
        sort_keys=True,
    )


# ------------------------------------------------------------
# 11. Save XGBoost experiment protocol
# ------------------------------------------------------------

xgboost_protocol = {
    "model":
        "XGBRegressor",

    "xgboost_version":
        xgb.__version__,

    "target":
        "P_Solar[kW]",

    "sampling_resolution_minutes":
        5,

    "forecast_horizons_minutes":
        [
            15,
            30,
            60,
        ],

    "feature_families":
        [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ],

    "feature_counts":
        {
            family:
                int(
                    primary_protocol[
                        "feature_counts"
                    ][family]
                )
            for family in [
                "F0",
                "F1",
                "F2",
                "F3",
                "F4",
            ]
        },

    "selection_metric":
        "validation MAE",

    "selection_tiebreaker":
        "validation RMSE",

    "validation_grid": {
        "n_estimators":
            [
                300,
                600,
            ],

        "learning_rate":
            [
                0.03,
                0.05,
            ],

        "max_depth":
            [
                4,
                6,
            ],
    },

    "fixed_model_parameters": {
        key:
            (
                value.item()
                if isinstance(
                    value,
                    np.generic,
                )
                else value
            )
        for key, value
        in XGB_FIXED_PARAMS.items()
    },

    "number_of_grid_configurations":
        8,

    "test_policy":
        (
            "All 15 horizon × feature-family configurations "
            "were selected using chronological validation MAE "
            "before any XGBoost test metric was inspected. "
            "Each frozen model was then refitted once on "
            "train + validation and evaluated on the held-out "
            "chronological test period."
        ),

    "bootstrap": {
        "comparison":
            "F3_minus_F4_MAE",

        "positive_difference_means":
            "F4 lower MAE than F3",

        "block_unit":
            "forecast-valid UTC calendar day",

        "daily_blocks":
            92,

        "replicates":
            10_000,

        "random_seed":
            42,

        "confidence_interval":
            "95% percentile",

        "retraining":
            False,

        "scopes":
            [
                "all",
                "daylight",
            ],

        "aggregation":
            (
                "pooled MAE reconstructed from sampled "
                "daily absolute-error sums and sampled "
                "row counts"
            ),
    },

    "daylight_definition":
        "forecast-valid solar_elevation_deg > 0",

    "numerical_note":
        (
            "Observed XGBoost F3/F4 MAEs were converted from "
            "float32-derived NumPy scalars to Python float "
            "before relative-percentage arithmetic. This "
            "avoids negligible float32/float64 cross-check "
            "differences and does not alter predictions or "
            "the bootstrap procedure."
        ),
}


with open(
    artifact_paths[
        "protocol"
    ],
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        xgboost_protocol,
        f,
        indent=2,
        sort_keys=True,
    )


# ------------------------------------------------------------
# 12. File-hashing helper
# ------------------------------------------------------------

def sha256_file(
    file_path: Path,
) -> str:

    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# ------------------------------------------------------------
# 13. Build artifact manifest
#
# The manifest itself is generated LAST and therefore does not
# hash itself.
# ------------------------------------------------------------

manifest_rows = []


for artifact_name, artifact_path in (
    artifact_paths.items()
):

    if artifact_name == "manifest":
        continue


    assert artifact_path.exists(), (
        f"Missing artifact: "
        f"{artifact_path}"
    )


    manifest_rows.append(
        {
            "artifact_name":
                artifact_name,

            "file_name":
                artifact_path.name,

            "size_bytes":
                artifact_path.stat().st_size,

            "sha256":
                sha256_file(
                    artifact_path
                ),
        }
    )


xgboost_results_manifest = (
    pd.DataFrame(
        manifest_rows
    )
    .sort_values(
        "artifact_name"
    )
    .reset_index(
        drop=True
    )
)


xgboost_results_manifest.to_csv(
    artifact_paths[
        "manifest"
    ],
    index=False,
)


# ------------------------------------------------------------
# 14. Final persistence integrity checks
# ------------------------------------------------------------

expected_artifact_count_before_manifest = 11


assert (
    len(
        xgboost_results_manifest
    )
    == expected_artifact_count_before_manifest
)


assert artifact_paths[
    "manifest"
].exists()


assert (
    pd.read_csv(
        artifact_paths[
            "validation_summary"
        ]
    ).shape[0]
    == 15
)


assert (
    pd.read_csv(
        artifact_paths[
            "frozen_selection"
        ]
    ).shape[0]
    == 15
)


assert (
    pd.read_csv(
        artifact_paths[
            "final_test_results"
        ]
    ).shape[0]
    == 15
)


assert (
    pd.read_csv(
        artifact_paths[
            "f3_f4_test_summary"
        ]
    ).shape[0]
    == 3
)


assert (
    pd.read_csv(
        artifact_paths[
            "bootstrap_summary"
        ]
    ).shape[0]
    == 6
)


assert (
    pd.read_csv(
        artifact_paths[
            "bootstrap_distributions"
        ]
    ).shape[0]
    == 60_000
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    assert (
        pd.read_csv(
            artifact_paths[
                f"test_predictions_{horizon_minutes}"
            ]
        ).shape[0]
        == 26_496
    )


# ------------------------------------------------------------
# 15. Display manifest
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "XGBOOST ARTIFACT MANIFEST"
)

print(
    "============================================\n"
)


display(
    xgboost_results_manifest
)


print(
    "\nSaved files:"
)


for artifact_path in sorted(
    results_dir.glob(
        "xgboost_*"
    )
):

    print(
        f"  {artifact_path.name:<48} "
        f"{artifact_path.stat().st_size:>12,} bytes"
    )


print(
    "\nXGBoost artifact persistence: PASSED"
)

Project root: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project
Results directory: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook10_secondary_ml_models

XGBOOST ARTIFACT MANIFEST



,artifact_name,file_name,size_bytes,sha256
0,bootstrap_distributions,xgboost_bootstrap_distributions.csv.gz,1298410,383a9ae32edd91dbe53d3b965997d44216147fb8211f8b...
1,bootstrap_summary,xgboost_bootstrap_summary.csv,1500,88bc127319f524078b1811ceb87e46df692af9fcb9e7f2...
2,f3_f4_test_summary,xgboost_f3_f4_test_summary.csv,494,11ef2f4abc561ed6ce950c7fe08c7029396de6af1c4b72...
3,final_test_results,xgboost_final_test_results.csv,4287,99835b9e6775ff17d7e8a9bbceb4044fd44dedd335013f...
4,frozen_selection,xgboost_frozen_selection.csv,1141,89ac6f2b70d5bff0b12e22407d111f5424acbff5c80264...
5,protocol,xgboost_protocol.json,2065,28cd629542995d7876d0f7473a669950db6384657f1a4a...
6,selected_configs,xgboost_selected_configs.json,5849,fb56a205b3789f4f1562c005f41107db892f230e679087...
7,test_predictions_15,xgboost_test_predictions_15min.csv,4736184,cc96b19e927400925ee385f2447d1a315595dd8a7ee9ad...
8,test_predictions_30,xgboost_test_predictions_30min.csv,4730545,e5d0ef39eb58a39f6a66ec107bf8da518c0c8b1a8c5d5b...
9,test_predictions_60,xgboost_test_predictions_60min.csv,4698630,006a5c6a3a65071380e0c79735776df4fc1e106f2f7581...



Saved files:
  xgboost_bootstrap_distributions.csv.gz              1,298,410 bytes
  xgboost_bootstrap_summary.csv                           1,500 bytes
  xgboost_f3_f4_test_summary.csv                            494 bytes
  xgboost_final_test_results.csv                          4,287 bytes
  xgboost_frozen_selection.csv                            1,141 bytes
  xgboost_protocol.json                                   2,065 bytes
  xgboost_results_manifest.csv                            1,373 bytes
  xgboost_selected_configs.json                           5,849 bytes
  xgboost_test_predictions_15min.csv                  4,736,184 bytes
  xgboost_test_predictions_30min.csv                  4,730,545 bytes
  xgboost_test_predictions_60min.csv                  4,698,630 bytes
  xgboost_validation_summary.csv                          1,264 bytes

XGBoost artifact persistence: PASSED


In [75]:
# ============================================================
# Step 65: Check LightGBM installation
# ============================================================
#
# Before defining the LightGBM experiment protocol, verify
# whether LightGBM is available in the active project
# environment and record its version.
# ============================================================

try:
    import lightgbm as lgb

    print("LightGBM import: PASSED")
    print("LightGBM version:", lgb.__version__)

except ImportError as exc:

    print("LightGBM import: FAILED")
    print()
    print("Import error:")
    print(exc)

LightGBM import: PASSED
LightGBM version: 4.7.0


In [76]:
# ============================================================
# Step 66: Verify LightGBM in the active notebook kernel
# ============================================================
#
# LightGBM was installed into the project .venv.
#
# This cell verifies that the currently active Jupyter kernel
# is using the same environment before we freeze the LightGBM
# experimental protocol.
# ============================================================


import lightgbm as lgb

from lightgbm import LGBMRegressor


print(
    "LightGBM import: PASSED"
)

print(
    "LightGBM version:",
    lgb.__version__,
)

print(
    "LGBMRegressor import: PASSED"
)


assert (
    lgb.__version__
    == "4.7.0"
)


print(
    "\nLightGBM environment check: PASSED"
)

LightGBM import: PASSED
LightGBM version: 4.7.0
LGBMRegressor import: PASSED

LightGBM environment check: PASSED


In [77]:
# ============================================================
# Step 67: Freeze LightGBM validation protocol
# ============================================================
#
# IMPORTANT:
#
# This grid is defined BEFORE observing any LightGBM
# validation or test performance.
#
# The same frozen grid will be used for every:
#
#       horizon × feature family
#
# combination:
#
#       15, 30, 60 minutes
#       F0, F1, F2, F3, F4
#
#
# Model-selection rule
# --------------------
#
# Primary:
#     minimum chronological validation MAE
#
# Secondary / tie-break:
#     validation RMSE
#
#
# Test policy
# -----------
#
# The TEST period remains untouched until all 15 LightGBM
# configurations have been selected and frozen.
#
#
# Grid dimensions
# ---------------
#
# n_estimators:
#     300, 600
#
# learning_rate:
#     0.03, 0.05
#
# num_leaves:
#     15, 31
#
# Total:
#
#     2 × 2 × 2 = 8 configurations
#
#
# Why num_leaves?
# ---------------
#
# LightGBM grows trees leaf-wise. Therefore num_leaves is one
# of its main model-complexity controls.
#
# We use 15 and 31 as moderate values rather than performing
# a large hyperparameter search.
# ============================================================


# ------------------------------------------------------------
# 1. Fixed LightGBM parameters
# ------------------------------------------------------------

LIGHTGBM_FIXED_PARAMS = {
    # Standard regression objective.
    "objective":
        "regression",

    # Standard gradient-boosted decision trees.
    "boosting_type":
        "gbdt",

    # Unlimited explicit depth.
    #
    # Complexity is controlled mainly through num_leaves.
    "max_depth":
        -1,

    # Row subsampling.
    "subsample":
        0.8,

    # Required for subsample < 1.0 to actually be used.
    "subsample_freq":
        1,

    # Feature subsampling at each tree.
    "colsample_bytree":
        0.8,

    # Minimum observations allowed in a leaf.
    "min_child_samples":
        20,

    # No L1 penalty.
    "reg_alpha":
        0.0,

    # Mild L2 regularization.
    "reg_lambda":
        1.0,

    # Reproducibility.
    "random_state":
        42,

    # Encourage deterministic CPU behaviour.
    "deterministic":
        True,

    # Avoid automatic row-wise / column-wise switching.
    "force_col_wise":
        True,

    # Use available CPU cores.
    "n_jobs":
        -1,

    # Suppress LightGBM training messages.
    "verbosity":
        -1,
}


# ------------------------------------------------------------
# 2. Frozen hyperparameter values
# ------------------------------------------------------------

LIGHTGBM_N_ESTIMATORS = [
    300,
    600,
]

LIGHTGBM_LEARNING_RATES = [
    0.03,
    0.05,
]

LIGHTGBM_NUM_LEAVES = [
    15,
    31,
]


# ------------------------------------------------------------
# 3. Construct the complete frozen 8-configuration grid
# ------------------------------------------------------------

lightgbm_configs = []


for n_estimators in LIGHTGBM_N_ESTIMATORS:

    for learning_rate in LIGHTGBM_LEARNING_RATES:

        for num_leaves in LIGHTGBM_NUM_LEAVES:

            lightgbm_configs.append(
                {
                    "n_estimators":
                        n_estimators,

                    "learning_rate":
                        learning_rate,

                    "num_leaves":
                        num_leaves,
                }
            )


# ------------------------------------------------------------
# 4. Verify exact grid size
# ------------------------------------------------------------

assert len(
    lightgbm_configs
) == 8


# ------------------------------------------------------------
# 5. Human-readable frozen grid
# ------------------------------------------------------------

lightgbm_grid_table = pd.DataFrame(
    lightgbm_configs
)


lightgbm_grid_table.index = (
    np.arange(
        1,
        len(lightgbm_grid_table) + 1,
    )
)


lightgbm_grid_table.index.name = (
    "configuration"
)


print(
    "LightGBM frozen validation grid:\n"
)

display(
    lightgbm_grid_table
)


# ------------------------------------------------------------
# 6. Display fixed parameters
# ------------------------------------------------------------

print(
    "\nFixed LightGBM parameters:"
)


for parameter, value in (
    LIGHTGBM_FIXED_PARAMS.items()
):

    print(
        f"  {parameter:<20} = {value}"
    )


# ------------------------------------------------------------
# 7. Final protocol checks
# ------------------------------------------------------------

assert set(
    lightgbm_grid_table[
        "n_estimators"
    ]
) == {
    300,
    600,
}


assert set(
    lightgbm_grid_table[
        "learning_rate"
    ]
) == {
    0.03,
    0.05,
}


assert set(
    lightgbm_grid_table[
        "num_leaves"
    ]
) == {
    15,
    31,
}


print(
    "\nNumber of frozen configurations:",
    len(lightgbm_configs),
)

print(
    "Selection metric: chronological validation MAE"
)

print(
    "Tie-break metric: validation RMSE"
)

print(
    "Early stopping: NOT used"
)

print(
    "Test set used for selection: NO"
)

print(
    "\nLightGBM validation protocol: FROZEN"
)

LightGBM frozen validation grid:



,n_estimators,learning_rate,num_leaves
configuration,,,
1,300,0.03,15
2,300,0.03,31
3,300,0.05,15
4,300,0.05,31
5,600,0.03,15
6,600,0.03,31
7,600,0.05,15
8,600,0.05,31



Fixed LightGBM parameters:
  objective            = regression
  boosting_type        = gbdt
  max_depth            = -1
  subsample            = 0.8
  subsample_freq       = 1
  colsample_bytree     = 0.8
  min_child_samples    = 20
  reg_alpha            = 0.0
  reg_lambda           = 1.0
  random_state         = 42
  deterministic        = True
  force_col_wise       = True
  n_jobs               = -1
  verbosity            = -1

Number of frozen configurations: 8
Selection metric: chronological validation MAE
Tie-break metric: validation RMSE
Early stopping: NOT used
Test set used for selection: NO

LightGBM validation protocol: FROZEN


In [78]:
# ============================================================
# Step 68: Reusable LightGBM validation-grid helper
# ============================================================
#
# This function evaluates the SAME frozen 8-configuration
# LightGBM grid for any:
#
#       horizon × feature family
#
# combination.
#
# Model selection uses ONLY:
#
#       TRAIN       -> model fitting
#       VALIDATION  -> hyperparameter selection
#
# The TEST set is never accessed by this helper.
#
#
# Selection rule
# --------------
#
# Primary:
#       lowest validation MAE
#
# Secondary:
#       lowest validation RMSE
#
#
# Input matrices
# --------------
#
# The feature matrices were already constructed earlier as:
#
#       feature_matrices[horizon][feature_family]
#
# Targets:
#
#       targets[horizon]
#
# Chronological labels:
#
#       split_labels[horizon]
#
#
# Numerical representation
# ------------------------
#
# Features and targets are converted to float32 before model
# fitting, matching the approach used for XGBoost and reducing
# memory usage.
# ============================================================


def run_lightgbm_validation_grid(
    horizon_minutes: int,
    feature_family: str,
) -> tuple[pd.DataFrame, float]:
    """
    Evaluate the frozen LightGBM hyperparameter grid using
    chronological train/validation data only.

    Parameters
    ----------
    horizon_minutes
        Forecast horizon in minutes.
        Expected values: 15, 30, or 60.

    feature_family
        Feature family identifier.
        Expected values: F0, F1, F2, F3, or F4.

    Returns
    -------
    results_df
        One row per frozen LightGBM configuration, ordered by
        validation MAE and then validation RMSE.

    total_seconds
        Total runtime for the complete 8-configuration grid.
    """

    # --------------------------------------------------------
    # 1. Input validation
    # --------------------------------------------------------

    assert horizon_minutes in {
        15,
        30,
        60,
    }

    assert feature_family in {
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    }


    # --------------------------------------------------------
    # 2. Retrieve modelling objects
    # --------------------------------------------------------

    X = (
        feature_matrices[
            horizon_minutes
        ][
            feature_family
        ]
    )

    y = targets[
        horizon_minutes
    ]

    split = split_labels[
        horizon_minutes
    ]


    # --------------------------------------------------------
    # 3. Chronological TRAIN / VALIDATION masks
    #
    # TEST is deliberately not retrieved or evaluated.
    # --------------------------------------------------------

    train_mask = (
        split == "train"
    )

    validation_mask = (
        split == "validation"
    )


    # --------------------------------------------------------
    # 4. Construct model arrays
    # --------------------------------------------------------

    X_train = (
        X.loc[
            train_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    X_validation = (
        X.loc[
            validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    y_train = (
        y.loc[
            train_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    y_validation = (
        y.loc[
            validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # 5. Structural integrity checks
    # --------------------------------------------------------

    expected_n_features = (
        primary_protocol[
            "feature_counts"
        ][
            feature_family
        ]
    )


    assert (
        X_train.shape[1]
        == expected_n_features
    )

    assert (
        X_validation.shape[1]
        == expected_n_features
    )


    assert (
        X_train.shape[0]
        == int(
            train_mask.sum()
        )
    )

    assert (
        X_validation.shape[0]
        == int(
            validation_mask.sum()
        )
    )


    assert np.isfinite(
        X_train
    ).all()

    assert np.isfinite(
        X_validation
    ).all()

    assert np.isfinite(
        y_train
    ).all()

    assert np.isfinite(
        y_validation
    ).all()


    # --------------------------------------------------------
    # 6. Describe the experiment before fitting
    # --------------------------------------------------------

    print(
        "\nLightGBM validation dataset:"
    )

    print(
        "  Horizon:",
        horizon_minutes,
        "minutes",
    )

    print(
        "  Feature family:",
        feature_family,
    )

    print(
        "  Train:",
        X_train.shape,
    )

    print(
        "  Validation:",
        X_validation.shape,
    )

    print(
        "  Frozen configurations:",
        len(
            lightgbm_configs
        ),
    )


    # --------------------------------------------------------
    # 7. Storage
    # --------------------------------------------------------

    validation_rows = []


    total_start_time = (
        time.perf_counter()
    )


    # --------------------------------------------------------
    # 8. Evaluate all frozen configurations
    # --------------------------------------------------------

    for (
        configuration_number,
        variable_config,
    ) in enumerate(
        lightgbm_configs,
        start=1,
    ):

        # ----------------------------------------------------
        # Combine variable grid parameters with the already
        # frozen fixed LightGBM parameters.
        # ----------------------------------------------------

        model_params = {
            **LIGHTGBM_FIXED_PARAMS,
            **variable_config,
        }


        # ----------------------------------------------------
        # Instantiate model
        # ----------------------------------------------------

        model = LGBMRegressor(
            **model_params
        )


        # ----------------------------------------------------
        # Fit + validation prediction timer
        # ----------------------------------------------------

        fit_start_time = (
            time.perf_counter()
        )


        model.fit(
            X_train,
            y_train,
        )


        validation_predictions = (
            model.predict(
                X_validation
            )
        )


        fit_predict_seconds = (
            time.perf_counter()
            - fit_start_time
        )


        # ----------------------------------------------------
        # 9. Validation metrics
        # ----------------------------------------------------

        validation_mae = (
            mean_absolute_error(
                y_validation,
                validation_predictions,
            )
        )


        validation_rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                validation_predictions,
            )
        )


        # ----------------------------------------------------
        # 10. Store result
        # ----------------------------------------------------

        validation_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_estimators":
                    variable_config[
                        "n_estimators"
                    ],

                "learning_rate":
                    variable_config[
                        "learning_rate"
                    ],

                "num_leaves":
                    variable_config[
                        "num_leaves"
                    ],

                "subsample":
                    LIGHTGBM_FIXED_PARAMS[
                        "subsample"
                    ],

                "subsample_freq":
                    LIGHTGBM_FIXED_PARAMS[
                        "subsample_freq"
                    ],

                "colsample_bytree":
                    LIGHTGBM_FIXED_PARAMS[
                        "colsample_bytree"
                    ],

                "min_child_samples":
                    LIGHTGBM_FIXED_PARAMS[
                        "min_child_samples"
                    ],

                "reg_alpha":
                    LIGHTGBM_FIXED_PARAMS[
                        "reg_alpha"
                    ],

                "reg_lambda":
                    LIGHTGBM_FIXED_PARAMS[
                        "reg_lambda"
                    ],

                "validation_mae":
                    validation_mae,

                "validation_rmse":
                    validation_rmse,

                "fit_predict_seconds":
                    fit_predict_seconds,
            }
        )


        # ----------------------------------------------------
        # 11. Compact progress output
        # ----------------------------------------------------

        print(
            f"\nConfiguration "
            f"{configuration_number}/"
            f"{len(lightgbm_configs)}"
        )

        print(
            "  n_estimators     =",
            variable_config[
                "n_estimators"
            ],
        )

        print(
            "  learning_rate    =",
            variable_config[
                "learning_rate"
            ],
        )

        print(
            "  num_leaves       =",
            variable_config[
                "num_leaves"
            ],
        )

        print(
            f"  validation MAE  = "
            f"{validation_mae:.6f}"
        )

        print(
            f"  validation RMSE = "
            f"{validation_rmse:.6f}"
        )

        print(
            f"  runtime         = "
            f"{fit_predict_seconds:.1f} s"
        )


        # ----------------------------------------------------
        # Release fitted model before next configuration
        # ----------------------------------------------------

        del model
        del validation_predictions

        gc.collect()


    # --------------------------------------------------------
    # 12. Complete grid runtime
    # --------------------------------------------------------

    total_seconds = (
        time.perf_counter()
        - total_start_time
    )


    # --------------------------------------------------------
    # 13. Assemble results
    # --------------------------------------------------------

    results_df = pd.DataFrame(
        validation_rows
    )


    # --------------------------------------------------------
    # 14. Verify exactly eight configurations were evaluated
    # --------------------------------------------------------

    assert len(
        results_df
    ) == 8


    # --------------------------------------------------------
    # 15. Verify every result is finite
    # --------------------------------------------------------

    assert np.isfinite(
        results_df[
            [
                "validation_mae",
                "validation_rmse",
                "fit_predict_seconds",
            ]
        ]
        .to_numpy()
    ).all()


    # --------------------------------------------------------
    # 16. Sort according to the PRE-SPECIFIED selection rule
    #
    # Primary:
    #       validation MAE
    #
    # Secondary:
    #       validation RMSE
    # --------------------------------------------------------

    results_df = (
        results_df
        .sort_values(
            [
                "validation_mae",
                "validation_rmse",
            ],
            ascending=[
                True,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # 17. Cleanup large temporary arrays
    # --------------------------------------------------------

    del X_train
    del X_validation
    del y_train
    del y_validation

    gc.collect()


    return (
        results_df,
        total_seconds,
    )


# ------------------------------------------------------------
# 18. Confirm helper definition
# ------------------------------------------------------------

print(
    "LightGBM validation-grid helper: DEFINED"
)

print(
    "Frozen grid configurations:",
    len(
        lightgbm_configs
    ),
)

print(
    "Selection order:"
    " validation MAE -> validation RMSE"
)

print(
    "Test-set access inside helper: NONE"
)

LightGBM validation-grid helper: DEFINED
Frozen grid configurations: 8
Selection order: validation MAE -> validation RMSE
Test-set access inside helper: NONE


In [79]:
# ============================================================
# Step 69: LightGBM validation
#          60-minute horizon + F0
# ============================================================
#
# F0 contains the historical/autoregressive baseline:
#
#   - recent observed measurements through issue time t;
#   - lagged PV / irradiance / meteorological variables;
#   - 24-hour PV lag.
#
# The same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_60_f0_df, lightgbm_60_f0_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=60,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f0_60 = (
    lightgbm_60_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 60-minute F0 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_60_f0_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f0_60[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f0_60[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f0_60[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f0_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f0_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F0 grid runtime:"
)

print(
    f"  {lightgbm_60_f0_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_60_f0_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 60-minute F0 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 60 minutes
  Feature family: F0
  Train: (87252, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.453819
  validation RMSE = 0.824223
  runtime         = 2.4 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.446507
  validation RMSE = 0.816261
  runtime         = 3.1 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.460221
  validation RMSE = 0.830166
  runtime         = 2.1 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.460770
  validation RMSE = 0.841006
  runtime         = 2.9 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.463120
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,60,F0,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.446507,0.816261,3.086339
1,60,F0,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.453819,0.824223,2.424457
2,60,F0,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.457213,0.829089,5.437396
3,60,F0,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.460221,0.830166,2.116019
4,60,F0,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.460770,0.841006,2.900820
5,60,F0,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.463120,0.838071,4.012778
6,60,F0,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.474686,0.859148,5.357549
7,60,F0,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.475899,0.852627,4.009076



Selected 60-minute F0 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 31
  validation MAE: 0.44650656986330634
  validation RMSE: 0.816261059507079

Total 60-minute F0 grid runtime:
  29.7 seconds
  0.49 minutes

LightGBM 60-minute F0 experiment: PASSED


In [80]:
# ============================================================
# Step 70: LightGBM validation
#          60-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + conventional Gregorian / calendar-time features
#
# The same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_60_f1_df, lightgbm_60_f1_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=60,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f1_60 = (
    lightgbm_60_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 60-minute F1 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_60_f1_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f1_60[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f1_60[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f1_60[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f1_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f1_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_60_mae_lgbm = float(
    best_lightgbm_f0_60[
        "validation_mae"
    ]
)

f1_60_mae_lgbm = float(
    best_lightgbm_f1_60[
        "validation_mae"
    ]
)

f0_to_f1_60_pct_lgbm = (
    (
        f0_60_mae_lgbm
        - f1_60_mae_lgbm
    )
    / f0_60_mae_lgbm
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_60_mae_lgbm:.6f}"
)

print(
    f"  F1 MAE: {f1_60_mae_lgbm:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_60_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F1 grid runtime:"
)

print(
    f"  {lightgbm_60_f1_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_60_f1_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 60-minute F1 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 60 minutes
  Feature family: F1
  Train: (87252, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.369038
  validation RMSE = 0.717467
  runtime         = 3.1 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.368057
  validation RMSE = 0.717782
  runtime         = 3.8 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.369561
  validation RMSE = 0.720196
  runtime         = 2.6 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.376988
  validation RMSE = 0.726424
  runtime         = 3.6 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.374270
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,60,F1,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.368057,0.717782,3.761798
1,60,F1,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.369038,0.717467,3.099125
2,60,F1,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.369561,0.720196,2.631304
3,60,F1,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.374270,0.720313,4.901959
4,60,F1,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.376988,0.726424,3.588179
5,60,F1,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.380351,0.729828,6.575269
6,60,F1,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.383567,0.733482,4.749593
7,60,F1,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.393995,0.747389,6.313790



Selected 60-minute F1 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 31
  validation MAE: 0.3680572240671563
  validation RMSE: 0.7177818631750083

F0 -> F1 validation comparison:
  F0 MAE: 0.446507
  F1 MAE: 0.368057
  Relative MAE improvement: +17.5696%

Total 60-minute F1 grid runtime:
  35.9 seconds
  0.60 minutes

LightGBM 60-minute F1 experiment: PASSED


In [81]:
# ============================================================
# Step 71: LightGBM validation
#          60-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + deterministic solar-geometry features
#
# The same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_60_f2_df, lightgbm_60_f2_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=60,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f2_60 = (
    lightgbm_60_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 60-minute F2 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_60_f2_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f2_60[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f2_60[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f2_60[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f2_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f2_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_60_mae_lgbm = float(
    best_lightgbm_f1_60[
        "validation_mae"
    ]
)

f2_60_mae_lgbm = float(
    best_lightgbm_f2_60[
        "validation_mae"
    ]
)

f1_to_f2_60_pct_lgbm = (
    (
        f1_60_mae_lgbm
        - f2_60_mae_lgbm
    )
    / f1_60_mae_lgbm
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_60_mae_lgbm:.6f}"
)

print(
    f"  F2 MAE: {f2_60_mae_lgbm:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_60_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F2 grid runtime:"
)

print(
    f"  {lightgbm_60_f2_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_60_f2_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 60-minute F2 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 60 minutes
  Feature family: F2
  Train: (87252, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.370246
  validation RMSE = 0.722357
  runtime         = 2.8 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.373981
  validation RMSE = 0.721455
  runtime         = 3.6 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.375904
  validation RMSE = 0.726057
  runtime         = 2.5 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.391326
  validation RMSE = 0.748107
  runtime         = 3.5 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.381528
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,60,F2,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.370246,0.722357,2.801853
1,60,F2,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.373981,0.721455,3.603317
2,60,F2,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.375904,0.726057,2.543061
3,60,F2,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.381528,0.732123,4.717645
4,60,F2,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.390327,0.738304,6.506605
5,60,F2,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.391326,0.748107,3.485488
6,60,F2,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.395351,0.746728,4.591837
7,60,F2,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.406453,0.770399,6.661123



Selected 60-minute F2 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 15
  validation MAE: 0.3702458373941838
  validation RMSE: 0.7223571064692753

F1 -> F2 validation comparison:
  F1 MAE: 0.368057
  F2 MAE: 0.370246
  Relative MAE improvement: -0.5946%

Total 60-minute F2 grid runtime:
  35.2 seconds
  0.59 minutes

LightGBM 60-minute F2 experiment: PASSED


In [82]:
# ============================================================
# Step 72: LightGBM validation
#          60-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + continuous lunar / Sun-Moon astronomical features
#
# The same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_60_f3_df, lightgbm_60_f3_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=60,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f3_60 = (
    lightgbm_60_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 60-minute F3 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_60_f3_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f3_60[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f3_60[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f3_60[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f3_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f3_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_60_mae_lgbm = float(
    best_lightgbm_f2_60[
        "validation_mae"
    ]
)

f3_60_mae_lgbm = float(
    best_lightgbm_f3_60[
        "validation_mae"
    ]
)

f2_to_f3_60_pct_lgbm = (
    (
        f2_60_mae_lgbm
        - f3_60_mae_lgbm
    )
    / f2_60_mae_lgbm
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_60_mae_lgbm:.6f}"
)

print(
    f"  F3 MAE: {f3_60_mae_lgbm:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_60_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F3 grid runtime:"
)

print(
    f"  {lightgbm_60_f3_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_60_f3_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 60-minute F3 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 60 minutes
  Feature family: F3
  Train: (87252, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.385131
  validation RMSE = 0.733488
  runtime         = 3.0 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.385598
  validation RMSE = 0.733149
  runtime         = 4.0 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.390950
  validation RMSE = 0.743922
  runtime         = 2.7 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.396938
  validation RMSE = 0.744360
  runtime         = 3.7 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.392636
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,60,F3,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.385131,0.733488,2.979102
1,60,F3,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.385598,0.733149,3.951248
2,60,F3,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.390950,0.743922,2.739219
3,60,F3,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.392636,0.740610,5.139845
4,60,F3,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.396938,0.744360,3.665036
5,60,F3,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.397474,0.748191,7.042613
6,60,F3,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.401686,0.755031,5.198551
7,60,F3,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.413758,0.765596,6.782410



Selected 60-minute F3 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 15
  validation MAE: 0.3851312899483005
  validation RMSE: 0.733487838411866

F2 -> F3 validation comparison:
  F2 MAE: 0.370246
  F3 MAE: 0.385131
  Relative MAE improvement: -4.0204%

Total 60-minute F3 grid runtime:
  37.8 seconds
  0.63 minutes

LightGBM 60-minute F3 experiment: PASSED


In [83]:
# ============================================================
# Step 73: LightGBM validation
#          60-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + frozen one-hot encoded Vedic/Panchang features
#
# This gives the central:
#
#       F3 -> F4
#
# comparison for the 60-minute horizon.
#
# The same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_60_f4_df, lightgbm_60_f4_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=60,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f4_60 = (
    lightgbm_60_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 60-minute F4 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_60_f4_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f4_60[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f4_60[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f4_60[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f4_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f4_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Central F3 -> F4 validation comparison
# ------------------------------------------------------------

f3_60_mae_lgbm = float(
    best_lightgbm_f3_60[
        "validation_mae"
    ]
)

f4_60_mae_lgbm = float(
    best_lightgbm_f4_60[
        "validation_mae"
    ]
)


f3_to_f4_60_absolute_lgbm = (
    f3_60_mae_lgbm
    - f4_60_mae_lgbm
)


f3_to_f4_60_pct_lgbm = (
    f3_to_f4_60_absolute_lgbm
    / f3_60_mae_lgbm
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_60_mae_lgbm:.6f}"
)

print(
    f"  F4 MAE: {f4_60_mae_lgbm:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_60_absolute_lgbm:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_60_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F4 grid runtime:"
)

print(
    f"  {lightgbm_60_f4_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_60_f4_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 60-minute F4 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 60 minutes
  Feature family: F4
  Train: (87252, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.381773
  validation RMSE = 0.730913
  runtime         = 3.3 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.381886
  validation RMSE = 0.729677
  runtime         = 4.1 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.387569
  validation RMSE = 0.735148
  runtime         = 3.0 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.403218
  validation RMSE = 0.747707
  runtime         = 4.1 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.391389
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,60,F4,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.381773,0.730913,3.259371
1,60,F4,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.381886,0.729677,4.136863
2,60,F4,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.387569,0.735148,2.961949
3,60,F4,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.391389,0.738277,5.454835
4,60,F4,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.394562,0.741241,7.308214
5,60,F4,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.396094,0.742357,5.337152
6,60,F4,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.403218,0.747707,4.059486
7,60,F4,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.416682,0.761996,7.189425



Selected 60-minute F4 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 15
  validation MAE: 0.38177286355500634
  validation RMSE: 0.7309132039926226

F3 -> F4 validation comparison:
  F3 MAE: 0.385131
  F4 MAE: 0.381773
  Absolute MAE reduction: +0.003358
  Relative MAE improvement: +0.8720%

Total 60-minute F4 grid runtime:
  40.0 seconds
  0.67 minutes

LightGBM 60-minute F4 experiment: PASSED


In [84]:
# ============================================================
# Step 74: Summarize the complete 60-minute LightGBM
#          F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect selected configurations
# ------------------------------------------------------------

best_rows_lgbm_60 = {
    "F0": best_lightgbm_f0_60,
    "F1": best_lightgbm_f1_60,
    "F2": best_lightgbm_f2_60,
    "F3": best_lightgbm_f3_60,
    "F4": best_lightgbm_f4_60,
}


summary_rows_lgbm_60 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_lgbm_60[
        feature_family
    ]

    summary_rows_lgbm_60.append(
        {
            "horizon_minutes":
                60,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][feature_family],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    row[
                        "learning_rate"
                    ]
                ),

            "num_leaves":
                int(
                    row[
                        "num_leaves"
                    ]
                ),

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


lightgbm_60_summary = pd.DataFrame(
    summary_rows_lgbm_60
)


# ------------------------------------------------------------
# 2. Add incremental MAE improvement
#
# Positive:
#     new feature family reduced MAE.
#
# Negative:
#     new feature family increased MAE.
# ------------------------------------------------------------

lightgbm_60_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(lightgbm_60_summary),
):

    previous_mae = (
        lightgbm_60_summary.loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = (
        lightgbm_60_summary.loc[
            i,
            "validation_mae",
        ]
    )

    lightgbm_60_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (previous_mae - current_mae)
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Verify central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_lgbm_60 = (
    lightgbm_60_summary.loc[
        lightgbm_60_summary[
            "feature_family"
        ] == "F3",
        "validation_mae",
    ]
    .iloc[0]
)

f4_summary_mae_lgbm_60 = (
    lightgbm_60_summary.loc[
        lightgbm_60_summary[
            "feature_family"
        ] == "F4",
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_lgbm_60 = (
    (
        f3_summary_mae_lgbm_60
        - f4_summary_mae_lgbm_60
    )
    / f3_summary_mae_lgbm_60
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_lgbm_60,
    0.8720,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete summary
# ------------------------------------------------------------

print(
    "LightGBM — 60-minute validation summary\n"
)

display(
    lightgbm_60_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)

for i in range(
    1,
    len(lightgbm_60_summary),
):

    previous_family = (
        lightgbm_60_summary.loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        lightgbm_60_summary.loc[
            i,
            "feature_family",
        ]
    )

    change = (
        lightgbm_60_summary.loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )

    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_lgbm_60 = (
    lightgbm_60_summary
    .sort_values(
        "validation_mae"
    )
    .iloc[0]
)


print(
    "\nLowest 60-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_lgbm_60[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_lgbm_60[
        "validation_mae"
    ],
)


print(
    "\nCentral LightGBM comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_lgbm_60:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_lgbm_60:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_lgbm_60:+.4f}%"
)


print(
    "\nLightGBM 60-minute summary: PASSED"
)

LightGBM — 60-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,num_leaves,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,60,F0,118,300,0.03,31,0.446507,0.816261,NaN
1,60,F1,133,300,0.03,31,0.368057,0.717782,17.569584
2,60,F2,138,300,0.03,15,0.370246,0.722357,-0.594639
3,60,F3,147,300,0.03,15,0.385131,0.733488,-4.020424
4,60,F4,244,300,0.03,15,0.381773,0.730913,0.872021


Incremental validation MAE changes:
  F0 -> F1: +17.5696%
  F1 -> F2: -0.5946%
  F2 -> F3: -4.0204%
  F3 -> F4: +0.8720%

Lowest 60-minute validation MAE:
  Feature family: F1
  MAE: 0.3680572240671563

Central LightGBM comparison:
  F3 MAE: 0.385131
  F4 MAE: 0.381773
  F3 -> F4 change: +0.8720%

LightGBM 60-minute summary: PASSED


In [85]:
# ============================================================
# Step 75: LightGBM validation
#          30-minute horizon + F0
# ============================================================
#
# F0 contains the historical/autoregressive baseline:
#
#   - recent observed measurements through issue time t;
#   - lagged PV / irradiance / meteorological variables;
#   - 24-hour PV lag.
#
# The exact same frozen 8-configuration LightGBM grid is
# reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_30_f0_df, lightgbm_30_f0_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=30,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f0_30 = (
    lightgbm_30_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 30-minute F0 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_30_f0_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f0_30[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f0_30[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f0_30[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f0_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f0_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F0 grid runtime:"
)

print(
    f"  {lightgbm_30_f0_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_30_f0_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 30-minute F0 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 30 minutes
  Feature family: F0
  Train: (87258, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.352128
  validation RMSE = 0.697519
  runtime         = 2.5 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.351821
  validation RMSE = 0.700336
  runtime         = 3.0 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.358480
  validation RMSE = 0.707066
  runtime         = 2.1 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.365055
  validation RMSE = 0.720698
  runtime         = 2.8 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.357563
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,30,F0,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.351821,0.700336,3.029064
1,30,F0,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.352128,0.697519,2.470623
2,30,F0,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.357563,0.710284,4.043422
3,30,F0,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.358480,0.707066,2.082127
4,30,F0,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.362705,0.717302,5.364530
5,30,F0,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.365055,0.720698,2.825796
6,30,F0,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.368494,0.722917,3.814282
7,30,F0,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.379600,0.742398,5.188003



Selected 30-minute F0 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 31
  validation MAE: 0.35182081734714643
  validation RMSE: 0.7003364277639352

Total 30-minute F0 grid runtime:
  29.1 seconds
  0.49 minutes

LightGBM 30-minute F0 experiment: PASSED


In [86]:
# ============================================================
# Step 76: LightGBM validation
#          30-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + conventional Gregorian / calendar-time features
#
# The exact same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_30_f1_df, lightgbm_30_f1_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=30,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f1_30 = (
    lightgbm_30_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 30-minute F1 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_30_f1_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f1_30[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f1_30[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f1_30[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f1_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f1_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_30_mae_lgbm = float(
    best_lightgbm_f0_30[
        "validation_mae"
    ]
)

f1_30_mae_lgbm = float(
    best_lightgbm_f1_30[
        "validation_mae"
    ]
)


f0_to_f1_30_pct_lgbm = (
    (
        f0_30_mae_lgbm
        - f1_30_mae_lgbm
    )
    / f0_30_mae_lgbm
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_30_mae_lgbm:.6f}"
)

print(
    f"  F1 MAE: {f1_30_mae_lgbm:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_30_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F1 grid runtime:"
)

print(
    f"  {lightgbm_30_f1_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_30_f1_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 30-minute F1 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 30 minutes
  Feature family: F1
  Train: (87258, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.310495
  validation RMSE = 0.640372
  runtime         = 3.0 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.314441
  validation RMSE = 0.645223
  runtime         = 3.7 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.314878
  validation RMSE = 0.645918
  runtime         = 2.5 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.321758
  validation RMSE = 0.655685
  runtime         = 3.4 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.319994
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,30,F1,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.310495,0.640372,2.962275
1,30,F1,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.314441,0.645223,3.671643
2,30,F1,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.314878,0.645918,2.498154
3,30,F1,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.319994,0.647621,4.686134
4,30,F1,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.321758,0.655685,3.409585
5,30,F1,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.327672,0.658404,6.336723
6,30,F1,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.332549,0.664453,4.448659
7,30,F1,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.338928,0.679675,6.159400



Selected 30-minute F1 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 15
  validation MAE: 0.31049477501479233
  validation RMSE: 0.6403719061135197

F0 -> F1 validation comparison:
  F0 MAE: 0.351821
  F1 MAE: 0.310495
  Relative MAE improvement: +11.7463%

Total 30-minute F1 grid runtime:
  34.5 seconds
  0.57 minutes

LightGBM 30-minute F1 experiment: PASSED


In [87]:
# ============================================================
# Step 77: LightGBM validation
#          30-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + deterministic solar-geometry features
#
# The exact same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_30_f2_df, lightgbm_30_f2_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=30,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f2_30 = (
    lightgbm_30_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 30-minute F2 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_30_f2_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f2_30[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f2_30[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f2_30[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f2_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f2_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_30_mae_lgbm = float(
    best_lightgbm_f1_30[
        "validation_mae"
    ]
)

f2_30_mae_lgbm = float(
    best_lightgbm_f2_30[
        "validation_mae"
    ]
)


f1_to_f2_30_pct_lgbm = (
    (
        f1_30_mae_lgbm
        - f2_30_mae_lgbm
    )
    / f1_30_mae_lgbm
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_30_mae_lgbm:.6f}"
)

print(
    f"  F2 MAE: {f2_30_mae_lgbm:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_30_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F2 grid runtime:"
)

print(
    f"  {lightgbm_30_f2_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_30_f2_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 30-minute F2 experiment: PASSED"
)



LightGBM validation dataset:
  Horizon: 30 minutes
  Feature family: F2
  Train: (87258, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.310520
  validation RMSE = 0.642112
  runtime         = 2.8 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.311133
  validation RMSE = 0.642911
  runtime         = 3.6 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.316761
  validation RMSE = 0.640576
  runtime         = 2.6 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.319157
  validation RMSE = 0.648123
  runtime         = 3.5 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.318491
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,30,F2,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.310520,0.642112,2.792415
1,30,F2,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.311133,0.642911,3.581372
2,30,F2,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.316761,0.640576,2.577949
3,30,F2,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.318491,0.647616,4.769633
4,30,F2,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.319157,0.648123,3.469366
5,30,F2,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.321482,0.653335,6.449915
6,30,F2,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.328535,0.652755,4.579626
7,30,F2,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.332883,0.665687,6.261836



Selected 30-minute F2 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 15
  validation MAE: 0.3105199890647684
  validation RMSE: 0.642112224492222

F1 -> F2 validation comparison:
  F1 MAE: 0.310495
  F2 MAE: 0.310520
  Relative MAE improvement: -0.0081%

Total 30-minute F2 grid runtime:
  34.8 seconds
  0.58 minutes

LightGBM 30-minute F2 experiment: PASSED


In [88]:
# ============================================================
# Step 78: LightGBM validation
#          30-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + continuous lunar / Sun-Moon astronomical features
#
# The exact same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_30_f3_df, lightgbm_30_f3_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=30,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f3_30 = (
    lightgbm_30_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 30-minute F3 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_30_f3_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f3_30[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f3_30[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f3_30[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f3_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f3_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_30_mae_lgbm = float(
    best_lightgbm_f2_30[
        "validation_mae"
    ]
)

f3_30_mae_lgbm = float(
    best_lightgbm_f3_30[
        "validation_mae"
    ]
)


f2_to_f3_30_pct_lgbm = (
    (
        f2_30_mae_lgbm
        - f3_30_mae_lgbm
    )
    / f2_30_mae_lgbm
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_30_mae_lgbm:.6f}"
)

print(
    f"  F3 MAE: {f3_30_mae_lgbm:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_30_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F3 grid runtime:"
)

print(
    f"  {lightgbm_30_f3_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_30_f3_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 30-minute F3 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 30 minutes
  Feature family: F3
  Train: (87258, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.314208
  validation RMSE = 0.643084
  runtime         = 3.1 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.318301
  validation RMSE = 0.646866
  runtime         = 3.8 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.323232
  validation RMSE = 0.650472
  runtime         = 2.7 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.324466
  validation RMSE = 0.653912
  runtime         = 3.6 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.325969
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,30,F3,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.314208,0.643084,3.074613
1,30,F3,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.318301,0.646866,3.813861
2,30,F3,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.323232,0.650472,2.701655
3,30,F3,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.324466,0.653912,3.590303
4,30,F3,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.325969,0.651973,5.056083
5,30,F3,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.330624,0.659847,6.941811
6,30,F3,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.334164,0.665848,6.700668
7,30,F3,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.337022,0.665338,4.890959



Selected 30-minute F3 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 15
  validation MAE: 0.31420833051316605
  validation RMSE: 0.6430836970514776

F2 -> F3 validation comparison:
  F2 MAE: 0.310520
  F3 MAE: 0.314208
  Relative MAE improvement: -1.1878%

Total 30-minute F3 grid runtime:
  37.0 seconds
  0.62 minutes

LightGBM 30-minute F3 experiment: PASSED


In [89]:
# ============================================================
# Step 79: LightGBM validation
#          30-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + frozen one-hot encoded Vedic/Panchang features
#
# This gives the central:
#
#       F3 -> F4
#
# comparison for the 30-minute horizon.
#
# The exact same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_30_f4_df, lightgbm_30_f4_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=30,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f4_30 = (
    lightgbm_30_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 30-minute F4 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_30_f4_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f4_30[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f4_30[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f4_30[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f4_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f4_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Central F3 -> F4 validation comparison
# ------------------------------------------------------------

f3_30_mae_lgbm = float(
    best_lightgbm_f3_30[
        "validation_mae"
    ]
)

f4_30_mae_lgbm = float(
    best_lightgbm_f4_30[
        "validation_mae"
    ]
)


f3_to_f4_30_absolute_lgbm = (
    f3_30_mae_lgbm
    - f4_30_mae_lgbm
)


f3_to_f4_30_pct_lgbm = (
    f3_to_f4_30_absolute_lgbm
    / f3_30_mae_lgbm
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_30_mae_lgbm:.6f}"
)

print(
    f"  F4 MAE: {f4_30_mae_lgbm:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_30_absolute_lgbm:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_30_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F4 grid runtime:"
)

print(
    f"  {lightgbm_30_f4_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_30_f4_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 30-minute F4 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 30 minutes
  Feature family: F4
  Train: (87258, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.315966
  validation RMSE = 0.645094
  runtime         = 3.3 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.321085
  validation RMSE = 0.648775
  runtime         = 4.0 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.325259
  validation RMSE = 0.652749
  runtime         = 3.0 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.327592
  validation RMSE = 0.656869
  runtime         = 3.8 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.328344
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,30,F4,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.315966,0.645094,3.332276
1,30,F4,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.321085,0.648775,3.954136
2,30,F4,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.325259,0.652749,2.982149
3,30,F4,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.327592,0.656869,3.788407
4,30,F4,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.328344,0.654386,5.273222
5,30,F4,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.334080,0.663667,7.106463
6,30,F4,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.338356,0.669664,5.387308
7,30,F4,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.343549,0.677702,6.990370



Selected 30-minute F4 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 15
  validation MAE: 0.31596607733784954
  validation RMSE: 0.6450938295632191

F3 -> F4 validation comparison:
  F3 MAE: 0.314208
  F4 MAE: 0.315966
  Absolute MAE reduction: -0.001758
  Relative MAE improvement: -0.5594%

Total 30-minute F4 grid runtime:
  39.1 seconds
  0.65 minutes

LightGBM 30-minute F4 experiment: PASSED


In [90]:
# ============================================================
# Step 80: Summarize the complete 30-minute LightGBM
#          F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect selected configurations
# ------------------------------------------------------------

best_rows_lgbm_30 = {
    "F0": best_lightgbm_f0_30,
    "F1": best_lightgbm_f1_30,
    "F2": best_lightgbm_f2_30,
    "F3": best_lightgbm_f3_30,
    "F4": best_lightgbm_f4_30,
}


summary_rows_lgbm_30 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_lgbm_30[
        feature_family
    ]

    summary_rows_lgbm_30.append(
        {
            "horizon_minutes":
                30,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][feature_family],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    row[
                        "learning_rate"
                    ]
                ),

            "num_leaves":
                int(
                    row[
                        "num_leaves"
                    ]
                ),

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


lightgbm_30_summary = pd.DataFrame(
    summary_rows_lgbm_30
)


# ------------------------------------------------------------
# 2. Add incremental MAE improvement
#
# Positive:
#     new feature family reduced MAE.
#
# Negative:
#     new feature family increased MAE.
# ------------------------------------------------------------

lightgbm_30_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(lightgbm_30_summary),
):

    previous_mae = (
        lightgbm_30_summary.loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = (
        lightgbm_30_summary.loc[
            i,
            "validation_mae",
        ]
    )

    lightgbm_30_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (previous_mae - current_mae)
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Verify central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_lgbm_30 = (
    lightgbm_30_summary.loc[
        lightgbm_30_summary[
            "feature_family"
        ] == "F3",
        "validation_mae",
    ]
    .iloc[0]
)

f4_summary_mae_lgbm_30 = (
    lightgbm_30_summary.loc[
        lightgbm_30_summary[
            "feature_family"
        ] == "F4",
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_lgbm_30 = (
    (
        f3_summary_mae_lgbm_30
        - f4_summary_mae_lgbm_30
    )
    / f3_summary_mae_lgbm_30
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_lgbm_30,
    -0.5594,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete summary
# ------------------------------------------------------------

print(
    "LightGBM — 30-minute validation summary\n"
)

display(
    lightgbm_30_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)


for i in range(
    1,
    len(lightgbm_30_summary),
):

    previous_family = (
        lightgbm_30_summary.loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        lightgbm_30_summary.loc[
            i,
            "feature_family",
        ]
    )

    change = (
        lightgbm_30_summary.loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )

    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_lgbm_30 = (
    lightgbm_30_summary
    .sort_values(
        "validation_mae"
    )
    .iloc[0]
)


print(
    "\nLowest 30-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_lgbm_30[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_lgbm_30[
        "validation_mae"
    ],
)


# ------------------------------------------------------------
# 7. Central F3 -> F4 comparison
# ------------------------------------------------------------

print(
    "\nCentral LightGBM comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_lgbm_30:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_lgbm_30:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_lgbm_30:+.4f}%"
)


print(
    "\nLightGBM 30-minute summary: PASSED"
)

LightGBM — 30-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,num_leaves,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,30,F0,118,300,0.03,31,0.351821,0.700336,NaN
1,30,F1,133,300,0.03,15,0.310495,0.640372,11.746332
2,30,F2,138,300,0.03,15,0.310520,0.642112,-0.008121
3,30,F3,147,300,0.03,15,0.314208,0.643084,-1.187795
4,30,F4,244,300,0.03,15,0.315966,0.645094,-0.559421


Incremental validation MAE changes:
  F0 -> F1: +11.7463%
  F1 -> F2: -0.0081%
  F2 -> F3: -1.1878%
  F3 -> F4: -0.5594%

Lowest 30-minute validation MAE:
  Feature family: F1
  MAE: 0.31049477501479233

Central LightGBM comparison:
  F3 MAE: 0.314208
  F4 MAE: 0.315966
  F3 -> F4 change: -0.5594%

LightGBM 30-minute summary: PASSED


In [91]:
# ============================================================
# Step 81: LightGBM validation
#          15-minute horizon + F0
# ============================================================
#
# F0 contains the historical/autoregressive baseline:
#
#   - recent observed measurements through issue time t;
#   - lagged PV / irradiance / meteorological variables;
#   - 24-hour PV lag.
#
# The exact same frozen 8-configuration LightGBM grid is
# reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_15_f0_df, lightgbm_15_f0_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=15,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f0_15 = (
    lightgbm_15_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 15-minute F0 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_15_f0_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f0_15[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f0_15[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f0_15[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f0_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f0_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F0 grid runtime:"
)

print(
    f"  {lightgbm_15_f0_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_15_f0_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 15-minute F0 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 15 minutes
  Feature family: F0
  Train: (87261, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.285558
  validation RMSE = 0.609015
  runtime         = 2.5 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.282612
  validation RMSE = 0.607759
  runtime         = 2.8 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.289019
  validation RMSE = 0.617813
  runtime         = 2.0 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.292670
  validation RMSE = 0.619920
  runtime         = 2.7 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.290521
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,15,F0,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.282612,0.607759,2.806006
1,15,F0,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.285558,0.609015,2.510649
2,15,F0,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.289019,0.617813,2.004191
3,15,F0,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.290350,0.619813,5.170485
4,15,F0,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.290521,0.617701,3.805495
5,15,F0,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.292670,0.619920,2.669034
6,15,F0,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.294799,0.628148,3.717532
7,15,F0,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.301405,0.631213,4.973460



Selected 15-minute F0 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 31
  validation MAE: 0.28261201141027903
  validation RMSE: 0.6077593710784291

Total 15-minute F0 grid runtime:
  27.9 seconds
  0.47 minutes

LightGBM 15-minute F0 experiment: PASSED


In [92]:
# ============================================================
# Step 82: LightGBM validation
#          15-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + conventional Gregorian / calendar-time features
#
# The exact same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_15_f1_df, lightgbm_15_f1_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=15,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f1_15 = (
    lightgbm_15_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 15-minute F1 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_15_f1_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f1_15[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f1_15[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f1_15[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f1_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f1_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_15_mae_lgbm = float(
    best_lightgbm_f0_15[
        "validation_mae"
    ]
)

f1_15_mae_lgbm = float(
    best_lightgbm_f1_15[
        "validation_mae"
    ]
)


f0_to_f1_15_pct_lgbm = (
    (
        f0_15_mae_lgbm
        - f1_15_mae_lgbm
    )
    / f0_15_mae_lgbm
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_15_mae_lgbm:.6f}"
)

print(
    f"  F1 MAE: {f1_15_mae_lgbm:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_15_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F1 grid runtime:"
)

print(
    f"  {lightgbm_15_f1_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_15_f1_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 15-minute F1 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 15 minutes
  Feature family: F1
  Train: (87261, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.264461
  validation RMSE = 0.583356
  runtime         = 2.9 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.265378
  validation RMSE = 0.582204
  runtime         = 3.5 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.272096
  validation RMSE = 0.594666
  runtime         = 2.5 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.271261
  validation RMSE = 0.588304
  runtime         = 3.4 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.272154
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,15,F1,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.264461,0.583356,2.862164
1,15,F1,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.265378,0.582204,3.457789
2,15,F1,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.271261,0.588304,3.423044
3,15,F1,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.272096,0.594666,2.490149
4,15,F1,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.272154,0.590287,4.556821
5,15,F1,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.274414,0.591651,6.284280
6,15,F1,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.279983,0.601410,4.504739
7,15,F1,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.283817,0.601606,6.099766



Selected 15-minute F1 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 15
  validation MAE: 0.26446127242253603
  validation RMSE: 0.5833560584633894

F0 -> F1 validation comparison:
  F0 MAE: 0.282612
  F1 MAE: 0.264461
  Relative MAE improvement: +6.4225%

Total 15-minute F1 grid runtime:
  34.0 seconds
  0.57 minutes

LightGBM 15-minute F1 experiment: PASSED


In [93]:
# ============================================================
# Step 83: LightGBM validation
#          15-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + deterministic solar-geometry features
#
# The exact same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_15_f2_df, lightgbm_15_f2_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=15,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f2_15 = (
    lightgbm_15_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 15-minute F2 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_15_f2_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f2_15[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f2_15[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f2_15[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f2_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f2_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_15_mae_lgbm = float(
    best_lightgbm_f1_15[
        "validation_mae"
    ]
)

f2_15_mae_lgbm = float(
    best_lightgbm_f2_15[
        "validation_mae"
    ]
)


f1_to_f2_15_pct_lgbm = (
    (
        f1_15_mae_lgbm
        - f2_15_mae_lgbm
    )
    / f1_15_mae_lgbm
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_15_mae_lgbm:.6f}"
)

print(
    f"  F2 MAE: {f2_15_mae_lgbm:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_15_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F2 grid runtime:"
)

print(
    f"  {lightgbm_15_f2_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_15_f2_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 15-minute F2 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 15 minutes
  Feature family: F2
  Train: (87261, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.269910
  validation RMSE = 0.585209
  runtime         = 3.0 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.268188
  validation RMSE = 0.584637
  runtime         = 3.5 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.274302
  validation RMSE = 0.593253
  runtime         = 2.5 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.275555
  validation RMSE = 0.596049
  runtime         = 3.5 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.278174
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,15,F2,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.268188,0.584637,3.488897
1,15,F2,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.269910,0.585209,2.981139
2,15,F2,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.274302,0.593253,2.529001
3,15,F2,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.275555,0.596049,3.505320
4,15,F2,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.276760,0.592012,6.454734
5,15,F2,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.278174,0.593697,4.639604
6,15,F2,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.285776,0.607609,4.575008
7,15,F2,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.286115,0.607970,6.277986



Selected 15-minute F2 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 31
  validation MAE: 0.2681883846366148
  validation RMSE: 0.58463746504452

F1 -> F2 validation comparison:
  F1 MAE: 0.264461
  F2 MAE: 0.268188
  Relative MAE improvement: -1.4093%

Total 15-minute F2 grid runtime:
  34.7 seconds
  0.58 minutes

LightGBM 15-minute F2 experiment: PASSED


In [94]:
# ============================================================
# Step 84: LightGBM validation
#          15-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + continuous lunar / Sun-Moon astronomical features
#
# The exact same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_15_f3_df, lightgbm_15_f3_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=15,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f3_15 = (
    lightgbm_15_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 15-minute F3 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_15_f3_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f3_15[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f3_15[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f3_15[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f3_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f3_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_15_mae_lgbm = float(
    best_lightgbm_f2_15[
        "validation_mae"
    ]
)

f3_15_mae_lgbm = float(
    best_lightgbm_f3_15[
        "validation_mae"
    ]
)


f2_to_f3_15_pct_lgbm = (
    (
        f2_15_mae_lgbm
        - f3_15_mae_lgbm
    )
    / f2_15_mae_lgbm
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_15_mae_lgbm:.6f}"
)

print(
    f"  F3 MAE: {f3_15_mae_lgbm:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_15_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F3 grid runtime:"
)

print(
    f"  {lightgbm_15_f3_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_15_f3_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 15-minute F3 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 15 minutes
  Feature family: F3
  Train: (87261, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.271470
  validation RMSE = 0.584401
  runtime         = 3.1 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.268416
  validation RMSE = 0.583245
  runtime         = 3.7 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.281034
  validation RMSE = 0.594818
  runtime         = 2.8 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.283126
  validation RMSE = 0.596787
  runtime         = 3.6 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.281886
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,15,F3,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.268416,0.583245,3.670070
1,15,F3,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.271470,0.584401,3.108506
2,15,F3,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.278808,0.593564,6.727205
3,15,F3,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.281034,0.594818,2.768613
4,15,F3,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.281886,0.595131,5.101476
5,15,F3,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.283126,0.596787,3.581471
6,15,F3,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.292621,0.609399,6.880500
7,15,F3,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.296988,0.617010,4.863224



Selected 15-minute F3 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 31
  validation MAE: 0.268416106663904
  validation RMSE: 0.583244789045758

F2 -> F3 validation comparison:
  F2 MAE: 0.268188
  F3 MAE: 0.268416
  Relative MAE improvement: -0.0849%

Total 15-minute F3 grid runtime:
  37.0 seconds
  0.62 minutes

LightGBM 15-minute F3 experiment: PASSED


In [95]:
# ============================================================
# Step 85: LightGBM validation
#          15-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + frozen one-hot encoded Vedic/Panchang features
#
# This gives the central:
#
#       F3 -> F4
#
# comparison for the 15-minute horizon.
#
# The exact same frozen 8-configuration LightGBM grid is used.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LightGBM validation grid
# ------------------------------------------------------------

lightgbm_15_f4_df, lightgbm_15_f4_seconds = (
    run_lightgbm_validation_grid(
        horizon_minutes=15,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_lightgbm_f4_15 = (
    lightgbm_15_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "LightGBM — 15-minute F4 validation results"
)

print(
    "============================================\n"
)


display(
    lightgbm_15_f4_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_lightgbm_f4_15[
            "n_estimators"
        ]
    ),
)

print(
    "  learning_rate:",
    float(
        best_lightgbm_f4_15[
            "learning_rate"
        ]
    ),
)

print(
    "  num_leaves:",
    int(
        best_lightgbm_f4_15[
            "num_leaves"
        ]
    ),
)

print(
    "  validation MAE:",
    best_lightgbm_f4_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_lightgbm_f4_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Central F3 -> F4 validation comparison
# ------------------------------------------------------------

f3_15_mae_lgbm = float(
    best_lightgbm_f3_15[
        "validation_mae"
    ]
)

f4_15_mae_lgbm = float(
    best_lightgbm_f4_15[
        "validation_mae"
    ]
)


f3_to_f4_15_absolute_lgbm = (
    f3_15_mae_lgbm
    - f4_15_mae_lgbm
)


f3_to_f4_15_pct_lgbm = (
    f3_to_f4_15_absolute_lgbm
    / f3_15_mae_lgbm
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_15_mae_lgbm:.6f}"
)

print(
    f"  F4 MAE: {f4_15_mae_lgbm:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_15_absolute_lgbm:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_15_pct_lgbm:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F4 grid runtime:"
)

print(
    f"  {lightgbm_15_f4_seconds:.1f} seconds"
)

print(
    f"  {lightgbm_15_f4_seconds / 60:.2f} minutes"
)


print(
    "\nLightGBM 15-minute F4 experiment: PASSED"
)


LightGBM validation dataset:
  Horizon: 15 minutes
  Feature family: F4
  Train: (87261, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.269531
  validation RMSE = 0.583488
  runtime         = 3.2 s

Configuration 2/8
  n_estimators     = 300
  learning_rate    = 0.03
  num_leaves       = 31
  validation MAE  = 0.269555
  validation RMSE = 0.583051
  runtime         = 4.0 s

Configuration 3/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 15
  validation MAE  = 0.273850
  validation RMSE = 0.590517
  runtime         = 2.9 s

Configuration 4/8
  n_estimators     = 300
  learning_rate    = 0.05
  num_leaves       = 31
  validation MAE  = 0.279568
  validation RMSE = 0.595523
  runtime         = 3.8 s

Configuration 5/8
  n_estimators     = 600
  learning_rate    = 0.03
  num_leaves       = 15
  validation MAE  = 0.281583
  validation RMS

,horizon_minutes,feature_family,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,reg_alpha,reg_lambda,validation_mae,validation_rmse,fit_predict_seconds
0,15,F4,300,0.03,15,0.8,1,0.8,20,0.0,1.0,0.269531,0.583488,3.209620
1,15,F4,300,0.03,31,0.8,1,0.8,20,0.0,1.0,0.269555,0.583051,4.028985
2,15,F4,300,0.05,15,0.8,1,0.8,20,0.0,1.0,0.273850,0.590517,2.861929
3,15,F4,300,0.05,31,0.8,1,0.8,20,0.0,1.0,0.279568,0.595523,3.830240
4,15,F4,600,0.03,31,0.8,1,0.8,20,0.0,1.0,0.280968,0.593299,7.044203
5,15,F4,600,0.03,15,0.8,1,0.8,20,0.0,1.0,0.281583,0.594501,5.506122
6,15,F4,600,0.05,15,0.8,1,0.8,20,0.0,1.0,0.287817,0.606134,5.263962
7,15,F4,600,0.05,31,0.8,1,0.8,20,0.0,1.0,0.290173,0.607564,6.833704



Selected 15-minute F4 configuration:
  n_estimators: 300
  learning_rate: 0.03
  num_leaves: 15
  validation MAE: 0.26953138199384785
  validation RMSE: 0.583487976009674

F3 -> F4 validation comparison:
  F3 MAE: 0.268416
  F4 MAE: 0.269531
  Absolute MAE reduction: -0.001115
  Relative MAE improvement: -0.4155%

Total 15-minute F4 grid runtime:
  38.9 seconds
  0.65 minutes

LightGBM 15-minute F4 experiment: PASSED


In [96]:
# ============================================================
# Step 86: Summarize the complete 15-minute LightGBM
#          F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect selected configurations
# ------------------------------------------------------------

best_rows_lgbm_15 = {
    "F0": best_lightgbm_f0_15,
    "F1": best_lightgbm_f1_15,
    "F2": best_lightgbm_f2_15,
    "F3": best_lightgbm_f3_15,
    "F4": best_lightgbm_f4_15,
}


summary_rows_lgbm_15 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_lgbm_15[
        feature_family
    ]

    summary_rows_lgbm_15.append(
        {
            "horizon_minutes":
                15,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][feature_family],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    row[
                        "learning_rate"
                    ]
                ),

            "num_leaves":
                int(
                    row[
                        "num_leaves"
                    ]
                ),

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


lightgbm_15_summary = pd.DataFrame(
    summary_rows_lgbm_15
)


# ------------------------------------------------------------
# 2. Add incremental MAE improvement
#
# Positive:
#     new feature family reduced MAE.
#
# Negative:
#     new feature family increased MAE.
# ------------------------------------------------------------

lightgbm_15_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(lightgbm_15_summary),
):

    previous_mae = (
        lightgbm_15_summary.loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = (
        lightgbm_15_summary.loc[
            i,
            "validation_mae",
        ]
    )

    lightgbm_15_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (previous_mae - current_mae)
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Verify central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_lgbm_15 = (
    lightgbm_15_summary.loc[
        lightgbm_15_summary[
            "feature_family"
        ] == "F3",
        "validation_mae",
    ]
    .iloc[0]
)

f4_summary_mae_lgbm_15 = (
    lightgbm_15_summary.loc[
        lightgbm_15_summary[
            "feature_family"
        ] == "F4",
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_lgbm_15 = (
    (
        f3_summary_mae_lgbm_15
        - f4_summary_mae_lgbm_15
    )
    / f3_summary_mae_lgbm_15
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_lgbm_15,
    -0.4155,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete summary
# ------------------------------------------------------------

print(
    "LightGBM — 15-minute validation summary\n"
)

display(
    lightgbm_15_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)


for i in range(
    1,
    len(lightgbm_15_summary),
):

    previous_family = (
        lightgbm_15_summary.loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        lightgbm_15_summary.loc[
            i,
            "feature_family",
        ]
    )

    change = (
        lightgbm_15_summary.loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )

    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_lgbm_15 = (
    lightgbm_15_summary
    .sort_values(
        "validation_mae"
    )
    .iloc[0]
)


print(
    "\nLowest 15-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_lgbm_15[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_lgbm_15[
        "validation_mae"
    ],
)


# ------------------------------------------------------------
# 7. Central F3 -> F4 comparison
# ------------------------------------------------------------

print(
    "\nCentral LightGBM comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_lgbm_15:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_lgbm_15:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_lgbm_15:+.4f}%"
)


print(
    "\nLightGBM 15-minute summary: PASSED"
)

LightGBM — 15-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,num_leaves,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,15,F0,118,300,0.03,31,0.282612,0.607759,NaN
1,15,F1,133,300,0.03,15,0.264461,0.583356,6.422494
2,15,F2,138,300,0.03,31,0.268188,0.584637,-1.409322
3,15,F3,147,300,0.03,31,0.268416,0.583245,-0.084911
4,15,F4,244,300,0.03,15,0.269531,0.583488,-0.415502


Incremental validation MAE changes:
  F0 -> F1: +6.4225%
  F1 -> F2: -1.4093%
  F2 -> F3: -0.0849%
  F3 -> F4: -0.4155%

Lowest 15-minute validation MAE:
  Feature family: F1
  MAE: 0.26446127242253603

Central LightGBM comparison:
  F3 MAE: 0.268416
  F4 MAE: 0.269531
  F3 -> F4 change: -0.4155%

LightGBM 15-minute summary: PASSED


In [97]:
# ============================================================
# Step 87: Combine the complete LightGBM validation study
#          and freeze all selected configurations
# ============================================================
#
# At this checkpoint:
#
#   - all 15 LightGBM validation experiments are complete;
#   - model selection used chronological validation MAE only;
#   - validation RMSE was the secondary / tie-break metric;
#   - no LightGBM test metric has been examined.
#
# We now freeze one selected configuration for every:
#
#       horizon × feature family
#
# combination.
#
# Only AFTER this registry is frozen may we refit on:
#
#       TRAIN + VALIDATION
#
# and evaluate the held-out chronological TEST period.
# ============================================================


# ------------------------------------------------------------
# 1. Combine 15-, 30-, and 60-minute validation summaries
# ------------------------------------------------------------

lightgbm_validation_summary = (
    pd.concat(
        [
            lightgbm_15_summary,
            lightgbm_30_summary,
            lightgbm_60_summary,
        ],
        axis=0,
        ignore_index=True,
    )
)


# ------------------------------------------------------------
# Natural F0 -> F4 ordering
# ------------------------------------------------------------

feature_family_order = {
    "F0": 0,
    "F1": 1,
    "F2": 2,
    "F3": 3,
    "F4": 4,
}


lightgbm_validation_summary[
    "_family_order"
] = (
    lightgbm_validation_summary[
        "feature_family"
    ]
    .map(
        feature_family_order
    )
)


lightgbm_validation_summary = (
    lightgbm_validation_summary
    .sort_values(
        [
            "horizon_minutes",
            "_family_order",
        ]
    )
    .drop(
        columns="_family_order"
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        lightgbm_validation_summary
    )
    == 15
)


print(
    "Combined LightGBM validation summary:\n"
)


display(
    lightgbm_validation_summary
)


# ------------------------------------------------------------
# 2. Explicit F3 -> F4 validation comparison
# ------------------------------------------------------------

lightgbm_f3_f4_validation_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_summary = (
        lightgbm_validation_summary[
            lightgbm_validation_summary[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
    )


    f3_mae = float(
        horizon_summary.loc[
            horizon_summary[
                "feature_family"
            ]
            == "F3",
            "validation_mae",
        ]
        .iloc[0]
    )


    f4_mae = float(
        horizon_summary.loc[
            horizon_summary[
                "feature_family"
            ]
            == "F4",
            "validation_mae",
        ]
        .iloc[0]
    )


    absolute_reduction = (
        f3_mae
        - f4_mae
    )


    relative_improvement_pct = (
        absolute_reduction
        / f3_mae
        * 100.0
    )


    lightgbm_f3_f4_validation_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "f3_validation_mae":
                f3_mae,

            "f4_validation_mae":
                f4_mae,

            "absolute_mae_reduction":
                absolute_reduction,

            "relative_mae_improvement_pct":
                relative_improvement_pct,
        }
    )


lightgbm_f3_f4_validation = (
    pd.DataFrame(
        lightgbm_f3_f4_validation_rows
    )
)


print(
    "\nLightGBM F3 -> F4 validation comparison:\n"
)


display(
    lightgbm_f3_f4_validation
)


# ------------------------------------------------------------
# 3. Verify already-observed F3 -> F4 effects
# ------------------------------------------------------------

expected_lgbm_f3_f4_effects = {
    15: -0.415502,
    30: -0.559421,
    60: +0.872021,
}


for (
    horizon_minutes,
    expected_value,
) in expected_lgbm_f3_f4_effects.items():

    observed_value = float(
        lightgbm_f3_f4_validation.loc[
            lightgbm_f3_f4_validation[
                "horizon_minutes"
            ]
            == horizon_minutes,
            "relative_mae_improvement_pct",
        ]
        .iloc[0]
    )


    assert np.isclose(
        observed_value,
        expected_value,
        atol=1e-5,
    )


# ------------------------------------------------------------
# 4. Gather selected rows from each horizon
# ------------------------------------------------------------

best_rows_lgbm_by_horizon = {
    15: best_rows_lgbm_15,
    30: best_rows_lgbm_30,
    60: best_rows_lgbm_60,
}


# ------------------------------------------------------------
# 5. Freeze selected executable configurations
# ------------------------------------------------------------

lightgbm_selected_configs = {}


for horizon_minutes in [
    15,
    30,
    60,
]:

    lightgbm_selected_configs[
        horizon_minutes
    ] = {}


    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        selected_row = (
            best_rows_lgbm_by_horizon[
                horizon_minutes
            ][
                feature_family
            ]
        )


        lightgbm_selected_configs[
            horizon_minutes
        ][
            feature_family
        ] = {
            "n_estimators":
                int(
                    selected_row[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    selected_row[
                        "learning_rate"
                    ]
                ),

            "num_leaves":
                int(
                    selected_row[
                        "num_leaves"
                    ]
                ),

            # All remaining parameters retain the values
            # frozen before any LightGBM result was observed.
            **LIGHTGBM_FIXED_PARAMS,
        }


# ------------------------------------------------------------
# 6. Create human-readable frozen-selection table
# ------------------------------------------------------------

lightgbm_frozen_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        config = (
            lightgbm_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )


        validation_row = (
            lightgbm_validation_summary[
                (
                    lightgbm_validation_summary[
                        "horizon_minutes"
                    ]
                    == horizon_minutes
                )
                &
                (
                    lightgbm_validation_summary[
                        "feature_family"
                    ]
                    == feature_family
                )
            ]
            .iloc[0]
        )


        lightgbm_frozen_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_features":
                    primary_protocol[
                        "feature_counts"
                    ][
                        feature_family
                    ],

                "n_estimators":
                    config[
                        "n_estimators"
                    ],

                "learning_rate":
                    config[
                        "learning_rate"
                    ],

                "num_leaves":
                    config[
                        "num_leaves"
                    ],

                "subsample":
                    config[
                        "subsample"
                    ],

                "subsample_freq":
                    config[
                        "subsample_freq"
                    ],

                "colsample_bytree":
                    config[
                        "colsample_bytree"
                    ],

                "min_child_samples":
                    config[
                        "min_child_samples"
                    ],

                "validation_mae":
                    float(
                        validation_row[
                            "validation_mae"
                        ]
                    ),

                "validation_rmse":
                    float(
                        validation_row[
                            "validation_rmse"
                        ]
                    ),
            }
        )


lightgbm_frozen_selection = (
    pd.DataFrame(
        lightgbm_frozen_rows
    )
)


# ------------------------------------------------------------
# 7. Strong integrity checks
# ------------------------------------------------------------

assert (
    lightgbm_frozen_selection.shape[0]
    == 15
)


assert set(
    lightgbm_frozen_selection[
        "horizon_minutes"
    ]
) == {
    15,
    30,
    60,
}


assert set(
    lightgbm_frozen_selection[
        "feature_family"
    ]
) == {
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
}


for horizon_minutes in [
    15,
    30,
    60,
]:

    assert (
        len(
            lightgbm_selected_configs[
                horizon_minutes
            ]
        )
        == 5
    )


# ------------------------------------------------------------
# Every selected variable configuration must belong to the
# originally frozen 8-configuration grid.
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        selected_config = (
            lightgbm_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )


        assert any(
            (
                selected_config[
                    "n_estimators"
                ]
                == grid_config[
                    "n_estimators"
                ]
            )
            and
            (
                selected_config[
                    "learning_rate"
                ]
                == grid_config[
                    "learning_rate"
                ]
            )
            and
            (
                selected_config[
                    "num_leaves"
                ]
                == grid_config[
                    "num_leaves"
                ]
            )
            for grid_config
            in lightgbm_configs
        )


# ------------------------------------------------------------
# 8. Display frozen registry
# ------------------------------------------------------------

print(
    "\nFrozen LightGBM configuration registry:\n"
)


display(
    lightgbm_frozen_selection
)


# ------------------------------------------------------------
# 9. Lowest validation MAE within each horizon
# ------------------------------------------------------------

print(
    "\nLowest validation MAE by horizon:"
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    best_row = (
        lightgbm_validation_summary[
            lightgbm_validation_summary[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
        .sort_values(
            [
                "validation_mae",
                "validation_rmse",
            ]
        )
        .iloc[0]
    )


    print(
        f"  {horizon_minutes:2d} min: "
        f"{best_row['feature_family']} "
        f"(MAE = "
        f"{best_row['validation_mae']:.6f})"
    )


# ------------------------------------------------------------
# 10. Final F3 -> F4 pre-test checkpoint
# ------------------------------------------------------------

print(
    "\nF3 -> F4 validation effects:"
)


for _, row in (
    lightgbm_f3_f4_validation
    .iterrows()
):

    print(
        f"  "
        f"{int(row['horizon_minutes']):2d} min: "
        f"{row['relative_mae_improvement_pct']:+.4f}%"
    )


# ------------------------------------------------------------
# 11. Final checkpoint
# ------------------------------------------------------------

print(
    "\nAll 15 LightGBM configurations are now frozen."
)

print(
    "No LightGBM test-set metric has been used for selection."
)

print(
    "\nLightGBM validation-selection checkpoint: PASSED"
)

Combined LightGBM validation summary:



,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,num_leaves,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,15,F0,118,300,0.03,31,0.282612,0.607759,NaN
1,15,F1,133,300,0.03,15,0.264461,0.583356,6.422494
2,15,F2,138,300,0.03,31,0.268188,0.584637,-1.409322
3,15,F3,147,300,0.03,31,0.268416,0.583245,-0.084911
4,15,F4,244,300,0.03,15,0.269531,0.583488,-0.415502
5,30,F0,118,300,0.03,31,0.351821,0.700336,NaN
6,30,F1,133,300,0.03,15,0.310495,0.640372,11.746332
7,30,F2,138,300,0.03,15,0.310520,0.642112,-0.008121
8,30,F3,147,300,0.03,15,0.314208,0.643084,-1.187795
9,30,F4,244,300,0.03,15,0.315966,0.645094,-0.559421



LightGBM F3 -> F4 validation comparison:



,horizon_minutes,f3_validation_mae,f4_validation_mae,absolute_mae_reduction,relative_mae_improvement_pct
0,15,0.268416,0.269531,-0.001115,-0.415502
1,30,0.314208,0.315966,-0.001758,-0.559421
2,60,0.385131,0.381773,0.003358,0.872021



Frozen LightGBM configuration registry:



,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,validation_mae,validation_rmse
0,15,F0,118,300,0.03,31,0.8,1,0.8,20,0.282612,0.607759
1,15,F1,133,300,0.03,15,0.8,1,0.8,20,0.264461,0.583356
2,15,F2,138,300,0.03,31,0.8,1,0.8,20,0.268188,0.584637
3,15,F3,147,300,0.03,31,0.8,1,0.8,20,0.268416,0.583245
4,15,F4,244,300,0.03,15,0.8,1,0.8,20,0.269531,0.583488
5,30,F0,118,300,0.03,31,0.8,1,0.8,20,0.351821,0.700336
6,30,F1,133,300,0.03,15,0.8,1,0.8,20,0.310495,0.640372
7,30,F2,138,300,0.03,15,0.8,1,0.8,20,0.310520,0.642112
8,30,F3,147,300,0.03,15,0.8,1,0.8,20,0.314208,0.643084
9,30,F4,244,300,0.03,15,0.8,1,0.8,20,0.315966,0.645094



Lowest validation MAE by horizon:
  15 min: F1 (MAE = 0.264461)
  30 min: F1 (MAE = 0.310495)
  60 min: F1 (MAE = 0.368057)

F3 -> F4 validation effects:
  15 min: -0.4155%
  30 min: -0.5594%
  60 min: +0.8720%

All 15 LightGBM configurations are now frozen.
No LightGBM test-set metric has been used for selection.

LightGBM validation-selection checkpoint: PASSED


In [98]:
# ============================================================
# Step 88: Final LightGBM test evaluation
# ============================================================
#
# IMPORTANT:
#
# All 15 LightGBM configurations were frozen using
# chronological validation MAE BEFORE this cell.
#
# For every:
#
#       horizon × feature family
#
# combination we now:
#
#   1. refit the frozen LightGBM model on TRAIN + VALIDATION;
#   2. predict the held-out chronological TEST period once;
#   3. calculate all-time test metrics;
#   4. calculate daylight-only test metrics;
#   5. retain timestamp-level predictions for the later
#      paired F3-vs-F4 daily-block bootstrap.
#
# No hyperparameter will be modified using test performance.
# ============================================================


# ------------------------------------------------------------
# 1. Rated PV capacity for normalized metrics
# ------------------------------------------------------------

LGBM_PV_RATED_KW = float(
    primary_protocol[
        "pv_rated_kw"
    ]
)

assert LGBM_PV_RATED_KW == 10.0


# ------------------------------------------------------------
# 2. Result and prediction containers
# ------------------------------------------------------------

lightgbm_test_result_rows = []

lightgbm_test_predictions = {}


# ------------------------------------------------------------
# 3. Complete final-evaluation timer
# ------------------------------------------------------------

lightgbm_final_test_start_time = (
    time.perf_counter()
)


# ------------------------------------------------------------
# 4. Evaluate all frozen horizon × family configurations
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    y = targets[
        horizon_minutes
    ]

    split = split_labels[
        horizon_minutes
    ]


    # --------------------------------------------------------
    # Final development population:
    #
    #       TRAIN + VALIDATION
    #
    # TEST remains chronologically later.
    # --------------------------------------------------------

    train_validation_mask = (
        split.isin(
            [
                "train",
                "validation",
            ]
        )
    )

    test_mask = (
        split == "test"
    )


    assert int(
        test_mask.sum()
    ) == 26_496


    # --------------------------------------------------------
    # Daylight definition
    #
    # Same frozen definition used throughout:
    #
    #       forecast-valid solar elevation > 0 degrees
    # --------------------------------------------------------

    daylight_test_mask = (
        solar_valid_features[
            horizon_minutes
        ][
            "solar_elevation_deg"
        ]
        > 0.0
    ) & test_mask


    n_test = int(
        test_mask.sum()
    )

    n_test_daylight = int(
        daylight_test_mask.sum()
    )


    print(
        "\n"
        "============================================"
    )

    print(
        f"{horizon_minutes}-minute horizon"
    )

    print(
        "============================================"
    )

    print(
        "Test rows:",
        n_test,
    )

    print(
        "Daylight test rows:",
        n_test_daylight,
    )


    # --------------------------------------------------------
    # Evaluate F0 -> F4
    # --------------------------------------------------------

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        print(
            f"\n{horizon_minutes} min — "
            f"{feature_family}"
        )


        # ----------------------------------------------------
        # Frozen configuration
        # ----------------------------------------------------

        config = (
            lightgbm_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )


        # ----------------------------------------------------
        # Feature matrix
        # ----------------------------------------------------

        X = (
            feature_matrices[
                horizon_minutes
            ][
                feature_family
            ]
        )


        # ----------------------------------------------------
        # Final train+validation arrays
        # ----------------------------------------------------

        X_train_validation = (
            X.loc[
                train_validation_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        X_test = (
            X.loc[
                test_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        y_train_validation = (
            y.loc[
                train_validation_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        y_test = (
            y.loc[
                test_mask
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        # ----------------------------------------------------
        # Structural checks
        # ----------------------------------------------------

        expected_feature_count = (
            primary_protocol[
                "feature_counts"
            ][
                feature_family
            ]
        )


        assert (
            X_train_validation.shape[1]
            == expected_feature_count
        )

        assert (
            X_test.shape
            == (
                26_496,
                expected_feature_count,
            )
        )


        assert np.isfinite(
            X_train_validation
        ).all()

        assert np.isfinite(
            X_test
        ).all()

        assert np.isfinite(
            y_train_validation
        ).all()

        assert np.isfinite(
            y_test
        ).all()


        # ----------------------------------------------------
        # Refit frozen LightGBM model
        # ----------------------------------------------------

        model = LGBMRegressor(
            **config
        )


        fit_start_time = (
            time.perf_counter()
        )


        model.fit(
            X_train_validation,
            y_train_validation,
        )


        test_predictions = (
            model.predict(
                X_test
            )
        )


        fit_predict_seconds = (
            time.perf_counter()
            - fit_start_time
        )


        # ----------------------------------------------------
        # All-time metrics
        # ----------------------------------------------------

        test_mae = (
            mean_absolute_error(
                y_test,
                test_predictions,
            )
        )


        test_rmse = np.sqrt(
            mean_squared_error(
                y_test,
                test_predictions,
            )
        )


        test_nmae = (
            test_mae
            / LGBM_PV_RATED_KW
        )


        test_nrmse = (
            test_rmse
            / LGBM_PV_RATED_KW
        )


        test_r2 = (
            r2_score(
                y_test,
                test_predictions,
            )
        )


        # ----------------------------------------------------
        # Daylight-only metrics
        # ----------------------------------------------------

        test_issue_index = (
            y.index[
                test_mask
            ]
        )


        daylight_on_test = (
            daylight_test_mask
            .loc[
                test_issue_index
            ]
            .to_numpy(
                dtype=bool
            )
        )


        assert int(
            daylight_on_test.sum()
        ) == n_test_daylight


        y_test_daylight = (
            y_test[
                daylight_on_test
            ]
        )

        predictions_daylight = (
            test_predictions[
                daylight_on_test
            ]
        )


        daylight_mae = (
            mean_absolute_error(
                y_test_daylight,
                predictions_daylight,
            )
        )


        daylight_rmse = np.sqrt(
            mean_squared_error(
                y_test_daylight,
                predictions_daylight,
            )
        )


        daylight_nmae = (
            daylight_mae
            / LGBM_PV_RATED_KW
        )


        daylight_nrmse = (
            daylight_rmse
            / LGBM_PV_RATED_KW
        )


        daylight_r2 = (
            r2_score(
                y_test_daylight,
                predictions_daylight,
            )
        )


        # ----------------------------------------------------
        # Store final metrics
        # ----------------------------------------------------

        lightgbm_test_result_rows.append(
            {
                "model":
                    "LGBMRegressor",

                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_features":
                    X.shape[1],

                "n_estimators":
                    config[
                        "n_estimators"
                    ],

                "learning_rate":
                    config[
                        "learning_rate"
                    ],

                "num_leaves":
                    config[
                        "num_leaves"
                    ],

                "subsample":
                    config[
                        "subsample"
                    ],

                "subsample_freq":
                    config[
                        "subsample_freq"
                    ],

                "colsample_bytree":
                    config[
                        "colsample_bytree"
                    ],

                "min_child_samples":
                    config[
                        "min_child_samples"
                    ],

                "test_rows":
                    n_test,

                "test_mae":
                    test_mae,

                "test_rmse":
                    test_rmse,

                "test_nmae":
                    test_nmae,

                "test_nrmse":
                    test_nrmse,

                "test_r2":
                    test_r2,

                "daylight_test_rows":
                    n_test_daylight,

                "daylight_mae":
                    daylight_mae,

                "daylight_rmse":
                    daylight_rmse,

                "daylight_nmae":
                    daylight_nmae,

                "daylight_nrmse":
                    daylight_nrmse,

                "daylight_r2":
                    daylight_r2,

                "fit_predict_seconds":
                    fit_predict_seconds,
            }
        )


        # ----------------------------------------------------
        # Retain timestamp-level predictions
        # ----------------------------------------------------

        prediction_table = pd.DataFrame(
            {
                "valid_time":
                    base_modelling_tables[
                        horizon_minutes
                    ]
                    .loc[
                        test_issue_index,
                        "valid_time",
                    ],

                "y_true":
                    y_test,

                "y_pred":
                    test_predictions,

                "absolute_error":
                    np.abs(
                        y_test
                        - test_predictions
                    ),

                "daylight":
                    daylight_on_test,
            },
            index=test_issue_index,
        )


        prediction_table.index.name = (
            "issue_time"
        )


        lightgbm_test_predictions[
            (
                horizon_minutes,
                feature_family,
            )
        ] = prediction_table


        # ----------------------------------------------------
        # Compact live output
        # ----------------------------------------------------

        print(
            f"  all-time MAE:     "
            f"{test_mae:.6f}"
        )

        print(
            f"  all-time RMSE:    "
            f"{test_rmse:.6f}"
        )

        print(
            f"  daylight MAE:     "
            f"{daylight_mae:.6f}"
        )

        print(
            f"  daylight RMSE:    "
            f"{daylight_rmse:.6f}"
        )

        print(
            f"  runtime:          "
            f"{fit_predict_seconds:.1f} s"
        )


        # ----------------------------------------------------
        # Release fitted model memory
        # ----------------------------------------------------

        del model
        del X_train_validation
        del X_test
        del y_train_validation
        del y_test
        del test_predictions

        gc.collect()


# ------------------------------------------------------------
# 5. Assemble final results table
# ------------------------------------------------------------

lightgbm_final_test_results = (
    pd.DataFrame(
        lightgbm_test_result_rows
    )
)


assert len(
    lightgbm_final_test_results
) == 15


# ------------------------------------------------------------
# 6. Natural horizon / family ordering
# ------------------------------------------------------------

lightgbm_final_test_results[
    "_family_order"
] = (
    lightgbm_final_test_results[
        "feature_family"
    ]
    .map(
        feature_family_order
    )
)


lightgbm_final_test_results = (
    lightgbm_final_test_results
    .sort_values(
        [
            "horizon_minutes",
            "_family_order",
        ]
    )
    .drop(
        columns="_family_order"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 7. Structural checks
# ------------------------------------------------------------

assert (
    len(
        lightgbm_test_predictions
    )
    == 15
)


assert (
    lightgbm_final_test_results[
        "test_rows"
    ]
    .eq(
        26_496
    )
    .all()
)


assert (
    lightgbm_final_test_results[
        "test_mae"
    ]
    .notna()
    .all()
)


assert (
    lightgbm_final_test_results[
        "daylight_mae"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 8. Display complete final test table
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "FINAL LIGHTGBM TEST RESULTS"
)

print(
    "============================================\n"
)


display(
    lightgbm_final_test_results
)


# ------------------------------------------------------------
# 9. Explicit F3 -> F4 final-test comparison
# ------------------------------------------------------------

lightgbm_f3_f4_test_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_results = (
        lightgbm_final_test_results[
            lightgbm_final_test_results[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
    )


    f3_row = (
        horizon_results[
            horizon_results[
                "feature_family"
            ]
            == "F3"
        ]
        .iloc[0]
    )


    f4_row = (
        horizon_results[
            horizon_results[
                "feature_family"
            ]
            == "F4"
        ]
        .iloc[0]
    )


    all_time_relative_pct = (
        (
            f3_row[
                "test_mae"
            ]
            - f4_row[
                "test_mae"
            ]
        )
        / f3_row[
            "test_mae"
        ]
        * 100.0
    )


    daylight_relative_pct = (
        (
            f3_row[
                "daylight_mae"
            ]
            - f4_row[
                "daylight_mae"
            ]
        )
        / f3_row[
            "daylight_mae"
        ]
        * 100.0
    )


    lightgbm_f3_f4_test_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "f3_test_mae":
                f3_row[
                    "test_mae"
                ],

            "f4_test_mae":
                f4_row[
                    "test_mae"
                ],

            "f3_to_f4_test_improvement_pct":
                all_time_relative_pct,

            "f3_daylight_mae":
                f3_row[
                    "daylight_mae"
                ],

            "f4_daylight_mae":
                f4_row[
                    "daylight_mae"
                ],

            "f3_to_f4_daylight_improvement_pct":
                daylight_relative_pct,
        }
    )


lightgbm_f3_f4_test_summary = (
    pd.DataFrame(
        lightgbm_f3_f4_test_rows
    )
)


print(
    "\nLightGBM F3 -> F4 FINAL test comparison:\n"
)


display(
    lightgbm_f3_f4_test_summary
)


# ------------------------------------------------------------
# 10. Lowest final test MAE within each horizon
# ------------------------------------------------------------

print(
    "\nLowest LightGBM test MAE by horizon:"
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    best_test_row = (
        lightgbm_final_test_results[
            lightgbm_final_test_results[
                "horizon_minutes"
            ]
            == horizon_minutes
        ]
        .sort_values(
            "test_mae"
        )
        .iloc[0]
    )


    print(
        f"  {horizon_minutes:2d} min: "
        f"{best_test_row['feature_family']} "
        f"(MAE = "
        f"{best_test_row['test_mae']:.6f})"
    )


# ------------------------------------------------------------
# 11. Runtime
# ------------------------------------------------------------

lightgbm_final_test_total_seconds = (
    time.perf_counter()
    - lightgbm_final_test_start_time
)


print(
    "\nTotal final LightGBM evaluation runtime:"
)

print(
    f"  "
    f"{lightgbm_final_test_total_seconds:.1f} "
    f"seconds"
)

print(
    f"  "
    f"{lightgbm_final_test_total_seconds / 60:.2f} "
    f"minutes"
)


print(
    "\nFinal LightGBM test evaluation: PASSED"
)


15-minute horizon
Test rows: 26496
Daylight test rows: 17851

15 min — F0
  all-time MAE:     0.275620
  all-time RMSE:    0.578978
  daylight MAE:     0.406420
  daylight RMSE:    0.705359
  runtime:          3.4 s

15 min — F1
  all-time MAE:     0.260576
  all-time RMSE:    0.562322
  daylight MAE:     0.384331
  daylight RMSE:    0.685068
  runtime:          2.9 s

15 min — F2
  all-time MAE:     0.261805
  all-time RMSE:    0.559667
  daylight MAE:     0.387011
  daylight RMSE:    0.681840
  runtime:          3.6 s

15 min — F3
  all-time MAE:     0.262211
  all-time RMSE:    0.560043
  daylight MAE:     0.387584
  daylight RMSE:    0.682298
  runtime:          4.0 s

15 min — F4
  all-time MAE:     0.262057
  all-time RMSE:    0.561712
  daylight MAE:     0.386933
  daylight RMSE:    0.684327
  runtime:          3.2 s

30-minute horizon
Test rows: 26496
Daylight test rows: 17851

30 min — F0
  all-time MAE:     0.332928
  all-time RMSE:    0.656880
  daylight MAE:     0.489568
 

,model,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,...,test_nmae,test_nrmse,test_r2,daylight_test_rows,daylight_mae,daylight_rmse,daylight_nmae,daylight_nrmse,daylight_r2,fit_predict_seconds
0,LGBMRegressor,15,F0,118,300,0.03,31,0.8,1,0.8,...,0.027562,0.057898,0.876970,17851,0.406420,0.705359,0.040642,0.070536,0.833311,3.439075
1,LGBMRegressor,15,F1,133,300,0.03,15,0.8,1,0.8,...,0.026058,0.056232,0.883947,17851,0.384331,0.685068,0.038433,0.068507,0.842763,2.903638
2,LGBMRegressor,15,F2,138,300,0.03,31,0.8,1,0.8,...,0.026180,0.055967,0.885040,17851,0.387011,0.681840,0.038701,0.068184,0.844242,3.621565
3,LGBMRegressor,15,F3,147,300,0.03,31,0.8,1,0.8,...,0.026221,0.056004,0.884886,17851,0.387584,0.682298,0.038758,0.068230,0.844032,3.954212
4,LGBMRegressor,15,F4,244,300,0.03,15,0.8,1,0.8,...,0.026206,0.056171,0.884199,17851,0.386933,0.684327,0.038693,0.068433,0.843103,3.245716
5,LGBMRegressor,30,F0,118,300,0.03,31,0.8,1,0.8,...,0.033293,0.065688,0.841635,17851,0.489568,0.800223,0.048957,0.080022,0.785460,3.188589
6,LGBMRegressor,30,F1,133,300,0.03,15,0.8,1,0.8,...,0.030318,0.062446,0.856883,17851,0.446121,0.760749,0.044612,0.076075,0.806104,3.017768
7,LGBMRegressor,30,F2,138,300,0.03,15,0.8,1,0.8,...,0.030119,0.062228,0.857878,17851,0.444428,0.758119,0.044443,0.075812,0.807442,2.984652
8,LGBMRegressor,30,F3,147,300,0.03,15,0.8,1,0.8,...,0.030241,0.062267,0.857702,17851,0.446240,0.758588,0.044624,0.075859,0.807204,3.220666
9,LGBMRegressor,30,F4,244,300,0.03,15,0.8,1,0.8,...,0.030065,0.062220,0.857916,17851,0.443675,0.758018,0.044368,0.075802,0.807493,3.272725



LightGBM F3 -> F4 FINAL test comparison:



,horizon_minutes,f3_test_mae,f4_test_mae,f3_to_f4_test_improvement_pct,f3_daylight_mae,f4_daylight_mae,f3_to_f4_daylight_improvement_pct
0,15,0.262211,0.262057,0.058989,0.387584,0.386933,0.168061
1,30,0.302415,0.300646,0.584866,0.446240,0.443675,0.574744
2,60,0.343349,0.343123,0.065940,0.506980,0.505740,0.244620



Lowest LightGBM test MAE by horizon:
  15 min: F1 (MAE = 0.260576)
  30 min: F4 (MAE = 0.300646)
  60 min: F2 (MAE = 0.341262)

Total final LightGBM evaluation runtime:
  50.3 seconds
  0.84 minutes

Final LightGBM test evaluation: PASSED


In [99]:
# ============================================================
# Step 89: Paired daily-block bootstrap for LightGBM
#          F3 vs F4
# ============================================================
#
# Purpose
# -------
# Quantify uncertainty in the final LightGBM F3 -> F4 test
# comparison while preserving temporal dependence.
#
# This uses the SAME frozen bootstrap design used for:
#
#   - Notebook 09 primary experiment;
#   - Extra Trees;
#   - XGBoost.
#
#
# Central comparison
# ------------------
#
#       |error_F3| - |error_F4|
#
# Positive delta:
#       F4 has LOWER MAE than F3
#
# Negative delta:
#       F4 has HIGHER MAE than F3
#
#
# Frozen bootstrap specification
# ------------------------------
#
#   block unit:
#       forecast-valid UTC calendar day
#
#   daily blocks:
#       92
#
#   bootstrap replicates:
#       10,000
#
#   random seed:
#       42
#
#   confidence interval:
#       95% percentile interval
#
#   model retraining:
#       NO
#
#   scopes:
#       - all test observations
#       - daylight-only observations
#
#
# Important:
#   daylight filtering occurs BEFORE daily aggregation.
#
#   Bootstrap MAE is reconstructed as pooled MAE using
#   sampled absolute-error sums and sampled row counts.
#
#   We do NOT average daily MAEs equally.
# ============================================================


# ------------------------------------------------------------
# 1. Recover frozen bootstrap settings
# ------------------------------------------------------------

N_BOOTSTRAP_LGBM = int(
    primary_protocol[
        "bootstrap"
    ][
        "replicates"
    ]
)

BOOTSTRAP_RANDOM_SEED_LGBM = int(
    primary_protocol[
        "bootstrap"
    ][
        "random_seed"
    ]
)

EXPECTED_DAILY_BLOCKS_LGBM = int(
    primary_protocol[
        "bootstrap"
    ][
        "daily_blocks"
    ]
)


assert N_BOOTSTRAP_LGBM == 10_000

assert BOOTSTRAP_RANDOM_SEED_LGBM == 42

assert EXPECTED_DAILY_BLOCKS_LGBM == 92


BOOTSTRAP_HORIZONS_LGBM = [
    15,
    30,
    60,
]

BOOTSTRAP_SCOPES_LGBM = [
    "all",
    "daylight",
]


# ------------------------------------------------------------
# One reproducible RNG for the complete LightGBM bootstrap.
# ------------------------------------------------------------

rng_lightgbm = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED_LGBM
)


# ------------------------------------------------------------
# 2. Storage
# ------------------------------------------------------------

lightgbm_bootstrap_summary_rows = []

lightgbm_bootstrap_distributions = {}


# ------------------------------------------------------------
# 3. Run one paired bootstrap for each:
#
#       horizon × scope
#
# Total:
#
#       3 × 2 = 6 analyses
# ------------------------------------------------------------

for horizon_minutes in BOOTSTRAP_HORIZONS_LGBM:

    # --------------------------------------------------------
    # Retrieve final frozen F3 and F4 test predictions
    # --------------------------------------------------------

    df_f3 = (
        lightgbm_test_predictions[
            (
                horizon_minutes,
                "F3",
            )
        ]
        .copy()
    )

    df_f4 = (
        lightgbm_test_predictions[
            (
                horizon_minutes,
                "F4",
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # 4. Verify exact paired alignment
    # --------------------------------------------------------

    assert df_f3.index.equals(
        df_f4.index
    )

    assert len(df_f3) == 26_496

    assert len(df_f4) == 26_496


    assert df_f3[
        "valid_time"
    ].equals(
        df_f4[
            "valid_time"
        ]
    )


    assert np.allclose(
        df_f3[
            "y_true"
        ].to_numpy(),
        df_f4[
            "y_true"
        ].to_numpy(),
        atol=0.0,
        rtol=0.0,
    )


    assert df_f3[
        "daylight"
    ].equals(
        df_f4[
            "daylight"
        ]
    )


    # --------------------------------------------------------
    # 5. Construct paired prediction table
    # --------------------------------------------------------

    df_predictions = pd.DataFrame(
        {
            "valid_time":
                df_f3[
                    "valid_time"
                ],

            "y_true":
                df_f3[
                    "y_true"
                ],

            "F3_prediction":
                df_f3[
                    "y_pred"
                ],

            "F4_prediction":
                df_f4[
                    "y_pred"
                ],

            "daylight":
                df_f3[
                    "daylight"
                ],
        },
        index=df_f3.index,
    )


    # --------------------------------------------------------
    # 6. Verify fixed chronological test interval
    # --------------------------------------------------------

    assert (
        df_predictions[
            "valid_time"
        ].min()
        == pd.Timestamp(
            "2019-06-01 00:00:00",
            tz="UTC",
        )
    )


    assert (
        df_predictions[
            "valid_time"
        ].max()
        == pd.Timestamp(
            "2019-08-31 23:55:00",
            tz="UTC",
        )
    )


    # --------------------------------------------------------
    # 7. Evaluate both scopes
    # --------------------------------------------------------

    for scope_name in BOOTSTRAP_SCOPES_LGBM:

        if scope_name == "all":

            df_scope = (
                df_predictions
                .copy()
            )


        elif scope_name == "daylight":

            df_scope = (
                df_predictions.loc[
                    df_predictions[
                        "daylight"
                    ]
                ]
                .copy()
            )


        else:

            raise ValueError(
                f"Unexpected scope: {scope_name}"
            )


        # ----------------------------------------------------
        # 8. Calculate paired absolute errors
        # ----------------------------------------------------

        df_scope[
            "abs_error_F3"
        ] = np.abs(
            df_scope[
                "y_true"
            ]
            - df_scope[
                "F3_prediction"
            ]
        )


        df_scope[
            "abs_error_F4"
        ] = np.abs(
            df_scope[
                "y_true"
            ]
            - df_scope[
                "F4_prediction"
            ]
        )


        df_scope[
            "paired_error_difference"
        ] = (
            df_scope[
                "abs_error_F3"
            ]
            - df_scope[
                "abs_error_F4"
            ]
        )


        # ----------------------------------------------------
        # 9. Forecast-valid UTC calendar-day blocks
        # ----------------------------------------------------

        df_scope[
            "valid_date"
        ] = (
            df_scope[
                "valid_time"
            ]
            .dt.floor("D")
        )


        # ----------------------------------------------------
        # 10. Collapse each day into sufficient statistics
        # ----------------------------------------------------

        daily_blocks = (
            df_scope
            .groupby(
                "valid_date",
                sort=True,
            )
            .agg(
                n_rows=(
                    "y_true",
                    "size",
                ),

                abs_error_F3_sum=(
                    "abs_error_F3",
                    "sum",
                ),

                abs_error_F4_sum=(
                    "abs_error_F4",
                    "sum",
                ),
            )
        )


        n_days = len(
            daily_blocks
        )


        assert (
            n_days
            == EXPECTED_DAILY_BLOCKS_LGBM
        ), (
            f"LightGBM / "
            f"{horizon_minutes} min / "
            f"{scope_name}: "
            f"expected "
            f"{EXPECTED_DAILY_BLOCKS_LGBM} "
            f"daily blocks, found {n_days}."
        )


        # ----------------------------------------------------
        # 11. Observed test-set point estimates
        #
        # Explicit Python float conversion gives consistent
        # double-precision percentage arithmetic.
        # ----------------------------------------------------

        observed_f3_mae = float(
            df_scope[
                "abs_error_F3"
            ]
            .mean()
        )


        observed_f4_mae = float(
            df_scope[
                "abs_error_F4"
            ]
            .mean()
        )


        observed_delta_mae = (
            observed_f3_mae
            - observed_f4_mae
        )


        observed_relative_change = (
            observed_delta_mae
            / observed_f3_mae
            * 100.0
        )


        # ----------------------------------------------------
        # 12. Verify against final LightGBM test summary
        # ----------------------------------------------------

        expected_row = (
            lightgbm_f3_f4_test_summary.loc[
                lightgbm_f3_f4_test_summary[
                    "horizon_minutes"
                ].eq(
                    horizon_minutes
                )
            ]
            .iloc[0]
        )


        if scope_name == "all":

            expected_f3_mae = float(
                expected_row[
                    "f3_test_mae"
                ]
            )

            expected_f4_mae = float(
                expected_row[
                    "f4_test_mae"
                ]
            )

            expected_relative_change = float(
                expected_row[
                    "f3_to_f4_test_improvement_pct"
                ]
            )


        else:

            expected_f3_mae = float(
                expected_row[
                    "f3_daylight_mae"
                ]
            )

            expected_f4_mae = float(
                expected_row[
                    "f4_daylight_mae"
                ]
            )

            expected_relative_change = float(
                expected_row[
                    "f3_to_f4_daylight_improvement_pct"
                ]
            )


        assert np.isclose(
            observed_f3_mae,
            expected_f3_mae,
            atol=1e-12,
            rtol=0.0,
        )


        assert np.isclose(
            observed_f4_mae,
            expected_f4_mae,
            atol=1e-12,
            rtol=0.0,
        )


        assert np.isclose(
            observed_relative_change,
            expected_relative_change,
            atol=1e-10,
            rtol=0.0,
        )


        # ----------------------------------------------------
        # 13. Convert daily sufficient statistics to arrays
        # ----------------------------------------------------

        daily_counts = (
            daily_blocks[
                "n_rows"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )


        daily_f3_error_sums = (
            daily_blocks[
                "abs_error_F3_sum"
            ]
            .to_numpy(
                dtype=float
            )
        )


        daily_f4_error_sums = (
            daily_blocks[
                "abs_error_F4_sum"
            ]
            .to_numpy(
                dtype=float
            )
        )


        # ----------------------------------------------------
        # 14. Sample 92 complete daily blocks with replacement
        #
        # Shape:
        #
        #       10,000 × 92
        # ----------------------------------------------------

        sampled_day_indices = (
            rng_lightgbm.integers(
                low=0,
                high=n_days,
                size=(
                    N_BOOTSTRAP_LGBM,
                    n_days,
                ),
            )
        )


        # ----------------------------------------------------
        # 15. Reconstruct pooled MAE for each replicate
        # ----------------------------------------------------

        bootstrap_row_counts = (
            daily_counts[
                sampled_day_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f3_error_sums = (
            daily_f3_error_sums[
                sampled_day_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f4_error_sums = (
            daily_f4_error_sums[
                sampled_day_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f3_mae = (
            bootstrap_f3_error_sums
            / bootstrap_row_counts
        )


        bootstrap_f4_mae = (
            bootstrap_f4_error_sums
            / bootstrap_row_counts
        )


        # ----------------------------------------------------
        # 16. Paired F3 -> F4 difference
        # ----------------------------------------------------

        bootstrap_delta_mae = (
            bootstrap_f3_mae
            - bootstrap_f4_mae
        )


        bootstrap_relative_change = (
            bootstrap_delta_mae
            / bootstrap_f3_mae
            * 100.0
        )


        # ----------------------------------------------------
        # 17. Frozen 95% percentile intervals
        # ----------------------------------------------------

        (
            delta_ci_lower,
            delta_ci_upper,
        ) = np.percentile(
            bootstrap_delta_mae,
            [
                2.5,
                97.5,
            ],
        )


        (
            relative_ci_lower,
            relative_ci_upper,
        ) = np.percentile(
            bootstrap_relative_change,
            [
                2.5,
                97.5,
            ],
        )


        # ----------------------------------------------------
        # 18. Store complete bootstrap distribution
        # ----------------------------------------------------

        distribution_key = (
            horizon_minutes,
            scope_name,
        )


        lightgbm_bootstrap_distributions[
            distribution_key
        ] = pd.DataFrame(
            {
                "delta_MAE_kW":
                    bootstrap_delta_mae,

                "relative_MAE_change_percent":
                    bootstrap_relative_change,
            }
        )


        # ----------------------------------------------------
        # 19. Store summary row
        # ----------------------------------------------------

        lightgbm_bootstrap_summary_rows.append(
            {
                "model":
                    "LGBMRegressor",

                "horizon_minutes":
                    horizon_minutes,

                "scope":
                    scope_name,

                "n_test_rows":
                    len(
                        df_scope
                    ),

                "n_daily_blocks":
                    n_days,

                "F3_MAE_kW":
                    observed_f3_mae,

                "F4_MAE_kW":
                    observed_f4_mae,

                "delta_MAE_kW":
                    observed_delta_mae,

                "delta_MAE_CI95_lower_kW":
                    delta_ci_lower,

                "delta_MAE_CI95_upper_kW":
                    delta_ci_upper,

                "relative_MAE_change_percent":
                    observed_relative_change,

                "relative_CI95_lower_percent":
                    relative_ci_lower,

                "relative_CI95_upper_percent":
                    relative_ci_upper,
            }
        )


# ------------------------------------------------------------
# 20. Assemble final LightGBM bootstrap summary
# ------------------------------------------------------------

lightgbm_bootstrap_summary = (
    pd.DataFrame(
        lightgbm_bootstrap_summary_rows
    )
)


# ------------------------------------------------------------
# 21. Structural and numerical checks
# ------------------------------------------------------------

assert (
    len(
        lightgbm_bootstrap_summary
    )
    == 6
)


assert (
    len(
        lightgbm_bootstrap_distributions
    )
    == 6
)


assert (
    lightgbm_bootstrap_summary[
        "n_daily_blocks"
    ]
    .eq(
        92
    )
    .all()
)


bootstrap_numeric_columns_lgbm = [
    "F3_MAE_kW",
    "F4_MAE_kW",
    "delta_MAE_kW",
    "delta_MAE_CI95_lower_kW",
    "delta_MAE_CI95_upper_kW",
    "relative_MAE_change_percent",
    "relative_CI95_lower_percent",
    "relative_CI95_upper_percent",
]


assert np.isfinite(
    lightgbm_bootstrap_summary[
        bootstrap_numeric_columns_lgbm
    ]
    .to_numpy()
).all()


# ------------------------------------------------------------
# Every stored distribution must contain exactly 10,000
# bootstrap replicates.
# ------------------------------------------------------------

for (
    distribution_key,
    distribution_df,
) in (
    lightgbm_bootstrap_distributions.items()
):

    assert (
        len(
            distribution_df
        )
        == N_BOOTSTRAP_LGBM
    ), (
        f"Unexpected bootstrap length for "
        f"{distribution_key}"
    )


# ------------------------------------------------------------
# 22. Sort output:
#
#       all:       15, 30, 60
#       daylight:  15, 30, 60
# ------------------------------------------------------------

scope_order_lgbm = {
    "all": 0,
    "daylight": 1,
}


lightgbm_bootstrap_summary[
    "_scope_order"
] = (
    lightgbm_bootstrap_summary[
        "scope"
    ]
    .map(
        scope_order_lgbm
    )
)


lightgbm_bootstrap_summary = (
    lightgbm_bootstrap_summary
    .sort_values(
        [
            "_scope_order",
            "horizon_minutes",
        ]
    )
    .drop(
        columns="_scope_order"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 23. Explicitly record whether confidence intervals
#     contain zero
# ------------------------------------------------------------

lightgbm_bootstrap_summary[
    "delta_CI_contains_zero"
] = (
    (
        lightgbm_bootstrap_summary[
            "delta_MAE_CI95_lower_kW"
        ]
        <= 0.0
    )
    &
    (
        lightgbm_bootstrap_summary[
            "delta_MAE_CI95_upper_kW"
        ]
        >= 0.0
    )
)


lightgbm_bootstrap_summary[
    "relative_CI_contains_zero"
] = (
    (
        lightgbm_bootstrap_summary[
            "relative_CI95_lower_percent"
        ]
        <= 0.0
    )
    &
    (
        lightgbm_bootstrap_summary[
            "relative_CI95_upper_percent"
        ]
        >= 0.0
    )
)


assert (
    lightgbm_bootstrap_summary[
        "delta_CI_contains_zero"
    ]
    .equals(
        lightgbm_bootstrap_summary[
            "relative_CI_contains_zero"
        ]
    )
)


# ------------------------------------------------------------
# 24. Final output
# ------------------------------------------------------------

print(
    "LightGBM paired daily-block bootstrap: PASSED"
)


print(
    "\n"
    "LightGBM F3 -> F4 paired daily-block "
    "bootstrap results:\n"
)


display(
    lightgbm_bootstrap_summary[
        [
            "model",
            "horizon_minutes",
            "scope",
            "n_test_rows",
            "n_daily_blocks",
            "F3_MAE_kW",
            "F4_MAE_kW",
            "delta_MAE_kW",
            "delta_MAE_CI95_lower_kW",
            "delta_MAE_CI95_upper_kW",
            "relative_MAE_change_percent",
            "relative_CI95_lower_percent",
            "relative_CI95_upper_percent",
            "delta_CI_contains_zero",
        ]
    ]
)


print(
    "\nInterpretation convention:"
)

print(
    "  positive delta / relative change = F4 better"
)

print(
    "  negative delta / relative change = F4 worse"
)

print(
    "  CI containing zero = no clear paired difference "
    "at the 95% percentile level"
)


print(
    "\nF3 -> F4 bootstrap summary:"
)


for _, row in (
    lightgbm_bootstrap_summary
    .iterrows()
):

    ci_status = (
        "contains zero"
        if row[
            "delta_CI_contains_zero"
        ]
        else
        "excludes zero"
    )


    print(
        f"  "
        f"{int(row['horizon_minutes']):2d} min | "
        f"{row['scope']:8s} | "
        f"point = "
        f"{row['relative_MAE_change_percent']:+.4f}% | "
        f"95% CI = ["
        f"{row['relative_CI95_lower_percent']:+.4f}%, "
        f"{row['relative_CI95_upper_percent']:+.4f}%] | "
        f"{ci_status}"
    )


print(
    "\nLightGBM bootstrap analysis: PASSED"
)

LightGBM paired daily-block bootstrap: PASSED

LightGBM F3 -> F4 paired daily-block bootstrap results:



,model,horizon_minutes,scope,n_test_rows,n_daily_blocks,F3_MAE_kW,F4_MAE_kW,delta_MAE_kW,delta_MAE_CI95_lower_kW,delta_MAE_CI95_upper_kW,relative_MAE_change_percent,relative_CI95_lower_percent,relative_CI95_upper_percent,delta_CI_contains_zero
0,LGBMRegressor,15,all,26496,92,0.262211,0.262057,0.000155,-0.001531,0.001945,0.058989,-0.586560,0.752677,True
1,LGBMRegressor,30,all,26496,92,0.302415,0.300646,0.001769,0.000501,0.003159,0.584866,0.163201,1.060181,False
2,LGBMRegressor,60,all,26496,92,0.343349,0.343123,0.000226,-0.001928,0.002666,0.065940,-0.558823,0.770824,True
3,LGBMRegressor,15,daylight,17851,92,0.387584,0.386933,0.000651,-0.001869,0.003276,0.168061,-0.483412,0.846177,True
4,LGBMRegressor,30,daylight,17851,92,0.446240,0.443675,0.002565,0.000702,0.004673,0.574744,0.154675,1.057407,False
5,LGBMRegressor,60,daylight,17851,92,0.506980,0.505740,0.001240,-0.001935,0.004740,0.244620,-0.382256,0.941645,True



Interpretation convention:
  positive delta / relative change = F4 better
  negative delta / relative change = F4 worse
  CI containing zero = no clear paired difference at the 95% percentile level

F3 -> F4 bootstrap summary:
  15 min | all      | point = +0.0590% | 95% CI = [-0.5866%, +0.7527%] | contains zero
  30 min | all      | point = +0.5849% | 95% CI = [+0.1632%, +1.0602%] | excludes zero
  60 min | all      | point = +0.0659% | 95% CI = [-0.5588%, +0.7708%] | contains zero
  15 min | daylight | point = +0.1681% | 95% CI = [-0.4834%, +0.8462%] | contains zero
  30 min | daylight | point = +0.5747% | 95% CI = [+0.1547%, +1.0574%] | excludes zero
  60 min | daylight | point = +0.2446% | 95% CI = [-0.3823%, +0.9416%] | contains zero

LightGBM bootstrap analysis: PASSED


## LightGBM final interpretation

The LightGBM experiment used the same frozen chronological forecasting
protocol as the preceding models. Hyperparameters were selected using only
the validation period, after which all 15 horizon × feature-family
configurations were frozen. Each selected model was then refitted on the
combined training + validation population and evaluated once on the held-out
test period.

### Overall feature-family behaviour

The conventional Gregorian/calendar feature family **F1** produced the
lowest LightGBM validation MAE at all three horizons:

- **15 min:** F1 MAE = 0.264461 kW
- **30 min:** F1 MAE = 0.310495 kW
- **60 min:** F1 MAE = 0.368057 kW

The held-out test ranking was not identical:

- **15 min:** F1 had the lowest test MAE = 0.260576 kW
- **30 min:** F4 had the lowest test MAE = 0.300646 kW
- **60 min:** F2 had the lowest test MAE = 0.341262 kW

Thus, the most complex feature family was not uniformly preferred across
forecast horizons.

### Central F3 → F4 comparison

The main research comparison asks whether adding the categorical
Vedic/Panchang representation in **F4** improves forecasting beyond the
continuous solar and lunar astronomical controls already contained in
**F3**.

On the validation period, the F3 → F4 relative MAE changes were:

- **15 min:** −0.4155%
- **30 min:** −0.5594%
- **60 min:** +0.8720%

Therefore, the validation behaviour was mixed and did not show a consistent
Panchang improvement.

On the held-out test period, the corresponding point estimates were:

| Horizon | All-time F3 → F4 | Daylight F3 → F4 |
|---|---:|---:|
| 15 min | +0.0590% | +0.1681% |
| 30 min | +0.5849% | +0.5747% |
| 60 min | +0.0659% | +0.2446% |

Positive values mean that F4 produced lower MAE than F3.

### Paired daily-block bootstrap

To account for temporal dependence, uncertainty was estimated using the
pre-specified paired bootstrap over **92 forecast-valid UTC calendar days**,
with **10,000 bootstrap replicates** and pooled MAE reconstruction.

For the all-time test population:

- **15 min:** +0.0590%, 95% CI [−0.5866%, +0.7527%]
- **30 min:** +0.5849%, 95% CI [+0.1632%, +1.0602%]
- **60 min:** +0.0659%, 95% CI [−0.5588%, +0.7708%]

For daylight-only observations:

- **15 min:** +0.1681%, 95% CI [−0.4834%, +0.8462%]
- **30 min:** +0.5747%, 95% CI [+0.1547%, +1.0574%]
- **60 min:** +0.2446%, 95% CI [−0.3823%, +0.9416%]

The confidence intervals contain zero at the 15- and 60-minute horizons.
Therefore, these two horizons do not provide a clear paired test-set
difference between F3 and F4.

At the **30-minute horizon**, however, the confidence interval is entirely
above zero for both the all-time and daylight-only analyses. Within this
held-out test period and under the frozen LightGBM configuration, F4
therefore shows a small positive paired MAE reduction relative to F3.

### Interpretation relative to the hypothesis

This result should not be interpreted as general confirmation that Panchang
features improve renewable-energy forecasting.

The 30-minute LightGBM result is noteworthy because it is the first current
secondary-model comparison in which the pre-specified paired daily-block
bootstrap interval excludes zero. However:

1. the effect is small, approximately **0.6% MAE**;
2. the corresponding 30-minute validation comparison had the opposite sign
   (**−0.5594%**);
3. the 15- and 60-minute LightGBM intervals contain zero; and
4. the preceding Extra Trees and XGBoost experiments did not show clear
   paired F3 → F4 improvements under the same bootstrap criterion.

The present evidence therefore supports a more cautious conclusion:
**Panchang features may provide a small model- and horizon-specific
incremental signal, particularly for 30-minute LightGBM forecasting, but a
robust and general improvement beyond continuous astronomical features has
not yet been established.**

Further models and, especially, external-dataset validation are required to
determine whether the 30-minute LightGBM result is reproducible rather than
specific to this model, horizon, or test interval.

In [100]:
# ============================================================
# Step 90: Persist the complete LightGBM experiment
# ============================================================
#
# This saves:
#
#   1. validation summary
#   2. frozen model selections
#   3. final held-out test results
#   4. F3-vs-F4 final test summary
#   5. paired daily-block bootstrap summary
#   6. complete bootstrap distributions (gzip)
#   7. 15-minute test predictions
#   8. 30-minute test predictions
#   9. 60-minute test predictions
#  10. selected LightGBM configurations
#  11. LightGBM experiment protocol
#  12. SHA-256 results manifest
#
# No fitted model binaries are stored because the current
# experiment is for scientific evaluation rather than
# deployment.
# ============================================================

from pathlib import Path
import json
import hashlib


# ------------------------------------------------------------
# 1. Output directory
# ------------------------------------------------------------

LIGHTGBM_RESULTS_DIR = Path(
    "results/notebook10_secondary_ml_models"
)

LIGHTGBM_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 2. Save tabular result summaries
# ------------------------------------------------------------

lightgbm_validation_path = (
    LIGHTGBM_RESULTS_DIR
    / "lightgbm_validation_summary.csv"
)

lightgbm_frozen_path = (
    LIGHTGBM_RESULTS_DIR
    / "lightgbm_frozen_selection.csv"
)

lightgbm_final_test_path = (
    LIGHTGBM_RESULTS_DIR
    / "lightgbm_final_test_results.csv"
)

lightgbm_f3_f4_path = (
    LIGHTGBM_RESULTS_DIR
    / "lightgbm_f3_f4_test_summary.csv"
)

lightgbm_bootstrap_summary_path = (
    LIGHTGBM_RESULTS_DIR
    / "lightgbm_bootstrap_summary.csv"
)


lightgbm_validation_summary.to_csv(
    lightgbm_validation_path,
    index=False,
)

lightgbm_frozen_selection.to_csv(
    lightgbm_frozen_path,
    index=False,
)

lightgbm_final_test_results.to_csv(
    lightgbm_final_test_path,
    index=False,
)

lightgbm_f3_f4_test_summary.to_csv(
    lightgbm_f3_f4_path,
    index=False,
)

lightgbm_bootstrap_summary.to_csv(
    lightgbm_bootstrap_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 3. Combine and save all 60,000 bootstrap replicates
#
# 3 horizons × 2 scopes × 10,000 replicates
# ------------------------------------------------------------

lightgbm_bootstrap_distribution_frames = []


for (
    horizon_minutes,
    scope_name,
), distribution_df in (
    lightgbm_bootstrap_distributions.items()
):

    distribution_export = (
        distribution_df.copy()
    )

    distribution_export.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            1,
            len(distribution_export) + 1,
            dtype=np.int64,
        ),
    )

    distribution_export.insert(
        0,
        "scope",
        scope_name,
    )

    distribution_export.insert(
        0,
        "horizon_minutes",
        horizon_minutes,
    )


    lightgbm_bootstrap_distribution_frames.append(
        distribution_export
    )


lightgbm_bootstrap_distribution_export = (
    pd.concat(
        lightgbm_bootstrap_distribution_frames,
        axis=0,
        ignore_index=True,
    )
)


assert (
    len(
        lightgbm_bootstrap_distribution_export
    )
    == 60_000
)


lightgbm_bootstrap_distribution_path = (
    LIGHTGBM_RESULTS_DIR
    / "lightgbm_bootstrap_distributions.csv.gz"
)


lightgbm_bootstrap_distribution_export.to_csv(
    lightgbm_bootstrap_distribution_path,
    index=False,
    compression="gzip",
)


# ------------------------------------------------------------
# 4. Build one prediction file per forecast horizon
#
# Each file contains:
#
#   issue_time
#   valid_time
#   y_true
#   daylight
#   F0_prediction ... F4_prediction
#   F0_absolute_error ... F4_absolute_error
#
# This preserves enough information to reproduce model
# comparisons and diagnostic analyses without saving fitted
# model binaries.
# ------------------------------------------------------------

lightgbm_prediction_paths = {}


for horizon_minutes in [
    15,
    30,
    60,
]:

    reference_prediction_table = (
        lightgbm_test_predictions[
            (
                horizon_minutes,
                "F0",
            )
        ]
    )


    prediction_export = pd.DataFrame(
        {
            "issue_time":
                reference_prediction_table.index,

            "valid_time":
                reference_prediction_table[
                    "valid_time"
                ].to_numpy(),

            "y_true":
                reference_prediction_table[
                    "y_true"
                ].to_numpy(),

            "daylight":
                reference_prediction_table[
                    "daylight"
                ].to_numpy(),
        }
    )


    # --------------------------------------------------------
    # Add predictions / errors from every feature family
    # --------------------------------------------------------

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        family_prediction_table = (
            lightgbm_test_predictions[
                (
                    horizon_minutes,
                    feature_family,
                )
            ]
        )


        # ----------------------------------------------------
        # Exact pairing checks
        # ----------------------------------------------------

        assert (
            family_prediction_table.index.equals(
                reference_prediction_table.index
            )
        )

        assert family_prediction_table[
            "valid_time"
        ].equals(
            reference_prediction_table[
                "valid_time"
            ]
        )

        assert np.allclose(
            family_prediction_table[
                "y_true"
            ].to_numpy(),
            reference_prediction_table[
                "y_true"
            ].to_numpy(),
            atol=0.0,
            rtol=0.0,
        )

        assert family_prediction_table[
            "daylight"
        ].equals(
            reference_prediction_table[
                "daylight"
            ]
        )


        prediction_export[
            f"{feature_family}_prediction"
        ] = (
            family_prediction_table[
                "y_pred"
            ]
            .to_numpy()
        )


        prediction_export[
            f"{feature_family}_absolute_error"
        ] = (
            family_prediction_table[
                "absolute_error"
            ]
            .to_numpy()
        )


    # --------------------------------------------------------
    # Final prediction-file checks
    # --------------------------------------------------------

    assert (
        len(
            prediction_export
        )
        == 26_496
    )

    assert (
        prediction_export[
            "issue_time"
        ]
        .is_monotonic_increasing
    )

    assert (
        prediction_export[
            "valid_time"
        ]
        .is_monotonic_increasing
    )


    prediction_path = (
        LIGHTGBM_RESULTS_DIR
        / (
            "lightgbm_test_predictions_"
            f"{horizon_minutes}min.csv"
        )
    )


    prediction_export.to_csv(
        prediction_path,
        index=False,
    )


    lightgbm_prediction_paths[
        horizon_minutes
    ] = prediction_path


# ------------------------------------------------------------
# 5. JSON-safe helper
#
# Converts NumPy scalar types recursively so all protocol /
# configuration information is ordinary JSON.
# ------------------------------------------------------------

def make_json_safe_lightgbm(value):

    if isinstance(
        value,
        dict,
    ):

        return {
            str(key):
                make_json_safe_lightgbm(
                    item
                )
            for key, item
            in value.items()
        }


    if isinstance(
        value,
        (list, tuple),
    ):

        return [
            make_json_safe_lightgbm(
                item
            )
            for item in value
        ]


    if isinstance(
        value,
        np.integer,
    ):

        return int(
            value
        )


    if isinstance(
        value,
        np.floating,
    ):

        return float(
            value
        )


    if isinstance(
        value,
        np.bool_,
    ):

        return bool(
            value
        )


    return value


# ------------------------------------------------------------
# 6. Save frozen selected configurations
# ------------------------------------------------------------

lightgbm_selected_configs_path = (
    LIGHTGBM_RESULTS_DIR
    / "lightgbm_selected_configs.json"
)


with open(
    lightgbm_selected_configs_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        make_json_safe_lightgbm(
            lightgbm_selected_configs
        ),
        file,
        indent=2,
        sort_keys=True,
    )


# ------------------------------------------------------------
# 7. Record LightGBM-specific protocol
# ------------------------------------------------------------

lightgbm_protocol = {
    "model": "LGBMRegressor",

    "experiment_role":
        "secondary machine-learning model",

    "target":
        "P_Solar[kW]",

    "forecast_horizons_minutes":
        [
            15,
            30,
            60,
        ],

    "feature_family_order":
        [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ],

    "feature_counts":
        {
            feature_family:
                int(
                    primary_protocol[
                        "feature_counts"
                    ][
                        feature_family
                    ]
                )
            for feature_family in [
                "F0",
                "F1",
                "F2",
                "F3",
                "F4",
            ]
        },

    "validation_selection_metric":
        "MAE",

    "validation_secondary_metric":
        "RMSE",

    "test_selection_use":
        "none",

    "final_refit_population":
        "train + validation",

    "test_population":
        "held-out chronological test period",

    "fixed_parameters":
        make_json_safe_lightgbm(
            LIGHTGBM_FIXED_PARAMS
        ),

    "hyperparameter_grid":
        make_json_safe_lightgbm(
            lightgbm_configs
        ),

    "n_grid_configurations":
        int(
            len(
                lightgbm_configs
            )
        ),

    "early_stopping":
        False,

    "central_ablation":
        "F3 -> F4",

    "central_interpretation":
        (
            "incremental categorical Vedic/Panchang "
            "representation beyond continuous solar/lunar "
            "astronomical controls"
        ),

    "daylight_definition":
        "forecast-valid solar_elevation_deg > 0",

    "bootstrap": {
        "comparison":
            "F3 absolute error minus F4 absolute error",

        "positive_direction":
            "positive means F4 has lower MAE",

        "block_unit":
            "forecast-valid UTC calendar day",

        "daily_blocks":
            92,

        "replicates":
            10_000,

        "random_seed":
            42,

        "confidence_interval":
            "95% percentile",

        "model_retraining":
            False,

        "aggregation":
            (
                "pooled MAE reconstructed from sampled "
                "daily row counts and absolute-error sums"
            ),

        "scopes":
            [
                "all",
                "daylight",
            ],
    },
}


lightgbm_protocol_path = (
    LIGHTGBM_RESULTS_DIR
    / "lightgbm_protocol.json"
)


with open(
    lightgbm_protocol_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        make_json_safe_lightgbm(
            lightgbm_protocol
        ),
        file,
        indent=2,
        sort_keys=True,
    )


# ------------------------------------------------------------
# 8. Define complete pre-manifest artifact set
# ------------------------------------------------------------

lightgbm_artifact_paths = [
    lightgbm_validation_path,
    lightgbm_frozen_path,
    lightgbm_final_test_path,
    lightgbm_f3_f4_path,
    lightgbm_bootstrap_summary_path,
    lightgbm_bootstrap_distribution_path,
    lightgbm_prediction_paths[15],
    lightgbm_prediction_paths[30],
    lightgbm_prediction_paths[60],
    lightgbm_selected_configs_path,
    lightgbm_protocol_path,
]


assert len(
    lightgbm_artifact_paths
) == 11


# ------------------------------------------------------------
# 9. Verify every artifact exists and is non-empty
# ------------------------------------------------------------

for artifact_path in (
    lightgbm_artifact_paths
):

    assert (
        artifact_path.exists()
    ), (
        f"Missing artifact: "
        f"{artifact_path}"
    )

    assert (
        artifact_path.stat().st_size
        > 0
    ), (
        f"Empty artifact: "
        f"{artifact_path}"
    )


# ------------------------------------------------------------
# 10. SHA-256 helper
# ------------------------------------------------------------

def sha256_file_lightgbm(
    file_path,
):

    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as file:

        for chunk in iter(
            lambda:
                file.read(
                    1024 * 1024
                ),
            b"",
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


# ------------------------------------------------------------
# 11. Build reproducibility manifest
# ------------------------------------------------------------

lightgbm_manifest_rows = []


for artifact_path in sorted(
    lightgbm_artifact_paths,
    key=lambda path:
        path.name,
):

    lightgbm_manifest_rows.append(
        {
            "filename":
                artifact_path.name,

            "size_bytes":
                artifact_path.stat().st_size,

            "sha256":
                sha256_file_lightgbm(
                    artifact_path
                ),
        }
    )


lightgbm_results_manifest = (
    pd.DataFrame(
        lightgbm_manifest_rows
    )
)


assert (
    len(
        lightgbm_results_manifest
    )
    == 11
)


assert (
    lightgbm_results_manifest[
        "sha256"
    ]
    .str.len()
    .eq(
        64
    )
    .all()
)


# ------------------------------------------------------------
# 12. Save manifest itself
# ------------------------------------------------------------

lightgbm_manifest_path = (
    LIGHTGBM_RESULTS_DIR
    / "lightgbm_results_manifest.csv"
)


lightgbm_results_manifest.to_csv(
    lightgbm_manifest_path,
    index=False,
)


assert (
    lightgbm_manifest_path.exists()
)

assert (
    lightgbm_manifest_path.stat().st_size
    > 0
)


# ------------------------------------------------------------
# 13. Final artifact verification
# ------------------------------------------------------------

expected_lightgbm_artifact_names = {
    "lightgbm_validation_summary.csv",
    "lightgbm_frozen_selection.csv",
    "lightgbm_final_test_results.csv",
    "lightgbm_f3_f4_test_summary.csv",
    "lightgbm_bootstrap_summary.csv",
    "lightgbm_bootstrap_distributions.csv.gz",
    "lightgbm_test_predictions_15min.csv",
    "lightgbm_test_predictions_30min.csv",
    "lightgbm_test_predictions_60min.csv",
    "lightgbm_selected_configs.json",
    "lightgbm_protocol.json",
    "lightgbm_results_manifest.csv",
}


actual_lightgbm_artifact_names = {
    path.name
    for path in
    LIGHTGBM_RESULTS_DIR.glob(
        "lightgbm_*"
    )
}


assert (
    expected_lightgbm_artifact_names
    <= actual_lightgbm_artifact_names
)


# ------------------------------------------------------------
# 14. Display manifest
# ------------------------------------------------------------

print(
    "LightGBM artifacts persisted successfully.\n"
)


display(
    lightgbm_results_manifest
)


print(
    "\nManifest file:"
)

print(
    " ",
    lightgbm_manifest_path,
)


print(
    "\nArtifact count:"
)

print(
    "  11 experiment artifacts"
)

print(
    "  + 1 manifest"
)


print(
    "\nLightGBM persistence checkpoint: PASSED"
)

LightGBM artifacts persisted successfully.



,filename,size_bytes,sha256
0,lightgbm_bootstrap_distributions.csv.gz,1303311,d5f0495ac17d40d545a9bb29e2f92285cf3bc063513268...
1,lightgbm_bootstrap_summary.csv,1504,a67708b9337215c95f1d725f7f06628d5c496f6304d7ea...
2,lightgbm_f3_f4_test_summary.csv,492,b6611669e5b0ce365e5e82e4b9e2b6077a6df74b5cafa0...
3,lightgbm_final_test_results.csv,4437,c71b7d4bd72830190f0e20679861182953a71bd7c4812a...
4,lightgbm_frozen_selection.csv,1267,0f07119d62e6176813b423bfbe538b147ba2c0ebffacbf...
5,lightgbm_protocol.json,2519,1d9d25ec5d34b92ac87068d46c104357abb1c7666f77e1...
6,lightgbm_selected_configs.json,7169,1843a8178bf7ee5c31a86b301b7da2ed35bbb05a5b3fe6...
7,lightgbm_test_predictions_15min.csv,7062387,dfb1487074807e71da16750330549d24c61ce337553339...
8,lightgbm_test_predictions_30min.csv,7028478,ec7aea054fee5dc46da579fb7f0ca575f8b26c4a77cb83...
9,lightgbm_test_predictions_60min.csv,7018943,df91c2a778b2880ecdd2ebefdff8ae3548b2e9f3727096...



Manifest file:
  results/notebook10_secondary_ml_models/lightgbm_results_manifest.csv

Artifact count:
  11 experiment artifacts
  + 1 manifest

LightGBM persistence checkpoint: PASSED


In [101]:
# ============================================================
# Step 91: Freeze the Random Forest validation protocol
# ============================================================
#
# Random Forest is the fourth and final secondary classical
# machine-learning model in Notebook 10.
#
# The purpose of this cell is ONLY to freeze:
#
#   - model family;
#   - fixed parameters;
#   - hyperparameter search space;
#   - validation-selection rule.
#
# No Random Forest validation result is generated here.
#
#
# Experimental principle
# ----------------------
#
# We deliberately use the same structural search dimensions
# previously used for Extra Trees:
#
#       max_features
#       min_samples_leaf
#       max_depth
#
# This gives a cleaner algorithmic comparison between:
#
#       Extra Trees
#       vs
#       Random Forest
#
# while Random Forest retains its defining bootstrap-based
# tree construction.
#
#
# Selection rule
# --------------
#
# Primary:
#       chronological validation MAE
#
# Secondary / tie-break:
#       validation RMSE
#
# TEST remains completely untouched during selection.
# ============================================================


# ------------------------------------------------------------
# 1. Import Random Forest estimator
# ------------------------------------------------------------

from sklearn.ensemble import RandomForestRegressor


# ------------------------------------------------------------
# 2. Fixed Random Forest parameters
# ------------------------------------------------------------

RANDOM_FOREST_FIXED_PARAMS = {
    "n_estimators":
        300,

    "criterion":
        "squared_error",

    "bootstrap":
        True,

    "max_samples":
        None,

    "random_state":
        42,

    "n_jobs":
        -1,

    "verbose":
        0,
}


# ------------------------------------------------------------
# 3. Frozen hyperparameter search space
#
# 2 × 2 × 2 = 8 configurations
# ------------------------------------------------------------

RANDOM_FOREST_MAX_FEATURES = [
    0.5,
    1.0,
]

RANDOM_FOREST_MIN_SAMPLES_LEAF = [
    1,
    3,
]

RANDOM_FOREST_MAX_DEPTH = [
    None,
    30,
]


# ------------------------------------------------------------
# 4. Explicitly construct the frozen 8-configuration grid
# ------------------------------------------------------------

random_forest_configs = []


for max_features in (
    RANDOM_FOREST_MAX_FEATURES
):

    for min_samples_leaf in (
        RANDOM_FOREST_MIN_SAMPLES_LEAF
    ):

        for max_depth in (
            RANDOM_FOREST_MAX_DEPTH
        ):

            random_forest_configs.append(
                {
                    "max_features":
                        max_features,

                    "min_samples_leaf":
                        min_samples_leaf,

                    "max_depth":
                        max_depth,
                }
            )


# ------------------------------------------------------------
# 5. Integrity checks
# ------------------------------------------------------------

assert (
    len(
        random_forest_configs
    )
    == 8
)


assert {
    config[
        "max_features"
    ]
    for config in random_forest_configs
} == {
    0.5,
    1.0,
}


assert {
    config[
        "min_samples_leaf"
    ]
    for config in random_forest_configs
} == {
    1,
    3,
}


assert {
    config[
        "max_depth"
    ]
    for config in random_forest_configs
} == {
    None,
    30,
}


# ------------------------------------------------------------
# 6. Record model-selection protocol
# ------------------------------------------------------------

random_forest_validation_protocol = {
    "model":
        "RandomForestRegressor",

    "n_estimators":
        300,

    "n_configurations":
        len(
            random_forest_configs
        ),

    "validation_primary_metric":
        "MAE",

    "validation_secondary_metric":
        "RMSE",

    "selection_population":
        "training + chronological validation only",

    "test_used_for_selection":
        False,

    "early_stopping":
        False,

    "random_state":
        42,

    "n_jobs":
        -1,
}


# ------------------------------------------------------------
# 7. Display frozen protocol
# ------------------------------------------------------------

print(
    "Random Forest fixed parameters:\n"
)

for (
    parameter,
    value,
) in RANDOM_FOREST_FIXED_PARAMS.items():

    print(
        f"  {parameter}: {value}"
    )


print(
    "\nFrozen Random Forest configurations:"
)


for (
    config_number,
    config,
) in enumerate(
    random_forest_configs,
    start=1,
):

    print(
        f"  Configuration "
        f"{config_number}/8:"
    )

    print(
        "    max_features      =",
        config[
            "max_features"
        ],
    )

    print(
        "    min_samples_leaf  =",
        config[
            "min_samples_leaf"
        ],
    )

    print(
        "    max_depth         =",
        config[
            "max_depth"
        ],
    )


print(
    "\nSelection rule:"
)

print(
    "  Primary   = validation MAE"
)

print(
    "  Secondary = validation RMSE"
)

print(
    "  TEST      = untouched"
)


print(
    "\nRandom Forest protocol freeze: PASSED"
)

Random Forest fixed parameters:

  n_estimators: 300
  criterion: squared_error
  bootstrap: True
  max_samples: None
  random_state: 42
  n_jobs: -1
  verbose: 0

Frozen Random Forest configurations:
  Configuration 1/8:
    max_features      = 0.5
    min_samples_leaf  = 1
    max_depth         = None
  Configuration 2/8:
    max_features      = 0.5
    min_samples_leaf  = 1
    max_depth         = 30
  Configuration 3/8:
    max_features      = 0.5
    min_samples_leaf  = 3
    max_depth         = None
  Configuration 4/8:
    max_features      = 0.5
    min_samples_leaf  = 3
    max_depth         = 30
  Configuration 5/8:
    max_features      = 1.0
    min_samples_leaf  = 1
    max_depth         = None
  Configuration 6/8:
    max_features      = 1.0
    min_samples_leaf  = 1
    max_depth         = 30
  Configuration 7/8:
    max_features      = 1.0
    min_samples_leaf  = 3
    max_depth         = None
  Configuration 8/8:
    max_features      = 1.0
    min_samples_leaf  = 3
  

In [102]:
# ============================================================
# Step 92: Define reusable Random Forest validation helper
# ============================================================
#
# This function evaluates the already-frozen 8-configuration
# Random Forest grid for ONE:
#
#       forecast horizon × feature family
#
# combination.
#
# Model selection uses:
#
#   Primary:
#       chronological validation MAE
#
#   Secondary / tie-break:
#       chronological validation RMSE
#
# Only TRAIN is used for fitting.
# Only VALIDATION is used for model selection.
#
# TEST is never accessed inside this function.
# ============================================================


def run_random_forest_validation_grid(
    horizon_minutes,
    feature_family,
):

    # --------------------------------------------------------
    # 1. Retrieve the requested feature matrix, target,
    #    and chronological split labels
    # --------------------------------------------------------

    X = (
        feature_matrices[
            horizon_minutes
        ][
            feature_family
        ]
    )

    y = targets[
        horizon_minutes
    ]

    split = split_labels[
        horizon_minutes
    ]


    # --------------------------------------------------------
    # 2. Define TRAIN and VALIDATION populations only
    # --------------------------------------------------------

    train_mask = (
        split == "train"
    )

    validation_mask = (
        split == "validation"
    )


    # --------------------------------------------------------
    # 3. Convert to dense float32 NumPy arrays
    #
    # Random Forest does not require scaling.
    #
    # float32 reduces memory usage while retaining more than
    # enough precision for the present tree-based experiment.
    # --------------------------------------------------------

    X_train = (
        X.loc[
            train_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    X_validation = (
        X.loc[
            validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    y_train = (
        y.loc[
            train_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    y_validation = (
        y.loc[
            validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # 4. Structural integrity checks
    # --------------------------------------------------------

    expected_feature_count = (
        primary_protocol[
            "feature_counts"
        ][
            feature_family
        ]
    )


    assert (
        X_train.shape[1]
        == expected_feature_count
    )

    assert (
        X_validation.shape[1]
        == expected_feature_count
    )


    assert (
        X_train.shape[0]
        == int(
            train_mask.sum()
        )
    )

    assert (
        X_validation.shape[0]
        == int(
            validation_mask.sum()
        )
    )


    assert np.isfinite(
        X_train
    ).all()

    assert np.isfinite(
        X_validation
    ).all()

    assert np.isfinite(
        y_train
    ).all()

    assert np.isfinite(
        y_validation
    ).all()


    # --------------------------------------------------------
    # 5. Report the exact validation experiment
    # --------------------------------------------------------

    print(
        "\nRandom Forest validation dataset:"
    )

    print(
        "  Horizon:",
        horizon_minutes,
        "minutes",
    )

    print(
        "  Feature family:",
        feature_family,
    )

    print(
        "  Train:",
        X_train.shape,
    )

    print(
        "  Validation:",
        X_validation.shape,
    )

    print(
        "  Frozen configurations:",
        len(
            random_forest_configs
        ),
    )


    # --------------------------------------------------------
    # 6. Evaluate the frozen grid
    # --------------------------------------------------------

    result_rows = []

    total_start_time = (
        time.perf_counter()
    )


    for (
        config_number,
        variable_config,
    ) in enumerate(
        random_forest_configs,
        start=1,
    ):

        # ----------------------------------------------------
        # Combine fixed parameters with the current frozen
        # variable configuration
        # ----------------------------------------------------

        model_config = {
            **RANDOM_FOREST_FIXED_PARAMS,
            **variable_config,
        }


        print(
            f"\nConfiguration "
            f"{config_number}/"
            f"{len(random_forest_configs)}"
        )

        print(
            "  max_features      =",
            variable_config[
                "max_features"
            ],
        )

        print(
            "  min_samples_leaf  =",
            variable_config[
                "min_samples_leaf"
            ],
        )

        print(
            "  max_depth         =",
            variable_config[
                "max_depth"
            ],
        )


        # ----------------------------------------------------
        # Fit only on TRAIN
        # ----------------------------------------------------

        model = RandomForestRegressor(
            **model_config
        )


        fit_start_time = (
            time.perf_counter()
        )


        model.fit(
            X_train,
            y_train,
        )


        # ----------------------------------------------------
        # Predict only VALIDATION
        # ----------------------------------------------------

        validation_predictions = (
            model.predict(
                X_validation
            )
        )


        fit_predict_seconds = (
            time.perf_counter()
            - fit_start_time
        )


        # ----------------------------------------------------
        # 7. Validation metrics
        # ----------------------------------------------------

        validation_mae = (
            mean_absolute_error(
                y_validation,
                validation_predictions,
            )
        )


        validation_rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                validation_predictions,
            )
        )


        # ----------------------------------------------------
        # 8. Store current configuration result
        # ----------------------------------------------------

        result_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_estimators":
                    RANDOM_FOREST_FIXED_PARAMS[
                        "n_estimators"
                    ],

                "max_features":
                    variable_config[
                        "max_features"
                    ],

                "min_samples_leaf":
                    variable_config[
                        "min_samples_leaf"
                    ],

                "max_depth":
                    variable_config[
                        "max_depth"
                    ],

                "bootstrap":
                    RANDOM_FOREST_FIXED_PARAMS[
                        "bootstrap"
                    ],

                "validation_mae":
                    validation_mae,

                "validation_rmse":
                    validation_rmse,

                "fit_predict_seconds":
                    fit_predict_seconds,
            }
        )


        # ----------------------------------------------------
        # 9. Compact live result
        # ----------------------------------------------------

        print(
            f"  validation MAE  = "
            f"{validation_mae:.6f}"
        )

        print(
            f"  validation RMSE = "
            f"{validation_rmse:.6f}"
        )

        print(
            f"  runtime         = "
            f"{fit_predict_seconds:.1f} s"
        )


        # ----------------------------------------------------
        # Release fitted estimator before next configuration
        # ----------------------------------------------------

        del model
        del validation_predictions

        gc.collect()


    # --------------------------------------------------------
    # 10. Assemble complete ordered result table
    # --------------------------------------------------------

    results_df = pd.DataFrame(
        result_rows
    )


    assert (
        len(
            results_df
        )
        == 8
    )


    # --------------------------------------------------------
    # Selection rule:
    #
    #   1. lowest validation MAE
    #   2. lowest validation RMSE if MAE ties
    # --------------------------------------------------------

    results_df = (
        results_df
        .sort_values(
            [
                "validation_mae",
                "validation_rmse",
            ],
            ascending=True,
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # 11. Total grid runtime
    # --------------------------------------------------------

    total_seconds = (
        time.perf_counter()
        - total_start_time
    )


    # --------------------------------------------------------
    # 12. Final integrity checks
    # --------------------------------------------------------

    assert (
        results_df[
            "validation_mae"
        ]
        .notna()
        .all()
    )

    assert (
        results_df[
            "validation_rmse"
        ]
        .notna()
        .all()
    )


    assert np.isfinite(
        results_df[
            [
                "validation_mae",
                "validation_rmse",
                "fit_predict_seconds",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    ).all()


    return (
        results_df,
        total_seconds,
    )


print(
    "Random Forest validation helper defined successfully."
)

print(
    "TEST data are not accessed by this helper."
)

print(
    "Random Forest validation helper: PASSED"
)

Random Forest validation helper defined successfully.
TEST data are not accessed by this helper.
Random Forest validation helper: PASSED


In [103]:
# ============================================================
# Step 93: Random Forest validation
#          60-minute horizon + F0
# ============================================================
#
# F0 contains the historical/autoregressive baseline:
#
#   - recent observed measurements through issue time t;
#   - lagged PV / irradiance / meteorological variables;
#   - 24-hour PV lag.
#
# The previously frozen 8-configuration Random Forest grid
# is evaluated using TRAIN + chronological VALIDATION only.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_60_f0_df, random_forest_60_f0_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=60,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f0_60 = (
    random_forest_60_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 60-minute F0 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_60_f0_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f0_60[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f0_60[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f0_60[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    best_random_forest_f0_60[
        "max_depth"
    ],
)

print(
    "  validation MAE:",
    best_random_forest_f0_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f0_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F0 grid runtime:"
)

print(
    f"  {random_forest_60_f0_seconds:.1f} seconds"
)

print(
    f"  {random_forest_60_f0_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 60-minute F0 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 60 minutes
  Feature family: F0
  Train: (87252, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.453747
  validation RMSE = 0.803828
  runtime         = 164.5 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.453314
  validation RMSE = 0.803577
  runtime         = 199.9 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.441826
  validation RMSE = 0.795699
  runtime         = 315.9 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.442538
  validation RMSE = 0.795910
  runtime         = 305.5 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.4563

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,60,F0,300,0.5,3,NaN,True,0.441826,0.795699,315.935030
1,60,F0,300,0.5,3,30.0,True,0.442538,0.795910,305.529869
2,60,F0,300,1.0,3,30.0,True,0.444948,0.801274,509.676227
3,60,F0,300,1.0,3,NaN,True,0.445398,0.801537,591.882095
4,60,F0,300,0.5,1,30.0,True,0.453314,0.803577,199.941508
5,60,F0,300,0.5,1,NaN,True,0.453747,0.803828,164.539068
6,60,F0,300,1.0,1,30.0,True,0.456122,0.809569,598.665925
7,60,F0,300,1.0,1,NaN,True,0.456361,0.809765,658.341928



Selected 60-minute F0 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: nan
  validation MAE: 0.44182583593395247
  validation RMSE: 0.7956985776280951

Total 60-minute F0 grid runtime:
  3345.2 seconds
  55.75 minutes

Random Forest 60-minute F0 experiment: PASSED


In [104]:
# ============================================================
# Step 94: Random Forest validation
#          60-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + conventional Gregorian / calendar-time features
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_60_f1_df, random_forest_60_f1_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=60,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f1_60 = (
    random_forest_60_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 60-minute F1 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_60_f1_df
)


# ------------------------------------------------------------
# 4. Display selected configuration
#
# Note:
# pandas may display max_depth=None as NaN because the same
# column also contains numeric depth values.
# ------------------------------------------------------------

selected_max_depth_f1_60 = (
    None
    if pd.isna(
        best_random_forest_f1_60[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f1_60[
            "max_depth"
        ]
    )
)


print(
    "\nSelected 60-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f1_60[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f1_60[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f1_60[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f1_60,
)

print(
    "  validation MAE:",
    best_random_forest_f1_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f1_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 5. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_60_mae_rf = float(
    best_random_forest_f0_60[
        "validation_mae"
    ]
)

f1_60_mae_rf = float(
    best_random_forest_f1_60[
        "validation_mae"
    ]
)


f0_to_f1_60_pct_rf = (
    (
        f0_60_mae_rf
        - f1_60_mae_rf
    )
    / f0_60_mae_rf
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_60_mae_rf:.6f}"
)

print(
    f"  F1 MAE: {f1_60_mae_rf:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_60_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F1 grid runtime:"
)

print(
    f"  {random_forest_60_f1_seconds:.1f} seconds"
)

print(
    f"  {random_forest_60_f1_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 60-minute F1 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 60 minutes
  Feature family: F1
  Train: (87252, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.386665
  validation RMSE = 0.730008
  runtime         = 327.0 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.386463
  validation RMSE = 0.728889
  runtime         = 323.5 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.377246
  validation RMSE = 0.724708
  runtime         = 292.7 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.376561
  validation RMSE = 0.723506
  runtime         = 287.0 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.3862

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,60,F1,300,0.5,3,30.0,True,0.376561,0.723506,286.957938
1,60,F1,300,0.5,3,NaN,True,0.377246,0.724708,292.658207
2,60,F1,300,1.0,3,30.0,True,0.378322,0.728043,526.204662
3,60,F1,300,1.0,3,NaN,True,0.378795,0.728573,526.936754
4,60,F1,300,1.0,1,NaN,True,0.386243,0.733074,589.725390
5,60,F1,300,1.0,1,30.0,True,0.386441,0.732916,567.745343
6,60,F1,300,0.5,1,30.0,True,0.386463,0.728889,323.464803
7,60,F1,300,0.5,1,NaN,True,0.386665,0.730008,327.013793



Selected 60-minute F1 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.37656141512362173
  validation RMSE: 0.7235060781097488

F0 -> F1 validation comparison:
  F0 MAE: 0.441826
  F1 MAE: 0.376561
  Relative MAE improvement: +14.7715%

Total 60-minute F1 grid runtime:
  3441.4 seconds
  57.36 minutes

Random Forest 60-minute F1 experiment: PASSED


In [105]:
# ============================================================
# Step 95: Random Forest validation
#          60-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + deterministic solar-geometry features
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_60_f2_df, random_forest_60_f2_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=60,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f2_60 = (
    random_forest_60_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 60-minute F2 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_60_f2_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
#
# pandas may represent None as NaN in a mixed numeric column.
# ------------------------------------------------------------

selected_max_depth_f2_60 = (
    None
    if pd.isna(
        best_random_forest_f2_60[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f2_60[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f2_60[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f2_60[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f2_60[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f2_60,
)

print(
    "  validation MAE:",
    best_random_forest_f2_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f2_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_60_mae_rf = float(
    best_random_forest_f1_60[
        "validation_mae"
    ]
)

f2_60_mae_rf = float(
    best_random_forest_f2_60[
        "validation_mae"
    ]
)


f1_to_f2_60_pct_rf = (
    (
        f1_60_mae_rf
        - f2_60_mae_rf
    )
    / f1_60_mae_rf
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_60_mae_rf:.6f}"
)

print(
    f"  F2 MAE: {f2_60_mae_rf:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_60_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F2 grid runtime:"
)

print(
    f"  {random_forest_60_f2_seconds:.1f} seconds"
)

print(
    f"  {random_forest_60_f2_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 60-minute F2 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 60 minutes
  Feature family: F2
  Train: (87252, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.386733
  validation RMSE = 0.732091
  runtime         = 356.0 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.385796
  validation RMSE = 0.730703
  runtime         = 353.1 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.378712
  validation RMSE = 0.726223
  runtime         = 331.7 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.376613
  validation RMSE = 0.724660
  runtime         = 310.4 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.3844

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,60,F2,300,1.0,3,30.0,True,0.376448,0.725366,633.824154
1,60,F2,300,0.5,3,30.0,True,0.376613,0.724660,310.413621
2,60,F2,300,1.0,3,NaN,True,0.376645,0.725692,635.180744
3,60,F2,300,0.5,3,NaN,True,0.378712,0.726223,331.678995
4,60,F2,300,1.0,1,NaN,True,0.384420,0.730349,766.811864
5,60,F2,300,1.0,1,30.0,True,0.384809,0.731283,670.992793
6,60,F2,300,0.5,1,30.0,True,0.385796,0.730703,353.056154
7,60,F2,300,0.5,1,NaN,True,0.386733,0.732091,355.960945



Selected 60-minute F2 configuration:
  n_estimators: 300
  max_features: 1.0
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.3764482831479939
  validation RMSE: 0.7253664962564347

F1 -> F2 validation comparison:
  F1 MAE: 0.376561
  F2 MAE: 0.376448
  Relative MAE improvement: +0.0300%

Total 60-minute F2 grid runtime:
  4058.7 seconds
  67.64 minutes

Random Forest 60-minute F2 experiment: PASSED


In [106]:
# ============================================================
# Step 96: Random Forest validation
#          60-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + continuous lunar / Sun-Moon astronomical features
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_60_f3_df, random_forest_60_f3_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=60,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f3_60 = (
    random_forest_60_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 60-minute F3 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_60_f3_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
# ------------------------------------------------------------

selected_max_depth_f3_60 = (
    None
    if pd.isna(
        best_random_forest_f3_60[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f3_60[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f3_60[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f3_60[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f3_60[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f3_60,
)

print(
    "  validation MAE:",
    best_random_forest_f3_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f3_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_60_mae_rf = float(
    best_random_forest_f2_60[
        "validation_mae"
    ]
)

f3_60_mae_rf = float(
    best_random_forest_f3_60[
        "validation_mae"
    ]
)


f2_to_f3_60_pct_rf = (
    (
        f2_60_mae_rf
        - f3_60_mae_rf
    )
    / f2_60_mae_rf
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_60_mae_rf:.6f}"
)

print(
    f"  F3 MAE: {f3_60_mae_rf:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_60_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F3 grid runtime:"
)

print(
    f"  {random_forest_60_f3_seconds:.1f} seconds"
)

print(
    f"  {random_forest_60_f3_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 60-minute F3 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 60 minutes
  Feature family: F3
  Train: (87252, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.389709
  validation RMSE = 0.732746
  runtime         = 377.6 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.389520
  validation RMSE = 0.732556
  runtime         = 360.7 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.381455
  validation RMSE = 0.727151
  runtime         = 262.1 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.379673
  validation RMSE = 0.726033
  runtime         = 197.2 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.3897

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,60,F3,300,0.5,3,30.0,True,0.379673,0.726033,197.199778
1,60,F3,300,0.5,3,NaN,True,0.381455,0.727151,262.071442
2,60,F3,300,1.0,3,30.0,True,0.381956,0.730781,466.851371
3,60,F3,300,1.0,3,NaN,True,0.382389,0.730911,500.992098
4,60,F3,300,0.5,1,30.0,True,0.389520,0.732556,360.722415
5,60,F3,300,0.5,1,NaN,True,0.389709,0.732746,377.586833
6,60,F3,300,1.0,1,NaN,True,0.389781,0.736045,492.308643
7,60,F3,300,1.0,1,30.0,True,0.390085,0.735898,485.236371



Selected 60-minute F3 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.37967255792116683
  validation RMSE: 0.7260326569921006

F2 -> F3 validation comparison:
  F2 MAE: 0.376448
  F3 MAE: 0.379673
  Relative MAE improvement: -0.8565%

Total 60-minute F3 grid runtime:
  3143.5 seconds
  52.39 minutes

Random Forest 60-minute F3 experiment: PASSED


In [107]:
# ============================================================
# Step 97: Random Forest validation
#          60-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + one-hot encoded Vedic/Panchang categorical features
#
# This is the central F3 -> F4 comparison for the
# 60-minute horizon.
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_60_f4_df, random_forest_60_f4_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=60,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f4_60 = (
    random_forest_60_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 60-minute F4 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_60_f4_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
# ------------------------------------------------------------

selected_max_depth_f4_60 = (
    None
    if pd.isna(
        best_random_forest_f4_60[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f4_60[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 60-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f4_60[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f4_60[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f4_60[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f4_60,
)

print(
    "  validation MAE:",
    best_random_forest_f4_60[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f4_60[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. Central F3 -> F4 validation comparison
# ------------------------------------------------------------

f3_60_mae_rf = float(
    best_random_forest_f3_60[
        "validation_mae"
    ]
)

f4_60_mae_rf = float(
    best_random_forest_f4_60[
        "validation_mae"
    ]
)


f3_to_f4_60_absolute_rf = (
    f3_60_mae_rf
    - f4_60_mae_rf
)


f3_to_f4_60_pct_rf = (
    f3_to_f4_60_absolute_rf
    / f3_60_mae_rf
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_60_mae_rf:.6f}"
)

print(
    f"  F4 MAE: {f4_60_mae_rf:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_60_absolute_rf:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_60_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 60-minute F4 grid runtime:"
)

print(
    f"  {random_forest_60_f4_seconds:.1f} seconds"
)

print(
    f"  {random_forest_60_f4_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 60-minute F4 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 60 minutes
  Feature family: F4
  Train: (87252, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.389908
  validation RMSE = 0.734725
  runtime         = 440.7 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.387333
  validation RMSE = 0.732796
  runtime         = 495.5 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.380803
  validation RMSE = 0.727983
  runtime         = 443.6 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.381681
  validation RMSE = 0.729707
  runtime         = 413.7 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.3899

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,60,F4,300,0.5,3,NaN,True,0.380803,0.727983,443.600954
1,60,F4,300,1.0,3,NaN,True,0.381569,0.732782,935.370602
2,60,F4,300,0.5,3,30.0,True,0.381681,0.729707,413.695272
3,60,F4,300,1.0,3,30.0,True,0.381915,0.732768,849.053224
4,60,F4,300,0.5,1,30.0,True,0.387333,0.732796,495.465463
5,60,F4,300,0.5,1,NaN,True,0.389908,0.734725,440.690830
6,60,F4,300,1.0,1,NaN,True,0.389910,0.738408,1025.720705
7,60,F4,300,1.0,1,30.0,True,0.390312,0.739501,826.443148



Selected 60-minute F4 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.3808030130575144
  validation RMSE: 0.7279827571925431

F3 -> F4 validation comparison:
  F3 MAE: 0.379673
  F4 MAE: 0.380803
  Absolute MAE reduction: -0.001130
  Relative MAE improvement: -0.2977%

Total 60-minute F4 grid runtime:
  5430.8 seconds
  90.51 minutes

Random Forest 60-minute F4 experiment: PASSED


In [108]:
# ============================================================
# Step 98: Summarize the complete 60-minute Random Forest
#          F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect selected configurations
# ------------------------------------------------------------

best_rows_rf_60 = {
    "F0": best_random_forest_f0_60,
    "F1": best_random_forest_f1_60,
    "F2": best_random_forest_f2_60,
    "F3": best_random_forest_f3_60,
    "F4": best_random_forest_f4_60,
}


summary_rows_rf_60 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_rf_60[
        feature_family
    ]


    # --------------------------------------------------------
    # pandas converts None -> NaN in the mixed max_depth
    # column, so recover the original semantic value here.
    # --------------------------------------------------------

    selected_max_depth = (
        None
        if pd.isna(
            row[
                "max_depth"
            ]
        )
        else int(
            row[
                "max_depth"
            ]
        )
    )


    summary_rows_rf_60.append(
        {
            "horizon_minutes":
                60,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][
                    feature_family
                ],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "max_features":
                float(
                    row[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    row[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                selected_max_depth,

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


random_forest_60_summary = pd.DataFrame(
    summary_rows_rf_60
)


# ------------------------------------------------------------
# 2. Add incremental MAE improvement
#
# Positive:
#     newly added feature family reduced MAE.
#
# Negative:
#     newly added feature family increased MAE.
# ------------------------------------------------------------

random_forest_60_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(
        random_forest_60_summary
    ),
):

    previous_mae = float(
        random_forest_60_summary.loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = float(
        random_forest_60_summary.loc[
            i,
            "validation_mae",
        ]
    )


    random_forest_60_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (
            previous_mae
            - current_mae
        )
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_rf_60 = float(
    random_forest_60_summary.loc[
        random_forest_60_summary[
            "feature_family"
        ].eq(
            "F3"
        ),
        "validation_mae",
    ]
    .iloc[0]
)


f4_summary_mae_rf_60 = float(
    random_forest_60_summary.loc[
        random_forest_60_summary[
            "feature_family"
        ].eq(
            "F4"
        ),
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_rf_60 = (
    (
        f3_summary_mae_rf_60
        - f4_summary_mae_rf_60
    )
    / f3_summary_mae_rf_60
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_rf_60,
    -0.2977,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete summary
# ------------------------------------------------------------

print(
    "Random Forest — 60-minute validation summary\n"
)


display(
    random_forest_60_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)


for i in range(
    1,
    len(
        random_forest_60_summary
    ),
):

    previous_family = (
        random_forest_60_summary.loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        random_forest_60_summary.loc[
            i,
            "feature_family",
        ]
    )

    change = float(
        random_forest_60_summary.loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )


    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_rf_60 = (
    random_forest_60_summary
    .sort_values(
        [
            "validation_mae",
            "validation_rmse",
        ]
    )
    .iloc[0]
)


print(
    "\nLowest 60-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_rf_60[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_rf_60[
        "validation_mae"
    ],
)


# ------------------------------------------------------------
# 7. Central F3 -> F4 comparison
# ------------------------------------------------------------

print(
    "\nCentral Random Forest comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_rf_60:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_rf_60:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_rf_60:+.4f}%"
)


print(
    "\nRandom Forest 60-minute summary: PASSED"
)

Random Forest — 60-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,60,F0,118,300,0.5,3,NaN,0.441826,0.795699,NaN
1,60,F1,133,300,0.5,3,30.0,0.376561,0.723506,14.771527
2,60,F2,138,300,1.0,3,30.0,0.376448,0.725366,0.030043
3,60,F3,147,300,0.5,3,30.0,0.379673,0.726033,-0.856499
4,60,F4,244,300,0.5,3,NaN,0.380803,0.727983,-0.297745


Incremental validation MAE changes:
  F0 -> F1: +14.7715%
  F1 -> F2: +0.0300%
  F2 -> F3: -0.8565%
  F3 -> F4: -0.2977%

Lowest 60-minute validation MAE:
  Feature family: F2
  MAE: 0.3764482831479939

Central Random Forest comparison:
  F3 MAE: 0.379673
  F4 MAE: 0.380803
  F3 -> F4 change: -0.2977%

Random Forest 60-minute summary: PASSED


In [109]:
# ============================================================
# Step 99: Random Forest validation
#          30-minute horizon + F0
# ============================================================
#
# F0 is the historical/autoregressive baseline.
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_30_f0_df, random_forest_30_f0_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=30,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f0_30 = (
    random_forest_30_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 30-minute F0 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_30_f0_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
# ------------------------------------------------------------

selected_max_depth_f0_30 = (
    None
    if pd.isna(
        best_random_forest_f0_30[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f0_30[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f0_30[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f0_30[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f0_30[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f0_30,
)

print(
    "  validation MAE:",
    best_random_forest_f0_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f0_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F0 grid runtime:"
)

print(
    f"  {random_forest_30_f0_seconds:.1f} seconds"
)

print(
    f"  {random_forest_30_f0_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 30-minute F0 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 30 minutes
  Feature family: F0
  Train: (87258, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.352663
  validation RMSE = 0.684732
  runtime         = 331.2 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.352583
  validation RMSE = 0.685040
  runtime         = 304.2 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.343069
  validation RMSE = 0.677982
  runtime         = 281.5 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.342799
  validation RMSE = 0.677943
  runtime         = 286.4 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.3529

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,30,F0,300,0.5,3,30.0,True,0.342799,0.677943,286.436711
1,30,F0,300,0.5,3,NaN,True,0.343069,0.677982,281.481812
2,30,F0,300,1.0,3,30.0,True,0.343235,0.678669,582.760496
3,30,F0,300,1.0,3,NaN,True,0.343309,0.678758,577.030007
4,30,F0,300,1.0,1,30.0,True,0.352219,0.686105,550.581967
5,30,F0,300,0.5,1,30.0,True,0.352583,0.685040,304.201032
6,30,F0,300,0.5,1,NaN,True,0.352663,0.684732,331.206674
7,30,F0,300,1.0,1,NaN,True,0.352915,0.687442,615.369276



Selected 30-minute F0 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.34279927243968805
  validation RMSE: 0.6779428291572036

Total 30-minute F0 grid runtime:
  3529.7 seconds
  58.83 minutes

Random Forest 30-minute F0 experiment: PASSED


In [110]:
# ============================================================
# Step 100: Random Forest validation
#           30-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + conventional Gregorian / calendar-time features
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_30_f1_df, random_forest_30_f1_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=30,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f1_30 = (
    random_forest_30_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 30-minute F1 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_30_f1_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
# ------------------------------------------------------------

selected_max_depth_f1_30 = (
    None
    if pd.isna(
        best_random_forest_f1_30[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f1_30[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f1_30[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f1_30[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f1_30[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f1_30,
)

print(
    "  validation MAE:",
    best_random_forest_f1_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f1_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_30_mae_rf = float(
    best_random_forest_f0_30[
        "validation_mae"
    ]
)

f1_30_mae_rf = float(
    best_random_forest_f1_30[
        "validation_mae"
    ]
)


f0_to_f1_30_pct_rf = (
    (
        f0_30_mae_rf
        - f1_30_mae_rf
    )
    / f0_30_mae_rf
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_30_mae_rf:.6f}"
)

print(
    f"  F1 MAE: {f1_30_mae_rf:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_30_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F1 grid runtime:"
)

print(
    f"  {random_forest_30_f1_seconds:.1f} seconds"
)

print(
    f"  {random_forest_30_f1_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 30-minute F1 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 30 minutes
  Feature family: F1
  Train: (87258, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.327896
  validation RMSE = 0.654093
  runtime         = 181.7 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.325687
  validation RMSE = 0.650688
  runtime         = 234.2 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.315656
  validation RMSE = 0.644300
  runtime         = 238.3 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.316606
  validation RMSE = 0.644982
  runtime         = 211.4 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.3287

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,30,F1,300,0.5,3,NaN,True,0.315656,0.644300,238.312540
1,30,F1,300,0.5,3,30.0,True,0.316606,0.644982,211.396681
2,30,F1,300,1.0,3,30.0,True,0.317681,0.648015,468.547265
3,30,F1,300,1.0,3,NaN,True,0.318091,0.648297,724.191997
4,30,F1,300,0.5,1,30.0,True,0.325687,0.650688,234.211905
5,30,F1,300,0.5,1,NaN,True,0.327896,0.654093,181.700874
6,30,F1,300,1.0,1,30.0,True,0.328615,0.656723,789.768758
7,30,F1,300,1.0,1,NaN,True,0.328709,0.656931,626.917077



Selected 30-minute F1 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.3156559614174901
  validation RMSE: 0.644300490622674

F0 -> F1 validation comparison:
  F0 MAE: 0.342799
  F1 MAE: 0.315656
  Relative MAE improvement: +7.9181%

Total 30-minute F1 grid runtime:
  3475.7 seconds
  57.93 minutes

Random Forest 30-minute F1 experiment: PASSED


In [111]:
# ============================================================
# Step 101: Random Forest validation
#           30-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + deterministic solar-geometry features
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_30_f2_df, random_forest_30_f2_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=30,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f2_30 = (
    random_forest_30_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 30-minute F2 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_30_f2_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
# ------------------------------------------------------------

selected_max_depth_f2_30 = (
    None
    if pd.isna(
        best_random_forest_f2_30[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f2_30[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f2_30[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f2_30[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f2_30[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f2_30,
)

print(
    "  validation MAE:",
    best_random_forest_f2_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f2_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_30_mae_rf = float(
    best_random_forest_f1_30[
        "validation_mae"
    ]
)

f2_30_mae_rf = float(
    best_random_forest_f2_30[
        "validation_mae"
    ]
)


f1_to_f2_30_pct_rf = (
    (
        f1_30_mae_rf
        - f2_30_mae_rf
    )
    / f1_30_mae_rf
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_30_mae_rf:.6f}"
)

print(
    f"  F2 MAE: {f2_30_mae_rf:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_30_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F2 grid runtime:"
)

print(
    f"  {random_forest_30_f2_seconds:.1f} seconds"
)

print(
    f"  {random_forest_30_f2_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 30-minute F2 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 30 minutes
  Feature family: F2
  Train: (87258, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.327642
  validation RMSE = 0.653283
  runtime         = 187.6 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.327286
  validation RMSE = 0.651826
  runtime         = 211.6 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.315629
  validation RMSE = 0.643351
  runtime         = 201.7 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.316665
  validation RMSE = 0.644835
  runtime         = 206.9 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.3274

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,30,F2,300,0.5,3,NaN,True,0.315629,0.643351,201.693910
1,30,F2,300,0.5,3,30.0,True,0.316665,0.644835,206.875707
2,30,F2,300,1.0,3,NaN,True,0.317273,0.646833,410.997791
3,30,F2,300,1.0,3,30.0,True,0.317556,0.647025,405.735495
4,30,F2,300,0.5,1,30.0,True,0.327286,0.651826,211.636405
5,30,F2,300,1.0,1,NaN,True,0.327407,0.654987,435.447500
6,30,F2,300,0.5,1,NaN,True,0.327642,0.653283,187.621393
7,30,F2,300,1.0,1,30.0,True,0.327648,0.655901,427.352115



Selected 30-minute F2 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.3156289306719466
  validation RMSE: 0.6433506681684523

F1 -> F2 validation comparison:
  F1 MAE: 0.315656
  F2 MAE: 0.315629
  Relative MAE improvement: +0.0086%

Total 30-minute F2 grid runtime:
  2487.9 seconds
  41.46 minutes

Random Forest 30-minute F2 experiment: PASSED


In [112]:
# ============================================================
# Step 102: Random Forest validation
#           30-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + continuous lunar / Sun-Moon astronomical features
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_30_f3_df, random_forest_30_f3_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=30,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f3_30 = (
    random_forest_30_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 30-minute F3 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_30_f3_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
# ------------------------------------------------------------

selected_max_depth_f3_30 = (
    None
    if pd.isna(
        best_random_forest_f3_30[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f3_30[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f3_30[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f3_30[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f3_30[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f3_30,
)

print(
    "  validation MAE:",
    best_random_forest_f3_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f3_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_30_mae_rf = float(
    best_random_forest_f2_30[
        "validation_mae"
    ]
)

f3_30_mae_rf = float(
    best_random_forest_f3_30[
        "validation_mae"
    ]
)


f2_to_f3_30_pct_rf = (
    (
        f2_30_mae_rf
        - f3_30_mae_rf
    )
    / f2_30_mae_rf
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_30_mae_rf:.6f}"
)

print(
    f"  F3 MAE: {f3_30_mae_rf:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_30_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F3 grid runtime:"
)

print(
    f"  {random_forest_30_f3_seconds:.1f} seconds"
)

print(
    f"  {random_forest_30_f3_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 30-minute F3 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 30 minutes
  Feature family: F3
  Train: (87258, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.327516
  validation RMSE = 0.653257
  runtime         = 213.5 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.328891
  validation RMSE = 0.653611
  runtime         = 224.9 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.318915
  validation RMSE = 0.646633
  runtime         = 214.6 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.318716
  validation RMSE = 0.646182
  runtime         = 211.6 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.3320

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,30,F3,300,0.5,3,30.0,True,0.318716,0.646182,211.599190
1,30,F3,300,0.5,3,NaN,True,0.318915,0.646633,214.600353
2,30,F3,300,1.0,3,30.0,True,0.323577,0.653140,432.588893
3,30,F3,300,1.0,3,NaN,True,0.324589,0.654448,448.443524
4,30,F3,300,0.5,1,NaN,True,0.327516,0.653257,213.474467
5,30,F3,300,0.5,1,30.0,True,0.328891,0.653611,224.918752
6,30,F3,300,1.0,1,NaN,True,0.332012,0.660037,476.457987
7,30,F3,300,1.0,1,30.0,True,0.332301,0.660559,465.057608



Selected 30-minute F3 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.31871570872060384
  validation RMSE: 0.6461817707581039

F2 -> F3 validation comparison:
  F2 MAE: 0.315629
  F3 MAE: 0.318716
  Relative MAE improvement: -0.9780%

Total 30-minute F3 grid runtime:
  2687.6 seconds
  44.79 minutes

Random Forest 30-minute F3 experiment: PASSED


In [113]:
# ============================================================
# Step 103: Random Forest validation
#           30-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + one-hot encoded Vedic/Panchang categorical features
#
# This is the central F3 -> F4 comparison for the
# 30-minute horizon.
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_30_f4_df, random_forest_30_f4_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=30,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f4_30 = (
    random_forest_30_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 30-minute F4 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_30_f4_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
# ------------------------------------------------------------

selected_max_depth_f4_30 = (
    None
    if pd.isna(
        best_random_forest_f4_30[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f4_30[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 30-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f4_30[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f4_30[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f4_30[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f4_30,
)

print(
    "  validation MAE:",
    best_random_forest_f4_30[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f4_30[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. Central F3 -> F4 validation comparison
# ------------------------------------------------------------

f3_30_mae_rf = float(
    best_random_forest_f3_30[
        "validation_mae"
    ]
)

f4_30_mae_rf = float(
    best_random_forest_f4_30[
        "validation_mae"
    ]
)


f3_to_f4_30_absolute_rf = (
    f3_30_mae_rf
    - f4_30_mae_rf
)


f3_to_f4_30_pct_rf = (
    f3_to_f4_30_absolute_rf
    / f3_30_mae_rf
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_30_mae_rf:.6f}"
)

print(
    f"  F4 MAE: {f4_30_mae_rf:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_30_absolute_rf:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_30_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 30-minute F4 grid runtime:"
)

print(
    f"  {random_forest_30_f4_seconds:.1f} seconds"
)

print(
    f"  {random_forest_30_f4_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 30-minute F4 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 30 minutes
  Feature family: F4
  Train: (87258, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.327494
  validation RMSE = 0.653361
  runtime         = 226.9 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.328906
  validation RMSE = 0.654332
  runtime         = 236.1 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.319752
  validation RMSE = 0.647262
  runtime         = 241.0 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.318482
  validation RMSE = 0.645627
  runtime         = 234.7 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.3328

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,30,F4,300,0.5,3,30.0,True,0.318482,0.645627,234.746906
1,30,F4,300,0.5,3,NaN,True,0.319752,0.647262,241.029061
2,30,F4,300,1.0,3,30.0,True,0.322312,0.652209,634.916052
3,30,F4,300,1.0,3,NaN,True,0.322363,0.652231,487.395737
4,30,F4,300,0.5,1,NaN,True,0.327494,0.653361,226.928856
5,30,F4,300,0.5,1,30.0,True,0.328906,0.654332,236.088359
6,30,F4,300,1.0,1,30.0,True,0.332420,0.660850,515.096966
7,30,F4,300,1.0,1,NaN,True,0.332881,0.661842,562.029294



Selected 30-minute F4 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.31848188574215014
  validation RMSE: 0.6456266055290256

F3 -> F4 validation comparison:
  F3 MAE: 0.318716
  F4 MAE: 0.318482
  Absolute MAE reduction: +0.000234
  Relative MAE improvement: +0.0734%

Total 30-minute F4 grid runtime:
  3138.8 seconds
  52.31 minutes

Random Forest 30-minute F4 experiment: PASSED


In [114]:
# ============================================================
# Step 104: Summarize the complete 30-minute Random Forest
#           F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect selected configurations
# ------------------------------------------------------------

best_rows_rf_30 = {
    "F0": best_random_forest_f0_30,
    "F1": best_random_forest_f1_30,
    "F2": best_random_forest_f2_30,
    "F3": best_random_forest_f3_30,
    "F4": best_random_forest_f4_30,
}


summary_rows_rf_30 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_rf_30[
        feature_family
    ]


    selected_max_depth = (
        None
        if pd.isna(
            row[
                "max_depth"
            ]
        )
        else int(
            row[
                "max_depth"
            ]
        )
    )


    summary_rows_rf_30.append(
        {
            "horizon_minutes":
                30,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][
                    feature_family
                ],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "max_features":
                float(
                    row[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    row[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                selected_max_depth,

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


random_forest_30_summary = pd.DataFrame(
    summary_rows_rf_30
)


# ------------------------------------------------------------
# 2. Add incremental MAE improvement
# ------------------------------------------------------------

random_forest_30_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(
        random_forest_30_summary
    ),
):

    previous_mae = float(
        random_forest_30_summary.loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = float(
        random_forest_30_summary.loc[
            i,
            "validation_mae",
        ]
    )


    random_forest_30_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (
            previous_mae
            - current_mae
        )
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_rf_30 = float(
    random_forest_30_summary.loc[
        random_forest_30_summary[
            "feature_family"
        ].eq(
            "F3"
        ),
        "validation_mae",
    ]
    .iloc[0]
)


f4_summary_mae_rf_30 = float(
    random_forest_30_summary.loc[
        random_forest_30_summary[
            "feature_family"
        ].eq(
            "F4"
        ),
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_rf_30 = (
    (
        f3_summary_mae_rf_30
        - f4_summary_mae_rf_30
    )
    / f3_summary_mae_rf_30
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_rf_30,
    +0.0734,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete summary
# ------------------------------------------------------------

print(
    "Random Forest — 30-minute validation summary\n"
)


display(
    random_forest_30_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)


for i in range(
    1,
    len(
        random_forest_30_summary
    ),
):

    previous_family = (
        random_forest_30_summary.loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        random_forest_30_summary.loc[
            i,
            "feature_family",
        ]
    )

    change = float(
        random_forest_30_summary.loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )


    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_rf_30 = (
    random_forest_30_summary
    .sort_values(
        [
            "validation_mae",
            "validation_rmse",
        ]
    )
    .iloc[0]
)


print(
    "\nLowest 30-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_rf_30[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_rf_30[
        "validation_mae"
    ],
)


# ------------------------------------------------------------
# 7. Central F3 -> F4 comparison
# ------------------------------------------------------------

print(
    "\nCentral Random Forest comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_rf_30:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_rf_30:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_rf_30:+.4f}%"
)


print(
    "\nRandom Forest 30-minute summary: PASSED"
)

Random Forest — 30-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,30,F0,118,300,0.5,3,30.0,0.342799,0.677943,NaN
1,30,F1,133,300,0.5,3,NaN,0.315656,0.644300,7.918136
2,30,F2,138,300,0.5,3,NaN,0.315629,0.643351,0.008563
3,30,F3,147,300,0.5,3,30.0,0.318716,0.646182,-0.977977
4,30,F4,244,300,0.5,3,30.0,0.318482,0.645627,0.073364


Incremental validation MAE changes:
  F0 -> F1: +7.9181%
  F1 -> F2: +0.0086%
  F2 -> F3: -0.9780%
  F3 -> F4: +0.0734%

Lowest 30-minute validation MAE:
  Feature family: F2
  MAE: 0.3156289306719466

Central Random Forest comparison:
  F3 MAE: 0.318716
  F4 MAE: 0.318482
  F3 -> F4 change: +0.0734%

Random Forest 30-minute summary: PASSED


In [115]:
# ============================================================
# Step 105: Random Forest validation
#           15-minute horizon + F0
# ============================================================
#
# F0 is the historical/autoregressive baseline:
#
#   - recent observed measurements through issue time t;
#   - lagged PV / irradiance / meteorological variables;
#   - 24-hour PV lag.
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_15_f0_df, random_forest_15_f0_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=15,
        feature_family="F0",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f0_15 = (
    random_forest_15_f0_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 15-minute F0 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_15_f0_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
#
# pandas represents None as NaN when the same column also
# contains numeric values such as 30.
# ------------------------------------------------------------

selected_max_depth_f0_15 = (
    None
    if pd.isna(
        best_random_forest_f0_15[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f0_15[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F0 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f0_15[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f0_15[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f0_15[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f0_15,
)

print(
    "  validation MAE:",
    best_random_forest_f0_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f0_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F0 grid runtime:"
)

print(
    f"  {random_forest_15_f0_seconds:.1f} seconds"
)

print(
    f"  {random_forest_15_f0_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 15-minute F0 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 15 minutes
  Feature family: F0
  Train: (87261, 118)
  Validation: (17568, 118)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.293030
  validation RMSE = 0.611796
  runtime         = 177.7 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.291600
  validation RMSE = 0.609721
  runtime         = 187.7 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.279704
  validation RMSE = 0.597196
  runtime         = 174.5 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.281389
  validation RMSE = 0.599721
  runtime         = 178.3 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.2926

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,15,F0,300,0.5,3,NaN,True,0.279704,0.597196,174.476937
1,15,F0,300,0.5,3,30.0,True,0.281389,0.599721,178.329968
2,15,F0,300,1.0,3,NaN,True,0.282544,0.603168,357.917439
3,15,F0,300,1.0,3,30.0,True,0.282584,0.603350,360.582434
4,15,F0,300,0.5,1,30.0,True,0.291600,0.609721,187.716864
5,15,F0,300,1.0,1,NaN,True,0.292600,0.613852,392.283995
6,15,F0,300,1.0,1,30.0,True,0.292726,0.613929,395.803892
7,15,F0,300,0.5,1,NaN,True,0.293030,0.611796,177.746512



Selected 15-minute F0 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.27970374728073316
  validation RMSE: 0.5971962864693757

Total 15-minute F0 grid runtime:
  2225.3 seconds
  37.09 minutes

Random Forest 15-minute F0 experiment: PASSED


In [116]:
# ============================================================
# Step 106: Random Forest validation
#           15-minute horizon + F1
# ============================================================
#
# F1 = F0
#      + conventional Gregorian / calendar-time features
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_15_f1_df, random_forest_15_f1_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=15,
        feature_family="F1",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f1_15 = (
    random_forest_15_f1_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 15-minute F1 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_15_f1_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
# ------------------------------------------------------------

selected_max_depth_f1_15 = (
    None
    if pd.isna(
        best_random_forest_f1_15[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f1_15[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F1 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f1_15[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f1_15[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f1_15[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f1_15,
)

print(
    "  validation MAE:",
    best_random_forest_f1_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f1_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. F0 -> F1 incremental validation comparison
# ------------------------------------------------------------

f0_15_mae_rf = float(
    best_random_forest_f0_15[
        "validation_mae"
    ]
)

f1_15_mae_rf = float(
    best_random_forest_f1_15[
        "validation_mae"
    ]
)


f0_to_f1_15_pct_rf = (
    (
        f0_15_mae_rf
        - f1_15_mae_rf
    )
    / f0_15_mae_rf
    * 100.0
)


print(
    "\nF0 -> F1 validation comparison:"
)

print(
    f"  F0 MAE: {f0_15_mae_rf:.6f}"
)

print(
    f"  F1 MAE: {f1_15_mae_rf:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f0_to_f1_15_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F1 grid runtime:"
)

print(
    f"  {random_forest_15_f1_seconds:.1f} seconds"
)

print(
    f"  {random_forest_15_f1_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 15-minute F1 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 15 minutes
  Feature family: F1
  Train: (87261, 133)
  Validation: (17568, 133)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.279685
  validation RMSE = 0.592644
  runtime         = 173.4 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.279410
  validation RMSE = 0.592033
  runtime         = 173.6 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.267656
  validation RMSE = 0.582131
  runtime         = 163.5 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.268058
  validation RMSE = 0.581523
  runtime         = 172.8 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.2816

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,15,F1,300,0.5,3,NaN,True,0.267656,0.582131,163.475631
1,15,F1,300,0.5,3,30.0,True,0.268058,0.581523,172.808501
2,15,F1,300,1.0,3,NaN,True,0.268987,0.586544,331.779184
3,15,F1,300,1.0,3,30.0,True,0.269096,0.587031,357.596694
4,15,F1,300,0.5,1,30.0,True,0.279410,0.592033,173.615028
5,15,F1,300,0.5,1,NaN,True,0.279685,0.592644,173.437255
6,15,F1,300,1.0,1,30.0,True,0.281271,0.598449,394.505988
7,15,F1,300,1.0,1,NaN,True,0.281662,0.598845,439.136665



Selected 15-minute F1 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.26765629302013666
  validation RMSE: 0.5821309137562362

F0 -> F1 validation comparison:
  F0 MAE: 0.279704
  F1 MAE: 0.267656
  Relative MAE improvement: +4.3072%

Total 15-minute F1 grid runtime:
  2206.8 seconds
  36.78 minutes

Random Forest 15-minute F1 experiment: PASSED


In [117]:
# ============================================================
# Step 107: Random Forest validation
#           15-minute horizon + F2
# ============================================================
#
# F2 = F1
#      + deterministic solar-geometry features
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_15_f2_df, random_forest_15_f2_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=15,
        feature_family="F2",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f2_15 = (
    random_forest_15_f2_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 15-minute F2 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_15_f2_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
# ------------------------------------------------------------

selected_max_depth_f2_15 = (
    None
    if pd.isna(
        best_random_forest_f2_15[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f2_15[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F2 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f2_15[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f2_15[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f2_15[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f2_15,
)

print(
    "  validation MAE:",
    best_random_forest_f2_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f2_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. F1 -> F2 incremental validation comparison
# ------------------------------------------------------------

f1_15_mae_rf = float(
    best_random_forest_f1_15[
        "validation_mae"
    ]
)

f2_15_mae_rf = float(
    best_random_forest_f2_15[
        "validation_mae"
    ]
)


f1_to_f2_15_pct_rf = (
    (
        f1_15_mae_rf
        - f2_15_mae_rf
    )
    / f1_15_mae_rf
    * 100.0
)


print(
    "\nF1 -> F2 validation comparison:"
)

print(
    f"  F1 MAE: {f1_15_mae_rf:.6f}"
)

print(
    f"  F2 MAE: {f2_15_mae_rf:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f1_to_f2_15_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F2 grid runtime:"
)

print(
    f"  {random_forest_15_f2_seconds:.1f} seconds"
)

print(
    f"  {random_forest_15_f2_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 15-minute F2 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 15 minutes
  Feature family: F2
  Train: (87261, 138)
  Validation: (17568, 138)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.279041
  validation RMSE = 0.591406
  runtime         = 191.3 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.280490
  validation RMSE = 0.593088
  runtime         = 201.9 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.267526
  validation RMSE = 0.581843
  runtime         = 200.7 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.267966
  validation RMSE = 0.581610
  runtime         = 198.1 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.2809

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,15,F2,300,0.5,3,NaN,True,0.267526,0.581843,200.720301
1,15,F2,300,0.5,3,30.0,True,0.267966,0.581610,198.099957
2,15,F2,300,1.0,3,30.0,True,0.268642,0.585719,482.289727
3,15,F2,300,1.0,3,NaN,True,0.269028,0.586145,414.712909
4,15,F2,300,0.5,1,NaN,True,0.279041,0.591406,191.333929
5,15,F2,300,1.0,1,30.0,True,0.279838,0.596470,416.851274
6,15,F2,300,0.5,1,30.0,True,0.280490,0.593088,201.934780
7,15,F2,300,1.0,1,NaN,True,0.280914,0.596854,432.347419



Selected 15-minute F2 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.26752608234864533
  validation RMSE: 0.5818433395838126

F1 -> F2 validation comparison:
  F1 MAE: 0.267656
  F2 MAE: 0.267526
  Relative MAE improvement: +0.0486%

Total 15-minute F2 grid runtime:
  2538.8 seconds
  42.31 minutes

Random Forest 15-minute F2 experiment: PASSED


In [118]:
# ============================================================
# Step 108: Random Forest validation
#           15-minute horizon + F3
# ============================================================
#
# F3 = F2
#      + continuous lunar / Sun-Moon astronomical features
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_15_f3_df, random_forest_15_f3_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=15,
        feature_family="F3",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f3_15 = (
    random_forest_15_f3_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 15-minute F3 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_15_f3_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
#
# pandas may represent Python None as NaN because the same
# column also contains numeric values such as 30.
# ------------------------------------------------------------

selected_max_depth_f3_15 = (
    None
    if pd.isna(
        best_random_forest_f3_15[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f3_15[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F3 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f3_15[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f3_15[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f3_15[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f3_15,
)

print(
    "  validation MAE:",
    best_random_forest_f3_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f3_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. F2 -> F3 incremental validation comparison
# ------------------------------------------------------------

f2_15_mae_rf = float(
    best_random_forest_f2_15[
        "validation_mae"
    ]
)

f3_15_mae_rf = float(
    best_random_forest_f3_15[
        "validation_mae"
    ]
)


f2_to_f3_15_pct_rf = (
    (
        f2_15_mae_rf
        - f3_15_mae_rf
    )
    / f2_15_mae_rf
    * 100.0
)


print(
    "\nF2 -> F3 validation comparison:"
)

print(
    f"  F2 MAE: {f2_15_mae_rf:.6f}"
)

print(
    f"  F3 MAE: {f3_15_mae_rf:.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f2_to_f3_15_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F3 grid runtime:"
)

print(
    f"  {random_forest_15_f3_seconds:.1f} seconds"
)

print(
    f"  {random_forest_15_f3_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 15-minute F3 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 15 minutes
  Feature family: F3
  Train: (87261, 147)
  Validation: (17568, 147)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.280845
  validation RMSE = 0.593790
  runtime         = 261.4 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.279763
  validation RMSE = 0.592215
  runtime         = 252.1 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.269019
  validation RMSE = 0.582445
  runtime         = 238.6 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.270307
  validation RMSE = 0.584478
  runtime         = 207.6 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.2829

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,15,F3,300,0.5,3,NaN,True,0.269019,0.582445,238.550160
1,15,F3,300,0.5,3,30.0,True,0.270307,0.584478,207.578899
2,15,F3,300,1.0,3,30.0,True,0.271888,0.587365,464.997350
3,15,F3,300,1.0,3,NaN,True,0.271922,0.587440,470.685809
4,15,F3,300,0.5,1,30.0,True,0.279763,0.592215,252.149712
5,15,F3,300,0.5,1,NaN,True,0.280845,0.593790,261.414004
6,15,F3,300,1.0,1,30.0,True,0.282364,0.597867,480.761820
7,15,F3,300,1.0,1,NaN,True,0.282915,0.598746,474.539838



Selected 15-minute F3 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: None
  validation MAE: 0.2690186583004718
  validation RMSE: 0.5824454088999481

F2 -> F3 validation comparison:
  F2 MAE: 0.267526
  F3 MAE: 0.269019
  Relative MAE improvement: -0.5579%

Total 15-minute F3 grid runtime:
  2851.2 seconds
  47.52 minutes

Random Forest 15-minute F3 experiment: PASSED


In [119]:
# ============================================================
# Step 109: Random Forest validation
#           15-minute horizon + F4
# ============================================================
#
# F4 = F3
#      + one-hot encoded Vedic/Panchang categorical features
#
# This is the central F3 -> F4 comparison for the
# 15-minute horizon.
#
# The exact same frozen 8-configuration Random Forest grid
# is reused without modification.
#
# TEST remains untouched.
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen Random Forest validation grid
# ------------------------------------------------------------

random_forest_15_f4_df, random_forest_15_f4_seconds = (
    run_random_forest_validation_grid(
        horizon_minutes=15,
        feature_family="F4",
    )
)


# ------------------------------------------------------------
# 2. Select minimum-validation-MAE configuration
# ------------------------------------------------------------

best_random_forest_f4_15 = (
    random_forest_15_f4_df.iloc[0]
)


# ------------------------------------------------------------
# 3. Display ordered validation results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "Random Forest — 15-minute F4 validation results"
)

print(
    "============================================\n"
)


display(
    random_forest_15_f4_df
)


# ------------------------------------------------------------
# 4. Recover max_depth cleanly
# ------------------------------------------------------------

selected_max_depth_f4_15 = (
    None
    if pd.isna(
        best_random_forest_f4_15[
            "max_depth"
        ]
    )
    else int(
        best_random_forest_f4_15[
            "max_depth"
        ]
    )
)


# ------------------------------------------------------------
# 5. Display selected configuration
# ------------------------------------------------------------

print(
    "\nSelected 15-minute F4 configuration:"
)

print(
    "  n_estimators:",
    int(
        best_random_forest_f4_15[
            "n_estimators"
        ]
    ),
)

print(
    "  max_features:",
    float(
        best_random_forest_f4_15[
            "max_features"
        ]
    ),
)

print(
    "  min_samples_leaf:",
    int(
        best_random_forest_f4_15[
            "min_samples_leaf"
        ]
    ),
)

print(
    "  max_depth:",
    selected_max_depth_f4_15,
)

print(
    "  validation MAE:",
    best_random_forest_f4_15[
        "validation_mae"
    ],
)

print(
    "  validation RMSE:",
    best_random_forest_f4_15[
        "validation_rmse"
    ],
)


# ------------------------------------------------------------
# 6. Central F3 -> F4 validation comparison
# ------------------------------------------------------------

f3_15_mae_rf = float(
    best_random_forest_f3_15[
        "validation_mae"
    ]
)

f4_15_mae_rf = float(
    best_random_forest_f4_15[
        "validation_mae"
    ]
)


f3_to_f4_15_absolute_rf = (
    f3_15_mae_rf
    - f4_15_mae_rf
)


f3_to_f4_15_pct_rf = (
    f3_to_f4_15_absolute_rf
    / f3_15_mae_rf
    * 100.0
)


print(
    "\nF3 -> F4 validation comparison:"
)

print(
    f"  F3 MAE: {f3_15_mae_rf:.6f}"
)

print(
    f"  F4 MAE: {f4_15_mae_rf:.6f}"
)

print(
    "  Absolute MAE reduction:"
    f" {f3_to_f4_15_absolute_rf:+.6f}"
)

print(
    "  Relative MAE improvement:"
    f" {f3_to_f4_15_pct_rf:+.4f}%"
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

print(
    "\nTotal 15-minute F4 grid runtime:"
)

print(
    f"  {random_forest_15_f4_seconds:.1f} seconds"
)

print(
    f"  {random_forest_15_f4_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 15-minute F4 experiment: PASSED"
)


Random Forest validation dataset:
  Horizon: 15 minutes
  Feature family: F4
  Train: (87261, 244)
  Validation: (17568, 244)
  Frozen configurations: 8

Configuration 1/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.280993
  validation RMSE = 0.593269
  runtime         = 247.4 s

Configuration 2/8
  max_features      = 0.5
  min_samples_leaf  = 1
  max_depth         = 30
  validation MAE  = 0.279685
  validation RMSE = 0.591986
  runtime         = 279.0 s

Configuration 3/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = None
  validation MAE  = 0.269569
  validation RMSE = 0.583144
  runtime         = 258.3 s

Configuration 4/8
  max_features      = 0.5
  min_samples_leaf  = 3
  max_depth         = 30
  validation MAE  = 0.268776
  validation RMSE = 0.581911
  runtime         = 249.3 s

Configuration 5/8
  max_features      = 1.0
  min_samples_leaf  = 1
  max_depth         = None
  validation MAE  = 0.2831

,horizon_minutes,feature_family,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse,fit_predict_seconds
0,15,F4,300,0.5,3,30.0,True,0.268776,0.581911,249.267783
1,15,F4,300,0.5,3,NaN,True,0.269569,0.583144,258.333955
2,15,F4,300,1.0,3,NaN,True,0.272392,0.588270,505.721114
3,15,F4,300,1.0,3,30.0,True,0.272969,0.588751,474.967343
4,15,F4,300,0.5,1,30.0,True,0.279685,0.591986,279.010730
5,15,F4,300,0.5,1,NaN,True,0.280993,0.593269,247.393658
6,15,F4,300,1.0,1,30.0,True,0.282613,0.597904,540.075929
7,15,F4,300,1.0,1,NaN,True,0.283183,0.598735,615.742624



Selected 15-minute F4 configuration:
  n_estimators: 300
  max_features: 0.5
  min_samples_leaf: 3
  max_depth: 30
  validation MAE: 0.2687755557033245
  validation RMSE: 0.5819112898034691

F3 -> F4 validation comparison:
  F3 MAE: 0.269019
  F4 MAE: 0.268776
  Absolute MAE reduction: +0.000243
  Relative MAE improvement: +0.0904%

Total 15-minute F4 grid runtime:
  3171.0 seconds
  52.85 minutes

Random Forest 15-minute F4 experiment: PASSED


In [120]:
# ============================================================
# Step 110: Summarize the complete 15-minute Random Forest
#           F0 -> F4 validation experiment
# ============================================================


# ------------------------------------------------------------
# 1. Collect selected configurations
# ------------------------------------------------------------

best_rows_rf_15 = {
    "F0": best_random_forest_f0_15,
    "F1": best_random_forest_f1_15,
    "F2": best_random_forest_f2_15,
    "F3": best_random_forest_f3_15,
    "F4": best_random_forest_f4_15,
}


summary_rows_rf_15 = []


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    row = best_rows_rf_15[
        feature_family
    ]


    # --------------------------------------------------------
    # pandas may display None as NaN because max_depth also
    # contains numeric values such as 30.
    # --------------------------------------------------------

    selected_max_depth = (
        None
        if pd.isna(
            row[
                "max_depth"
            ]
        )
        else int(
            row[
                "max_depth"
            ]
        )
    )


    summary_rows_rf_15.append(
        {
            "horizon_minutes":
                15,

            "feature_family":
                feature_family,

            "n_features":
                primary_protocol[
                    "feature_counts"
                ][
                    feature_family
                ],

            "n_estimators":
                int(
                    row[
                        "n_estimators"
                    ]
                ),

            "max_features":
                float(
                    row[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    row[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                selected_max_depth,

            "validation_mae":
                float(
                    row[
                        "validation_mae"
                    ]
                ),

            "validation_rmse":
                float(
                    row[
                        "validation_rmse"
                    ]
                ),
        }
    )


random_forest_15_summary = pd.DataFrame(
    summary_rows_rf_15
)


# ------------------------------------------------------------
# 2. Add incremental validation-MAE changes
#
# Positive:
#     newly added feature family reduced MAE.
#
# Negative:
#     newly added feature family increased MAE.
# ------------------------------------------------------------

random_forest_15_summary[
    "incremental_mae_improvement_pct"
] = np.nan


for i in range(
    1,
    len(
        random_forest_15_summary
    ),
):

    previous_mae = float(
        random_forest_15_summary.loc[
            i - 1,
            "validation_mae",
        ]
    )

    current_mae = float(
        random_forest_15_summary.loc[
            i,
            "validation_mae",
        ]
    )


    random_forest_15_summary.loc[
        i,
        "incremental_mae_improvement_pct",
    ] = (
        (
            previous_mae
            - current_mae
        )
        / previous_mae
        * 100.0
    )


# ------------------------------------------------------------
# 3. Central F3 -> F4 comparison
# ------------------------------------------------------------

f3_summary_mae_rf_15 = float(
    random_forest_15_summary.loc[
        random_forest_15_summary[
            "feature_family"
        ].eq(
            "F3"
        ),
        "validation_mae",
    ]
    .iloc[0]
)


f4_summary_mae_rf_15 = float(
    random_forest_15_summary.loc[
        random_forest_15_summary[
            "feature_family"
        ].eq(
            "F4"
        ),
        "validation_mae",
    ]
    .iloc[0]
)


f3_to_f4_pct_rf_15 = (
    (
        f3_summary_mae_rf_15
        - f4_summary_mae_rf_15
    )
    / f3_summary_mae_rf_15
    * 100.0
)


assert np.isclose(
    f3_to_f4_pct_rf_15,
    +0.0904,
    atol=0.001,
)


# ------------------------------------------------------------
# 4. Display complete summary
# ------------------------------------------------------------

print(
    "Random Forest — 15-minute validation summary\n"
)


display(
    random_forest_15_summary
)


# ------------------------------------------------------------
# 5. Print incremental effects
# ------------------------------------------------------------

print(
    "Incremental validation MAE changes:"
)


for i in range(
    1,
    len(
        random_forest_15_summary
    ),
):

    previous_family = (
        random_forest_15_summary.loc[
            i - 1,
            "feature_family",
        ]
    )

    current_family = (
        random_forest_15_summary.loc[
            i,
            "feature_family",
        ]
    )

    change = float(
        random_forest_15_summary.loc[
            i,
            "incremental_mae_improvement_pct",
        ]
    )


    print(
        f"  {previous_family} -> "
        f"{current_family}: "
        f"{change:+.4f}%"
    )


# ------------------------------------------------------------
# 6. Identify lowest validation MAE
# ------------------------------------------------------------

best_overall_rf_15 = (
    random_forest_15_summary
    .sort_values(
        [
            "validation_mae",
            "validation_rmse",
        ]
    )
    .iloc[0]
)


print(
    "\nLowest 15-minute validation MAE:"
)

print(
    "  Feature family:",
    best_overall_rf_15[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_overall_rf_15[
        "validation_mae"
    ],
)


# ------------------------------------------------------------
# 7. Central F3 -> F4 comparison
# ------------------------------------------------------------

print(
    "\nCentral Random Forest comparison:"
)

print(
    f"  F3 MAE: "
    f"{f3_summary_mae_rf_15:.6f}"
)

print(
    f"  F4 MAE: "
    f"{f4_summary_mae_rf_15:.6f}"
)

print(
    f"  F3 -> F4 change: "
    f"{f3_to_f4_pct_rf_15:+.4f}%"
)


print(
    "\nRandom Forest 15-minute summary: PASSED"
)

Random Forest — 15-minute validation summary



,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,15,F0,118,300,0.5,3,NaN,0.279704,0.597196,NaN
1,15,F1,133,300,0.5,3,NaN,0.267656,0.582131,4.307219
2,15,F2,138,300,0.5,3,NaN,0.267526,0.581843,0.048648
3,15,F3,147,300,0.5,3,NaN,0.269019,0.582445,-0.557918
4,15,F4,244,300,0.5,3,30.0,0.268776,0.581911,0.090366


Incremental validation MAE changes:
  F0 -> F1: +4.3072%
  F1 -> F2: +0.0486%
  F2 -> F3: -0.5579%
  F3 -> F4: +0.0904%

Lowest 15-minute validation MAE:
  Feature family: F2
  MAE: 0.26752608234864533

Central Random Forest comparison:
  F3 MAE: 0.269019
  F4 MAE: 0.268776
  F3 -> F4 change: +0.0904%

Random Forest 15-minute summary: PASSED


In [121]:
# ============================================================
# Step 111: Combine the complete Random Forest validation
#           study and freeze all 15 selected configurations
# ============================================================
#
# At this checkpoint:
#
#   - all 15 Random Forest validation experiments are complete;
#   - model selection used chronological validation MAE only;
#   - validation RMSE was the secondary / tie-break metric;
#   - no Random Forest test metric has been examined.
#
# We now freeze one selected configuration for every:
#
#       horizon × feature family
#
# combination.
#
# Only AFTER this checkpoint may the held-out TEST population
# be opened.
# ============================================================


# ------------------------------------------------------------
# 1. Combine 15-, 30-, and 60-minute validation summaries
# ------------------------------------------------------------

random_forest_validation_summary = (
    pd.concat(
        [
            random_forest_15_summary,
            random_forest_30_summary,
            random_forest_60_summary,
        ],
        axis=0,
        ignore_index=True,
    )
)


# ------------------------------------------------------------
# Natural F0 -> F4 ordering
# ------------------------------------------------------------

rf_feature_family_order = {
    "F0": 0,
    "F1": 1,
    "F2": 2,
    "F3": 3,
    "F4": 4,
}


random_forest_validation_summary[
    "_family_order"
] = (
    random_forest_validation_summary[
        "feature_family"
    ]
    .map(
        rf_feature_family_order
    )
)


random_forest_validation_summary = (
    random_forest_validation_summary
    .sort_values(
        [
            "horizon_minutes",
            "_family_order",
        ]
    )
    .drop(
        columns="_family_order"
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        random_forest_validation_summary
    )
    == 15
)


print(
    "Combined Random Forest validation summary:\n"
)


display(
    random_forest_validation_summary
)


# ------------------------------------------------------------
# 2. Explicit central F3 -> F4 validation comparison
# ------------------------------------------------------------

random_forest_f3_f4_validation_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_summary = (
        random_forest_validation_summary[
            random_forest_validation_summary[
                "horizon_minutes"
            ].eq(
                horizon_minutes
            )
        ]
    )


    f3_mae = float(
        horizon_summary.loc[
            horizon_summary[
                "feature_family"
            ].eq(
                "F3"
            ),
            "validation_mae",
        ]
        .iloc[0]
    )


    f4_mae = float(
        horizon_summary.loc[
            horizon_summary[
                "feature_family"
            ].eq(
                "F4"
            ),
            "validation_mae",
        ]
        .iloc[0]
    )


    absolute_reduction = (
        f3_mae
        - f4_mae
    )


    relative_improvement_pct = (
        absolute_reduction
        / f3_mae
        * 100.0
    )


    random_forest_f3_f4_validation_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "f3_validation_mae":
                f3_mae,

            "f4_validation_mae":
                f4_mae,

            "absolute_mae_reduction":
                absolute_reduction,

            "relative_mae_improvement_pct":
                relative_improvement_pct,
        }
    )


random_forest_f3_f4_validation = (
    pd.DataFrame(
        random_forest_f3_f4_validation_rows
    )
)


print(
    "\nRandom Forest F3 -> F4 validation comparison:\n"
)


display(
    random_forest_f3_f4_validation
)


# ------------------------------------------------------------
# 3. Verify the already-observed F3 -> F4 effects
# ------------------------------------------------------------

expected_rf_f3_f4_effects = {
    15: +0.090366,
    30: +0.073364,
    60: -0.297745,
}


for (
    horizon_minutes,
    expected_value,
) in expected_rf_f3_f4_effects.items():

    observed_value = float(
        random_forest_f3_f4_validation.loc[
            random_forest_f3_f4_validation[
                "horizon_minutes"
            ].eq(
                horizon_minutes
            ),
            "relative_mae_improvement_pct",
        ]
        .iloc[0]
    )


    assert np.isclose(
        observed_value,
        expected_value,
        atol=1e-5,
    )


# ------------------------------------------------------------
# 4. Gather selected rows from every horizon
# ------------------------------------------------------------

best_rows_rf_by_horizon = {
    15: best_rows_rf_15,
    30: best_rows_rf_30,
    60: best_rows_rf_60,
}


# ------------------------------------------------------------
# 5. Freeze executable selected configurations
#
# IMPORTANT:
#   pandas represents max_depth=None as NaN inside the
#   validation DataFrames.
#
#   Here we explicitly restore NaN -> Python None before
#   constructing the executable configuration.
# ------------------------------------------------------------

random_forest_selected_configs = {}


for horizon_minutes in [
    15,
    30,
    60,
]:

    random_forest_selected_configs[
        horizon_minutes
    ] = {}


    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        selected_row = (
            best_rows_rf_by_horizon[
                horizon_minutes
            ][
                feature_family
            ]
        )


        selected_max_depth = (
            None
            if pd.isna(
                selected_row[
                    "max_depth"
                ]
            )
            else int(
                selected_row[
                    "max_depth"
                ]
            )
        )


        random_forest_selected_configs[
            horizon_minutes
        ][
            feature_family
        ] = {
            **RANDOM_FOREST_FIXED_PARAMS,

            "max_features":
                float(
                    selected_row[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    selected_row[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                selected_max_depth,
        }


# ------------------------------------------------------------
# 6. Create human-readable frozen-selection table
# ------------------------------------------------------------

random_forest_frozen_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        config = (
            random_forest_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )


        validation_row = (
            random_forest_validation_summary[
                (
                    random_forest_validation_summary[
                        "horizon_minutes"
                    ].eq(
                        horizon_minutes
                    )
                )
                &
                (
                    random_forest_validation_summary[
                        "feature_family"
                    ].eq(
                        feature_family
                    )
                )
            ]
            .iloc[0]
        )


        random_forest_frozen_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_features":
                    int(
                        primary_protocol[
                            "feature_counts"
                        ][
                            feature_family
                        ]
                    ),

                "n_estimators":
                    int(
                        config[
                            "n_estimators"
                        ]
                    ),

                "max_features":
                    float(
                        config[
                            "max_features"
                        ]
                    ),

                "min_samples_leaf":
                    int(
                        config[
                            "min_samples_leaf"
                        ]
                    ),

                "max_depth":
                    config[
                        "max_depth"
                    ],

                "bootstrap":
                    bool(
                        config[
                            "bootstrap"
                        ]
                    ),

                "validation_mae":
                    float(
                        validation_row[
                            "validation_mae"
                        ]
                    ),

                "validation_rmse":
                    float(
                        validation_row[
                            "validation_rmse"
                        ]
                    ),
            }
        )


random_forest_frozen_selection = (
    pd.DataFrame(
        random_forest_frozen_rows
    )
)


# ------------------------------------------------------------
# Preserve None rather than allowing pandas to silently
# convert it back to floating NaN for presentation.
# ------------------------------------------------------------

random_forest_frozen_selection[
    "max_depth"
] = pd.Series(
    [
        row[
            "max_depth"
        ]
        for row in random_forest_frozen_rows
    ],
    dtype="object",
)


# ------------------------------------------------------------
# 7. Strong structural checks
# ------------------------------------------------------------

assert (
    random_forest_frozen_selection.shape[0]
    == 15
)


assert set(
    random_forest_frozen_selection[
        "horizon_minutes"
    ]
) == {
    15,
    30,
    60,
}


assert set(
    random_forest_frozen_selection[
        "feature_family"
    ]
) == {
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
}


for horizon_minutes in [
    15,
    30,
    60,
]:

    assert (
        len(
            random_forest_selected_configs[
                horizon_minutes
            ]
        )
        == 5
    )


# ------------------------------------------------------------
# 8. Verify each selected variable configuration belongs
#    exactly to the originally frozen 8-configuration grid
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        selected_config = (
            random_forest_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )


        assert any(
            (
                selected_config[
                    "max_features"
                ]
                == grid_config[
                    "max_features"
                ]
            )
            and
            (
                selected_config[
                    "min_samples_leaf"
                ]
                == grid_config[
                    "min_samples_leaf"
                ]
            )
            and
            (
                selected_config[
                    "max_depth"
                ]
                == grid_config[
                    "max_depth"
                ]
            )
            for grid_config
            in random_forest_configs
        )


# ------------------------------------------------------------
# 9. Verify fixed parameters were not altered
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        config = (
            random_forest_selected_configs[
                horizon_minutes
            ][
                feature_family
            ]
        )


        for (
            fixed_parameter,
            fixed_value,
        ) in RANDOM_FOREST_FIXED_PARAMS.items():

            assert (
                config[
                    fixed_parameter
                ]
                == fixed_value
            )


# ------------------------------------------------------------
# 10. Display frozen registry
# ------------------------------------------------------------

print(
    "\nFrozen Random Forest configuration registry:\n"
)


display(
    random_forest_frozen_selection
)


# ------------------------------------------------------------
# 11. Lowest validation MAE within each horizon
# ------------------------------------------------------------

print(
    "\nLowest validation MAE by horizon:"
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    best_row = (
        random_forest_validation_summary[
            random_forest_validation_summary[
                "horizon_minutes"
            ].eq(
                horizon_minutes
            )
        ]
        .sort_values(
            [
                "validation_mae",
                "validation_rmse",
            ]
        )
        .iloc[0]
    )


    print(
        f"  {horizon_minutes:2d} min: "
        f"{best_row['feature_family']} "
        f"(MAE = "
        f"{best_row['validation_mae']:.6f})"
    )


# ------------------------------------------------------------
# 12. Final F3 -> F4 pre-test checkpoint
# ------------------------------------------------------------

print(
    "\nF3 -> F4 validation effects:"
)


for _, row in (
    random_forest_f3_f4_validation
    .iterrows()
):

    print(
        f"  "
        f"{int(row['horizon_minutes']):2d} min: "
        f"{row['relative_mae_improvement_pct']:+.4f}%"
    )


# ------------------------------------------------------------
# 13. Final checkpoint
# ------------------------------------------------------------

print(
    "\nAll 15 Random Forest configurations are now frozen."
)

print(
    "No Random Forest test-set metric has been used for selection."
)

print(
    "\nRandom Forest validation-selection checkpoint: PASSED"
)

Combined Random Forest validation summary:



,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,validation_mae,validation_rmse,incremental_mae_improvement_pct
0,15,F0,118,300,0.5,3,NaN,0.279704,0.597196,NaN
1,15,F1,133,300,0.5,3,NaN,0.267656,0.582131,4.307219
2,15,F2,138,300,0.5,3,NaN,0.267526,0.581843,0.048648
3,15,F3,147,300,0.5,3,NaN,0.269019,0.582445,-0.557918
4,15,F4,244,300,0.5,3,30.0,0.268776,0.581911,0.090366
5,30,F0,118,300,0.5,3,30.0,0.342799,0.677943,NaN
6,30,F1,133,300,0.5,3,NaN,0.315656,0.644300,7.918136
7,30,F2,138,300,0.5,3,NaN,0.315629,0.643351,0.008563
8,30,F3,147,300,0.5,3,30.0,0.318716,0.646182,-0.977977
9,30,F4,244,300,0.5,3,30.0,0.318482,0.645627,0.073364



Random Forest F3 -> F4 validation comparison:



,horizon_minutes,f3_validation_mae,f4_validation_mae,absolute_mae_reduction,relative_mae_improvement_pct
0,15,0.269019,0.268776,0.000243,0.090366
1,30,0.318716,0.318482,0.000234,0.073364
2,60,0.379673,0.380803,-0.001130,-0.297745



Frozen Random Forest configuration registry:



,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,validation_mae,validation_rmse
0,15,F0,118,300,0.5,3,None,True,0.279704,0.597196
1,15,F1,133,300,0.5,3,None,True,0.267656,0.582131
2,15,F2,138,300,0.5,3,None,True,0.267526,0.581843
3,15,F3,147,300,0.5,3,None,True,0.269019,0.582445
4,15,F4,244,300,0.5,3,30,True,0.268776,0.581911
5,30,F0,118,300,0.5,3,30,True,0.342799,0.677943
6,30,F1,133,300,0.5,3,None,True,0.315656,0.644300
7,30,F2,138,300,0.5,3,None,True,0.315629,0.643351
8,30,F3,147,300,0.5,3,30,True,0.318716,0.646182
9,30,F4,244,300,0.5,3,30,True,0.318482,0.645627



Lowest validation MAE by horizon:
  15 min: F2 (MAE = 0.267526)
  30 min: F2 (MAE = 0.315629)
  60 min: F2 (MAE = 0.376448)

F3 -> F4 validation effects:
  15 min: +0.0904%
  30 min: +0.0734%
  60 min: -0.2977%

All 15 Random Forest configurations are now frozen.
No Random Forest test-set metric has been used for selection.

Random Forest validation-selection checkpoint: PASSED


In [122]:
# ============================================================
# Step 112: Final Random Forest test evaluation
#           15-minute horizon
# ============================================================
#
# IMPORTANT
# ---------
# All Random Forest hyperparameters were frozen using
# TRAIN + chronological VALIDATION before this cell.
#
# We now open the held-out TEST population for the first time.
#
# For F0 -> F4:
#
#   1. refit the frozen model on TRAIN + VALIDATION;
#   2. predict TEST exactly once;
#   3. calculate all-time metrics;
#   4. calculate daylight-only metrics;
#   5. retain timestamp-level predictions for the later
#      paired F3-vs-F4 daily-block bootstrap.
#
# No hyperparameter may be modified based on these results.
# ============================================================


# ------------------------------------------------------------
# 1. Final-evaluation constants
# ------------------------------------------------------------

RF_PV_RATED_KW = float(
    primary_protocol[
        "pv_rated_kw"
    ]
)

assert RF_PV_RATED_KW == 10.0


# ------------------------------------------------------------
# 2. Create final result containers
#
# These will remain in memory and will be extended later by
# the 30-minute and 60-minute final-test cells.
# ------------------------------------------------------------

random_forest_test_result_rows = []

random_forest_test_predictions = {}


# ------------------------------------------------------------
# 3. Horizon for this cell
# ------------------------------------------------------------

horizon_minutes = 15


y = targets[
    horizon_minutes
]

split = split_labels[
    horizon_minutes
]


# ------------------------------------------------------------
# 4. Final development and held-out populations
# ------------------------------------------------------------

train_validation_mask = (
    split.isin(
        [
            "train",
            "validation",
        ]
    )
)

test_mask = (
    split == "test"
)


assert int(
    test_mask.sum()
) == 26_496


# ------------------------------------------------------------
# 5. Daylight definition
#
# Same frozen definition used throughout the project:
#
#       forecast-valid solar elevation > 0 degrees
# ------------------------------------------------------------

daylight_test_mask = (
    solar_valid_features[
        horizon_minutes
    ][
        "solar_elevation_deg"
    ]
    > 0.0
) & test_mask


n_test = int(
    test_mask.sum()
)

n_test_daylight = int(
    daylight_test_mask.sum()
)


assert n_test == 26_496

assert n_test_daylight == 17_851


print(
    "\n"
    "============================================"
)

print(
    "FINAL RANDOM FOREST TEST — 15 MINUTES"
)

print(
    "============================================"
)

print(
    "Test rows:",
    n_test,
)

print(
    "Daylight test rows:",
    n_test_daylight,
)


# ------------------------------------------------------------
# 6. Evaluate frozen F0 -> F4 configurations
# ------------------------------------------------------------

rf_15_test_start_time = (
    time.perf_counter()
)


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    print(
        "\n"
        f"15 min — {feature_family}"
    )


    # --------------------------------------------------------
    # Frozen configuration selected before test evaluation
    # --------------------------------------------------------

    config = (
        random_forest_selected_configs[
            horizon_minutes
        ][
            feature_family
        ]
    )


    print(
        "  Frozen config:"
    )

    print(
        "    n_estimators      =",
        config[
            "n_estimators"
        ],
    )

    print(
        "    max_features      =",
        config[
            "max_features"
        ],
    )

    print(
        "    min_samples_leaf  =",
        config[
            "min_samples_leaf"
        ],
    )

    print(
        "    max_depth         =",
        config[
            "max_depth"
        ],
    )


    # --------------------------------------------------------
    # Feature matrix
    # --------------------------------------------------------

    X = (
        feature_matrices[
            horizon_minutes
        ][
            feature_family
        ]
    )


    # --------------------------------------------------------
    # TRAIN + VALIDATION arrays
    # --------------------------------------------------------

    X_train_validation = (
        X.loc[
            train_validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    y_train_validation = (
        y.loc[
            train_validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # TEST arrays
    # --------------------------------------------------------

    X_test = (
        X.loc[
            test_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    y_test = (
        y.loc[
            test_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # Structural checks
    # --------------------------------------------------------

    expected_feature_count = int(
        primary_protocol[
            "feature_counts"
        ][
            feature_family
        ]
    )


    assert (
        X_train_validation.shape[1]
        == expected_feature_count
    )

    assert (
        X_test.shape
        == (
            26_496,
            expected_feature_count,
        )
    )


    assert np.isfinite(
        X_train_validation
    ).all()

    assert np.isfinite(
        X_test
    ).all()

    assert np.isfinite(
        y_train_validation
    ).all()

    assert np.isfinite(
        y_test
    ).all()


    # --------------------------------------------------------
    # 7. Refit frozen Random Forest on TRAIN + VALIDATION
    # --------------------------------------------------------

    model = RandomForestRegressor(
        **config
    )


    fit_start_time = (
        time.perf_counter()
    )


    model.fit(
        X_train_validation,
        y_train_validation,
    )


    test_predictions = (
        model.predict(
            X_test
        )
    )


    fit_predict_seconds = (
        time.perf_counter()
        - fit_start_time
    )


    # --------------------------------------------------------
    # 8. All-time held-out test metrics
    # --------------------------------------------------------

    test_mae = (
        mean_absolute_error(
            y_test,
            test_predictions,
        )
    )


    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_predictions,
        )
    )


    test_nmae = (
        test_mae
        / RF_PV_RATED_KW
    )


    test_nrmse = (
        test_rmse
        / RF_PV_RATED_KW
    )


    test_r2 = (
        r2_score(
            y_test,
            test_predictions,
        )
    )


    # --------------------------------------------------------
    # 9. Daylight-only test metrics
    # --------------------------------------------------------

    test_issue_index = (
        y.index[
            test_mask
        ]
    )


    daylight_on_test = (
        daylight_test_mask
        .loc[
            test_issue_index
        ]
        .to_numpy(
            dtype=bool
        )
    )


    assert int(
        daylight_on_test.sum()
    ) == 17_851


    y_test_daylight = (
        y_test[
            daylight_on_test
        ]
    )


    predictions_daylight = (
        test_predictions[
            daylight_on_test
        ]
    )


    daylight_mae = (
        mean_absolute_error(
            y_test_daylight,
            predictions_daylight,
        )
    )


    daylight_rmse = np.sqrt(
        mean_squared_error(
            y_test_daylight,
            predictions_daylight,
        )
    )


    daylight_nmae = (
        daylight_mae
        / RF_PV_RATED_KW
    )


    daylight_nrmse = (
        daylight_rmse
        / RF_PV_RATED_KW
    )


    daylight_r2 = (
        r2_score(
            y_test_daylight,
            predictions_daylight,
        )
    )


    # --------------------------------------------------------
    # 10. Store final metric row
    # --------------------------------------------------------

    random_forest_test_result_rows.append(
        {
            "model":
                "RandomForestRegressor",

            "horizon_minutes":
                horizon_minutes,

            "feature_family":
                feature_family,

            "n_features":
                expected_feature_count,

            "n_estimators":
                int(
                    config[
                        "n_estimators"
                    ]
                ),

            "max_features":
                float(
                    config[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    config[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                config[
                    "max_depth"
                ],

            "bootstrap":
                bool(
                    config[
                        "bootstrap"
                    ]
                ),

            "test_rows":
                n_test,

            "test_mae":
                test_mae,

            "test_rmse":
                test_rmse,

            "test_nmae":
                test_nmae,

            "test_nrmse":
                test_nrmse,

            "test_r2":
                test_r2,

            "daylight_test_rows":
                n_test_daylight,

            "daylight_mae":
                daylight_mae,

            "daylight_rmse":
                daylight_rmse,

            "daylight_nmae":
                daylight_nmae,

            "daylight_nrmse":
                daylight_nrmse,

            "daylight_r2":
                daylight_r2,

            "fit_predict_seconds":
                fit_predict_seconds,
        }
    )


    # --------------------------------------------------------
    # 11. Retain timestamp-level predictions
    # --------------------------------------------------------

    prediction_table = pd.DataFrame(
        {
            "valid_time":
                base_modelling_tables[
                    horizon_minutes
                ]
                .loc[
                    test_issue_index,
                    "valid_time",
                ],

            "y_true":
                y_test,

            "y_pred":
                test_predictions,

            "absolute_error":
                np.abs(
                    y_test
                    - test_predictions
                ),

            "daylight":
                daylight_on_test,
        },
        index=test_issue_index,
    )


    prediction_table.index.name = (
        "issue_time"
    )


    random_forest_test_predictions[
        (
            horizon_minutes,
            feature_family,
        )
    ] = prediction_table


    # --------------------------------------------------------
    # 12. Compact live output
    # --------------------------------------------------------

    print(
        f"  all-time MAE:     "
        f"{test_mae:.6f}"
    )

    print(
        f"  all-time RMSE:    "
        f"{test_rmse:.6f}"
    )

    print(
        f"  daylight MAE:     "
        f"{daylight_mae:.6f}"
    )

    print(
        f"  daylight RMSE:    "
        f"{daylight_rmse:.6f}"
    )

    print(
        f"  runtime:          "
        f"{fit_predict_seconds:.1f} s"
    )


    # --------------------------------------------------------
    # 13. Release fitted-model memory
    # --------------------------------------------------------

    del model
    del X_train_validation
    del X_test
    del y_train_validation
    del y_test
    del test_predictions

    gc.collect()


# ------------------------------------------------------------
# 14. Assemble 15-minute final-test table
# ------------------------------------------------------------

random_forest_15_test_results = (
    pd.DataFrame(
        random_forest_test_result_rows
    )
)


assert (
    len(
        random_forest_15_test_results
    )
    == 5
)


assert (
    len(
        random_forest_test_predictions
    )
    == 5
)


# ------------------------------------------------------------
# 15. Display results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "RANDOM FOREST — 15-MINUTE FINAL TEST RESULTS"
)

print(
    "============================================\n"
)


display(
    random_forest_15_test_results
)


# ------------------------------------------------------------
# 16. Central F3 -> F4 held-out comparison
# ------------------------------------------------------------

f3_test_row_rf_15 = (
    random_forest_15_test_results.loc[
        random_forest_15_test_results[
            "feature_family"
        ].eq(
            "F3"
        )
    ]
    .iloc[0]
)


f4_test_row_rf_15 = (
    random_forest_15_test_results.loc[
        random_forest_15_test_results[
            "feature_family"
        ].eq(
            "F4"
        )
    ]
    .iloc[0]
)


rf_15_f3_f4_all_pct = (
    (
        float(
            f3_test_row_rf_15[
                "test_mae"
            ]
        )
        -
        float(
            f4_test_row_rf_15[
                "test_mae"
            ]
        )
    )
    /
    float(
        f3_test_row_rf_15[
            "test_mae"
        ]
    )
    * 100.0
)


rf_15_f3_f4_daylight_pct = (
    (
        float(
            f3_test_row_rf_15[
                "daylight_mae"
            ]
        )
        -
        float(
            f4_test_row_rf_15[
                "daylight_mae"
            ]
        )
    )
    /
    float(
        f3_test_row_rf_15[
            "daylight_mae"
        ]
    )
    * 100.0
)


print(
    "\n15-minute F3 -> F4 final-test comparison:"
)

print(
    f"  All-time: "
    f"{rf_15_f3_f4_all_pct:+.4f}%"
)

print(
    f"  Daylight: "
    f"{rf_15_f3_f4_daylight_pct:+.4f}%"
)


# ------------------------------------------------------------
# 17. Lowest test MAE
# ------------------------------------------------------------

best_rf_15_test_row = (
    random_forest_15_test_results
    .sort_values(
        "test_mae"
    )
    .iloc[0]
)


print(
    "\nLowest 15-minute Random Forest test MAE:"
)

print(
    "  Feature family:",
    best_rf_15_test_row[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_rf_15_test_row[
        "test_mae"
    ],
)


# ------------------------------------------------------------
# 18. Runtime
# ------------------------------------------------------------

rf_15_test_seconds = (
    time.perf_counter()
    - rf_15_test_start_time
)


print(
    "\nTotal 15-minute final-test runtime:"
)

print(
    f"  {rf_15_test_seconds:.1f} seconds"
)

print(
    f"  {rf_15_test_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 15-minute final test: PASSED"
)


FINAL RANDOM FOREST TEST — 15 MINUTES
Test rows: 26496
Daylight test rows: 17851

15 min — F0
  Frozen config:
    n_estimators      = 300
    max_features      = 0.5
    min_samples_leaf  = 3
    max_depth         = None
  all-time MAE:     0.269186
  all-time RMSE:    0.574705
  daylight MAE:     0.399088
  daylight RMSE:    0.700165
  runtime:          185.4 s

15 min — F1
  Frozen config:
    n_estimators      = 300
    max_features      = 0.5
    min_samples_leaf  = 3
    max_depth         = None
  all-time MAE:     0.258081
  all-time RMSE:    0.558742
  daylight MAE:     0.382711
  daylight RMSE:    0.680720
  runtime:          222.0 s

15 min — F2
  Frozen config:
    n_estimators      = 300
    max_features      = 0.5
    min_samples_leaf  = 3
    max_depth         = None
  all-time MAE:     0.257491
  all-time RMSE:    0.559197
  daylight MAE:     0.381917
  daylight RMSE:    0.681276
  runtime:          263.4 s

15 min — F3
  Frozen config:
    n_estimators      = 300
    m

,model,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,test_rows,...,test_nmae,test_nrmse,test_r2,daylight_test_rows,daylight_mae,daylight_rmse,daylight_nmae,daylight_nrmse,daylight_r2,fit_predict_seconds
0,RandomForestRegressor,15,F0,118,300,0.5,3,NaN,True,26496,...,0.026919,0.057470,0.878779,17851,0.399088,0.700165,0.039909,0.070017,0.835757,185.411075
1,RandomForestRegressor,15,F1,133,300,0.5,3,NaN,True,26496,...,0.025808,0.055874,0.885420,17851,0.382711,0.680720,0.038271,0.068072,0.844753,221.966198
2,RandomForestRegressor,15,F2,138,300,0.5,3,NaN,True,26496,...,0.025749,0.055920,0.885233,17851,0.381917,0.681276,0.038192,0.068128,0.844499,263.395866
3,RandomForestRegressor,15,F3,147,300,0.5,3,NaN,True,26496,...,0.025816,0.055976,0.885003,17851,0.382895,0.681960,0.038289,0.068196,0.844187,323.616059
4,RandomForestRegressor,15,F4,244,300,0.5,3,30.0,True,26496,...,0.025863,0.056031,0.884775,17851,0.383601,0.682634,0.038360,0.068263,0.843879,337.577627



15-minute F3 -> F4 final-test comparison:
  All-time: -0.1831%
  Daylight: -0.1845%

Lowest 15-minute Random Forest test MAE:
  Feature family: F2
  MAE: 0.25749132220956744

Total 15-minute final-test runtime:
  1333.0 seconds
  22.22 minutes

Random Forest 15-minute final test: PASSED


In [123]:
# ============================================================
# Step 113: Final Random Forest test evaluation
#           30-minute horizon
# ============================================================
#
# The 30-minute Random Forest configurations were frozen
# BEFORE any Random Forest test-set metric was examined.
#
# This cell:
#
#   1. preserves the already-completed 15-minute results;
#   2. refits each frozen 30-minute F0 -> F4 model on
#      TRAIN + VALIDATION;
#   3. predicts the held-out TEST set exactly once;
#   4. calculates all-time and daylight metrics;
#   5. retains timestamp-level predictions for the later
#      paired F3-vs-F4 daily-block bootstrap.
#
# No configuration may be changed based on test performance.
# ============================================================


# ------------------------------------------------------------
# 1. Confirm that the previous 15-minute test block exists
#    and that 30-minute test results have NOT already been
#    appended.
# ------------------------------------------------------------

assert len(
    random_forest_test_result_rows
) == 5


assert {
    int(
        row[
            "horizon_minutes"
        ]
    )
    for row in random_forest_test_result_rows
} == {
    15
}


assert len(
    random_forest_test_predictions
) == 5


assert all(
    key[
        0
    ] == 15
    for key in random_forest_test_predictions
)


# ------------------------------------------------------------
# 2. Horizon for this cell
# ------------------------------------------------------------

horizon_minutes = 30


y = targets[
    horizon_minutes
]

split = split_labels[
    horizon_minutes
]


# ------------------------------------------------------------
# 3. Final development and held-out populations
# ------------------------------------------------------------

train_validation_mask = (
    split.isin(
        [
            "train",
            "validation",
        ]
    )
)

test_mask = (
    split == "test"
)


n_train_validation = int(
    train_validation_mask.sum()
)

n_test = int(
    test_mask.sum()
)


assert n_test == 26_496


# ------------------------------------------------------------
# 4. Frozen daylight definition
#
# Daylight is evaluated at forecast-valid time:
#
#       solar_elevation_deg > 0
# ------------------------------------------------------------

daylight_test_mask = (
    solar_valid_features[
        horizon_minutes
    ][
        "solar_elevation_deg"
    ]
    > 0.0
) & test_mask


n_test_daylight = int(
    daylight_test_mask.sum()
)


assert n_test_daylight == 17_851


print(
    "\n"
    "============================================"
)

print(
    "FINAL RANDOM FOREST TEST — 30 MINUTES"
)

print(
    "============================================"
)

print(
    "Train + validation rows:",
    n_train_validation,
)

print(
    "Test rows:",
    n_test,
)

print(
    "Daylight test rows:",
    n_test_daylight,
)


# ------------------------------------------------------------
# 5. Start total horizon runtime
# ------------------------------------------------------------

rf_30_test_start_time = (
    time.perf_counter()
)


# ------------------------------------------------------------
# 6. Evaluate the five frozen feature families
# ------------------------------------------------------------

for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    print(
        "\n"
        f"30 min — {feature_family}"
    )


    # --------------------------------------------------------
    # Frozen configuration selected using validation only
    # --------------------------------------------------------

    config = (
        random_forest_selected_configs[
            horizon_minutes
        ][
            feature_family
        ]
    )


    print(
        "  Frozen config:"
    )

    print(
        "    n_estimators      =",
        config[
            "n_estimators"
        ],
    )

    print(
        "    max_features      =",
        config[
            "max_features"
        ],
    )

    print(
        "    min_samples_leaf  =",
        config[
            "min_samples_leaf"
        ],
    )

    print(
        "    max_depth         =",
        config[
            "max_depth"
        ],
    )


    # --------------------------------------------------------
    # Feature matrix
    # --------------------------------------------------------

    X = (
        feature_matrices[
            horizon_minutes
        ][
            feature_family
        ]
    )


    expected_feature_count = int(
        primary_protocol[
            "feature_counts"
        ][
            feature_family
        ]
    )


    # --------------------------------------------------------
    # TRAIN + VALIDATION arrays
    # --------------------------------------------------------

    X_train_validation = (
        X.loc[
            train_validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    y_train_validation = (
        y.loc[
            train_validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # Held-out TEST arrays
    # --------------------------------------------------------

    X_test = (
        X.loc[
            test_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    y_test = (
        y.loc[
            test_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # Structural checks
    # --------------------------------------------------------

    assert (
        X_train_validation.shape[1]
        == expected_feature_count
    )


    assert (
        X_test.shape
        == (
            26_496,
            expected_feature_count,
        )
    )


    assert (
        len(
            y_test
        )
        == 26_496
    )


    assert np.isfinite(
        X_train_validation
    ).all()

    assert np.isfinite(
        X_test
    ).all()

    assert np.isfinite(
        y_train_validation
    ).all()

    assert np.isfinite(
        y_test
    ).all()


    # --------------------------------------------------------
    # 7. Refit frozen model on TRAIN + VALIDATION
    # --------------------------------------------------------

    model = RandomForestRegressor(
        **config
    )


    fit_start_time = (
        time.perf_counter()
    )


    model.fit(
        X_train_validation,
        y_train_validation,
    )


    test_predictions = (
        model.predict(
            X_test
        )
    )


    fit_predict_seconds = (
        time.perf_counter()
        - fit_start_time
    )


    # --------------------------------------------------------
    # 8. All-time held-out metrics
    # --------------------------------------------------------

    test_mae = (
        mean_absolute_error(
            y_test,
            test_predictions,
        )
    )


    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_predictions,
        )
    )


    test_nmae = (
        test_mae
        / RF_PV_RATED_KW
    )


    test_nrmse = (
        test_rmse
        / RF_PV_RATED_KW
    )


    test_r2 = (
        r2_score(
            y_test,
            test_predictions,
        )
    )


    # --------------------------------------------------------
    # 9. Daylight-only held-out metrics
    # --------------------------------------------------------

    test_issue_index = (
        y.index[
            test_mask
        ]
    )


    daylight_on_test = (
        daylight_test_mask
        .loc[
            test_issue_index
        ]
        .to_numpy(
            dtype=bool
        )
    )


    assert int(
        daylight_on_test.sum()
    ) == 17_851


    y_test_daylight = (
        y_test[
            daylight_on_test
        ]
    )


    predictions_daylight = (
        test_predictions[
            daylight_on_test
        ]
    )


    daylight_mae = (
        mean_absolute_error(
            y_test_daylight,
            predictions_daylight,
        )
    )


    daylight_rmse = np.sqrt(
        mean_squared_error(
            y_test_daylight,
            predictions_daylight,
        )
    )


    daylight_nmae = (
        daylight_mae
        / RF_PV_RATED_KW
    )


    daylight_nrmse = (
        daylight_rmse
        / RF_PV_RATED_KW
    )


    daylight_r2 = (
        r2_score(
            y_test_daylight,
            predictions_daylight,
        )
    )


    # --------------------------------------------------------
    # 10. Store final metric row
    # --------------------------------------------------------

    random_forest_test_result_rows.append(
        {
            "model":
                "RandomForestRegressor",

            "horizon_minutes":
                horizon_minutes,

            "feature_family":
                feature_family,

            "n_features":
                expected_feature_count,

            "n_estimators":
                int(
                    config[
                        "n_estimators"
                    ]
                ),

            "max_features":
                float(
                    config[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    config[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                config[
                    "max_depth"
                ],

            "bootstrap":
                bool(
                    config[
                        "bootstrap"
                    ]
                ),

            "test_rows":
                n_test,

            "test_mae":
                test_mae,

            "test_rmse":
                test_rmse,

            "test_nmae":
                test_nmae,

            "test_nrmse":
                test_nrmse,

            "test_r2":
                test_r2,

            "daylight_test_rows":
                n_test_daylight,

            "daylight_mae":
                daylight_mae,

            "daylight_rmse":
                daylight_rmse,

            "daylight_nmae":
                daylight_nmae,

            "daylight_nrmse":
                daylight_nrmse,

            "daylight_r2":
                daylight_r2,

            "fit_predict_seconds":
                fit_predict_seconds,
        }
    )


    # --------------------------------------------------------
    # 11. Retain timestamp-level predictions
    #
    # valid_time is essential for the later day-block
    # bootstrap because blocks are defined by forecast-valid
    # UTC calendar day.
    # --------------------------------------------------------

    prediction_table = pd.DataFrame(
        {
            "valid_time":
                base_modelling_tables[
                    horizon_minutes
                ]
                .loc[
                    test_issue_index,
                    "valid_time",
                ],

            "y_true":
                y_test,

            "y_pred":
                test_predictions,

            "absolute_error":
                np.abs(
                    y_test
                    - test_predictions
                ),

            "daylight":
                daylight_on_test,
        },
        index=test_issue_index,
    )


    prediction_table.index.name = (
        "issue_time"
    )


    random_forest_test_predictions[
        (
            horizon_minutes,
            feature_family,
        )
    ] = prediction_table


    # --------------------------------------------------------
    # 12. Compact live output
    # --------------------------------------------------------

    print(
        f"  all-time MAE:     "
        f"{test_mae:.6f}"
    )

    print(
        f"  all-time RMSE:    "
        f"{test_rmse:.6f}"
    )

    print(
        f"  daylight MAE:     "
        f"{daylight_mae:.6f}"
    )

    print(
        f"  daylight RMSE:    "
        f"{daylight_rmse:.6f}"
    )

    print(
        f"  runtime:          "
        f"{fit_predict_seconds:.1f} s"
    )


    # --------------------------------------------------------
    # 13. Release fitted-model memory
    # --------------------------------------------------------

    del model
    del X_train_validation
    del X_test
    del y_train_validation
    del y_test
    del test_predictions

    gc.collect()


# ------------------------------------------------------------
# 14. Verify cumulative result containers
#
# After completing 15 min + 30 min:
#
#       10 metric rows
#       10 prediction tables
# ------------------------------------------------------------

assert len(
    random_forest_test_result_rows
) == 10


assert len(
    random_forest_test_predictions
) == 10


# ------------------------------------------------------------
# 15. Build the 30-minute result table only
# ------------------------------------------------------------

random_forest_30_test_results = pd.DataFrame(
    [
        row
        for row
        in random_forest_test_result_rows
        if int(
            row[
                "horizon_minutes"
            ]
        ) == 30
    ]
)


assert (
    len(
        random_forest_30_test_results
    )
    == 5
)


# ------------------------------------------------------------
# 16. Display 30-minute final-test results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "RANDOM FOREST — 30-MINUTE FINAL TEST RESULTS"
)

print(
    "============================================\n"
)


display(
    random_forest_30_test_results
)


# ------------------------------------------------------------
# 17. Central F3 -> F4 held-out comparison
# ------------------------------------------------------------

f3_test_row_rf_30 = (
    random_forest_30_test_results.loc[
        random_forest_30_test_results[
            "feature_family"
        ].eq(
            "F3"
        )
    ]
    .iloc[0]
)


f4_test_row_rf_30 = (
    random_forest_30_test_results.loc[
        random_forest_30_test_results[
            "feature_family"
        ].eq(
            "F4"
        )
    ]
    .iloc[0]
)


rf_30_f3_f4_all_pct = (
    (
        float(
            f3_test_row_rf_30[
                "test_mae"
            ]
        )
        -
        float(
            f4_test_row_rf_30[
                "test_mae"
            ]
        )
    )
    /
    float(
        f3_test_row_rf_30[
            "test_mae"
        ]
    )
    * 100.0
)


rf_30_f3_f4_daylight_pct = (
    (
        float(
            f3_test_row_rf_30[
                "daylight_mae"
            ]
        )
        -
        float(
            f4_test_row_rf_30[
                "daylight_mae"
            ]
        )
    )
    /
    float(
        f3_test_row_rf_30[
            "daylight_mae"
        ]
    )
    * 100.0
)


print(
    "\n30-minute F3 -> F4 final-test comparison:"
)

print(
    f"  All-time: "
    f"{rf_30_f3_f4_all_pct:+.4f}%"
)

print(
    f"  Daylight: "
    f"{rf_30_f3_f4_daylight_pct:+.4f}%"
)


# ------------------------------------------------------------
# 18. Identify lowest 30-minute test MAE
# ------------------------------------------------------------

best_rf_30_test_row = (
    random_forest_30_test_results
    .sort_values(
        [
            "test_mae",
            "test_rmse",
        ]
    )
    .iloc[0]
)


print(
    "\nLowest 30-minute Random Forest test MAE:"
)

print(
    "  Feature family:",
    best_rf_30_test_row[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_rf_30_test_row[
        "test_mae"
    ],
)


# ------------------------------------------------------------
# 19. Total runtime
# ------------------------------------------------------------

rf_30_test_seconds = (
    time.perf_counter()
    - rf_30_test_start_time
)


print(
    "\nTotal 30-minute final-test runtime:"
)

print(
    f"  {rf_30_test_seconds:.1f} seconds"
)

print(
    f"  {rf_30_test_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 30-minute final test: PASSED"
)


FINAL RANDOM FOREST TEST — 30 MINUTES
Train + validation rows: 104826
Test rows: 26496
Daylight test rows: 17851

30 min — F0
  Frozen config:
    n_estimators      = 300
    max_features      = 0.5
    min_samples_leaf  = 3
    max_depth         = 30
  all-time MAE:     0.326990
  all-time RMSE:    0.653240
  daylight MAE:     0.484116
  daylight RMSE:    0.795826
  runtime:          178.0 s

30 min — F1
  Frozen config:
    n_estimators      = 300
    max_features      = 0.5
    min_samples_leaf  = 3
    max_depth         = None
  all-time MAE:     0.298233
  all-time RMSE:    0.620671
  daylight MAE:     0.441902
  daylight RMSE:    0.756158
  runtime:          200.8 s

30 min — F2
  Frozen config:
    n_estimators      = 300
    max_features      = 0.5
    min_samples_leaf  = 3
    max_depth         = None
  all-time MAE:     0.296665
  all-time RMSE:    0.619884
  daylight MAE:     0.439928
  daylight RMSE:    0.755210
  runtime:          250.2 s

30 min — F3
  Frozen config:
   

,model,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,test_rows,...,test_nmae,test_nrmse,test_r2,daylight_test_rows,daylight_mae,daylight_rmse,daylight_nmae,daylight_nrmse,daylight_r2,fit_predict_seconds
0,RandomForestRegressor,30,F0,118,300,0.5,3,30.0,True,26496,...,0.032699,0.065324,0.843385,17851,0.484116,0.795826,0.048412,0.079583,0.787811,177.950924
1,RandomForestRegressor,30,F1,133,300,0.5,3,NaN,True,26496,...,0.029823,0.062067,0.858613,17851,0.441902,0.756158,0.044190,0.075616,0.808437,200.780922
2,RandomForestRegressor,30,F2,138,300,0.5,3,NaN,True,26496,...,0.029667,0.061988,0.858972,17851,0.439928,0.755210,0.043993,0.075521,0.808917,250.233925
3,RandomForestRegressor,30,F3,147,300,0.5,3,30.0,True,26496,...,0.029810,0.062062,0.858637,17851,0.442046,0.756105,0.044205,0.075611,0.808463,243.973783
4,RandomForestRegressor,30,F4,244,300,0.5,3,30.0,True,26496,...,0.029748,0.062011,0.858868,17851,0.441128,0.755487,0.044113,0.075549,0.808777,272.466862



30-minute F3 -> F4 final-test comparison:
  All-time: +0.2080%
  Daylight: +0.2076%

Lowest 30-minute Random Forest test MAE:
  Feature family: F2
  MAE: 0.2966652050464309

Total 30-minute final-test runtime:
  1146.2 seconds
  19.10 minutes

Random Forest 30-minute final test: PASSED


In [124]:
# ============================================================
# Step 114: Final Random Forest test evaluation
#           60-minute horizon
# ============================================================
#
# This is the final held-out Random Forest horizon.
#
# The 60-minute configurations were frozen using validation
# BEFORE any Random Forest test-set metric was used for model
# selection.
#
# This cell:
#
#   1. preserves the already-completed 15- and 30-minute
#      final-test results;
#   2. refits each frozen 60-minute F0 -> F4 model on
#      TRAIN + VALIDATION;
#   3. predicts the held-out TEST set exactly once;
#   4. calculates all-time and daylight metrics;
#   5. retains timestamp-level predictions for the later
#      paired F3-vs-F4 daily-block bootstrap.
#
# No hyperparameter may be changed after seeing test results.
# ============================================================


# ------------------------------------------------------------
# 1. Confirm that exactly 15- and 30-minute final-test
#    results already exist.
# ------------------------------------------------------------

assert len(
    random_forest_test_result_rows
) == 10


assert {
    int(
        row[
            "horizon_minutes"
        ]
    )
    for row in random_forest_test_result_rows
} == {
    15,
    30,
}


assert len(
    random_forest_test_predictions
) == 10


assert {
    key[
        0
    ]
    for key in random_forest_test_predictions
} == {
    15,
    30,
}


# ------------------------------------------------------------
# 2. Horizon for this cell
# ------------------------------------------------------------

horizon_minutes = 60


y = targets[
    horizon_minutes
]

split = split_labels[
    horizon_minutes
]


# ------------------------------------------------------------
# 3. TRAIN + VALIDATION and held-out TEST masks
# ------------------------------------------------------------

train_validation_mask = (
    split.isin(
        [
            "train",
            "validation",
        ]
    )
)

test_mask = (
    split == "test"
)


n_train_validation = int(
    train_validation_mask.sum()
)

n_test = int(
    test_mask.sum()
)


assert n_train_validation == 104_820

assert n_test == 26_496


# ------------------------------------------------------------
# 4. Frozen daylight definition
#
# Daylight is defined using forecast-valid solar elevation:
#
#       solar_elevation_deg > 0
# ------------------------------------------------------------

daylight_test_mask = (
    solar_valid_features[
        horizon_minutes
    ][
        "solar_elevation_deg"
    ]
    > 0.0
) & test_mask


n_test_daylight = int(
    daylight_test_mask.sum()
)


assert n_test_daylight == 17_851


print(
    "\n"
    "============================================"
)

print(
    "FINAL RANDOM FOREST TEST — 60 MINUTES"
)

print(
    "============================================"
)

print(
    "Train + validation rows:",
    n_train_validation,
)

print(
    "Test rows:",
    n_test,
)

print(
    "Daylight test rows:",
    n_test_daylight,
)


# ------------------------------------------------------------
# 5. Start total horizon runtime
# ------------------------------------------------------------

rf_60_test_start_time = (
    time.perf_counter()
)


# ------------------------------------------------------------
# 6. Evaluate frozen F0 -> F4 configurations
# ------------------------------------------------------------

for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    print(
        "\n"
        f"60 min — {feature_family}"
    )


    # --------------------------------------------------------
    # Frozen configuration selected using validation only
    # --------------------------------------------------------

    config = (
        random_forest_selected_configs[
            horizon_minutes
        ][
            feature_family
        ]
    )


    print(
        "  Frozen config:"
    )

    print(
        "    n_estimators      =",
        config[
            "n_estimators"
        ],
    )

    print(
        "    max_features      =",
        config[
            "max_features"
        ],
    )

    print(
        "    min_samples_leaf  =",
        config[
            "min_samples_leaf"
        ],
    )

    print(
        "    max_depth         =",
        config[
            "max_depth"
        ],
    )


    # --------------------------------------------------------
    # Feature matrix
    # --------------------------------------------------------

    X = (
        feature_matrices[
            horizon_minutes
        ][
            feature_family
        ]
    )


    expected_feature_count = int(
        primary_protocol[
            "feature_counts"
        ][
            feature_family
        ]
    )


    # --------------------------------------------------------
    # TRAIN + VALIDATION arrays
    # --------------------------------------------------------

    X_train_validation = (
        X.loc[
            train_validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    y_train_validation = (
        y.loc[
            train_validation_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # Held-out TEST arrays
    # --------------------------------------------------------

    X_test = (
        X.loc[
            test_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    y_test = (
        y.loc[
            test_mask
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # Structural checks
    # --------------------------------------------------------

    assert (
        X_train_validation.shape
        == (
            104_820,
            expected_feature_count,
        )
    )


    assert (
        X_test.shape
        == (
            26_496,
            expected_feature_count,
        )
    )


    assert (
        len(
            y_train_validation
        )
        == 104_820
    )


    assert (
        len(
            y_test
        )
        == 26_496
    )


    assert np.isfinite(
        X_train_validation
    ).all()

    assert np.isfinite(
        X_test
    ).all()

    assert np.isfinite(
        y_train_validation
    ).all()

    assert np.isfinite(
        y_test
    ).all()


    # --------------------------------------------------------
    # 7. Refit frozen model on TRAIN + VALIDATION
    # --------------------------------------------------------

    model = RandomForestRegressor(
        **config
    )


    fit_start_time = (
        time.perf_counter()
    )


    model.fit(
        X_train_validation,
        y_train_validation,
    )


    test_predictions = (
        model.predict(
            X_test
        )
    )


    fit_predict_seconds = (
        time.perf_counter()
        - fit_start_time
    )


    # --------------------------------------------------------
    # 8. All-time held-out metrics
    # --------------------------------------------------------

    test_mae = (
        mean_absolute_error(
            y_test,
            test_predictions,
        )
    )


    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_predictions,
        )
    )


    test_nmae = (
        test_mae
        / RF_PV_RATED_KW
    )


    test_nrmse = (
        test_rmse
        / RF_PV_RATED_KW
    )


    test_r2 = (
        r2_score(
            y_test,
            test_predictions,
        )
    )


    # --------------------------------------------------------
    # 9. Daylight-only held-out metrics
    # --------------------------------------------------------

    test_issue_index = (
        y.index[
            test_mask
        ]
    )


    daylight_on_test = (
        daylight_test_mask
        .loc[
            test_issue_index
        ]
        .to_numpy(
            dtype=bool
        )
    )


    assert int(
        daylight_on_test.sum()
    ) == 17_851


    y_test_daylight = (
        y_test[
            daylight_on_test
        ]
    )


    predictions_daylight = (
        test_predictions[
            daylight_on_test
        ]
    )


    daylight_mae = (
        mean_absolute_error(
            y_test_daylight,
            predictions_daylight,
        )
    )


    daylight_rmse = np.sqrt(
        mean_squared_error(
            y_test_daylight,
            predictions_daylight,
        )
    )


    daylight_nmae = (
        daylight_mae
        / RF_PV_RATED_KW
    )


    daylight_nrmse = (
        daylight_rmse
        / RF_PV_RATED_KW
    )


    daylight_r2 = (
        r2_score(
            y_test_daylight,
            predictions_daylight,
        )
    )


    # --------------------------------------------------------
    # 10. Store final metric row
    # --------------------------------------------------------

    random_forest_test_result_rows.append(
        {
            "model":
                "RandomForestRegressor",

            "horizon_minutes":
                horizon_minutes,

            "feature_family":
                feature_family,

            "n_features":
                expected_feature_count,

            "n_estimators":
                int(
                    config[
                        "n_estimators"
                    ]
                ),

            "max_features":
                float(
                    config[
                        "max_features"
                    ]
                ),

            "min_samples_leaf":
                int(
                    config[
                        "min_samples_leaf"
                    ]
                ),

            "max_depth":
                config[
                    "max_depth"
                ],

            "bootstrap":
                bool(
                    config[
                        "bootstrap"
                    ]
                ),

            "test_rows":
                n_test,

            "test_mae":
                test_mae,

            "test_rmse":
                test_rmse,

            "test_nmae":
                test_nmae,

            "test_nrmse":
                test_nrmse,

            "test_r2":
                test_r2,

            "daylight_test_rows":
                n_test_daylight,

            "daylight_mae":
                daylight_mae,

            "daylight_rmse":
                daylight_rmse,

            "daylight_nmae":
                daylight_nmae,

            "daylight_nrmse":
                daylight_nrmse,

            "daylight_r2":
                daylight_r2,

            "fit_predict_seconds":
                fit_predict_seconds,
        }
    )


    # --------------------------------------------------------
    # 11. Retain timestamp-level predictions
    #
    # Blocks in the later bootstrap are defined by
    # forecast-valid UTC calendar day.
    # --------------------------------------------------------

    prediction_table = pd.DataFrame(
        {
            "valid_time":
                base_modelling_tables[
                    horizon_minutes
                ]
                .loc[
                    test_issue_index,
                    "valid_time",
                ],

            "y_true":
                y_test,

            "y_pred":
                test_predictions,

            "absolute_error":
                np.abs(
                    y_test
                    - test_predictions
                ),

            "daylight":
                daylight_on_test,
        },
        index=test_issue_index,
    )


    prediction_table.index.name = (
        "issue_time"
    )


    random_forest_test_predictions[
        (
            horizon_minutes,
            feature_family,
        )
    ] = prediction_table


    # --------------------------------------------------------
    # 12. Compact live output
    # --------------------------------------------------------

    print(
        f"  all-time MAE:     "
        f"{test_mae:.6f}"
    )

    print(
        f"  all-time RMSE:    "
        f"{test_rmse:.6f}"
    )

    print(
        f"  daylight MAE:     "
        f"{daylight_mae:.6f}"
    )

    print(
        f"  daylight RMSE:    "
        f"{daylight_rmse:.6f}"
    )

    print(
        f"  runtime:          "
        f"{fit_predict_seconds:.1f} s"
    )


    # --------------------------------------------------------
    # 13. Release fitted-model memory
    # --------------------------------------------------------

    del model
    del X_train_validation
    del X_test
    del y_train_validation
    del y_test
    del test_predictions

    gc.collect()


# ------------------------------------------------------------
# 14. Verify all 15 final-test experiments now exist
# ------------------------------------------------------------

assert len(
    random_forest_test_result_rows
) == 15


assert len(
    random_forest_test_predictions
) == 15


assert {
    int(
        row[
            "horizon_minutes"
        ]
    )
    for row in random_forest_test_result_rows
} == {
    15,
    30,
    60,
}


# ------------------------------------------------------------
# 15. Build 60-minute result table
# ------------------------------------------------------------

random_forest_60_test_results = pd.DataFrame(
    [
        row
        for row
        in random_forest_test_result_rows
        if int(
            row[
                "horizon_minutes"
            ]
        ) == 60
    ]
)


assert (
    len(
        random_forest_60_test_results
    )
    == 5
)


# ------------------------------------------------------------
# 16. Display 60-minute final-test results
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "RANDOM FOREST — 60-MINUTE FINAL TEST RESULTS"
)

print(
    "============================================\n"
)


display(
    random_forest_60_test_results
)


# ------------------------------------------------------------
# 17. Central F3 -> F4 held-out comparison
# ------------------------------------------------------------

f3_test_row_rf_60 = (
    random_forest_60_test_results.loc[
        random_forest_60_test_results[
            "feature_family"
        ].eq(
            "F3"
        )
    ]
    .iloc[0]
)


f4_test_row_rf_60 = (
    random_forest_60_test_results.loc[
        random_forest_60_test_results[
            "feature_family"
        ].eq(
            "F4"
        )
    ]
    .iloc[0]
)


rf_60_f3_f4_all_pct = (
    (
        float(
            f3_test_row_rf_60[
                "test_mae"
            ]
        )
        -
        float(
            f4_test_row_rf_60[
                "test_mae"
            ]
        )
    )
    /
    float(
        f3_test_row_rf_60[
            "test_mae"
        ]
    )
    * 100.0
)


rf_60_f3_f4_daylight_pct = (
    (
        float(
            f3_test_row_rf_60[
                "daylight_mae"
            ]
        )
        -
        float(
            f4_test_row_rf_60[
                "daylight_mae"
            ]
        )
    )
    /
    float(
        f3_test_row_rf_60[
            "daylight_mae"
        ]
    )
    * 100.0
)


print(
    "\n60-minute F3 -> F4 final-test comparison:"
)

print(
    f"  All-time: "
    f"{rf_60_f3_f4_all_pct:+.4f}%"
)

print(
    f"  Daylight: "
    f"{rf_60_f3_f4_daylight_pct:+.4f}%"
)


# ------------------------------------------------------------
# 18. Identify lowest 60-minute test MAE
# ------------------------------------------------------------

best_rf_60_test_row = (
    random_forest_60_test_results
    .sort_values(
        [
            "test_mae",
            "test_rmse",
        ]
    )
    .iloc[0]
)


print(
    "\nLowest 60-minute Random Forest test MAE:"
)

print(
    "  Feature family:",
    best_rf_60_test_row[
        "feature_family"
    ],
)

print(
    "  MAE:",
    best_rf_60_test_row[
        "test_mae"
    ],
)


# ------------------------------------------------------------
# 19. Total runtime
# ------------------------------------------------------------

rf_60_test_seconds = (
    time.perf_counter()
    - rf_60_test_start_time
)


print(
    "\nTotal 60-minute final-test runtime:"
)

print(
    f"  {rf_60_test_seconds:.1f} seconds"
)

print(
    f"  {rf_60_test_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest 60-minute final test: PASSED"
)


FINAL RANDOM FOREST TEST — 60 MINUTES
Train + validation rows: 104820
Test rows: 26496
Daylight test rows: 17851

60 min — F0
  Frozen config:
    n_estimators      = 300
    max_features      = 0.5
    min_samples_leaf  = 3
    max_depth         = None
  all-time MAE:     0.409046
  all-time RMSE:    0.754151
  daylight MAE:     0.602189
  daylight RMSE:    0.918529
  runtime:          188.2 s

60 min — F1
  Frozen config:
    n_estimators      = 300
    max_features      = 0.5
    min_samples_leaf  = 3
    max_depth         = 30
  all-time MAE:     0.338990
  all-time RMSE:    0.679051
  daylight MAE:     0.501900
  daylight RMSE:    0.827282
  runtime:          218.0 s

60 min — F2
  Frozen config:
    n_estimators      = 300
    max_features      = 1.0
    min_samples_leaf  = 3
    max_depth         = 30
  all-time MAE:     0.339296
  all-time RMSE:    0.682305
  daylight MAE:     0.503120
  daylight RMSE:    0.831259
  runtime:          491.2 s

60 min — F3
  Frozen config:
    n

,model,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,test_rows,...,test_nmae,test_nrmse,test_r2,daylight_test_rows,daylight_mae,daylight_rmse,daylight_nmae,daylight_nrmse,daylight_r2,fit_predict_seconds
0,RandomForestRegressor,60,F0,118,300,0.5,3,NaN,True,26496,...,0.040905,0.075415,0.791261,17851,0.602189,0.918529,0.060219,0.091853,0.717335,188.210293
1,RandomForestRegressor,60,F1,133,300,0.5,3,30.0,True,26496,...,0.033899,0.067905,0.830764,17851,0.501900,0.827282,0.050190,0.082728,0.770705,217.992163
2,RandomForestRegressor,60,F2,138,300,1.0,3,30.0,True,26496,...,0.033930,0.068231,0.829139,17851,0.503120,0.831259,0.050312,0.083126,0.768495,491.226022
3,RandomForestRegressor,60,F3,147,300,0.5,3,30.0,True,26496,...,0.034105,0.068415,0.828214,17851,0.505728,0.833506,0.050573,0.083351,0.767242,270.076590
4,RandomForestRegressor,60,F4,244,300,0.5,3,NaN,True,26496,...,0.034061,0.068351,0.828535,17851,0.505067,0.832725,0.050507,0.083272,0.767678,326.431238



60-minute F3 -> F4 final-test comparison:
  All-time: +0.1301%
  Daylight: +0.1307%

Lowest 60-minute Random Forest test MAE:
  Feature family: F1
  MAE: 0.338990190812323

Total 60-minute final-test runtime:
  1494.7 seconds
  24.91 minutes

Random Forest 60-minute final test: PASSED


In [125]:
# ============================================================
# Step 115: Random Forest paired daily-block bootstrap
#           F3 versus F4 on the held-out TEST set
# ============================================================
#
# PURPOSE
# -------
# The final-test point estimates tell us whether F4 had lower
# MAE than F3 on this particular test period.
#
# They do NOT tell us whether that small difference is stable
# across the test period.
#
# We therefore apply the exact same uncertainty-analysis
# protocol used for the previous model families:
#
#   - comparison:
#         F3 MAE - F4 MAE
#
#   - positive value:
#         F4 has LOWER MAE than F3
#
#   - negative value:
#         F4 has HIGHER MAE than F3
#
#   - resampling unit:
#         forecast-valid UTC calendar day
#
#   - number of test-day blocks:
#         92
#
#   - bootstrap replicates:
#         10,000
#
#   - seed:
#         42
#
#   - confidence interval:
#         95% percentile interval
#
#   - no model retraining
#
#   - all-time analysis first
#
#   - daylight analysis second
#
# IMPORTANT
# ---------
# We resample WHOLE DAYS with replacement.
#
# Within each sampled bootstrap replicate, pooled MAE is
# reconstructed from:
#
#       total absolute-error sum
#       ------------------------
#             total rows
#
# This preserves row weighting.
#
# We do NOT compute an unweighted average of daily MAEs.
# ============================================================


# ------------------------------------------------------------
# 1. Bootstrap constants
# ------------------------------------------------------------

RF_BOOTSTRAP_REPLICATES = 10_000
RF_BOOTSTRAP_SEED = 42
RF_EXPECTED_TEST_DAY_BLOCKS = 92


# ------------------------------------------------------------
# 2. Verify all final Random Forest experiments exist
# ------------------------------------------------------------

assert len(
    random_forest_test_result_rows
) == 15


assert len(
    random_forest_test_predictions
) == 15


assert {
    int(
        row[
            "horizon_minutes"
        ]
    )
    for row in random_forest_test_result_rows
} == {
    15,
    30,
    60,
}


# ------------------------------------------------------------
# 3. Build complete final-test results table
# ------------------------------------------------------------

random_forest_final_test_results = (
    pd.DataFrame(
        random_forest_test_result_rows
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    random_forest_final_test_results.shape[0]
    == 15
)


print(
    "Random Forest complete final-test results:\n"
)


display(
    random_forest_final_test_results
)


# ------------------------------------------------------------
# 4. Containers for bootstrap outputs
# ------------------------------------------------------------

random_forest_bootstrap_summary_rows = []

random_forest_bootstrap_distribution_rows = []


# ------------------------------------------------------------
# 5. ONE RNG for the complete Random Forest bootstrap
#
# This matches the previously frozen bootstrap protocol:
# instantiate np.random.default_rng(42) once, then consume it
# sequentially across all horizon/population analyses.
# ------------------------------------------------------------

rf_bootstrap_rng = np.random.default_rng(
    RF_BOOTSTRAP_SEED
)


# ------------------------------------------------------------
# 6. Run paired day-block bootstrap
#
# Analysis order is deliberately fixed:
#
#   15 min all
#   15 min daylight
#   30 min all
#   30 min daylight
#   60 min all
#   60 min daylight
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    # --------------------------------------------------------
    # Retrieve F3 and F4 timestamp-level held-out predictions
    # --------------------------------------------------------

    f3_predictions = (
        random_forest_test_predictions[
            (
                horizon_minutes,
                "F3",
            )
        ]
        .copy()
    )


    f4_predictions = (
        random_forest_test_predictions[
            (
                horizon_minutes,
                "F4",
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Strong pairing checks
    #
    # F3 and F4 must represent exactly the same forecasts.
    # --------------------------------------------------------

    assert f3_predictions.index.equals(
        f4_predictions.index
    )


    assert np.array_equal(
        f3_predictions[
            "valid_time"
        ].to_numpy(),
        f4_predictions[
            "valid_time"
        ].to_numpy(),
    )


    assert np.allclose(
        f3_predictions[
            "y_true"
        ].to_numpy(),
        f4_predictions[
            "y_true"
        ].to_numpy(),
        rtol=0.0,
        atol=0.0,
    )


    assert np.array_equal(
        f3_predictions[
            "daylight"
        ].to_numpy(),
        f4_predictions[
            "daylight"
        ].to_numpy(),
    )


    # --------------------------------------------------------
    # Create one paired table
    # --------------------------------------------------------

    paired_predictions = pd.DataFrame(
        {
            "valid_time":
                f3_predictions[
                    "valid_time"
                ],

            "daylight":
                f3_predictions[
                    "daylight"
                ].astype(
                    bool
                ),

            "f3_absolute_error":
                f3_predictions[
                    "absolute_error"
                ].astype(
                    float
                ),

            "f4_absolute_error":
                f4_predictions[
                    "absolute_error"
                ].astype(
                    float
                ),
        },
        index=f3_predictions.index,
    )


    # --------------------------------------------------------
    # Forecast-valid UTC calendar-day block
    #
    # The valid_time should already be timezone-aware UTC.
    # floor("D") preserves the day boundary in UTC.
    # --------------------------------------------------------

    paired_predictions[
        "valid_day"
    ] = pd.to_datetime(
        paired_predictions[
            "valid_time"
        ],
        utc=True,
    ).dt.floor(
        "D"
    )


    # --------------------------------------------------------
    # Run both frozen evaluation populations
    # --------------------------------------------------------

    for population in [
        "all",
        "daylight",
    ]:

        if population == "all":

            population_table = (
                paired_predictions
                .copy()
            )

        else:

            # ------------------------------------------------
            # IMPORTANT:
            # filter daylight rows BEFORE constructing blocks.
            # ------------------------------------------------

            population_table = (
                paired_predictions.loc[
                    paired_predictions[
                        "daylight"
                    ]
                ]
                .copy()
            )


        # ----------------------------------------------------
        # 7. Daily sufficient statistics
        #
        # These are enough to reconstruct pooled MAE after
        # resampling days:
        #
        #   n_rows
        #   sum(|error_F3|)
        #   sum(|error_F4|)
        # ----------------------------------------------------

        daily_stats = (
            population_table
            .groupby(
                "valid_day",
                sort=True,
            )
            .agg(
                n_rows=(
                    "f3_absolute_error",
                    "size",
                ),

                f3_absolute_error_sum=(
                    "f3_absolute_error",
                    "sum",
                ),

                f4_absolute_error_sum=(
                    "f4_absolute_error",
                    "sum",
                ),
            )
            .reset_index()
        )


        n_blocks = int(
            len(
                daily_stats
            )
        )


        assert (
            n_blocks
            == RF_EXPECTED_TEST_DAY_BLOCKS
        )


        assert (
            daily_stats[
                "n_rows"
            ]
            > 0
        ).all()


        # ----------------------------------------------------
        # 8. Full-sample point estimate
        # ----------------------------------------------------

        full_n_rows = int(
            daily_stats[
                "n_rows"
            ].sum()
        )


        full_f3_error_sum = float(
            daily_stats[
                "f3_absolute_error_sum"
            ].sum()
        )


        full_f4_error_sum = float(
            daily_stats[
                "f4_absolute_error_sum"
            ].sum()
        )


        full_f3_mae = (
            full_f3_error_sum
            / full_n_rows
        )


        full_f4_mae = (
            full_f4_error_sum
            / full_n_rows
        )


        full_absolute_difference = (
            full_f3_mae
            - full_f4_mae
        )


        full_relative_improvement_pct = (
            full_absolute_difference
            / full_f3_mae
            * 100.0
        )


        # ----------------------------------------------------
        # 9. Cross-check point estimate against stored final
        #    test metrics.
        # ----------------------------------------------------

        final_f3_row = (
            random_forest_final_test_results.loc[
                (
                    random_forest_final_test_results[
                        "horizon_minutes"
                    ].eq(
                        horizon_minutes
                    )
                )
                &
                (
                    random_forest_final_test_results[
                        "feature_family"
                    ].eq(
                        "F3"
                    )
                )
            ]
            .iloc[0]
        )


        final_f4_row = (
            random_forest_final_test_results.loc[
                (
                    random_forest_final_test_results[
                        "horizon_minutes"
                    ].eq(
                        horizon_minutes
                    )
                )
                &
                (
                    random_forest_final_test_results[
                        "feature_family"
                    ].eq(
                        "F4"
                    )
                )
            ]
            .iloc[0]
        )


        if population == "all":

            stored_f3_mae = float(
                final_f3_row[
                    "test_mae"
                ]
            )

            stored_f4_mae = float(
                final_f4_row[
                    "test_mae"
                ]
            )

            expected_rows = 26_496

        else:

            stored_f3_mae = float(
                final_f3_row[
                    "daylight_mae"
                ]
            )

            stored_f4_mae = float(
                final_f4_row[
                    "daylight_mae"
                ]
            )

            expected_rows = 17_851


        assert (
            full_n_rows
            == expected_rows
        )


        assert np.isclose(
            full_f3_mae,
            stored_f3_mae,
            rtol=0.0,
            atol=1e-12,
        )


        assert np.isclose(
            full_f4_mae,
            stored_f4_mae,
            rtol=0.0,
            atol=1e-12,
        )


        # ----------------------------------------------------
        # 10. Convert daily sufficient statistics to NumPy
        # ----------------------------------------------------

        block_n_rows = (
            daily_stats[
                "n_rows"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )


        block_f3_error_sums = (
            daily_stats[
                "f3_absolute_error_sum"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        block_f4_error_sums = (
            daily_stats[
                "f4_absolute_error_sum"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        # ----------------------------------------------------
        # 11. Draw 10,000 paired bootstrap samples
        #
        # Each replicate samples 92 entire valid-time days
        # WITH replacement.
        # ----------------------------------------------------

        sampled_block_indices = (
            rf_bootstrap_rng.integers(
                low=0,
                high=n_blocks,
                size=(
                    RF_BOOTSTRAP_REPLICATES,
                    n_blocks,
                ),
            )
        )


        # ----------------------------------------------------
        # 12. Reconstruct pooled MAE for each replicate
        #
        # This is deliberately ROW-WEIGHTED pooled MAE,
        # not an average of daily MAEs.
        # ----------------------------------------------------

        sampled_row_counts = (
            block_n_rows[
                sampled_block_indices
            ]
            .sum(
                axis=1
            )
        )


        sampled_f3_error_sums = (
            block_f3_error_sums[
                sampled_block_indices
            ]
            .sum(
                axis=1
            )
        )


        sampled_f4_error_sums = (
            block_f4_error_sums[
                sampled_block_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f3_mae = (
            sampled_f3_error_sums
            / sampled_row_counts
        )


        bootstrap_f4_mae = (
            sampled_f4_error_sums
            / sampled_row_counts
        )


        # ----------------------------------------------------
        # Difference convention:
        #
        #     F3 MAE - F4 MAE
        #
        # Positive => F4 better
        # Negative => F4 worse
        # ----------------------------------------------------

        bootstrap_absolute_difference = (
            bootstrap_f3_mae
            - bootstrap_f4_mae
        )


        bootstrap_relative_improvement_pct = (
            bootstrap_absolute_difference
            / bootstrap_f3_mae
            * 100.0
        )


        # ----------------------------------------------------
        # 13. 95% percentile confidence intervals
        # ----------------------------------------------------

        absolute_ci_lower = float(
            np.quantile(
                bootstrap_absolute_difference,
                0.025,
            )
        )


        absolute_ci_upper = float(
            np.quantile(
                bootstrap_absolute_difference,
                0.975,
            )
        )


        relative_ci_lower = float(
            np.quantile(
                bootstrap_relative_improvement_pct,
                0.025,
            )
        )


        relative_ci_upper = float(
            np.quantile(
                bootstrap_relative_improvement_pct,
                0.975,
            )
        )


        relative_ci_contains_zero = bool(
            (
                relative_ci_lower
                <= 0.0
            )
            and
            (
                relative_ci_upper
                >= 0.0
            )
        )


        # ----------------------------------------------------
        # 14. Store summary row
        # ----------------------------------------------------

        random_forest_bootstrap_summary_rows.append(
            {
                "model":
                    "RandomForestRegressor",

                "horizon_minutes":
                    horizon_minutes,

                "population":
                    population,

                "comparison":
                    "F3_minus_F4_MAE",

                "positive_means":
                    "F4_lower_MAE",

                "n_rows":
                    full_n_rows,

                "n_day_blocks":
                    n_blocks,

                "n_bootstrap_replicates":
                    RF_BOOTSTRAP_REPLICATES,

                "bootstrap_seed":
                    RF_BOOTSTRAP_SEED,

                "f3_mae":
                    full_f3_mae,

                "f4_mae":
                    full_f4_mae,

                "absolute_mae_reduction":
                    full_absolute_difference,

                "relative_mae_improvement_pct":
                    full_relative_improvement_pct,

                "absolute_ci_2_5":
                    absolute_ci_lower,

                "absolute_ci_97_5":
                    absolute_ci_upper,

                "relative_ci_2_5_pct":
                    relative_ci_lower,

                "relative_ci_97_5_pct":
                    relative_ci_upper,

                "relative_ci_contains_zero":
                    relative_ci_contains_zero,
            }
        )


        # ----------------------------------------------------
        # 15. Store full bootstrap distribution
        #
        # Retaining the distributions allows later plotting,
        # auditing, and artifact persistence without rerunning
        # any model.
        # ----------------------------------------------------

        distribution_table = pd.DataFrame(
            {
                "model":
                    "RandomForestRegressor",

                "horizon_minutes":
                    horizon_minutes,

                "population":
                    population,

                "replicate":
                    np.arange(
                        RF_BOOTSTRAP_REPLICATES,
                        dtype=np.int32,
                    ),

                "f3_mae":
                    bootstrap_f3_mae,

                "f4_mae":
                    bootstrap_f4_mae,

                "f3_minus_f4_mae":
                    bootstrap_absolute_difference,

                "relative_mae_improvement_pct":
                    bootstrap_relative_improvement_pct,
            }
        )


        random_forest_bootstrap_distribution_rows.append(
            distribution_table
        )


        # ----------------------------------------------------
        # 16. Compact live output
        # ----------------------------------------------------

        print(
            "\n"
            f"{horizon_minutes:2d} min — "
            f"{population}"
        )

        print(
            f"  Rows: "
            f"{full_n_rows}"
        )

        print(
            f"  Day blocks: "
            f"{n_blocks}"
        )

        print(
            f"  F3 MAE: "
            f"{full_f3_mae:.6f}"
        )

        print(
            f"  F4 MAE: "
            f"{full_f4_mae:.6f}"
        )

        print(
            "  Relative F3 -> F4 improvement: "
            f"{full_relative_improvement_pct:+.6f}%"
        )

        print(
            "  95% bootstrap CI: "
            f"[{relative_ci_lower:+.6f}%, "
            f"{relative_ci_upper:+.6f}%]"
        )

        print(
            "  CI contains zero:",
            relative_ci_contains_zero,
        )


# ------------------------------------------------------------
# 17. Assemble final summary table
# ------------------------------------------------------------

random_forest_bootstrap_summary = (
    pd.DataFrame(
        random_forest_bootstrap_summary_rows
    )
)


assert (
    random_forest_bootstrap_summary.shape[0]
    == 6
)


# ------------------------------------------------------------
# 18. Assemble complete distribution table
#
# 3 horizons × 2 populations × 10,000 replicates
# = 60,000 rows
# ------------------------------------------------------------

random_forest_bootstrap_distributions = (
    pd.concat(
        random_forest_bootstrap_distribution_rows,
        axis=0,
        ignore_index=True,
    )
)


assert (
    random_forest_bootstrap_distributions.shape[0]
    == 60_000
)


# ------------------------------------------------------------
# 19. Final summary display
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)

print(
    "RANDOM FOREST — PAIRED DAILY-BLOCK BOOTSTRAP"
)

print(
    "============================================\n"
)


display(
    random_forest_bootstrap_summary
)


# ------------------------------------------------------------
# 20. Compact inference-oriented printout
# ------------------------------------------------------------

print(
    "\nRandom Forest F3 -> F4 bootstrap summary:"
)


for _, row in (
    random_forest_bootstrap_summary
    .iterrows()
):

    zero_status = (
        "CONTAINS ZERO"
        if bool(
            row[
                "relative_ci_contains_zero"
            ]
        )
        else
        "EXCLUDES ZERO"
    )


    print(
        f"  "
        f"{int(row['horizon_minutes']):2d} min "
        f"{row['population']:8s}: "
        f"{row['relative_mae_improvement_pct']:+.6f}% "
        f"["
        f"{row['relative_ci_2_5_pct']:+.6f}%, "
        f"{row['relative_ci_97_5_pct']:+.6f}%"
        f"]  "
        f"{zero_status}"
    )


# ------------------------------------------------------------
# 21. Protocol integrity checks
# ------------------------------------------------------------

assert (
    random_forest_bootstrap_summary[
        "n_day_blocks"
    ]
    == 92
).all()


assert (
    random_forest_bootstrap_summary[
        "n_bootstrap_replicates"
    ]
    == 10_000
).all()


assert (
    random_forest_bootstrap_summary[
        "bootstrap_seed"
    ]
    == 42
).all()


assert (
    random_forest_bootstrap_summary[
        "comparison"
    ]
    == "F3_minus_F4_MAE"
).all()


print(
    "\nRandom Forest paired daily-block bootstrap: PASSED"
)

Random Forest complete final-test results:



,model,horizon_minutes,feature_family,n_features,n_estimators,max_features,min_samples_leaf,max_depth,bootstrap,test_rows,...,test_nmae,test_nrmse,test_r2,daylight_test_rows,daylight_mae,daylight_rmse,daylight_nmae,daylight_nrmse,daylight_r2,fit_predict_seconds
0,RandomForestRegressor,15,F0,118,300,0.5,3,NaN,True,26496,...,0.026919,0.057470,0.878779,17851,0.399088,0.700165,0.039909,0.070017,0.835757,185.411075
1,RandomForestRegressor,15,F1,133,300,0.5,3,NaN,True,26496,...,0.025808,0.055874,0.885420,17851,0.382711,0.680720,0.038271,0.068072,0.844753,221.966198
2,RandomForestRegressor,15,F2,138,300,0.5,3,NaN,True,26496,...,0.025749,0.055920,0.885233,17851,0.381917,0.681276,0.038192,0.068128,0.844499,263.395866
3,RandomForestRegressor,15,F3,147,300,0.5,3,NaN,True,26496,...,0.025816,0.055976,0.885003,17851,0.382895,0.681960,0.038289,0.068196,0.844187,323.616059
4,RandomForestRegressor,15,F4,244,300,0.5,3,30.0,True,26496,...,0.025863,0.056031,0.884775,17851,0.383601,0.682634,0.038360,0.068263,0.843879,337.577627
5,RandomForestRegressor,30,F0,118,300,0.5,3,30.0,True,26496,...,0.032699,0.065324,0.843385,17851,0.484116,0.795826,0.048412,0.079583,0.787811,177.950924
6,RandomForestRegressor,30,F1,133,300,0.5,3,NaN,True,26496,...,0.029823,0.062067,0.858613,17851,0.441902,0.756158,0.044190,0.075616,0.808437,200.780922
7,RandomForestRegressor,30,F2,138,300,0.5,3,NaN,True,26496,...,0.029667,0.061988,0.858972,17851,0.439928,0.755210,0.043993,0.075521,0.808917,250.233925
8,RandomForestRegressor,30,F3,147,300,0.5,3,30.0,True,26496,...,0.029810,0.062062,0.858637,17851,0.442046,0.756105,0.044205,0.075611,0.808463,243.973783
9,RandomForestRegressor,30,F4,244,300,0.5,3,30.0,True,26496,...,0.029748,0.062011,0.858868,17851,0.441128,0.755487,0.044113,0.075549,0.808777,272.466862



15 min — all
  Rows: 26496
  Day blocks: 92
  F3 MAE: 0.258158
  F4 MAE: 0.258630
  Relative F3 -> F4 improvement: -0.183070%
  95% bootstrap CI: [-0.479548%, +0.100633%]
  CI contains zero: True

15 min — daylight
  Rows: 17851
  Day blocks: 92
  F3 MAE: 0.382895
  F4 MAE: 0.383601
  Relative F3 -> F4 improvement: -0.184481%
  95% bootstrap CI: [-0.474605%, +0.103043%]
  CI contains zero: True

30 min — all
  Rows: 26496
  Day blocks: 92
  F3 MAE: 0.298097
  F4 MAE: 0.297476
  Relative F3 -> F4 improvement: +0.208025%
  95% bootstrap CI: [-0.056944%, +0.472241%]
  CI contains zero: True

30 min — daylight
  Rows: 17851
  Day blocks: 92
  F3 MAE: 0.442046
  F4 MAE: 0.441128
  Relative F3 -> F4 improvement: +0.207598%
  95% bootstrap CI: [-0.057371%, +0.467351%]
  CI contains zero: True

60 min — all
  Rows: 26496
  Day blocks: 92
  F3 MAE: 0.341052
  F4 MAE: 0.340608
  Relative F3 -> F4 improvement: +0.130099%
  95% bootstrap CI: [-0.283526%, +0.552420%]
  CI contains zero: True

60 m

,model,horizon_minutes,population,comparison,positive_means,n_rows,n_day_blocks,n_bootstrap_replicates,bootstrap_seed,f3_mae,f4_mae,absolute_mae_reduction,relative_mae_improvement_pct,absolute_ci_2_5,absolute_ci_97_5,relative_ci_2_5_pct,relative_ci_97_5_pct,relative_ci_contains_zero
0,RandomForestRegressor,15,all,F3_minus_F4_MAE,F4_lower_MAE,26496,92,10000,42,0.258158,0.258630,-0.000473,-0.183070,-0.001218,0.000259,-0.479548,0.100633,True
1,RandomForestRegressor,15,daylight,F3_minus_F4_MAE,F4_lower_MAE,17851,92,10000,42,0.382895,0.383601,-0.000706,-0.184481,-0.001790,0.000392,-0.474605,0.103043,True
2,RandomForestRegressor,30,all,F3_minus_F4_MAE,F4_lower_MAE,26496,92,10000,42,0.298097,0.297476,0.000620,0.208025,-0.000168,0.001392,-0.056944,0.472241,True
3,RandomForestRegressor,30,daylight,F3_minus_F4_MAE,F4_lower_MAE,17851,92,10000,42,0.442046,0.441128,0.000918,0.207598,-0.000254,0.002064,-0.057371,0.467351,True
4,RandomForestRegressor,60,all,F3_minus_F4_MAE,F4_lower_MAE,26496,92,10000,42,0.341052,0.340608,0.000444,0.130099,-0.000956,0.001867,-0.283526,0.552420,True
5,RandomForestRegressor,60,daylight,F3_minus_F4_MAE,F4_lower_MAE,17851,92,10000,42,0.505728,0.505067,0.000661,0.130706,-0.001447,0.002720,-0.285442,0.546339,True



Random Forest F3 -> F4 bootstrap summary:
  15 min all     : -0.183070% [-0.479548%, +0.100633%]  CONTAINS ZERO
  15 min daylight: -0.184481% [-0.474605%, +0.103043%]  CONTAINS ZERO
  30 min all     : +0.208025% [-0.056944%, +0.472241%]  CONTAINS ZERO
  30 min daylight: +0.207598% [-0.057371%, +0.467351%]  CONTAINS ZERO
  60 min all     : +0.130099% [-0.283526%, +0.552420%]  CONTAINS ZERO
  60 min daylight: +0.130706% [-0.285442%, +0.546339%]  CONTAINS ZERO

Random Forest paired daily-block bootstrap: PASSED


## Random Forest: final interpretation

The Random Forest experiment followed the same controlled forecasting
protocol used for the other secondary machine-learning models.

Hyperparameters were selected exclusively from the chronological
validation period. All 15 configurations corresponding to

- three forecasting horizons: 15, 30, and 60 minutes, and
- five nested feature families: F0 through F4

were frozen before the held-out test set was evaluated.

The central scientific comparison remains **F3 versus F4**:

- **F3** contains the historical, Gregorian, solar, and continuous
  lunar/Sun-Moon astronomical predictors.
- **F4** adds the categorical Vedic/Panchang representation
  (Tithi, Paksha, Karana, Nakshatra, and Yoga).

Therefore, the F3 → F4 comparison tests whether the Panchang
representation contributes predictive information beyond the
underlying continuous astronomical quantities.

### Validation results

The validation-period F3 → F4 MAE changes were:

| Horizon | F3 → F4 validation MAE change |
|---:|---:|
| 15 min | +0.0904% |
| 30 min | +0.0734% |
| 60 min | -0.2977% |

Positive values mean that F4 produced lower MAE than F3.

These validation effects were very small and were not consistent in
sign across the three forecasting horizons.

The lowest validation MAE was obtained by **F2 at all three horizons**,
indicating that the continuous solar-geometry representation was
sufficient to obtain the best Random Forest validation performance in
this experiment.

### Held-out test results

On the held-out test period, the F3 → F4 MAE changes were:

| Horizon | All test rows | Daylight only |
|---:|---:|---:|
| 15 min | -0.1831% | -0.1845% |
| 30 min | +0.2080% | +0.2076% |
| 60 min | +0.1301% | +0.1307% |

The sign of the effect again depended on forecasting horizon.

At 15 minutes, adding the Panchang features slightly increased MAE.
At 30 and 60 minutes, F4 produced slightly lower point-estimate MAE
than F3. However, all differences were small, approximately two-tenths
of one percent or less.

The lowest held-out test MAE was obtained by:

- **F2 at 15 minutes:** approximately 0.25749 kW,
- **F2 at 30 minutes:** approximately 0.29667 kW,
- **F1 at 60 minutes:** approximately 0.33899 kW.

Thus, F4 was not the best-performing Random Forest feature family at
any of the three forecasting horizons.

### Paired daily-block bootstrap

To assess whether the small F3 → F4 differences were stable across the
test period, a paired forecast-valid-UTC-day block bootstrap was
performed using:

- 92 test-day blocks,
- 10,000 bootstrap replicates,
- seed 42,
- pooled row-weighted MAE,
- no model retraining,
- separate all-time and daylight analyses.

The 95% bootstrap confidence intervals for the relative F3 → F4 MAE
change were:

| Horizon | Population | Relative change | 95% bootstrap CI |
|---:|:---|---:|:---|
| 15 min | All | -0.1831% | [-0.4795%, +0.1006%] |
| 15 min | Daylight | -0.1845% | [-0.4746%, +0.1030%] |
| 30 min | All | +0.2080% | [-0.0569%, +0.4722%] |
| 30 min | Daylight | +0.2076% | [-0.0574%, +0.4674%] |
| 60 min | All | +0.1301% | [-0.2835%, +0.5524%] |
| 60 min | Daylight | +0.1307% | [-0.2854%, +0.5463%] |

**Every Random Forest confidence interval contains zero.**

Consequently, the Random Forest experiment does not provide evidence
of a stable held-out MAE improvement from adding the categorical
Panchang features beyond the continuous astronomical F3 predictors.

### Relation to the broader experiment

This result is especially relevant when compared with the other
secondary machine-learning models.

LightGBM produced a positive 30-minute F3 → F4 test effect whose paired
bootstrap confidence interval excluded zero. Random Forest does **not**
reproduce that result: its corresponding 30-minute point estimate is
positive, but the confidence interval crosses zero.

Therefore, the LightGBM 30-minute finding should currently be regarded
as **model- and horizon-specific**, rather than evidence of a general
forecasting advantage from the Panchang representation.

Overall, the Random Forest results are consistent with the broader
cautious conclusion of the controlled ablation study: any incremental
benefit from F4 is small, unstable across horizons and model classes,
and has not yet demonstrated a robust general improvement over the
continuous astronomical control representation.

In [126]:
# ============================================================
# Step 116: Persist complete Random Forest experiment
# ============================================================
#
# This cell saves the complete reproducible Random Forest
# experiment after:
#
#   - frozen validation selection;
#   - held-out testing;
#   - F3 -> F4 comparison;
#   - paired daily-block bootstrap;
#   - scientific interpretation.
#
# IMPORTANT PATH NOTE
# -------------------
# Notebook cwd:
#
#       Project/notebooks/
#
# Therefore the correct project-level result directory is:
#
#       ../results/notebook10_secondary_ml_models/
#
# NOT:
#
#       results/notebook10_secondary_ml_models/
#
# which would incorrectly create:
#
#       Project/notebooks/results/...
# ============================================================


from pathlib import Path
import hashlib
import json


# ------------------------------------------------------------
# 1. Correct project-level output directory
# ------------------------------------------------------------

rf_output_dir = Path(
    "../results/notebook10_secondary_ml_models"
).resolve()


rf_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Random Forest output directory:"
)

print(
    rf_output_dir
)


# ------------------------------------------------------------
# 2. Validation summary
# ------------------------------------------------------------

rf_validation_summary_path = (
    rf_output_dir
    / "random_forest_validation_summary.csv"
)


random_forest_validation_summary.to_csv(
    rf_validation_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 3. Frozen model-selection registry
# ------------------------------------------------------------

rf_frozen_selection_path = (
    rf_output_dir
    / "random_forest_frozen_selection.csv"
)


random_forest_frozen_selection.to_csv(
    rf_frozen_selection_path,
    index=False,
)


# ------------------------------------------------------------
# 4. Complete held-out final-test results
# ------------------------------------------------------------

rf_final_test_results_path = (
    rf_output_dir
    / "random_forest_final_test_results.csv"
)


random_forest_final_test_results.to_csv(
    rf_final_test_results_path,
    index=False,
)


# ------------------------------------------------------------
# 5. Compact F3 -> F4 final-test summary
#
# Store both:
#
#   - all test rows;
#   - daylight-only rows.
#
# Positive relative change means:
#
#       F4 has LOWER MAE than F3.
# ------------------------------------------------------------

rf_f3_f4_test_summary_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_results = (
        random_forest_final_test_results.loc[
            random_forest_final_test_results[
                "horizon_minutes"
            ].eq(
                horizon_minutes
            )
        ]
    )


    f3_row = (
        horizon_results.loc[
            horizon_results[
                "feature_family"
            ].eq(
                "F3"
            )
        ]
        .iloc[0]
    )


    f4_row = (
        horizon_results.loc[
            horizon_results[
                "feature_family"
            ].eq(
                "F4"
            )
        ]
        .iloc[0]
    )


    # --------------------------------------------------------
    # All-time comparison
    # --------------------------------------------------------

    f3_test_mae = float(
        f3_row[
            "test_mae"
        ]
    )

    f4_test_mae = float(
        f4_row[
            "test_mae"
        ]
    )


    all_absolute_reduction = (
        f3_test_mae
        - f4_test_mae
    )


    all_relative_improvement_pct = (
        all_absolute_reduction
        / f3_test_mae
        * 100.0
    )


    # --------------------------------------------------------
    # Daylight comparison
    # --------------------------------------------------------

    f3_daylight_mae = float(
        f3_row[
            "daylight_mae"
        ]
    )

    f4_daylight_mae = float(
        f4_row[
            "daylight_mae"
        ]
    )


    daylight_absolute_reduction = (
        f3_daylight_mae
        - f4_daylight_mae
    )


    daylight_relative_improvement_pct = (
        daylight_absolute_reduction
        / f3_daylight_mae
        * 100.0
    )


    rf_f3_f4_test_summary_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "f3_test_mae":
                f3_test_mae,

            "f4_test_mae":
                f4_test_mae,

            "all_absolute_mae_reduction":
                all_absolute_reduction,

            "all_relative_mae_improvement_pct":
                all_relative_improvement_pct,

            "f3_daylight_mae":
                f3_daylight_mae,

            "f4_daylight_mae":
                f4_daylight_mae,

            "daylight_absolute_mae_reduction":
                daylight_absolute_reduction,

            "daylight_relative_mae_improvement_pct":
                daylight_relative_improvement_pct,
        }
    )


random_forest_f3_f4_test_summary = pd.DataFrame(
    rf_f3_f4_test_summary_rows
)


rf_f3_f4_test_summary_path = (
    rf_output_dir
    / "random_forest_f3_f4_test_summary.csv"
)


random_forest_f3_f4_test_summary.to_csv(
    rf_f3_f4_test_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 6. Bootstrap summary
# ------------------------------------------------------------

rf_bootstrap_summary_path = (
    rf_output_dir
    / "random_forest_bootstrap_summary.csv"
)


random_forest_bootstrap_summary.to_csv(
    rf_bootstrap_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 7. Full bootstrap distributions
#
# gzip compression substantially reduces storage while
# preserving every one of the 60,000 bootstrap replicates.
# ------------------------------------------------------------

rf_bootstrap_distributions_path = (
    rf_output_dir
    / "random_forest_bootstrap_distributions.csv.gz"
)


random_forest_bootstrap_distributions.to_csv(
    rf_bootstrap_distributions_path,
    index=False,
    compression="gzip",
)


# ------------------------------------------------------------
# 8. Persist timestamp-level test predictions
#
# One file per forecasting horizon.
#
# Each file stores:
#
#   - issue time;
#   - valid time;
#   - observed target;
#   - daylight flag;
#   - prediction from F0 ... F4;
#   - absolute error from F0 ... F4.
#
# This allows future diagnostics without retraining models.
# ------------------------------------------------------------

rf_prediction_paths = {}


for horizon_minutes in [
    15,
    30,
    60,
]:

    # --------------------------------------------------------
    # Start from F0 as the canonical row/index structure.
    # --------------------------------------------------------

    base_prediction_table = (
        random_forest_test_predictions[
            (
                horizon_minutes,
                "F0",
            )
        ]
        .copy()
    )


    combined_predictions = pd.DataFrame(
        index=base_prediction_table.index
    )


    combined_predictions.index.name = (
        "issue_time"
    )


    combined_predictions[
        "valid_time"
    ] = (
        base_prediction_table[
            "valid_time"
        ]
    )


    combined_predictions[
        "y_true"
    ] = (
        base_prediction_table[
            "y_true"
        ]
    )


    combined_predictions[
        "daylight"
    ] = (
        base_prediction_table[
            "daylight"
        ]
    )


    # --------------------------------------------------------
    # Attach all five feature-family predictions.
    # --------------------------------------------------------

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        prediction_table = (
            random_forest_test_predictions[
                (
                    horizon_minutes,
                    feature_family,
                )
            ]
        )


        # ----------------------------------------------------
        # Strong pairing checks before merging.
        # ----------------------------------------------------

        assert prediction_table.index.equals(
            combined_predictions.index
        )


        assert np.array_equal(
            prediction_table[
                "valid_time"
            ].to_numpy(),
            combined_predictions[
                "valid_time"
            ].to_numpy(),
        )


        assert np.allclose(
            prediction_table[
                "y_true"
            ].to_numpy(),
            combined_predictions[
                "y_true"
            ].to_numpy(),
            rtol=0.0,
            atol=0.0,
        )


        assert np.array_equal(
            prediction_table[
                "daylight"
            ].to_numpy(),
            combined_predictions[
                "daylight"
            ].to_numpy(),
        )


        combined_predictions[
            f"y_pred_{feature_family}"
        ] = (
            prediction_table[
                "y_pred"
            ]
        )


        combined_predictions[
            f"absolute_error_{feature_family}"
        ] = (
            prediction_table[
                "absolute_error"
            ]
        )


    assert (
        combined_predictions.shape[0]
        == 26_496
    )


    prediction_path = (
        rf_output_dir
        / (
            "random_forest_test_predictions_"
            f"{horizon_minutes}min.csv"
        )
    )


    combined_predictions.to_csv(
        prediction_path,
        index=True,
    )


    rf_prediction_paths[
        horizon_minutes
    ] = prediction_path


# ------------------------------------------------------------
# 9. JSON-safe conversion helper
# ------------------------------------------------------------

def rf_json_default(value):

    if isinstance(
        value,
        (
            np.integer,
        ),
    ):
        return int(
            value
        )

    if isinstance(
        value,
        (
            np.floating,
        ),
    ):
        return float(
            value
        )

    if isinstance(
        value,
        (
            np.bool_,
        ),
    ):
        return bool(
            value
        )

    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    raise TypeError(
        f"Object of type "
        f"{type(value).__name__} "
        f"is not JSON serializable."
    )


# ------------------------------------------------------------
# 10. Persist executable selected configurations
# ------------------------------------------------------------

rf_selected_configs_path = (
    rf_output_dir
    / "random_forest_selected_configs.json"
)


with open(
    rf_selected_configs_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        random_forest_selected_configs,
        f,
        indent=2,
        sort_keys=True,
        default=rf_json_default,
    )


# ------------------------------------------------------------
# 11. Persist Random Forest experiment protocol
# ------------------------------------------------------------

random_forest_protocol = {
    "model":
        "RandomForestRegressor",

    "experiment_role":
        "Secondary classical machine-learning model",

    "target":
        "P_Solar[kW]",

    "forecast_horizons_minutes":
        [
            15,
            30,
            60,
        ],

    "feature_families":
        [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ],

    "feature_counts":
        {
            family:
                int(
                    primary_protocol[
                        "feature_counts"
                    ][
                        family
                    ]
                )
            for family in [
                "F0",
                "F1",
                "F2",
                "F3",
                "F4",
            ]
        },

    "selection_metric":
        "validation_MAE",

    "secondary_selection_metric":
        "validation_RMSE",

    "training_for_selection":
        "train_only",

    "final_refit":
        "train_plus_validation",

    "test_policy":
        (
            "Held-out test set evaluated only after all "
            "15 Random Forest configurations were frozen."
        ),

    "fixed_parameters":
        RANDOM_FOREST_FIXED_PARAMS,

    "validation_grid":
        random_forest_configs,

    "grid_size_per_horizon_feature_family":
        8,

    "total_validation_model_selections":
        15,

    "daylight_definition":
        (
            "Forecast-valid solar_elevation_deg > 0 degrees"
        ),

    "pv_rated_kw":
        float(
            RF_PV_RATED_KW
        ),

    "central_comparison":
        "F3 versus F4",

    "central_comparison_interpretation":
        (
            "Positive F3_minus_F4_MAE means F4 has lower MAE."
        ),

    "bootstrap": {
        "resampling_unit":
            "forecast-valid UTC calendar day",

        "n_day_blocks":
            92,

        "n_replicates":
            int(
                RF_BOOTSTRAP_REPLICATES
            ),

        "seed":
            int(
                RF_BOOTSTRAP_SEED
            ),

        "confidence_interval":
            "95% percentile",

        "retraining":
            False,

        "mae_reconstruction":
            "pooled row-weighted MAE",

        "analysis_order": [
            "15_all",
            "15_daylight",
            "30_all",
            "30_daylight",
            "60_all",
            "60_daylight",
        ],
    },

    "software_random_state":
        42,
}


rf_protocol_path = (
    rf_output_dir
    / "random_forest_protocol.json"
)


with open(
    rf_protocol_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        random_forest_protocol,
        f,
        indent=2,
        sort_keys=True,
        default=rf_json_default,
    )


# ------------------------------------------------------------
# 12. Define the 11 experiment artifacts that must exist
#     BEFORE creating the manifest.
# ------------------------------------------------------------

rf_artifact_paths = [
    rf_validation_summary_path,
    rf_frozen_selection_path,
    rf_final_test_results_path,
    rf_f3_f4_test_summary_path,
    rf_bootstrap_summary_path,
    rf_bootstrap_distributions_path,
    rf_prediction_paths[
        15
    ],
    rf_prediction_paths[
        30
    ],
    rf_prediction_paths[
        60
    ],
    rf_selected_configs_path,
    rf_protocol_path,
]


assert len(
    rf_artifact_paths
) == 11


assert all(
    path.exists()
    for path in rf_artifact_paths
)


# ------------------------------------------------------------
# 13. SHA-256 helper
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):

    digest = hashlib.sha256()


    with open(
        file_path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )


    return digest.hexdigest()


# ------------------------------------------------------------
# 14. Build integrity manifest
# ------------------------------------------------------------

rf_manifest_rows = []


for path in rf_artifact_paths:

    rf_manifest_rows.append(
        {
            "filename":
                path.name,

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


random_forest_results_manifest = pd.DataFrame(
    rf_manifest_rows
)


rf_manifest_path = (
    rf_output_dir
    / "random_forest_results_manifest.csv"
)


random_forest_results_manifest.to_csv(
    rf_manifest_path,
    index=False,
)


# ------------------------------------------------------------
# 15. Final persistence checks
# ------------------------------------------------------------

assert (
    random_forest_results_manifest.shape[0]
    == 11
)


assert rf_manifest_path.exists()


expected_rf_files = {
    path.name
    for path in rf_artifact_paths
} | {
    rf_manifest_path.name
}


actual_rf_files = {
    path.name
    for path in rf_output_dir.glob(
        "random_forest_*"
    )
}


assert (
    expected_rf_files
    <= actual_rf_files
)


# ------------------------------------------------------------
# 16. Display compact F3 -> F4 test summary
# ------------------------------------------------------------

print(
    "\nRandom Forest F3 -> F4 final-test summary:\n"
)


display(
    random_forest_f3_f4_test_summary
)


# ------------------------------------------------------------
# 17. Display bootstrap summary
# ------------------------------------------------------------

print(
    "\nRandom Forest bootstrap summary:\n"
)


display(
    random_forest_bootstrap_summary
)


# ------------------------------------------------------------
# 18. Display artifact manifest
# ------------------------------------------------------------

print(
    "\nRandom Forest artifact manifest:\n"
)


display(
    random_forest_results_manifest
)


# ------------------------------------------------------------
# 19. Final status
# ------------------------------------------------------------

print(
    "\nArtifacts written:"
)

print(
    f"  11 experiment artifacts"
)

print(
    f"  1 integrity manifest"
)

print(
    f"  Total: 12 Random Forest files"
)


print(
    "\nOutput directory:"
)

print(
    rf_output_dir
)


print(
    "\nRandom Forest artifact persistence: PASSED"
)

Random Forest output directory:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook10_secondary_ml_models

Random Forest F3 -> F4 final-test summary:



,horizon_minutes,f3_test_mae,f4_test_mae,all_absolute_mae_reduction,all_relative_mae_improvement_pct,f3_daylight_mae,f4_daylight_mae,daylight_absolute_mae_reduction,daylight_relative_mae_improvement_pct
0,15,0.258158,0.258630,-0.000473,-0.183070,0.382895,0.383601,-0.000706,-0.184481
1,30,0.298097,0.297476,0.000620,0.208025,0.442046,0.441128,0.000918,0.207598
2,60,0.341052,0.340608,0.000444,0.130099,0.505728,0.505067,0.000661,0.130706



Random Forest bootstrap summary:



,model,horizon_minutes,population,comparison,positive_means,n_rows,n_day_blocks,n_bootstrap_replicates,bootstrap_seed,f3_mae,f4_mae,absolute_mae_reduction,relative_mae_improvement_pct,absolute_ci_2_5,absolute_ci_97_5,relative_ci_2_5_pct,relative_ci_97_5_pct,relative_ci_contains_zero
0,RandomForestRegressor,15,all,F3_minus_F4_MAE,F4_lower_MAE,26496,92,10000,42,0.258158,0.258630,-0.000473,-0.183070,-0.001218,0.000259,-0.479548,0.100633,True
1,RandomForestRegressor,15,daylight,F3_minus_F4_MAE,F4_lower_MAE,17851,92,10000,42,0.382895,0.383601,-0.000706,-0.184481,-0.001790,0.000392,-0.474605,0.103043,True
2,RandomForestRegressor,30,all,F3_minus_F4_MAE,F4_lower_MAE,26496,92,10000,42,0.298097,0.297476,0.000620,0.208025,-0.000168,0.001392,-0.056944,0.472241,True
3,RandomForestRegressor,30,daylight,F3_minus_F4_MAE,F4_lower_MAE,17851,92,10000,42,0.442046,0.441128,0.000918,0.207598,-0.000254,0.002064,-0.057371,0.467351,True
4,RandomForestRegressor,60,all,F3_minus_F4_MAE,F4_lower_MAE,26496,92,10000,42,0.341052,0.340608,0.000444,0.130099,-0.000956,0.001867,-0.283526,0.552420,True
5,RandomForestRegressor,60,daylight,F3_minus_F4_MAE,F4_lower_MAE,17851,92,10000,42,0.505728,0.505067,0.000661,0.130706,-0.001447,0.002720,-0.285442,0.546339,True



Random Forest artifact manifest:



,filename,size_bytes,sha256
0,random_forest_validation_summary.csv,1321,0c9441759be94d2b84ee7a56fd1b3085b71a43f851e6cd...
1,random_forest_frozen_selection.csv,1118,e46ddb63f4b38a9004ef6baa380566aca5510e9ce37b14...
2,random_forest_final_test_results.csv,4396,638401686016c5c74c3d7e1fb4b655499855f7d9d5013a...
3,random_forest_f3_f4_test_summary.csv,696,e65a8b7a147a10cbc3d8dc5098631b37edbdc5608a8137...
4,random_forest_bootstrap_summary.csv,1782,edf64b838f4d6de6e1fcb1a6566e8a3074b9e567f4260c...
5,random_forest_bootstrap_distributions.csv.gz,2405994,b89702a1d2f819bbc36a0b10f9e305a0329af4f2f40e6d...
6,random_forest_test_predictions_15min.csv,6927577,fc2b68a5b54f1b0293b350c49f60fe5fa107d0cc556fe9...
7,random_forest_test_predictions_30min.csv,6954836,016d1658b9d5abdfd2a4e33e085b3dc06a222581d63eab...
8,random_forest_test_predictions_60min.csv,7030973,fe40dfef94c1bec213ee15e5b6c472093b95d8ae69fd56...
9,random_forest_selected_configs.json,4230,1e49c8b9e2a17e0ee4fa919041978efba1e80f4cdc6593...



Artifacts written:
  11 experiment artifacts
  1 integrity manifest
  Total: 12 Random Forest files

Output directory:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook10_secondary_ml_models

Random Forest artifact persistence: PASSED
